In [ ]:
# Cell 1 — Environment Setup (no cloud mutations)

import importlib.util
import shutil
import subprocess
import sys


def _module_exists(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


def _install_packages(packages: list[str]) -> None:
    if not packages:
        return
    cmd = [sys.executable, "-m", "pip", "install", "--upgrade", *packages]
    print(f"$ {' '.join(cmd)}")
    subprocess.check_call(cmd)


# Only Python dependencies actually needed by later cells
required_python = {
    "oci": "oci",
    "ipywidgets": "ipywidgets",
}

missing_python = [
    pip_name for mod, pip_name in required_python.items() if not _module_exists(mod)
]
if missing_python:
    print(f"Installing missing Python packages: {', '.join(missing_python)}")
    _install_packages(missing_python)

# Verify/import what is required later
still_missing = [mod for mod in required_python if not _module_exists(mod)]
if still_missing:
    raise RuntimeError(f"Missing Python modules after install: {still_missing}")

import oci  # used in later cells
import ipywidgets as widgets  # used in Cell 3
from IPython.display import HTML, display  # used in Cell 3

print("Python dependencies ready: oci, ipywidgets, IPython.display")


# CLI tools required by later cells
required_cli = ["terraform", "kubectl", "oci", "helm"]
missing_cli = []

for tool in required_cli:
    path = shutil.which(tool)
    if path:
        print(f"{tool}: {path}")
    else:
        print(f"{tool}: missing")
        missing_cli.append(tool)

if missing_cli:
    raise RuntimeError(
        "Missing required CLI tools: "
        + ", ".join(missing_cli)
        + ". Install them before running Cell 2+."
    )

print("Environment ready. Proceed to Cell 2.")

In [ ]:
# Cell 2 — SSOT Defaults + Local Credential Context (no network calls)
import os
import json
from pathlib import Path
from datetime import datetime, timezone
import oci


def _as_bool(value, default=False):
    if value is None:
        return default
    return str(value).strip().lower() in {"1", "true", "yes", "y", "on"}


def _as_int(value, default):
    try:
        return int(value)
    except Exception:
        return default


def _as_float(value, default):
    try:
        return float(value)
    except Exception:
        return default


def _setenv(k, v):
    os.environ[k] = "" if v is None else str(v)


def _bool_str(v: bool) -> str:
    return "true" if bool(v) else "false"


def _find_notebook_project_root(start: Path) -> Path:
    """
    Resolve notebook project root robustly whether cwd is:
    - this project root (contains this notebook), or
    - a parent containing OCIK8sClusterAutoscalerVsKarpenter/.
    """
    p = start.resolve()
    for c in [p] + list(p.parents):
        if (c / "Cluster_Autoscaler_Vs_Karpenter_On_OKE.ipynb").exists():
            return c
        nested = c / "OCIK8sClusterAutoscalerVsKarpenter"
        if (nested / "Cluster_Autoscaler_Vs_Karpenter_On_OKE.ipynb").exists():
            return nested.resolve()
    return p


def _module_requirements():
    return ["versions.tf", "module-cluster.tf", "module-workers.tf", "modules"]


def _missing_module_markers(path: Path):
    return [m for m in _module_requirements() if not (path / m).exists()]


def _is_valid_terraform_oke_module_dir(path: Path) -> bool:
    return path.exists() and path.is_dir() and not _missing_module_markers(path)


def _resolve_terraform_module_source(project_root: Path) -> str:
    explicit = os.environ.get("TERRAFORM_OCI_OKE_MODULE_SOURCE", "").strip()
    if explicit:
        p = Path(os.path.expanduser(explicit)).resolve()
        if not _is_valid_terraform_oke_module_dir(p):
            missing = (
                _missing_module_markers(p)
                if p.exists() and p.is_dir()
                else _module_requirements()
            )
            raise RuntimeError(
                "TERRAFORM_OCI_OKE_MODULE_SOURCE must point to terraform-oci-oke-main root "
                f"(missing markers: {missing}). Received: {p}"
            )
        return str(p)
    vendored = (project_root / "vendor" / "terraform-oci-oke-main").resolve()
    if _is_valid_terraform_oke_module_dir(vendored):
        return str(vendored)
    missing = (
        _missing_module_markers(vendored)
        if vendored.exists() and vendored.is_dir()
        else _module_requirements()
    )
    raise RuntimeError(
        "Could not resolve Terraform OKE module source.\n"
        "Expected default: PROJECT_ROOT/vendor/terraform-oci-oke-main\n"
        "Or set TERRAFORM_OCI_OKE_MODULE_SOURCE explicitly.\n"
        f"Checked default path: {vendored}\n"
        f"Missing required markers: {missing}"
    )


# ---------- Local credential/context load (no network calls) ----------
OCI_CONFIG_FILE = os.path.expanduser(os.environ.get("OCI_CONFIG_FILE", "~/.oci/config"))
OCI_PROFILE = os.environ.get("OCI_PROFILE", "DEFAULT")
if not Path(OCI_CONFIG_FILE).exists():
    raise RuntimeError(
        f"OCI config file not found: {OCI_CONFIG_FILE}. "
        "Set OCI_CONFIG_FILE or create ~/.oci/config."
    )
try:
    _cfg = oci.config.from_file(file_location=OCI_CONFIG_FILE, profile_name=OCI_PROFILE)
except Exception as e:
    raise RuntimeError(
        f"Failed to load OCI profile '{OCI_PROFILE}' from {OCI_CONFIG_FILE}: {e}"
    ) from e

TENANCY_OCID = os.environ.get("TENANCY_OCID", _cfg.get("tenancy", "")).strip()
USER_OCID = os.environ.get("USER_OCID", _cfg.get("user", "")).strip()
FINGERPRINT = os.environ.get("FINGERPRINT", _cfg.get("fingerprint", "")).strip()
REGION = os.environ.get("REGION", _cfg.get("region", "us-ashburn-1")).strip()
OCI_REGION = os.environ.get("OCI_REGION", REGION).strip() or REGION
if not TENANCY_OCID:
    raise RuntimeError(
        "TENANCY_OCID is empty. Set TENANCY_OCID or ensure OCI config profile has tenancy."
    )

# ---------- Workspace/module source ----------
PROJECT_ROOT = str(_find_notebook_project_root(Path.cwd()))
TF_STACK_ROOT = os.path.expanduser(
    os.environ.get("TF_STACK_ROOT", str((Path(PROJECT_ROOT) / "tf").resolve()))
)
Path(TF_STACK_ROOT).mkdir(parents=True, exist_ok=True)
TERRAFORM_OCI_OKE_MODULE_SOURCE = _resolve_terraform_module_source(Path(PROJECT_ROOT))
UTC_TIMESTAMP_FORMAT = os.environ.get("UTC_TIMESTAMP_FORMAT", "%Y%m%dT%H%M%SZ").strip()

# ---------- Benchmark/global controls ----------
BENCHMARK_MODE = os.environ.get("BENCHMARK_MODE", "parity").strip().lower()
if BENCHMARK_MODE not in {"parity", "native"}:
    BENCHMARK_MODE = "parity"

TARGET_CLUSTER = os.environ.get("TARGET_CLUSTER", "both").strip().lower()
if TARGET_CLUSTER not in {"ca", "kpo", "both"}:
    TARGET_CLUSTER = "both"

TARGET_SCENARIOS = os.environ.get("TARGET_SCENARIOS", "all").strip().lower()
if TARGET_SCENARIOS not in {"all", "burst", "steady", "scale_down"}:
    TARGET_SCENARIOS = "all"

REPEATS = _as_int(os.environ.get("REPEATS"), 5)
if REPEATS < 1:
    REPEATS = 1

IMAGE_MODE = os.environ.get("IMAGE_MODE", "pin").strip().lower()
if IMAGE_MODE not in {"pin", "filter"}:
    IMAGE_MODE = "pin"

ENABLE_METRICS_SERVER = _as_bool(os.environ.get("ENABLE_METRICS_SERVER"), False)
KUBERNETES_VERSION = os.environ.get("KUBERNETES_VERSION", "v1.34.2").strip()
RUN_ID = os.environ.get(
    "RUN_ID",
    f"ca-vs-kpo-{datetime.now(timezone.utc).strftime(UTC_TIMESTAMP_FORMAT)}",
)
OUTPUT_DIR = os.path.expanduser(
    os.environ.get("OUTPUT_DIR", str((Path(PROJECT_ROOT) / "artifacts").resolve()))
)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# ---------- Deferred selections for Cell 3 ----------
HOME_REGION = os.environ.get("HOME_REGION", "").strip()
PARENT_COMPARTMENT_OCID = os.environ.get("PARENT_COMPARTMENT_OCID", "").strip()
IAM_POLICY_COMPARTMENT_OCID = os.environ.get("IAM_POLICY_COMPARTMENT_OCID", "").strip()
BENCHMARK_REGION = os.environ.get("BENCHMARK_REGION", "").strip()
BENCHMARK_COMPARTMENT_OCID = os.environ.get("BENCHMARK_COMPARTMENT_OCID", "").strip()

# ---------- IAM/workload identity ----------
CA_NAMESPACE = os.environ.get("CA_NAMESPACE", "kube-system").strip()
CA_SERVICE_ACCOUNT = os.environ.get("CA_SERVICE_ACCOUNT", "cluster-autoscaler").strip()
KPO_NAMESPACE = os.environ.get("KPO_NAMESPACE", "karpenter").strip()
KPO_SERVICE_ACCOUNT = os.environ.get("KPO_SERVICE_ACCOUNT", "karpenter").strip()

TELEMETRY_PRINCIPAL_GROUP_NAME = os.environ.get(
    "TELEMETRY_PRINCIPAL_GROUP_NAME", ""
).strip()
CA_WORKLOAD_POLICY_NAME = os.environ.get("CA_WORKLOAD_POLICY_NAME", "").strip()
KPO_WORKLOAD_POLICY_NAME = os.environ.get("KPO_WORKLOAD_POLICY_NAME", "").strip()
KPO_NODES_POLICY_NAME = os.environ.get("KPO_NODES_POLICY_NAME", "").strip()
KPO_NODES_DYNAMIC_GROUP_NAME = os.environ.get(
    "KPO_NODES_DYNAMIC_GROUP_NAME", ""
).strip()
KPO_NODES_DYNAMIC_GROUP_RULE = os.environ.get(
    "KPO_NODES_DYNAMIC_GROUP_RULE", ""
).strip()

# ---------- Pool defaults ----------
CA_BASELINE_POOL_SIZE = _as_int(os.environ.get("CA_BASELINE_POOL_SIZE"), 2)
KPO_BASELINE_POOL_SIZE = _as_int(os.environ.get("KPO_BASELINE_POOL_SIZE"), 2)

CA_AUTOSCALED_POOL_MIN_SIZE = _as_int(os.environ.get("CA_AUTOSCALED_POOL_MIN_SIZE"), 0)
CA_AUTOSCALED_POOL_MAX_SIZE = _as_int(os.environ.get("CA_AUTOSCALED_POOL_MAX_SIZE"), 10)
KPO_AUTOSCALED_POOL_MIN_SIZE = _as_int(
    os.environ.get("KPO_AUTOSCALED_POOL_MIN_SIZE"), 0
)
KPO_AUTOSCALED_POOL_MAX_SIZE = _as_int(
    os.environ.get("KPO_AUTOSCALED_POOL_MAX_SIZE"), 10
)

CA_BASELINE_POOL_SHAPE = os.environ.get("CA_BASELINE_POOL_SHAPE", "").strip()
CA_BASELINE_POOL_IMAGE_ID = os.environ.get("CA_BASELINE_POOL_IMAGE_ID", "").strip()
CA_BASELINE_POOL_FLEX_OCPUS = os.environ.get("CA_BASELINE_POOL_FLEX_OCPUS", "").strip()
CA_BASELINE_POOL_FLEX_MEMORY_GB = os.environ.get(
    "CA_BASELINE_POOL_FLEX_MEMORY_GB", ""
).strip()

CA_AUTOSCALED_POOL_SHAPE = os.environ.get("CA_AUTOSCALED_POOL_SHAPE", "").strip()
CA_AUTOSCALED_POOL_IMAGE_ID = os.environ.get("CA_AUTOSCALED_POOL_IMAGE_ID", "").strip()
CA_AUTOSCALED_POOL_FLEX_OCPUS = os.environ.get(
    "CA_AUTOSCALED_POOL_FLEX_OCPUS", ""
).strip()
CA_AUTOSCALED_POOL_FLEX_MEMORY_GB = os.environ.get(
    "CA_AUTOSCALED_POOL_FLEX_MEMORY_GB", ""
).strip()

KPO_BASELINE_POOL_SHAPE = os.environ.get("KPO_BASELINE_POOL_SHAPE", "").strip()
KPO_BASELINE_POOL_IMAGE_ID = os.environ.get("KPO_BASELINE_POOL_IMAGE_ID", "").strip()
KPO_BASELINE_POOL_FLEX_OCPUS = os.environ.get(
    "KPO_BASELINE_POOL_FLEX_OCPUS", ""
).strip()
KPO_BASELINE_POOL_FLEX_MEMORY_GB = os.environ.get(
    "KPO_BASELINE_POOL_FLEX_MEMORY_GB", ""
).strip()

KPO_AUTOSCALED_POOL_SHAPE = os.environ.get("KPO_AUTOSCALED_POOL_SHAPE", "").strip()
KPO_AUTOSCALED_POOL_IMAGE_ID = os.environ.get(
    "KPO_AUTOSCALED_POOL_IMAGE_ID", ""
).strip()
KPO_AUTOSCALED_POOL_FLEX_OCPUS = os.environ.get(
    "KPO_AUTOSCALED_POOL_FLEX_OCPUS", ""
).strip()
KPO_AUTOSCALED_POOL_FLEX_MEMORY_GB = os.environ.get(
    "KPO_AUTOSCALED_POOL_FLEX_MEMORY_GB", ""
).strip()

# ---------- Endpoint/health ----------
CONTROL_PLANE_ENDPOINT_MODE = (
    os.environ.get("CONTROL_PLANE_ENDPOINT_MODE", "public").strip().lower()
)
CONTROL_PLANE_ALLOWED_CIDRS = os.environ.get("CONTROL_PLANE_ALLOWED_CIDRS", "").strip()
KUBECONFIG_ENDPOINT_MODE = (
    os.environ.get("KUBECONFIG_ENDPOINT_MODE", "auto").strip().lower()
)
K8S_API_CHECK_TIMEOUT_SECONDS = _as_int(
    os.environ.get("K8S_API_CHECK_TIMEOUT_SECONDS"), 15
)

HEALTHCHECK_ALLOW_API_UNREACHABLE = _as_bool(
    os.environ.get("HEALTHCHECK_ALLOW_API_UNREACHABLE"),
    False,
)
METRICS_SERVER_SKIP_ADDON_DEPENDENCIES_CHECK = _as_bool(
    os.environ.get("METRICS_SERVER_SKIP_ADDON_DEPENDENCIES_CHECK"),
    False,
)

# ---------- KPO setup ----------
KPO_HELM_RELEASE = os.environ.get("KPO_HELM_RELEASE", "karpenter").strip()
KPO_HELM_CHART = os.environ.get("KPO_HELM_CHART", "").strip()
KPO_APISERVER_ENDPOINT_SOURCE = (
    os.environ.get("KPO_APISERVER_ENDPOINT_SOURCE", "private").strip().lower()
)
if KPO_APISERVER_ENDPOINT_SOURCE not in {
    "auto",
    "selected",
    "provisioned",
    "public",
    "private",
    "kubeconfig",
}:
    raise RuntimeError(
        "KPO_APISERVER_ENDPOINT_SOURCE must be one of: "
        "provisioned|selected|auto|public|private|kubeconfig"
    )

KPO_OCINODECLASS_NAME = os.environ.get(
    "KPO_OCINODECLASS_NAME", "benchmark-ocinodeclass"
).strip()
KPO_WORKLOAD_NODEPOOL = os.environ.get(
    "KPO_WORKLOAD_NODEPOOL", "benchmark-nodepool"
).strip()
KPO_NPN_IP_COUNT = _as_int(os.environ.get("KPO_NPN_IP_COUNT"), 16)
KPO_CPU_PER_OCPU_FACTOR = _as_float(os.environ.get("KPO_CPU_PER_OCPU_FACTOR"), 2.0)
KPO_IMAGE_FILTER_COMPARTMENT_ID = os.environ.get(
    "KPO_IMAGE_FILTER_COMPARTMENT_ID", ""
).strip()
KPO_IMAGE_OS_VERSION_FILTER = os.environ.get("KPO_IMAGE_OS_VERSION_FILTER", "8").strip()

# ---------- Scenario controls ----------
SCENARIO_SCALE_UP_POLL_SECONDS = _as_int(
    os.environ.get("SCENARIO_SCALE_UP_POLL_SECONDS"), 1
)
SCENARIO_SCALE_DOWN_POLL_SECONDS = _as_int(
    os.environ.get("SCENARIO_SCALE_DOWN_POLL_SECONDS"), 5
)
SCENARIO_POLL_SECONDS = _as_int(
    os.environ.get("SCENARIO_POLL_SECONDS"),
    SCENARIO_SCALE_UP_POLL_SECONDS,
)
SCENARIO_PRECHECK_TIMEOUT_SECONDS = _as_int(
    os.environ.get("SCENARIO_PRECHECK_TIMEOUT_SECONDS"), 900
)
SCENARIO_PENDING_TIMEOUT_SECONDS = _as_int(
    os.environ.get("SCENARIO_PENDING_TIMEOUT_SECONDS"), 300
)
SCENARIO_READY_TIMEOUT_SECONDS = _as_int(
    os.environ.get("SCENARIO_READY_TIMEOUT_SECONDS"), 1800
)
SCENARIO_SCALE_DOWN_TIMEOUT_SECONDS = _as_int(
    os.environ.get("SCENARIO_SCALE_DOWN_TIMEOUT_SECONDS"), 1800
)

SCENARIO_BURST_REPLICAS = _as_int(os.environ.get("SCENARIO_BURST_REPLICAS"), 20)
SCENARIO_BURST_HOLD_SECONDS = _as_int(
    os.environ.get("SCENARIO_BURST_HOLD_SECONDS"), 120
)
SCENARIO_STEADY_REPLICAS = _as_int(os.environ.get("SCENARIO_STEADY_REPLICAS"), 8)
SCENARIO_STEADY_HOLD_SECONDS = _as_int(
    os.environ.get("SCENARIO_STEADY_HOLD_SECONDS"), 300
)

CA_EXPECTED_BENCHMARK_NODE_FLOOR = _as_int(
    os.environ.get(
        "CA_EXPECTED_BENCHMARK_NODE_FLOOR", str(CA_AUTOSCALED_POOL_MIN_SIZE)
    ),
    CA_AUTOSCALED_POOL_MIN_SIZE,
)
KPO_EXPECTED_BENCHMARK_NODE_FLOOR = _as_int(
    os.environ.get(
        "KPO_EXPECTED_BENCHMARK_NODE_FLOOR", str(KPO_AUTOSCALED_POOL_MIN_SIZE)
    ),
    KPO_AUTOSCALED_POOL_MIN_SIZE,
)
REQUIRE_KPO_NODECLAIMS_AT_FLOOR = _as_bool(
    os.environ.get("REQUIRE_KPO_NODECLAIMS_AT_FLOOR"),
    True,
)

# ---------- Workload ----------
WORKLOAD_NAME = os.environ.get("WORKLOAD_NAME", "benchmark-load").strip()
WORKLOAD_IMAGE = os.environ.get("WORKLOAD_IMAGE", "registry.k8s.io/pause:3.9").strip()
WORKLOAD_BASE_REPLICAS = _as_int(os.environ.get("WORKLOAD_BASE_REPLICAS"), 0)
WORKLOAD_CPU_REQUEST = os.environ.get("WORKLOAD_CPU_REQUEST", "500m").strip()
WORKLOAD_MEM_REQUEST = os.environ.get("WORKLOAD_MEM_REQUEST", "512Mi").strip()
WORKLOAD_CPU_LIMIT = os.environ.get("WORKLOAD_CPU_LIMIT", WORKLOAD_CPU_REQUEST).strip()
WORKLOAD_MEM_LIMIT = os.environ.get("WORKLOAD_MEM_LIMIT", WORKLOAD_MEM_REQUEST).strip()
WORKLOAD_NAMESPACE_PREFIX = os.environ.get(
    "WORKLOAD_NAMESPACE_PREFIX", "benchmark"
).strip()
WORKLOAD_NAMESPACE_KPO = os.environ.get(
    "WORKLOAD_NAMESPACE_KPO", "benchmark-kpo"
).strip()
WORKLOAD_DEPLOYMENT_KPO = os.environ.get(
    "WORKLOAD_DEPLOYMENT_KPO", WORKLOAD_NAME
).strip()

# ---------- Remediation ----------
KPO_REMEDIATION_APPLY_NSG = _as_bool(
    os.environ.get("KPO_REMEDIATION_APPLY_NSG", "false"),
    False,
)
KPO_REMEDIATION_APPLY_LIMITS = _as_bool(
    os.environ.get("KPO_REMEDIATION_APPLY_LIMITS", "false"),
    False,
)
KPO_REMEDIATION_LIMITS_CPU = os.environ.get("KPO_REMEDIATION_LIMITS_CPU", "").strip()
KPO_REMEDIATION_LIMITS_MEMORY = os.environ.get(
    "KPO_REMEDIATION_LIMITS_MEMORY", ""
).strip()
KPO_REMEDIATION_LIMITS_CPU_HEADROOM = _as_float(
    os.environ.get("KPO_REMEDIATION_LIMITS_CPU_HEADROOM", "1.0"),
    1.0,
)
KPO_REMEDIATION_LIMITS_MEMORY_HEADROOM_GI = _as_float(
    os.environ.get("KPO_REMEDIATION_LIMITS_MEMORY_HEADROOM_GI", "8.0"),
    8.0,
)

# ---------- Telemetry / cleanup ----------
TELEMETRY_METRIC_INTERVAL = os.environ.get("TELEMETRY_METRIC_INTERVAL", "1m").strip()
TELEMETRY_METRIC_RESOLUTION = os.environ.get(
    "TELEMETRY_METRIC_RESOLUTION", "1m"
).strip()
CLEANUP_DRY_RUN = _as_bool(os.environ.get("CLEANUP_DRY_RUN", "false"), False)
CLEANUP_SKIP_TERRAFORM_DESTROY = _as_bool(
    os.environ.get("CLEANUP_SKIP_TERRAFORM_DESTROY", "false"), False
)
CLEANUP_NODECLAIMS_TIMEOUT_SECONDS = _as_int(
    os.environ.get("CLEANUP_NODECLAIMS_TIMEOUT_SECONDS"), 1800
)
CLEANUP_NODES_TIMEOUT_SECONDS = _as_int(
    os.environ.get("CLEANUP_NODES_TIMEOUT_SECONDS"), 1200
)
CLEANUP_POLL_SECONDS = _as_int(os.environ.get("CLEANUP_POLL_SECONDS"), 10)

# ---------- Native mode optional ----------
KPO_NATIVE_ALLOWED_SHAPES = os.environ.get("KPO_NATIVE_ALLOWED_SHAPES", "").strip()
KPO_NATIVE_CAPACITY_TYPE_VALUES = os.environ.get(
    "KPO_NATIVE_CAPACITY_TYPE_VALUES", "on-demand"
).strip()
KPO_NATIVE_NODEPOOL_CPU_LIMIT = os.environ.get(
    "KPO_NATIVE_NODEPOOL_CPU_LIMIT", ""
).strip()
KPO_NATIVE_NODEPOOL_MEMORY_LIMIT_GI = os.environ.get(
    "KPO_NATIVE_NODEPOOL_MEMORY_LIMIT_GI", ""
).strip()
CA_NATIVE_WORKER_POOLS_JSON = os.environ.get("CA_NATIVE_WORKER_POOLS_JSON", "").strip()
BENCHMARK_PRICEBOOK_PATH = os.environ.get("BENCHMARK_PRICEBOOK_PATH", "").strip()

# ---------- SSOT export ----------
ssot_vars = {
    "OCI_CONFIG_FILE": OCI_CONFIG_FILE,
    "OCI_PROFILE": OCI_PROFILE,
    "TENANCY_OCID": TENANCY_OCID,
    "USER_OCID": USER_OCID,
    "FINGERPRINT": FINGERPRINT,
    "REGION": REGION,
    "OCI_REGION": OCI_REGION,
    "PROJECT_ROOT": PROJECT_ROOT,
    "TF_STACK_ROOT": TF_STACK_ROOT,
    "TERRAFORM_OCI_OKE_MODULE_SOURCE": TERRAFORM_OCI_OKE_MODULE_SOURCE,
    "UTC_TIMESTAMP_FORMAT": UTC_TIMESTAMP_FORMAT,
    "BENCHMARK_MODE": BENCHMARK_MODE,
    "TARGET_CLUSTER": TARGET_CLUSTER,
    "TARGET_SCENARIOS": TARGET_SCENARIOS,
    "REPEATS": REPEATS,
    "IMAGE_MODE": IMAGE_MODE,
    "ENABLE_METRICS_SERVER": _bool_str(ENABLE_METRICS_SERVER),
    "KUBERNETES_VERSION": KUBERNETES_VERSION,
    "RUN_ID": RUN_ID,
    "OUTPUT_DIR": OUTPUT_DIR,
    "HOME_REGION": HOME_REGION,
    "PARENT_COMPARTMENT_OCID": PARENT_COMPARTMENT_OCID,
    "IAM_POLICY_COMPARTMENT_OCID": IAM_POLICY_COMPARTMENT_OCID,
    "CA_NAMESPACE": CA_NAMESPACE,
    "CA_SERVICE_ACCOUNT": CA_SERVICE_ACCOUNT,
    "KPO_NAMESPACE": KPO_NAMESPACE,
    "KPO_SERVICE_ACCOUNT": KPO_SERVICE_ACCOUNT,
    "TELEMETRY_PRINCIPAL_GROUP_NAME": TELEMETRY_PRINCIPAL_GROUP_NAME,
    "CA_WORKLOAD_POLICY_NAME": CA_WORKLOAD_POLICY_NAME,
    "KPO_WORKLOAD_POLICY_NAME": KPO_WORKLOAD_POLICY_NAME,
    "KPO_NODES_POLICY_NAME": KPO_NODES_POLICY_NAME,
    "KPO_NODES_DYNAMIC_GROUP_NAME": KPO_NODES_DYNAMIC_GROUP_NAME,
    "KPO_NODES_DYNAMIC_GROUP_RULE": KPO_NODES_DYNAMIC_GROUP_RULE,
    "BENCHMARK_REGION": BENCHMARK_REGION,
    "BENCHMARK_COMPARTMENT_OCID": BENCHMARK_COMPARTMENT_OCID,
    "CA_BASELINE_POOL_SIZE": CA_BASELINE_POOL_SIZE,
    "KPO_BASELINE_POOL_SIZE": KPO_BASELINE_POOL_SIZE,
    "CA_AUTOSCALED_POOL_MIN_SIZE": CA_AUTOSCALED_POOL_MIN_SIZE,
    "CA_AUTOSCALED_POOL_MAX_SIZE": CA_AUTOSCALED_POOL_MAX_SIZE,
    "KPO_AUTOSCALED_POOL_MIN_SIZE": KPO_AUTOSCALED_POOL_MIN_SIZE,
    "KPO_AUTOSCALED_POOL_MAX_SIZE": KPO_AUTOSCALED_POOL_MAX_SIZE,
    "CA_BASELINE_POOL_SHAPE": CA_BASELINE_POOL_SHAPE,
    "CA_BASELINE_POOL_IMAGE_ID": CA_BASELINE_POOL_IMAGE_ID,
    "CA_BASELINE_POOL_FLEX_OCPUS": CA_BASELINE_POOL_FLEX_OCPUS,
    "CA_BASELINE_POOL_FLEX_MEMORY_GB": CA_BASELINE_POOL_FLEX_MEMORY_GB,
    "CA_AUTOSCALED_POOL_SHAPE": CA_AUTOSCALED_POOL_SHAPE,
    "CA_AUTOSCALED_POOL_IMAGE_ID": CA_AUTOSCALED_POOL_IMAGE_ID,
    "CA_AUTOSCALED_POOL_FLEX_OCPUS": CA_AUTOSCALED_POOL_FLEX_OCPUS,
    "CA_AUTOSCALED_POOL_FLEX_MEMORY_GB": CA_AUTOSCALED_POOL_FLEX_MEMORY_GB,
    "KPO_BASELINE_POOL_SHAPE": KPO_BASELINE_POOL_SHAPE,
    "KPO_BASELINE_POOL_IMAGE_ID": KPO_BASELINE_POOL_IMAGE_ID,
    "KPO_BASELINE_POOL_FLEX_OCPUS": KPO_BASELINE_POOL_FLEX_OCPUS,
    "KPO_BASELINE_POOL_FLEX_MEMORY_GB": KPO_BASELINE_POOL_FLEX_MEMORY_GB,
    "KPO_AUTOSCALED_POOL_SHAPE": KPO_AUTOSCALED_POOL_SHAPE,
    "KPO_AUTOSCALED_POOL_IMAGE_ID": KPO_AUTOSCALED_POOL_IMAGE_ID,
    "KPO_AUTOSCALED_POOL_FLEX_OCPUS": KPO_AUTOSCALED_POOL_FLEX_OCPUS,
    "KPO_AUTOSCALED_POOL_FLEX_MEMORY_GB": KPO_AUTOSCALED_POOL_FLEX_MEMORY_GB,
    "CONTROL_PLANE_ENDPOINT_MODE": CONTROL_PLANE_ENDPOINT_MODE,
    "CONTROL_PLANE_ALLOWED_CIDRS": CONTROL_PLANE_ALLOWED_CIDRS,
    "KUBECONFIG_ENDPOINT_MODE": KUBECONFIG_ENDPOINT_MODE,
    "K8S_API_CHECK_TIMEOUT_SECONDS": K8S_API_CHECK_TIMEOUT_SECONDS,
    "HEALTHCHECK_ALLOW_API_UNREACHABLE": _bool_str(HEALTHCHECK_ALLOW_API_UNREACHABLE),
    "METRICS_SERVER_SKIP_ADDON_DEPENDENCIES_CHECK": _bool_str(
        METRICS_SERVER_SKIP_ADDON_DEPENDENCIES_CHECK
    ),
    "KPO_HELM_RELEASE": KPO_HELM_RELEASE,
    "KPO_HELM_CHART": KPO_HELM_CHART,
    "KPO_APISERVER_ENDPOINT_SOURCE": KPO_APISERVER_ENDPOINT_SOURCE,
    "KPO_OCINODECLASS_NAME": KPO_OCINODECLASS_NAME,
    "KPO_WORKLOAD_NODEPOOL": KPO_WORKLOAD_NODEPOOL,
    "KPO_NPN_IP_COUNT": KPO_NPN_IP_COUNT,
    "KPO_CPU_PER_OCPU_FACTOR": KPO_CPU_PER_OCPU_FACTOR,
    "KPO_IMAGE_FILTER_COMPARTMENT_ID": KPO_IMAGE_FILTER_COMPARTMENT_ID,
    "KPO_IMAGE_OS_VERSION_FILTER": KPO_IMAGE_OS_VERSION_FILTER,
    "SCENARIO_SCALE_UP_POLL_SECONDS": SCENARIO_SCALE_UP_POLL_SECONDS,
    "SCENARIO_SCALE_DOWN_POLL_SECONDS": SCENARIO_SCALE_DOWN_POLL_SECONDS,
    "SCENARIO_POLL_SECONDS": SCENARIO_POLL_SECONDS,
    "SCENARIO_PRECHECK_TIMEOUT_SECONDS": SCENARIO_PRECHECK_TIMEOUT_SECONDS,
    "SCENARIO_PENDING_TIMEOUT_SECONDS": SCENARIO_PENDING_TIMEOUT_SECONDS,
    "SCENARIO_READY_TIMEOUT_SECONDS": SCENARIO_READY_TIMEOUT_SECONDS,
    "SCENARIO_SCALE_DOWN_TIMEOUT_SECONDS": SCENARIO_SCALE_DOWN_TIMEOUT_SECONDS,
    "SCENARIO_BURST_REPLICAS": SCENARIO_BURST_REPLICAS,
    "SCENARIO_BURST_HOLD_SECONDS": SCENARIO_BURST_HOLD_SECONDS,
    "SCENARIO_STEADY_REPLICAS": SCENARIO_STEADY_REPLICAS,
    "SCENARIO_STEADY_HOLD_SECONDS": SCENARIO_STEADY_HOLD_SECONDS,
    "CA_EXPECTED_BENCHMARK_NODE_FLOOR": CA_EXPECTED_BENCHMARK_NODE_FLOOR,
    "KPO_EXPECTED_BENCHMARK_NODE_FLOOR": KPO_EXPECTED_BENCHMARK_NODE_FLOOR,
    "REQUIRE_KPO_NODECLAIMS_AT_FLOOR": _bool_str(REQUIRE_KPO_NODECLAIMS_AT_FLOOR),
    "WORKLOAD_NAME": WORKLOAD_NAME,
    "WORKLOAD_IMAGE": WORKLOAD_IMAGE,
    "WORKLOAD_BASE_REPLICAS": WORKLOAD_BASE_REPLICAS,
    "WORKLOAD_CPU_REQUEST": WORKLOAD_CPU_REQUEST,
    "WORKLOAD_MEM_REQUEST": WORKLOAD_MEM_REQUEST,
    "WORKLOAD_CPU_LIMIT": WORKLOAD_CPU_LIMIT,
    "WORKLOAD_MEM_LIMIT": WORKLOAD_MEM_LIMIT,
    "WORKLOAD_NAMESPACE_PREFIX": WORKLOAD_NAMESPACE_PREFIX,
    "WORKLOAD_NAMESPACE_KPO": WORKLOAD_NAMESPACE_KPO,
    "WORKLOAD_DEPLOYMENT_KPO": WORKLOAD_DEPLOYMENT_KPO,
    "KPO_REMEDIATION_APPLY_NSG": _bool_str(KPO_REMEDIATION_APPLY_NSG),
    "KPO_REMEDIATION_APPLY_LIMITS": _bool_str(KPO_REMEDIATION_APPLY_LIMITS),
    "KPO_REMEDIATION_LIMITS_CPU": KPO_REMEDIATION_LIMITS_CPU,
    "KPO_REMEDIATION_LIMITS_MEMORY": KPO_REMEDIATION_LIMITS_MEMORY,
    "KPO_REMEDIATION_LIMITS_CPU_HEADROOM": KPO_REMEDIATION_LIMITS_CPU_HEADROOM,
    "KPO_REMEDIATION_LIMITS_MEMORY_HEADROOM_GI": KPO_REMEDIATION_LIMITS_MEMORY_HEADROOM_GI,
    "TELEMETRY_METRIC_INTERVAL": TELEMETRY_METRIC_INTERVAL,
    "TELEMETRY_METRIC_RESOLUTION": TELEMETRY_METRIC_RESOLUTION,
    "CLEANUP_DRY_RUN": _bool_str(CLEANUP_DRY_RUN),
    "CLEANUP_SKIP_TERRAFORM_DESTROY": _bool_str(CLEANUP_SKIP_TERRAFORM_DESTROY),
    "CLEANUP_NODECLAIMS_TIMEOUT_SECONDS": CLEANUP_NODECLAIMS_TIMEOUT_SECONDS,
    "CLEANUP_NODES_TIMEOUT_SECONDS": CLEANUP_NODES_TIMEOUT_SECONDS,
    "CLEANUP_POLL_SECONDS": CLEANUP_POLL_SECONDS,
    "KPO_NATIVE_ALLOWED_SHAPES": KPO_NATIVE_ALLOWED_SHAPES,
    "KPO_NATIVE_CAPACITY_TYPE_VALUES": KPO_NATIVE_CAPACITY_TYPE_VALUES,
    "KPO_NATIVE_NODEPOOL_CPU_LIMIT": KPO_NATIVE_NODEPOOL_CPU_LIMIT,
    "KPO_NATIVE_NODEPOOL_MEMORY_LIMIT_GI": KPO_NATIVE_NODEPOOL_MEMORY_LIMIT_GI,
    "CA_NATIVE_WORKER_POOLS_JSON": CA_NATIVE_WORKER_POOLS_JSON,
    "BENCHMARK_PRICEBOOK_PATH": BENCHMARK_PRICEBOOK_PATH,
}
for k, v in ssot_vars.items():
    _setenv(k, v)

summary = {
    "BENCHMARK_MODE": BENCHMARK_MODE,
    "TARGET_CLUSTER": TARGET_CLUSTER,
    "TARGET_SCENARIOS": TARGET_SCENARIOS,
    "REPEATS": REPEATS,
    "IMAGE_MODE": IMAGE_MODE,
    "ENABLE_METRICS_SERVER": ENABLE_METRICS_SERVER,
    "OCI_PROFILE": OCI_PROFILE,
    "REGION": REGION,
    "KUBERNETES_VERSION": KUBERNETES_VERSION,
    "PROJECT_ROOT": PROJECT_ROOT,
    "OUTPUT_DIR": OUTPUT_DIR,
    "RUN_ID": RUN_ID,
    "TERRAFORM_OCI_OKE_MODULE_SOURCE": TERRAFORM_OCI_OKE_MODULE_SOURCE,
    "TF_STACK_ROOT": TF_STACK_ROOT,
    "CA_NAMESPACE": CA_NAMESPACE,
    "KPO_NAMESPACE": KPO_NAMESPACE,
    "KPO_APISERVER_ENDPOINT_SOURCE": KPO_APISERVER_ENDPOINT_SOURCE,
    "SCENARIO_SCALE_UP_POLL_SECONDS": SCENARIO_SCALE_UP_POLL_SECONDS,
    "SCENARIO_SCALE_DOWN_POLL_SECONDS": SCENARIO_SCALE_DOWN_POLL_SECONDS,
    "SCENARIO_POLL_SECONDS": SCENARIO_POLL_SECONDS,
    "WORKLOAD_IMAGE": WORKLOAD_IMAGE,
}
print("Cell 2 complete: SSOT initialized (no network calls).")
print(json.dumps(summary, indent=2))
print("\nDeferred selections for Cell 3 UI:")
for name in [
    "BENCHMARK_REGION",
    "BENCHMARK_COMPARTMENT_OCID",
    "PARENT_COMPARTMENT_OCID",
    "IAM_POLICY_COMPARTMENT_OCID",
    "HOME_REGION",
]:
    print(f"- {name}")
print("\nNext: Run Cell 3 (interactive UI + validation).")

In [ ]:
# Cell 3 — Interactive UI + Validation (mode-aware, plan-aligned)
import os
import re
import json
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML
import oci

display(
    HTML(
        """
<style>
  .nb3-container { max-width: 1420px; margin: 0 auto; }
  .nb3-section { border: 1px solid #e0e0e0; background: #f8f9fb; padding: 12px; margin: 8px 0; border-left: 4px solid #d2d7e5; }
  .nb3-title { font-weight: 600; margin: 0 0 6px 0; text-align: left; }
  .nb3-lbl {
    white-space: normal !important;
    overflow: visible !important;
    text-overflow: clip !important;
    line-height: 1.2;
    font-weight: 500;
  }
  .widget-label {
    white-space: normal !important;
    overflow: visible !important;
    text-overflow: clip !important;
    min-width: 320px !important;
  }
  .issues-strip { background: #fdecea; color: #ba1a1a; padding: 6px 10px; border: 1px solid #f5c2c7; border-radius: 6px; font-size: 13px; margin: 0 0 6px 0; }
  .ok-strip { background: #e8f4fd; color: #0b5394; padding: 6px 10px; border: 1px solid #a4c2f4; border-radius: 6px; font-size: 13px; margin: 0 0 6px 0; }
  .hint-strip { background: #fff8e1; color: #6b4f00; padding: 6px 10px; border: 1px solid #f0d98c; border-radius: 6px; font-size: 13px; margin: 0 0 6px 0; }
</style>
"""
    )
)


def _env(k, d=""):
    return os.environ.get(k, d)


def _setenv(k, v):
    os.environ[k] = "" if v is None else str(v)


def _bool(v, default=False):
    if v is None:
        return default
    return str(v).strip().lower() in {"1", "true", "yes", "y", "on"}


def _to_int(v, d=0):
    try:
        return int(v)
    except Exception:
        return d


def _to_float(v, d=0.0):
    try:
        return float(v)
    except Exception:
        return d


def _csv_values(v: str):
    return [x.strip() for x in str(v).split(",") if x.strip()]


def _get_attr(obj, names, default=None):
    if obj is None:
        return default
    for n in names:
        if hasattr(obj, n):
            val = getattr(obj, n)
            if val is not None:
                return val
    return default


def _is_flex(shape_name):
    return bool(shape_name and str(shape_name).endswith(".Flex"))


def _k8_minor(version):
    m = re.search(r"v?(\d+)\.(\d+)", str(version))
    if not m:
        return ""
    return f"{m.group(1)}.{m.group(2)}"


def _k8_ge(version, major, minor):
    m = re.search(r"v?(\d+)\.(\d+)", str(version))
    if not m:
        return False
    vmaj, vmin = int(m.group(1)), int(m.group(2))
    return (vmaj, vmin) >= (major, minor)


def _looks_like_ocid(ocid: str, resource_hint: str = "") -> bool:
    v = str(ocid or "").strip()
    if not v:
        return False
    if not v.startswith("ocid1."):
        return False
    if resource_hint:
        return f"ocid1.{resource_hint}." in v
    return True


def _safe_set_options(dd, options, preferred=None):
    if not options:
        options = [("No options available", "")]
    values = [v for _, v in options]
    current = dd.value if dd.value in values else None
    if preferred in values:
        value = preferred
    elif current in values:
        value = current
    else:
        value = values[0]
    dd.options = options
    dd.value = value


def _list_all(fn, *args, **kwargs):
    return oci.pagination.list_call_get_all_results(fn, *args, **kwargs).data


def _make_cfg(base_cfg, region):
    c = dict(base_cfg)
    c["region"] = region
    return c


def _shape_bounds(shape_obj):
    ocpu_opts = _get_attr(shape_obj, ["ocpu_options"], None)
    mem_opts = _get_attr(shape_obj, ["memory_options"], None)
    min_ocpu = _to_float(
        _get_attr(ocpu_opts, ["min", "min_in_ocpus", "min_ocpu"], 1.0), 1.0
    )
    max_ocpu = _to_float(
        _get_attr(ocpu_opts, ["max", "max_in_ocpus", "max_ocpu"], 64.0), 64.0
    )
    min_mem = _to_float(
        _get_attr(mem_opts, ["min_in_g_bs", "min_in_gbs", "min_memory_in_g_bs"], 1.0),
        1.0,
    )
    max_mem = _to_float(
        _get_attr(
            mem_opts, ["max_in_g_bs", "max_in_gbs", "max_memory_in_g_bs"], 1024.0
        ),
        1024.0,
    )
    default_per_ocpu = _to_float(
        _get_attr(
            mem_opts, ["default_per_ocpu_in_g_bs", "default_per_ocpu_in_gbs"], 16.0
        ),
        16.0,
    )
    return min_ocpu, max_ocpu, min_mem, max_mem, default_per_ocpu


def _row(label_text: str, control: widgets.Widget, label_width="360px") -> widgets.HBox:
    lbl = widgets.HTML(
        f"<span class='nb3-lbl'>{label_text}</span>",
        layout=widgets.Layout(width=label_width, min_width=label_width),
    )
    if hasattr(control, "layout"):
        control.layout.flex = "1 1 auto"
        control.layout.width = "auto"
    return widgets.HBox(
        [lbl, control],
        layout=widgets.Layout(
            width="100%", justify_content="flex-start", align_items="center", gap="12px"
        ),
    )


def _section(title: str, *children: widgets.Widget) -> widgets.VBox:
    return widgets.VBox(
        [widgets.HTML(f"<div class='nb3-title'>{title}</div>"), *children],
        layout=widgets.Layout(width="100%", align_items="stretch"),
        _dom_classes=["nb3-section"],
    )


def _set_visible(widget: widgets.Widget, visible: bool):
    widget.layout.display = "" if visible else "none"


def _target_has_ca(target: str) -> bool:
    return target in {"ca", "both"}


def _target_has_kpo(target: str) -> bool:
    return target in {"kpo", "both"}


# OCI preemptible-supported shape families from OCI docs.
def _is_preemptible_supported_shape(shape_name: str) -> bool:
    s = str(shape_name or "").strip()
    if not s:
        return False
    if s == "VM.Standard.E2.1.Micro":
        return False
    prefixes = (
        "VM.Standard1.",
        "VM.Standard.B1.",
        "VM.Standard2.",
        "VM.Standard.E2.",
        "VM.DenseIO1.",
        "VM.DenseIO2.",
        "VM.GPU2.",
        "VM.GPU3.",
    )
    exact_or_starts = (
        "VM.Standard3.Flex",
        "VM.Standard.E3.Flex",
        "VM.Standard.E4.Flex",
        "VM.Standard.E5.Flex",
        "VM.Standard.E6.Flex",
        "VM.Standard.A1.Flex",
        "VM.Standard.A4.Flex",
        "VM.Optimized3.Flex",
    )
    if s.startswith(prefixes):
        return True
    return s in exact_or_starts or any(s.startswith(x + ".") for x in exact_or_starts)


OCI_CONFIG_FILE = os.path.expanduser(_env("OCI_CONFIG_FILE", "~/.oci/config"))
OCI_PROFILE = _env("OCI_PROFILE", "DEFAULT")
base_cfg = oci.config.from_file(file_location=OCI_CONFIG_FILE, profile_name=OCI_PROFILE)
TENANCY_OCID = _env("TENANCY_OCID", base_cfg.get("tenancy", ""))
if not TENANCY_OCID:
    raise RuntimeError(
        "TENANCY_OCID is empty. Set it in env or OCI config before running Cell 3."
    )
idc_home = oci.identity.IdentityClient(base_cfg)
subs = _list_all(idc_home.list_region_subscriptions, TENANCY_OCID)
region_names = sorted(
    {
        s.region_name
        for s in subs
        if getattr(s, "status", "READY") in ("READY", "ACTIVE", None)
    }
)
if not region_names:
    raise RuntimeError("No subscribed regions returned for tenancy.")
home_regions = [
    s.region_name for s in subs if bool(getattr(s, "is_home_region", False))
]
home_region_default = _env(
    "HOME_REGION", home_regions[0] if home_regions else region_names[0]
)
region_default = _env(
    "BENCHMARK_REGION", _env("REGION", base_cfg.get("region", region_names[0]))
)
if region_default not in region_names:
    region_default = region_names[0]
if home_region_default not in region_names:
    home_region_default = region_names[0]
comp_data = _list_all(
    idc_home.list_compartments,
    TENANCY_OCID,
    compartment_id_in_subtree=True,
    access_level="ACCESSIBLE",
)
comp_active = [c for c in comp_data if getattr(c, "lifecycle_state", "") == "ACTIVE"]
root_entry = ("TENANCY_ROOT", TENANCY_OCID)
comp_options = [root_entry] + sorted(
    [(f"{c.name} ({c.id[-8:]})", c.id) for c in comp_active],
    key=lambda x: x[0].lower(),
)
parent_comp_default = _env(
    "PARENT_COMPARTMENT_OCID", _env("BENCHMARK_COMPARTMENT_OCID", "")
)
if parent_comp_default not in [v for _, v in comp_options]:
    parent_comp_default = TENANCY_OCID
iam_policy_comp_default = _env("IAM_POLICY_COMPARTMENT_OCID", parent_comp_default)
if iam_policy_comp_default not in [v for _, v in comp_options]:
    iam_policy_comp_default = parent_comp_default
status_html = widgets.HTML(
    "<div class='ok-strip'><b>Cell 3</b>: loading regional options...</div>"
)
mode_note_html = widgets.HTML("")
target_note_html = widgets.HTML("")
image_mode_note_html = widgets.HTML("")
native_image_note_html = widgets.HTML("")
issues_html = widgets.HTML("")
apply_out = widgets.Output(
    layout=widgets.Layout(border="1px solid #ddd", padding="8px")
)
region_dd = widgets.Dropdown(
    options=[(r, r) for r in region_names], value=region_default
)
home_region_dd = widgets.Dropdown(
    options=[(r, r) for r in region_names], value=home_region_default
)
bench_comp_dd = widgets.Dropdown(options=comp_options, value=parent_comp_default)
parent_comp_dd = widgets.Dropdown(options=comp_options, value=parent_comp_default)
iam_comp_dd = widgets.Dropdown(options=comp_options, value=iam_policy_comp_default)
mode_default = _env("BENCHMARK_MODE", "parity").strip().lower()
if mode_default not in {"parity", "native"}:
    mode_default = "parity"
benchmark_mode_dd = widgets.Dropdown(
    options=[("parity", "parity"), ("native", "native")],
    value=mode_default,
)
target_default = _env("TARGET_CLUSTER", "both").strip().lower()
if target_default not in {"ca", "kpo", "both"}:
    target_default = "both"
target_dd = widgets.Dropdown(
    options=[("both", "both"), ("ca", "ca"), ("kpo", "kpo")],
    value=target_default,
)
scenario_default = _env("TARGET_SCENARIOS", "all").strip().lower()
if scenario_default not in {"all", "burst", "steady", "scale_down"}:
    scenario_default = "all"
scenario_dd = widgets.Dropdown(
    options=[
        ("all", "all"),
        ("burst", "burst"),
        ("steady", "steady"),
        ("scale_down", "scale_down"),
    ],
    value=scenario_default,
)
repeats_in = widgets.BoundedIntText(
    value=_to_int(_env("REPEATS", "5"), 5), min=1, max=20
)
image_mode_default = _env("IMAGE_MODE", "pin").strip().lower()
if image_mode_default not in {"pin", "filter"}:
    image_mode_default = "pin"
image_mode_dd = widgets.Dropdown(
    options=[("pin", "pin"), ("filter", "filter")],
    value=image_mode_default,
)
metrics_server_cb = widgets.Checkbox(
    value=_bool(_env("ENABLE_METRICS_SERVER", "false")),
    indent=False,
)
k8_dd = widgets.Dropdown(
    options=[
        (_env("KUBERNETES_VERSION", "v1.34.2"), _env("KUBERNETES_VERSION", "v1.34.2"))
    ]
)
scale_up_poll_in = widgets.BoundedIntText(
    value=_to_int(_env("SCENARIO_SCALE_UP_POLL_SECONDS", "1"), 1),
    min=1,
    max=60,
)
scale_down_poll_in = widgets.BoundedIntText(
    value=_to_int(_env("SCENARIO_SCALE_DOWN_POLL_SECONDS", "5"), 5),
    min=1,
    max=120,
)
burst_replicas_in = widgets.BoundedIntText(
    value=_to_int(_env("SCENARIO_BURST_REPLICAS", "20"), 20),
    min=1,
    max=500,
)
steady_replicas_in = widgets.BoundedIntText(
    value=_to_int(_env("SCENARIO_STEADY_REPLICAS", "8"), 8),
    min=1,
    max=500,
)
workload_image_in = widgets.Text(
    value=_env("WORKLOAD_IMAGE", "registry.k8s.io/pause:3.9").strip()
)
workload_base_replicas_in = widgets.BoundedIntText(
    value=_to_int(_env("WORKLOAD_BASE_REPLICAS", "0"), 0),
    min=0,
    max=500,
)
workload_cpu_request_in = widgets.Text(
    value=_env("WORKLOAD_CPU_REQUEST", "500m").strip()
)
workload_mem_request_in = widgets.Text(
    value=_env("WORKLOAD_MEM_REQUEST", "512Mi").strip()
)
workload_cpu_limit_in = widgets.Text(
    value=_env("WORKLOAD_CPU_LIMIT", _env("WORKLOAD_CPU_REQUEST", "500m")).strip()
)
workload_mem_limit_in = widgets.Text(
    value=_env("WORKLOAD_MEM_LIMIT", _env("WORKLOAD_MEM_REQUEST", "512Mi")).strip()
)
# Filter compartment: selector instead of free-text to reduce OCID mistakes.
filter_comp_options = [("Use KPO default compartment resolution", "")] + comp_options
kpo_filter_comp_default_env = _env("KPO_IMAGE_FILTER_COMPARTMENT_ID", "").strip()
# Keep filter compartment default unset unless user explicitly chooses one.
# This aligns with KPO default image resolution behavior in filter mode.
kpo_filter_comp_default = ""
kpo_filter_compartment_dd = widgets.Dropdown(
    options=filter_comp_options, value=kpo_filter_comp_default
)
kpo_filter_osver_in = widgets.Text(
    value=_env("KPO_IMAGE_OS_VERSION_FILTER", "8").strip()
)
# Filter compartment auto-sync state.
_filter_sync = {
    "suppress": False,
    "user_overridden": False,
    "last_auto": kpo_filter_comp_default,
}
shape_options = [("Loading shapes...", "")]
image_options = [("Loading images...", "")]
shape_map = {}
ui_shape_values = []
_shape_compat_cache = {}


def _pool_ui(title, prefix, autoscaled=False):
    shape_dd = widgets.Dropdown(options=shape_options)
    image_dd = widgets.Dropdown(options=image_options)
    if autoscaled:
        min_in = widgets.BoundedIntText(
            value=_to_int(_env(f"{prefix}_MIN_SIZE", "0"), 0),
            min=0,
            max=200,
        )
        max_in = widgets.BoundedIntText(
            value=_to_int(_env(f"{prefix}_MAX_SIZE", "5"), 5),
            min=0,
            max=300,
        )
        minmax_row = widgets.HBox(
            [
                _row("Minimum node count", min_in, label_width="240px"),
                _row("Maximum node count", max_in, label_width="240px"),
            ],
            layout=widgets.Layout(width="100%", gap="12px"),
        )
        size_in = None
    else:
        default_size = "2"
        size_in = widgets.BoundedIntText(
            value=_to_int(
                _env(f"{prefix}_SIZE", default_size), _to_int(default_size, 2)
            ),
            min=1,
            max=200,
        )
        min_in = None
        max_in = None
        minmax_row = _row("Node count", size_in)
    flex_ocpu = widgets.BoundedFloatText(
        value=_to_float(_env(f"{prefix}_FLEX_OCPUS", "2"), 2.0),
        min=1,
        max=128,
        step=1,
    )
    flex_mem = widgets.BoundedFloatText(
        value=_to_float(_env(f"{prefix}_FLEX_MEMORY_GB", "16"), 16.0),
        min=1,
        max=1024,
        step=1,
    )
    flex_box = widgets.VBox(
        [
            _row("Flex OCPUs", flex_ocpu),
            _row("Flex memory (GB)", flex_mem),
        ]
    )
    shape_row = _row("Shape", shape_dd)
    image_row = _row("Image", image_dd)
    state = {
        "title": title,
        "prefix": prefix,
        "autoscaled": autoscaled,
        "shape_dd": shape_dd,
        "image_dd": image_dd,
        "shape_row": shape_row,
        "image_row": image_row,
        "size_in": size_in,
        "min_in": min_in,
        "max_in": max_in,
        "flex_ocpu": flex_ocpu,
        "flex_mem": flex_mem,
        "flex_box": flex_box,
    }
    box = _section(
        title,
        shape_row,
        image_row,
        minmax_row,
        flex_box,
    )
    return state, box


ca_base_state, ca_base_box = _pool_ui(
    "CA Baseline Pool (non-autoscaled)", "CA_BASELINE_POOL", autoscaled=False
)
ca_auto_state, ca_auto_box = _pool_ui(
    "CA Autoscaled Benchmark Pool", "CA_AUTOSCALED_POOL", autoscaled=True
)
kpo_base_state, kpo_base_box = _pool_ui(
    "KPO Baseline Pool (non-autoscaled)", "KPO_BASELINE_POOL", autoscaled=False
)
kpo_auto_state, kpo_auto_box = _pool_ui(
    "KPO Autoscaled Profile", "KPO_AUTOSCALED_POOL", autoscaled=True
)
all_pools = [ca_base_state, ca_auto_state, kpo_base_state, kpo_auto_state]
native_instructions_html = widgets.HTML(
    "<div class='hint-strip'><b>Native mode guidance:</b> "
    "Select the KPO autoscaled image in <b>KPO Autoscaled Profile</b> when Image mode is <code>pin</code>. "
    "Pin shape policy below only controls how allowed shapes are derived from that selected image. "
    "For <code>Image mode=filter</code>, KPO resolves image via <code>imageFilter</code>. "
    "For reproducible tests use <code>on-demand</code>; <code>spot</code> maps to preemptible capacity.</div>"
)
kpo_pin_shape_policy_default = _env("KPO_NATIVE_PIN_SHAPE_POLICY", "").strip().lower()
if not kpo_pin_shape_policy_default:
    kpo_pin_shape_policy_default = (
        "auto"
        if _bool(_env("KPO_NATIVE_FILTER_SHAPES_BY_IMAGE", "true"), True)
        else "manual"
    )
if kpo_pin_shape_policy_default not in {"auto", "manual"}:
    kpo_pin_shape_policy_default = "auto"
kpo_pin_shape_policy_dd = widgets.Dropdown(
    options=[
        ("Auto-filter to selected image compatible shapes (recommended)", "auto"),
        ("Manual shape selection (validate on apply)", "manual"),
    ],
    value=kpo_pin_shape_policy_default,
)
kpo_native_allowed_shapes_sm = widgets.SelectMultiple(
    options=[],
    value=tuple(),
    rows=8,
)
_cap_default = (
    _csv_values(_env("KPO_NATIVE_CAPACITY_TYPE_VALUES", "on-demand")) or ["on-demand"]
)[0]
if _cap_default not in {"on-demand", "spot"}:
    _cap_default = "on-demand"
kpo_native_capacity_type_dd = widgets.Dropdown(
    options=[
        ("on-demand (recommended for benchmark reproducibility)", "on-demand"),
        ("spot / preemptible (interruption-tolerant tests)", "spot"),
    ],
    value=_cap_default,
)
kpo_native_limits_mode_dd = widgets.Dropdown(
    options=[("Auto-calculate limits", "auto"), ("Set manual limits", "manual")],
    value=(
        "manual"
        if _env("KPO_NATIVE_NODEPOOL_CPU_LIMIT", "").strip()
        or _env("KPO_NATIVE_NODEPOOL_MEMORY_LIMIT_GI", "").strip()
        else "auto"
    ),
)
kpo_native_nodepool_cpu_limit_in = widgets.Text(
    value=_env("KPO_NATIVE_NODEPOOL_CPU_LIMIT", "").strip()
)
kpo_native_nodepool_memory_limit_gi_in = widgets.Text(
    value=_env("KPO_NATIVE_NODEPOOL_MEMORY_LIMIT_GI", "").strip()
)
native_advanced_cb = widgets.Checkbox(
    value=bool(
        _env("CA_NATIVE_WORKER_POOLS_JSON", "").strip()
        or _env("BENCHMARK_PRICEBOOK_PATH", "").strip()
    ),
    indent=False,
)
ca_native_worker_pools_json_in = widgets.Textarea(
    value=_env("CA_NATIVE_WORKER_POOLS_JSON", "").strip(),
    layout=widgets.Layout(width="auto", height="120px"),
)
benchmark_pricebook_path_in = widgets.Text(
    value=_env("BENCHMARK_PRICEBOOK_PATH", "").strip()
)
kpo_pin_shape_policy_row = _row(
    "Pin shape policy (only for Image mode = pin)", kpo_pin_shape_policy_dd
)
kpo_native_allowed_shapes_row = _row(
    "KPO native allowed shapes", kpo_native_allowed_shapes_sm
)
kpo_native_capacity_row = _row("KPO native capacity type", kpo_native_capacity_type_dd)
kpo_native_manual_limits_box = widgets.VBox(
    [
        _row("KPO native NodePool CPU limit", kpo_native_nodepool_cpu_limit_in),
        _row(
            "KPO native NodePool memory limit (Gi)",
            kpo_native_nodepool_memory_limit_gi_in,
        ),
    ]
)
native_advanced_box = widgets.VBox(
    [
        _row("CA native worker pools JSON", ca_native_worker_pools_json_in),
        _row("Benchmark pricebook path", benchmark_pricebook_path_in),
    ]
)
native_section = _section(
    "Native Mode Options (optional)",
    native_instructions_html,
    kpo_pin_shape_policy_row,
    kpo_native_allowed_shapes_row,
    kpo_native_capacity_row,
    _row("KPO NodePool limits mode", kpo_native_limits_mode_dd),
    kpo_native_manual_limits_box,
    _row("Show advanced native options", native_advanced_cb),
    native_advanced_box,
)
image_filter_section = _section(
    "KPO Image Filter Inputs (used when Image mode = filter)",
    widgets.HTML(
        "<div class='hint-strip'><b>Image mode filter:</b> KPO resolves a compatible OKE image at runtime. "
        "Use this for drift experiments. For strict reproducibility, keep <code>Image mode=pin</code>. "
        "Compartment is optional; leave default to rely on KPO default resolution. "
        "If a compartment is set, KPO resolves images only from that compartment and workload identity "
        "must include <code>read instance-images</code> for that compartment.</div>"
    ),
    _row("KPO image filter compartment", kpo_filter_compartment_dd),
    _row("KPO image OS version filter", kpo_filter_osver_in),
)


def _pool_values(pool_state):
    d = {
        "shape": pool_state["shape_dd"].value,
        "image_id": pool_state["image_dd"].value,
        "flex_ocpus": (
            pool_state["flex_ocpu"].value
            if _is_flex(pool_state["shape_dd"].value)
            else None
        ),
        "flex_memory_gb": (
            pool_state["flex_mem"].value
            if _is_flex(pool_state["shape_dd"].value)
            else None
        ),
    }
    if pool_state["autoscaled"]:
        d["min_size"] = pool_state["min_in"].value
        d["max_size"] = pool_state["max_in"].value
    else:
        d["size"] = pool_state["size_in"].value
    return d


def _refresh_flex_for_pool(pool_state):
    shape_name = pool_state["shape_dd"].value
    is_flex = _is_flex(shape_name)
    pool_state["flex_box"].layout.display = "" if is_flex else "none"
    if not is_flex:
        return
    shp = shape_map.get(shape_name)
    if shp is None:
        return
    min_ocpu, max_ocpu, min_mem, max_mem, default_per_ocpu = _shape_bounds(shp)
    pool_state["flex_ocpu"].min = min_ocpu
    pool_state["flex_ocpu"].max = max_ocpu
    if (
        pool_state["flex_ocpu"].value < min_ocpu
        or pool_state["flex_ocpu"].value > max_ocpu
    ):
        pool_state["flex_ocpu"].value = min_ocpu
    pool_state["flex_mem"].min = min_mem
    pool_state["flex_mem"].max = max_mem
    suggested_mem = max(
        min_mem, min(max_mem, pool_state["flex_ocpu"].value * default_per_ocpu)
    )
    if pool_state["flex_mem"].value < min_mem or pool_state["flex_mem"].value > max_mem:
        pool_state["flex_mem"].value = suggested_mem


def _make_region_clients(region):
    cfg = _make_cfg(base_cfg, region)
    return (
        oci.container_engine.ContainerEngineClient(cfg),
        oci.core.ComputeClient(cfg),
    )


def _load_k8_versions(ce_client):
    try:
        resp = ce_client.get_cluster_options(cluster_option_id="all").data
        versions = list(getattr(resp, "kubernetes_versions", []) or [])
    except Exception:
        versions = []
    if not versions:
        versions = [_env("KUBERNETES_VERSION", "v1.34.2")]
    versions = [v for v in versions if _k8_ge(v, 1, 31)]
    if not versions:
        versions = [_env("KUBERNETES_VERSION", "v1.34.2")]
    return sorted(set(versions), reverse=True)


def _load_shapes(ce_client, comp_client, compartment_id):
    compute_data = _list_all(comp_client.list_shapes, compartment_id)
    compute_active = [
        s
        for s in compute_data
        if getattr(s, "lifecycle_state", "ACTIVE") in ("ACTIVE", None)
    ]
    compute_map = {s.shape: s for s in compute_active}
    oke_shapes = []
    try:
        np_opts = ce_client.get_node_pool_options(node_pool_option_id="all").data
        oke_shapes = list(getattr(np_opts, "shapes", []) or [])
    except Exception:
        oke_shapes = []
    if oke_shapes:
        names = sorted(set(oke_shapes))
    else:
        names = sorted(compute_map.keys())
    opts = [(n, n) for n in names] if names else [("No shapes found", "")]
    return opts, compute_map, names


def _load_oke_images(ce_client, comp_client, compartment_id, k8_version):
    minor = _k8_minor(k8_version)
    options = []
    try:
        np_opts = ce_client.get_node_pool_options(node_pool_option_id="all").data
        sources = list(getattr(np_opts, "sources", []) or [])
        for s in sources:
            image_id = getattr(s, "image_id", "")
            source_name = getattr(s, "source_name", "")
            if not image_id:
                continue
            if minor and minor not in source_name:
                continue
            options.append((f"{source_name} | {image_id[:24]}...", image_id))
    except Exception:
        pass
    if not options:
        try:
            imgs = _list_all(comp_client.list_images, compartment_id)
            for img in imgs:
                name = getattr(img, "display_name", "") or ""
                image_id = getattr(img, "id", "")
                if not image_id:
                    continue
                if "OKE" not in name.upper():
                    continue
                if minor and minor not in name:
                    continue
                options.append((f"{name} | {image_id[:24]}...", image_id))
        except Exception:
            pass
    if not options:
        return [("No compatible images discovered", "")]
    dedup = {}
    for lbl, iid in options:
        dedup[iid] = lbl
    return [(lbl, iid) for iid, lbl in dedup.items()]


def _shape_compat_info(region, image_id):
    if not image_id:
        return {"resolved": False, "shapes": set(), "error": "missing image id"}
    key = (region, image_id)
    if key in _shape_compat_cache:
        return _shape_compat_cache[key]
    _, comp_client = _make_region_clients(region)
    try:
        entries = _list_all(
            comp_client.list_image_shape_compatibility_entries, image_id=image_id
        )
        shapes = {e.shape for e in entries}
        info = {"resolved": True, "shapes": shapes, "error": ""}
    except Exception as e:
        info = {"resolved": False, "shapes": set(), "error": str(e)}
    _shape_compat_cache[key] = info
    return info


def _shape_selection_base_for_native():
    base = list(ui_shape_values)
    capacity = kpo_native_capacity_type_dd.value
    if capacity == "spot":
        base = [s for s in base if _is_preemptible_supported_shape(s)]
    return base


def _set_filter_compartment_value(value: str, auto=False):
    if value not in [v for _, v in kpo_filter_compartment_dd.options]:
        value = ""
    _filter_sync["suppress"] = True
    try:
        kpo_filter_compartment_dd.value = value
    finally:
        _filter_sync["suppress"] = False
    if auto:
        _filter_sync["last_auto"] = value


def _maybe_auto_sync_filter_compartment():
    if _filter_sync["user_overridden"]:
        return
    # Keep default filter compartment unset unless user explicitly chooses one.
    # This follows the KPO default image resolution path.
    _set_filter_compartment_value("", auto=True)


def _on_filter_compartment_change(change):
    if change.get("name") != "value":
        return
    if _filter_sync["suppress"]:
        return
    if change.get("new") != _filter_sync["last_auto"]:
        _filter_sync["user_overridden"] = True


def _sync_visibility():
    target = target_dd.value
    mode = benchmark_mode_dd.value
    image_mode = image_mode_dd.value
    has_ca = _target_has_ca(target)
    has_kpo = _target_has_kpo(target)
    _set_visible(ca_base_box, has_ca)
    _set_visible(ca_auto_box, has_ca)
    _set_visible(kpo_base_box, has_kpo)
    _set_visible(kpo_auto_box, has_kpo)
    _set_visible(image_filter_section, has_kpo and image_mode == "filter")
    _set_visible(native_section, has_kpo and mode == "native")
    _set_visible(
        kpo_pin_shape_policy_row, has_kpo and mode == "native" and image_mode == "pin"
    )
    _set_visible(
        kpo_native_manual_limits_box,
        has_kpo and mode == "native" and kpo_native_limits_mode_dd.value == "manual",
    )
    _set_visible(
        native_advanced_box, has_kpo and mode == "native" and native_advanced_cb.value
    )
    # KPO autoscaled pinned image is not used when image mode is filter.
    _set_visible(kpo_auto_state["image_row"], not (has_kpo and image_mode == "filter"))
    if mode == "native":
        mode_note_html.value = "<div class='ok-strip'><b>Mode:</b> native. Cross-target fairness checks are relaxed.</div>"
    else:
        mode_note_html.value = "<div class='ok-strip'><b>Mode:</b> parity. Cross-target fairness checks are enforced.</div>"
    if target == "both":
        target_note_html.value = "<div class='ok-strip'><b>Target:</b> both. CA and KPO inputs are required.</div>"
    elif target == "ca":
        target_note_html.value = "<div class='ok-strip'><b>Target:</b> ca only. KPO-specific settings are hidden and not required.</div>"
    else:
        target_note_html.value = "<div class='ok-strip'><b>Target:</b> kpo only. CA-specific settings are hidden and not required.</div>"
    if has_kpo and image_mode == "filter":
        image_mode_note_html.value = (
            "<div class='hint-strip'><b>Filter mode active:</b> "
            "KPO autoscaled pinned image field is ignored. OCINodeClass image is resolved via <code>imageFilter</code>.</div>"
        )
    else:
        image_mode_note_html.value = ""
    _refresh_native_shape_options()


def _refresh_native_shape_options():
    target = target_dd.value
    mode = benchmark_mode_dd.value
    if not (_target_has_kpo(target) and mode == "native"):
        native_image_note_html.value = ""
        return
    base_shapes = _shape_selection_base_for_native()
    selected_shape = kpo_auto_state["shape_dd"].value
    image_mode = image_mode_dd.value
    auto_filter_pin = kpo_pin_shape_policy_dd.value == "auto"
    current_selected = [
        x for x in list(kpo_native_allowed_shapes_sm.value) if x in base_shapes
    ]
    env_selected = [
        x
        for x in _csv_values(_env("KPO_NATIVE_ALLOWED_SHAPES", ""))
        if x in base_shapes
    ]
    if image_mode == "pin" and auto_filter_pin:
        compat_info = _shape_compat_info(
            region_dd.value, kpo_auto_state["image_dd"].value
        )
        if compat_info["resolved"]:
            filtered = [s for s in base_shapes if s in compat_info["shapes"]]
        else:
            filtered = base_shapes
        kpo_native_allowed_shapes_sm.options = filtered
        selected = (
            [x for x in current_selected if x in filtered]
            or [x for x in env_selected if x in filtered]
            or (
                [selected_shape]
                if selected_shape in filtered
                else (filtered[:1] if filtered else [])
            )
        )
        kpo_native_allowed_shapes_sm.value = tuple(selected)
        if not compat_info["resolved"]:
            native_image_note_html.value = (
                "<div class='hint-strip'><b>Native + pin:</b> compatibility lookup is temporarily unavailable. "
                "Allowed shapes are filtered by OKE-supported shapes"
                + (
                    " and spot/preemptible support."
                    if kpo_native_capacity_type_dd.value == "spot"
                    else "."
                )
                + "</div>"
            )
        elif filtered:
            native_image_note_html.value = (
                "<div class='ok-strip'><b>Native + pin:</b> allowed shapes are filtered by selected image compatibility"
                + (
                    " and spot/preemptible support."
                    if kpo_native_capacity_type_dd.value == "spot"
                    else "."
                )
                + "</div>"
            )
        else:
            native_image_note_html.value = (
                "<div class='issues-strip'><b>Native + pin:</b> selected KPO image has no compatible shapes after current filters. "
                "Change image or choose manual pin shape policy.</div>"
            )
    else:
        kpo_native_allowed_shapes_sm.options = base_shapes
        selected = (
            current_selected
            or env_selected
            or (
                [selected_shape]
                if selected_shape in base_shapes
                else (base_shapes[:1] if base_shapes else [])
            )
        )
        kpo_native_allowed_shapes_sm.value = tuple(selected)
        if image_mode == "pin":
            native_image_note_html.value = (
                "<div class='ok-strip'><b>Native + pin:</b> manual shape selection enabled. "
                "Shapes are still constrained by OKE-supported list"
                + (
                    " and spot/preemptible support."
                    if kpo_native_capacity_type_dd.value == "spot"
                    else "."
                )
                + " Compatibility is validated when available.</div>"
            )
        else:
            native_image_note_html.value = (
                "<div class='ok-strip'><b>Native + filter:</b> multi-shape selection enabled. "
                "Image is resolved at runtime via <code>imageFilter</code>.</div>"
            )


def _on_shape_change(pool_state):
    _refresh_flex_for_pool(pool_state)
    if pool_state is kpo_auto_state:
        _refresh_native_shape_options()


def _refresh_region_data(*_):
    issues_html.value = ""
    status_html.value = "<div class='ok-strip'><b>Cell 3</b>: loading region-scoped versions/shapes/images...</div>"
    region = region_dd.value
    comp_id = bench_comp_dd.value
    ce_client, comp_client = _make_region_clients(region)
    k8_versions = _load_k8_versions(ce_client)
    _safe_set_options(
        k8_dd,
        [(v, v) for v in k8_versions],
        preferred=_env("KUBERNETES_VERSION", k8_versions[0]),
    )
    global shape_map, ui_shape_values
    s_opts, s_map, s_names = _load_shapes(ce_client, comp_client, comp_id)
    shape_map = s_map
    ui_shape_values = s_names
    for pool in all_pools:
        _safe_set_options(
            pool["shape_dd"], s_opts, preferred=_env(f"{pool['prefix']}_SHAPE", "")
        )
        _refresh_flex_for_pool(pool)
    i_opts = _load_oke_images(ce_client, comp_client, comp_id, k8_dd.value)
    for pool in all_pools:
        _safe_set_options(
            pool["image_dd"], i_opts, preferred=_env(f"{pool['prefix']}_IMAGE_ID", "")
        )
    _maybe_auto_sync_filter_compartment()
    _refresh_native_shape_options()
    status_html.value = "<div class='ok-strip'><b>Cell 3</b>: ready. Review selections and click Apply.</div>"


def _refresh_images_only(*_):
    region = region_dd.value
    comp_id = bench_comp_dd.value
    ce_client, comp_client = _make_region_clients(region)
    i_opts = _load_oke_images(ce_client, comp_client, comp_id, k8_dd.value)
    for pool in all_pools:
        _safe_set_options(pool["image_dd"], i_opts, preferred=pool["image_dd"].value)
    _refresh_native_shape_options()


def _validate_and_apply(_):
    issues = []
    apply_out.clear_output()
    issues_html.value = ""
    region = region_dd.value
    home_region = home_region_dd.value
    bench_comp = bench_comp_dd.value
    parent_comp = parent_comp_dd.value
    iam_comp = iam_comp_dd.value
    benchmark_mode = benchmark_mode_dd.value
    target = target_dd.value
    scenarios = scenario_dd.value
    repeats = repeats_in.value
    image_mode = image_mode_dd.value
    enable_metrics = metrics_server_cb.value
    k8_version = k8_dd.value
    scale_up_poll = scale_up_poll_in.value
    scale_down_poll = scale_down_poll_in.value
    burst_replicas = burst_replicas_in.value
    steady_replicas = steady_replicas_in.value
    workload_image = workload_image_in.value.strip()
    workload_base_replicas = workload_base_replicas_in.value
    workload_cpu_request = workload_cpu_request_in.value.strip()
    workload_mem_request = workload_mem_request_in.value.strip()
    workload_cpu_limit = workload_cpu_limit_in.value.strip()
    workload_mem_limit = workload_mem_limit_in.value.strip()
    kpo_filter_compartment = str(kpo_filter_compartment_dd.value or "").strip()
    kpo_filter_osver = kpo_filter_osver_in.value.strip() or "8"
    if benchmark_mode not in {"parity", "native"}:
        issues.append(
            f"BENCHMARK_MODE must be parity|native. Selected: {benchmark_mode}"
        )
    if target not in {"ca", "kpo", "both"}:
        issues.append(f"TARGET_CLUSTER must be ca|kpo|both. Selected: {target}")
    if scenarios not in {"all", "burst", "steady", "scale_down"}:
        issues.append(
            f"TARGET_SCENARIOS must be all|burst|steady|scale_down. Selected: {scenarios}"
        )
    if image_mode not in {"pin", "filter"}:
        issues.append(f"IMAGE_MODE must be pin|filter. Selected: {image_mode}")
    if not _k8_ge(k8_version, 1, 31):
        issues.append(f"Kubernetes version must be >=1.31. Selected: {k8_version}")
    if not region:
        issues.append("Region is required.")
    if not bench_comp:
        issues.append("Benchmark compartment is required.")
    if not parent_comp:
        issues.append("Parent compartment is required.")
    if not iam_comp:
        issues.append("IAM policy compartment is required.")
    if not home_region:
        issues.append("Home region is required for strict IAM mode.")
    if scale_up_poll < 1:
        issues.append("SCENARIO_SCALE_UP_POLL_SECONDS must be >= 1")
    if scale_down_poll < 1:
        issues.append("SCENARIO_SCALE_DOWN_POLL_SECONDS must be >= 1")
    if burst_replicas < 1:
        issues.append("SCENARIO_BURST_REPLICAS must be >= 1")
    if steady_replicas < 1:
        issues.append("SCENARIO_STEADY_REPLICAS must be >= 1")
    if not workload_image:
        issues.append("WORKLOAD_IMAGE must not be empty")
    if not workload_cpu_request or not workload_mem_request:
        issues.append("WORKLOAD_CPU_REQUEST and WORKLOAD_MEM_REQUEST must not be empty")
    if not workload_cpu_limit or not workload_mem_limit:
        issues.append("WORKLOAD_CPU_LIMIT and WORKLOAD_MEM_LIMIT must not be empty")
    pools = {
        "CA_BASELINE_POOL": _pool_values(ca_base_state),
        "CA_AUTOSCALED_POOL": _pool_values(ca_auto_state),
        "KPO_BASELINE_POOL": _pool_values(kpo_base_state),
        "KPO_AUTOSCALED_POOL": _pool_values(kpo_auto_state),
    }
    target_pool_names = []
    if _target_has_ca(target):
        target_pool_names.extend(["CA_BASELINE_POOL", "CA_AUTOSCALED_POOL"])
    if _target_has_kpo(target):
        target_pool_names.extend(["KPO_BASELINE_POOL", "KPO_AUTOSCALED_POOL"])
    for name in target_pool_names:
        p = pools[name]
        if not p["shape"]:
            issues.append(f"{name}: shape is required.")
        image_required = not (
            name == "KPO_AUTOSCALED_POOL"
            and _target_has_kpo(target)
            and image_mode == "filter"
        )
        if image_required and not p["image_id"]:
            issues.append(f"{name}: image_id is required.")
        if "size" in p and p["size"] < 1:
            issues.append(f"{name}: size must be >=1.")
        if "min_size" in p:
            if p["min_size"] > p["max_size"]:
                issues.append(f"{name}: min_size cannot exceed max_size.")
            if p["min_size"] < 0 or p["max_size"] < 0:
                issues.append(f"{name}: min/max must be >=0.")
        if _is_flex(p["shape"]):
            if p["flex_ocpus"] is None or p["flex_ocpus"] <= 0:
                issues.append(f"{name}: flex_ocpus must be > 0 for Flex shapes.")
            if p["flex_memory_gb"] is None or p["flex_memory_gb"] <= 0:
                issues.append(f"{name}: flex_memory_gb must be > 0 for Flex shapes.")
    if _target_has_kpo(target) and pools["KPO_BASELINE_POOL"]["size"] < 2:
        issues.append(
            "KPO_BASELINE_POOL: size must be >=2 for default KPO chart readiness "
            "(replicaCount=2 with required pod anti-affinity)."
        )
    if benchmark_mode == "parity" and target == "both":
        ca_auto = pools["CA_AUTOSCALED_POOL"]
        kpo_auto = pools["KPO_AUTOSCALED_POOL"]
        if (
            ca_auto["min_size"] != kpo_auto["min_size"]
            or ca_auto["max_size"] != kpo_auto["max_size"]
        ):
            issues.append(
                "Fairness check failed (parity): CA and KPO autoscaled min/max must match in TARGET_CLUSTER=both."
            )
        if ca_auto["shape"] != kpo_auto["shape"]:
            issues.append(
                "Fairness check failed (parity): CA and KPO autoscaled shapes must match in TARGET_CLUSTER=both."
            )
        if _is_flex(ca_auto["shape"]) and _is_flex(kpo_auto["shape"]):
            if abs((ca_auto["flex_ocpus"] or 0) - (kpo_auto["flex_ocpus"] or 0)) > 1e-9:
                issues.append(
                    "Fairness check failed (parity): CA and KPO autoscaled Flex OCPU must match."
                )
            if (
                abs(
                    (ca_auto["flex_memory_gb"] or 0) - (kpo_auto["flex_memory_gb"] or 0)
                )
                > 1e-9
            ):
                issues.append(
                    "Fairness check failed (parity): CA and KPO autoscaled Flex memory must match."
                )
    kpo_native_allowed_shapes = list(kpo_native_allowed_shapes_sm.value)
    kpo_native_capacity_types = [kpo_native_capacity_type_dd.value]
    kpo_native_limits_mode = kpo_native_limits_mode_dd.value
    kpo_native_cpu_limit = kpo_native_nodepool_cpu_limit_in.value.strip()
    kpo_native_mem_limit = kpo_native_nodepool_memory_limit_gi_in.value.strip()
    ca_native_worker_pools_json = ca_native_worker_pools_json_in.value.strip()
    benchmark_pricebook_path = benchmark_pricebook_path_in.value.strip()
    pin_policy = kpo_pin_shape_policy_dd.value
    auto_filter_shapes = pin_policy == "auto"
    if benchmark_mode == "native" and _target_has_kpo(target):
        allowed_base = _shape_selection_base_for_native()
        if not kpo_native_allowed_shapes:
            issues.append("Native mode: select at least one KPO native allowed shape.")
        if pools["KPO_AUTOSCALED_POOL"]["shape"] not in kpo_native_allowed_shapes:
            issues.append(
                "Native mode: KPO autoscaled pool shape must be included in KPO native allowed shapes."
            )
        if kpo_native_capacity_types[0] not in {"on-demand", "spot"}:
            issues.append(
                "Native mode: KPO native capacity type must be on-demand or spot."
            )
        invalid_by_capacity = [
            s for s in kpo_native_allowed_shapes if s not in allowed_base
        ]
        if invalid_by_capacity:
            if kpo_native_capacity_types[0] == "spot":
                issues.append(
                    "Native mode spot: selected allowed shape(s) are not in preemptible-supported families: "
                    + ", ".join(invalid_by_capacity)
                )
            else:
                issues.append(
                    "Native mode: selected allowed shape(s) are not in current OKE-supported shape list: "
                    + ", ".join(invalid_by_capacity)
                )
        if image_mode == "pin":
            compat_info = _shape_compat_info(
                region, pools["KPO_AUTOSCALED_POOL"]["image_id"]
            )
            if compat_info["resolved"]:
                invalid = [
                    s
                    for s in kpo_native_allowed_shapes
                    if s not in compat_info["shapes"]
                ]
                if invalid:
                    issues.append(
                        "Native + pin: selected KPO image is incompatible with allowed shape(s): "
                        + ", ".join(invalid)
                    )
        if kpo_native_limits_mode == "manual":
            if not kpo_native_cpu_limit:
                issues.append("Native mode manual limits: CPU limit is required.")
            else:
                try:
                    if float(kpo_native_cpu_limit) <= 0:
                        issues.append(
                            "Native mode manual limits: CPU limit must be > 0."
                        )
                except Exception:
                    issues.append(
                        "Native mode manual limits: CPU limit must be numeric."
                    )
            if not kpo_native_mem_limit:
                issues.append(
                    "Native mode manual limits: memory limit (Gi) is required."
                )
            else:
                try:
                    if float(kpo_native_mem_limit) <= 0:
                        issues.append(
                            "Native mode manual limits: memory limit must be > 0."
                        )
                except Exception:
                    issues.append(
                        "Native mode manual limits: memory limit (Gi) must be numeric."
                    )
    if image_mode == "filter" and _target_has_kpo(target):
        if not kpo_filter_osver:
            issues.append(
                "KPO image OS version filter must not be empty when IMAGE_MODE=filter."
            )
        elif not re.fullmatch(r"[0-9]+(\.[0-9]+)?", kpo_filter_osver):
            issues.append(
                "KPO image OS version filter must be numeric (for example 8 or 8.10) when IMAGE_MODE=filter."
            )
        if kpo_filter_compartment and not _looks_like_ocid(
            kpo_filter_compartment, "compartment"
        ):
            issues.append(
                "KPO image filter compartment must be a compartment OCID or default resolution."
            )
    if ca_native_worker_pools_json:
        try:
            parsed = json.loads(ca_native_worker_pools_json)
            if not isinstance(parsed, (dict, list)):
                issues.append(
                    "CA_NATIVE_WORKER_POOLS_JSON must be a JSON object or array."
                )
        except Exception as e:
            issues.append(f"CA_NATIVE_WORKER_POOLS_JSON is invalid JSON: {e}")
    if (
        benchmark_pricebook_path
        and not Path(os.path.expanduser(benchmark_pricebook_path)).exists()
    ):
        issues.append(
            f"BENCHMARK_PRICEBOOK_PATH does not exist: {benchmark_pricebook_path}"
        )
    _, comp_client = _make_region_clients(region)
    for name in target_pool_names:
        p = pools[name]
        if (
            name == "KPO_AUTOSCALED_POOL"
            and _target_has_kpo(target)
            and image_mode == "filter"
        ):
            continue
        if p["image_id"] and p["shape"]:
            try:
                entries = _list_all(
                    comp_client.list_image_shape_compatibility_entries,
                    image_id=p["image_id"],
                )
                shapes = {e.shape for e in entries}
                if shapes and p["shape"] not in shapes:
                    issues.append(
                        f"{name}: selected image is not shape-compatible for {p['shape']}."
                    )
            except Exception:
                pass
    if issues:
        issues_html.value = (
            "<div class='issues-strip'><b>Validation failed:</b><br>- "
            + "<br>- ".join(issues)
            + "</div>"
        )
        with apply_out:
            print("Cell 3 apply failed. Fix validation issues and re-apply.")
        return
    _setenv("BENCHMARK_REGION", region)
    _setenv("REGION", region)
    _setenv("HOME_REGION", home_region)
    _setenv("BENCHMARK_COMPARTMENT_OCID", bench_comp)
    _setenv("PARENT_COMPARTMENT_OCID", parent_comp)
    _setenv("IAM_POLICY_COMPARTMENT_OCID", iam_comp)
    _setenv("BENCHMARK_MODE", benchmark_mode)
    _setenv("TARGET_CLUSTER", target)
    _setenv("TARGET_SCENARIOS", scenarios)
    _setenv("REPEATS", repeats)
    _setenv("IMAGE_MODE", image_mode)
    _setenv("ENABLE_METRICS_SERVER", str(enable_metrics).lower())
    _setenv("KUBERNETES_VERSION", k8_version)
    _setenv("SCENARIO_SCALE_UP_POLL_SECONDS", scale_up_poll)
    _setenv("SCENARIO_SCALE_DOWN_POLL_SECONDS", scale_down_poll)
    _setenv("SCENARIO_POLL_SECONDS", scale_up_poll)
    _setenv("SCENARIO_BURST_REPLICAS", burst_replicas)
    _setenv("SCENARIO_STEADY_REPLICAS", steady_replicas)
    _setenv("WORKLOAD_IMAGE", workload_image)
    _setenv("WORKLOAD_BASE_REPLICAS", workload_base_replicas)
    _setenv("WORKLOAD_CPU_REQUEST", workload_cpu_request)
    _setenv("WORKLOAD_MEM_REQUEST", workload_mem_request)
    _setenv("WORKLOAD_CPU_LIMIT", workload_cpu_limit)
    _setenv("WORKLOAD_MEM_LIMIT", workload_mem_limit)
    if image_mode == "filter" and _target_has_kpo(target):
        _setenv("KPO_IMAGE_FILTER_COMPARTMENT_ID", kpo_filter_compartment)
        _setenv("KPO_IMAGE_OS_VERSION_FILTER", kpo_filter_osver)
    else:
        # Avoid stale scope from prior filter-mode runs.
        _setenv("KPO_IMAGE_FILTER_COMPARTMENT_ID", "")
        _setenv("KPO_IMAGE_OS_VERSION_FILTER", kpo_filter_osver)
    _setenv("KPO_NATIVE_PIN_SHAPE_POLICY", pin_policy)
    _setenv("KPO_NATIVE_FILTER_SHAPES_BY_IMAGE", str(auto_filter_shapes).lower())
    _setenv("KPO_NATIVE_ALLOWED_SHAPES", ",".join(kpo_native_allowed_shapes))
    _setenv("KPO_NATIVE_CAPACITY_TYPE_VALUES", ",".join(kpo_native_capacity_types))
    if (
        benchmark_mode == "native"
        and _target_has_kpo(target)
        and kpo_native_limits_mode == "manual"
    ):
        _setenv("KPO_NATIVE_NODEPOOL_CPU_LIMIT", kpo_native_cpu_limit)
        _setenv("KPO_NATIVE_NODEPOOL_MEMORY_LIMIT_GI", kpo_native_mem_limit)
    else:
        _setenv("KPO_NATIVE_NODEPOOL_CPU_LIMIT", "")
        _setenv("KPO_NATIVE_NODEPOOL_MEMORY_LIMIT_GI", "")
    _setenv("CA_NATIVE_WORKER_POOLS_JSON", ca_native_worker_pools_json)
    _setenv("BENCHMARK_PRICEBOOK_PATH", benchmark_pricebook_path)
    for name, p in pools.items():
        _setenv(f"{name}_SHAPE", p["shape"])
        _setenv(f"{name}_IMAGE_ID", p["image_id"])
        if "size" in p:
            _setenv(f"{name}_SIZE", p["size"])
        if "min_size" in p:
            _setenv(f"{name}_MIN_SIZE", p["min_size"])
            _setenv(f"{name}_MAX_SIZE", p["max_size"])
        _setenv(
            f"{name}_FLEX_OCPUS", "" if p["flex_ocpus"] is None else p["flex_ocpus"]
        )
        _setenv(
            f"{name}_FLEX_MEMORY_GB",
            "" if p["flex_memory_gb"] is None else p["flex_memory_gb"],
        )
    _setenv("CA_EXPECTED_BENCHMARK_NODE_FLOOR", pools["CA_AUTOSCALED_POOL"]["min_size"])
    _setenv(
        "KPO_EXPECTED_BENCHMARK_NODE_FLOOR", pools["KPO_AUTOSCALED_POOL"]["min_size"]
    )
    summary = {
        "BENCHMARK_MODE": benchmark_mode,
        "BENCHMARK_REGION": region,
        "HOME_REGION": home_region,
        "BENCHMARK_COMPARTMENT_OCID": bench_comp,
        "PARENT_COMPARTMENT_OCID": parent_comp,
        "IAM_POLICY_COMPARTMENT_OCID": iam_comp,
        "TARGET_CLUSTER": target,
        "TARGET_SCENARIOS": scenarios,
        "REPEATS": repeats,
        "IMAGE_MODE": image_mode,
        "ENABLE_METRICS_SERVER": enable_metrics,
        "KUBERNETES_VERSION": k8_version,
        "SCENARIO_SCALE_UP_POLL_SECONDS": scale_up_poll,
        "SCENARIO_SCALE_DOWN_POLL_SECONDS": scale_down_poll,
        "SCENARIO_POLL_SECONDS": scale_up_poll,
        "SCENARIO_BURST_REPLICAS": burst_replicas,
        "SCENARIO_STEADY_REPLICAS": steady_replicas,
        "WORKLOAD_IMAGE": workload_image,
        "KPO_IMAGE_FILTER_COMPARTMENT_ID": _env("KPO_IMAGE_FILTER_COMPARTMENT_ID", ""),
        "KPO_IMAGE_OS_VERSION_FILTER": _env("KPO_IMAGE_OS_VERSION_FILTER", ""),
        "KPO_NATIVE_PIN_SHAPE_POLICY": pin_policy,
        "KPO_NATIVE_FILTER_SHAPES_BY_IMAGE": auto_filter_shapes,
        "KPO_NATIVE_ALLOWED_SHAPES": kpo_native_allowed_shapes,
        "KPO_NATIVE_CAPACITY_TYPE_VALUES": kpo_native_capacity_types,
        "KPO_NATIVE_LIMITS_MODE": kpo_native_limits_mode,
        "KPO_NATIVE_NODEPOOL_CPU_LIMIT": _env("KPO_NATIVE_NODEPOOL_CPU_LIMIT", ""),
        "KPO_NATIVE_NODEPOOL_MEMORY_LIMIT_GI": _env(
            "KPO_NATIVE_NODEPOOL_MEMORY_LIMIT_GI", ""
        ),
        "CA_NATIVE_WORKER_POOLS_JSON_SET": bool(ca_native_worker_pools_json),
        "BENCHMARK_PRICEBOOK_PATH": benchmark_pricebook_path,
        "CA_BASELINE_POOL": pools["CA_BASELINE_POOL"],
        "CA_AUTOSCALED_POOL": pools["CA_AUTOSCALED_POOL"],
        "KPO_BASELINE_POOL": pools["KPO_BASELINE_POOL"],
        "KPO_AUTOSCALED_POOL": pools["KPO_AUTOSCALED_POOL"],
    }
    issues_html.value = "<div class='ok-strip'><b>Validation passed.</b> Selections persisted to environment.</div>"
    with apply_out:
        print("Cell 3 complete: Selections applied.")
        print(json.dumps(summary, indent=2))
        if image_mode == "filter" and _target_has_kpo(target):
            if _env("KPO_IMAGE_FILTER_COMPARTMENT_ID", "").strip():
                print(
                    "Note: KPO filter mode uses a compartment-scoped imageFilter. "
                    "Ensure workload identity policy includes 'read instance-images' "
                    "for the selected compartment."
                )
            else:
                print(
                    "Note: KPO filter mode uses default KPO image resolution "
                    "(no explicit filter compartment)."
                )


top_section = _section(
    "Global Selection",
    _row("Benchmark region", region_dd),
    _row("Home region (IAM)", home_region_dd),
    _row("Benchmark compartment", bench_comp_dd),
    _row("Parent compartment", parent_comp_dd),
    _row("IAM policy compartment", iam_comp_dd),
    _row("Benchmark mode", benchmark_mode_dd),
    _row("Target cluster(s)", target_dd),
    _row("Scenarios", scenario_dd),
    _row("Repeats", repeats_in),
    _row("Image mode", image_mode_dd),
    _row("Enable Metrics Server", metrics_server_cb),
    _row("Kubernetes version", k8_dd),
)
benchmark_knobs_section = _section(
    "Benchmark Knobs",
    _row("Scale-up poll seconds", scale_up_poll_in),
    _row("Scale-down poll seconds", scale_down_poll_in),
    _row("Burst replicas", burst_replicas_in),
    _row("Steady replicas", steady_replicas_in),
)
workload_section = _section(
    "Workload Profile",
    _row("Workload image", workload_image_in),
    _row("Base replicas", workload_base_replicas_in),
    _row("CPU request", workload_cpu_request_in),
    _row("Memory request", workload_mem_request_in),
    _row("CPU limit", workload_cpu_limit_in),
    _row("Memory limit", workload_mem_limit_in),
)
apply_btn = widgets.Button(
    description="Apply Selections", button_style="primary", icon="check"
)
apply_btn.layout.width = "auto"
apply_btn.on_click(_validate_and_apply)
apply_row = widgets.HBox(
    [apply_btn], layout=widgets.Layout(width="100%", justify_content="center")
)
ui = widgets.VBox(
    [
        status_html,
        mode_note_html,
        target_note_html,
        image_mode_note_html,
        native_image_note_html,
        top_section,
        benchmark_knobs_section,
        workload_section,
        image_filter_section,
        ca_base_box,
        ca_auto_box,
        kpo_base_box,
        kpo_auto_box,
        native_section,
        apply_row,
        issues_html,
        apply_out,
    ],
    layout=widgets.Layout(width="100%"),
    _dom_classes=["nb3-container"],
)
region_dd.observe(_refresh_region_data, names="value")
bench_comp_dd.observe(_refresh_region_data, names="value")
k8_dd.observe(_refresh_images_only, names="value")
target_dd.observe(lambda ch: _sync_visibility(), names="value")
benchmark_mode_dd.observe(lambda ch: _sync_visibility(), names="value")
image_mode_dd.observe(lambda ch: _sync_visibility(), names="value")
kpo_pin_shape_policy_dd.observe(
    lambda ch: _refresh_native_shape_options(), names="value"
)
kpo_native_capacity_type_dd.observe(
    lambda ch: _refresh_native_shape_options(), names="value"
)
kpo_native_limits_mode_dd.observe(lambda ch: _sync_visibility(), names="value")
native_advanced_cb.observe(lambda ch: _sync_visibility(), names="value")
kpo_filter_compartment_dd.observe(_on_filter_compartment_change, names="value")
for pool in all_pools:
    pool["shape_dd"].observe(lambda ch, p=pool: _on_shape_change(p), names="value")
kpo_auto_state["image_dd"].observe(
    lambda ch: _refresh_native_shape_options(), names="value"
)
# Initial filter-compartment normalization (keeps default unset unless overridden).
_maybe_auto_sync_filter_compartment()
_refresh_region_data()
_sync_visibility()
display(ui)
print("Cell 3 loaded. Select values, then click 'Apply Selections'.")
print("Mode behavior: parity=enforce cross-target fairness, native=relaxed fairness.")
print("Target behavior: only selected target inputs are required.")
print("Next: Run Cell 4.")

In [ ]:
# Cell 4 — Generate both Terraform stacks (tf/ca + tf/kpo) in one pass from Cell 3 SSOT

# Updated: mode-aware fairness (parity/native), native KPO requirements wiring, filter-mode image handling.


import os

import re

import json

from pathlib import Path

from datetime import datetime, timezone


# ---------------------------

# Helpers

# ---------------------------


def _env(k, d=""):

    return os.environ.get(k, d)


def _require(k):

    v = _env(k, "").strip()

    if not v:

        raise RuntimeError(f"Missing required environment variable: {k}")

    return v


def _as_bool(v, default=False):

    if v is None:

        return default

    return str(v).strip().lower() in {"1", "true", "yes", "y", "on"}


def _as_int(v, name):

    try:

        return int(v)

    except Exception:

        raise RuntimeError(f"Expected integer for {name}, got: {v}")


def _as_float(v, name):

    try:

        return float(v)

    except Exception:

        raise RuntimeError(f"Expected numeric value for {name}, got: {v}")


def _csv_list(v: str):

    return [x.strip() for x in str(v).split(",") if x.strip()]


def _is_flex(shape):

    return str(shape).endswith(".Flex")


def _safe_name(s, max_len=36):

    x = re.sub(r"[^a-z0-9-]+", "-", s.lower())

    x = re.sub(r"-+", "-", x).strip("-")

    if not x:

        x = "bench"

    return x[:max_len].rstrip("-")


def _dns_label(seed, max_len=15):

    x = re.sub(r"[^a-z0-9]+", "", seed.lower())

    if not x:

        x = "oke"

    if not x[0].isalpha():

        x = "a" + x

    return x[:max_len]


def _fmt_num(v):

    fv = float(v)

    return str(int(fv)) if fv.is_integer() else str(fv)


def _normalize_capacity_values(values):

    out = []

    for raw in values:

        v = str(raw).strip().lower()

        if not v:

            continue

        # OCI docs: Karpenter 'spot' maps to OCI preemptible.

        if v == "preemptible":

            v = "spot"

        if v not in {"on-demand", "spot"}:

            raise RuntimeError(
                f"Unsupported capacity type: {raw}. Allowed: on-demand, spot."
            )

        if v not in out:

            out.append(v)

    return out or ["on-demand"]


def _load_pool(prefix: str, autoscaled: bool, require_image=True):

    shape = _require(f"{prefix}_SHAPE")

    image_id = _env(f"{prefix}_IMAGE_ID", "").strip()

    if require_image and not image_id:

        raise RuntimeError(f"Missing required environment variable: {prefix}_IMAGE_ID")

    out = {
        "shape": shape,
        "image_id": image_id,
    }

    if autoscaled:

        out["min_size"] = _as_int(_require(f"{prefix}_MIN_SIZE"), f"{prefix}_MIN_SIZE")

        out["max_size"] = _as_int(_require(f"{prefix}_MAX_SIZE"), f"{prefix}_MAX_SIZE")

        if out["min_size"] > out["max_size"]:

            raise RuntimeError(f"{prefix}: min_size cannot be greater than max_size")

    else:

        out["size"] = _as_int(_require(f"{prefix}_SIZE"), f"{prefix}_SIZE")

        if out["size"] < 1:

            raise RuntimeError(f"{prefix}: size must be >= 1")

    if _is_flex(shape):

        out["flex_ocpus"] = _as_float(
            _require(f"{prefix}_FLEX_OCPUS"), f"{prefix}_FLEX_OCPUS"
        )

        out["flex_memory_gb"] = _as_float(
            _require(f"{prefix}_FLEX_MEMORY_GB"), f"{prefix}_FLEX_MEMORY_GB"
        )

        if out["flex_ocpus"] <= 0 or out["flex_memory_gb"] <= 0:

            raise RuntimeError(f"{prefix}: flex ocpus/memory must be > 0")

    else:

        out["flex_ocpus"] = None

        out["flex_memory_gb"] = None

    return out


def _mk_worker_pool_entry(
    pool_name,
    pool,
    autoscaled,
    allow_autoscaler=False,
    autoscale=False,
    extra_labels=None,
):

    labels = {
        "benchmark.oracle.com/profile": pool_name,
    }

    if extra_labels:

        labels.update(extra_labels)

    entry = {
        "description": pool_name,
        "mode": "node-pool",
        "shape": pool["shape"],
        "image_type": "custom",
        "image_id": pool["image_id"],
        "node_labels": labels,
    }

    if _is_flex(pool["shape"]):

        entry["ocpus"] = pool["flex_ocpus"]

        entry["memory"] = pool["flex_memory_gb"]

    if autoscaled:

        entry["size"] = pool["min_size"]

        entry["min_size"] = pool["min_size"]

        entry["max_size"] = pool["max_size"]

        entry["autoscale"] = autoscale

        entry["allow_autoscaler"] = allow_autoscaler

        entry["ignore_initial_pool_size"] = True

    else:

        entry["size"] = pool["size"]

        entry["allow_autoscaler"] = allow_autoscaler

    return entry


def _write_text(path: Path, content: str):

    path.parent.mkdir(parents=True, exist_ok=True)

    path.write_text(content, encoding="utf-8")


def _write_json(path: Path, data):

    path.parent.mkdir(parents=True, exist_ok=True)

    path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def _apply_ca_native_worker_pools(default_worker_pools, json_text):

    text = str(json_text or "").strip()

    if not text:

        return default_worker_pools

    try:

        parsed = json.loads(text)

    except Exception as e:

        raise RuntimeError(f"CA_NATIVE_WORKER_POOLS_JSON is invalid JSON: {e}")

    custom = {}

    if isinstance(parsed, dict):

        for k, v in parsed.items():

            if not isinstance(v, dict):

                raise RuntimeError(
                    f"CA_NATIVE_WORKER_POOLS_JSON[{k}] must be an object, got {type(v).__name__}"
                )

        custom = parsed

    elif isinstance(parsed, list):

        for i, v in enumerate(parsed, start=1):

            if not isinstance(v, dict):

                raise RuntimeError(
                    f"CA_NATIVE_WORKER_POOLS_JSON list item {i} must be an object, got {type(v).__name__}"
                )

            custom[f"ca-native-{i}"] = v

    else:

        raise RuntimeError(
            "CA_NATIVE_WORKER_POOLS_JSON must be a JSON object or array of objects."
        )

    merged = dict(default_worker_pools)

    merged.update(custom)

    return merged


# ---------------------------

# Read SSOT values from Cell 2/3

# ---------------------------

TENANCY_ID = _require("TENANCY_OCID")

REGION = _require("BENCHMARK_REGION")

HOME_REGION = _require("HOME_REGION")

COMPARTMENT_ID = _require("BENCHMARK_COMPARTMENT_OCID")

PARENT_COMPARTMENT_OCID = _require("PARENT_COMPARTMENT_OCID")

IAM_POLICY_COMPARTMENT_OCID = _require("IAM_POLICY_COMPARTMENT_OCID")

OCI_PROFILE = _require("OCI_PROFILE")

OCI_CONFIG_FILE = _require("OCI_CONFIG_FILE")

K8S_VERSION = _require("KUBERNETES_VERSION")

RUN_ID = _require("RUN_ID")


BENCHMARK_MODE = _env("BENCHMARK_MODE", "parity").strip().lower()

TARGET_CLUSTER = _env("TARGET_CLUSTER", "both").strip().lower()

TARGET_SCENARIOS = _env("TARGET_SCENARIOS", "all").strip().lower()

REPEATS = _as_int(_env("REPEATS", "3"), "REPEATS")

IMAGE_MODE = _env("IMAGE_MODE", "pin").strip().lower()

ENABLE_METRICS_SERVER = _as_bool(_env("ENABLE_METRICS_SERVER", "false"), False)


CA_NAMESPACE = _env("CA_NAMESPACE", "kube-system")

CA_SERVICE_ACCOUNT = _env("CA_SERVICE_ACCOUNT", "cluster-autoscaler")

KPO_NAMESPACE = _env("KPO_NAMESPACE", "karpenter")

KPO_SERVICE_ACCOUNT = _env("KPO_SERVICE_ACCOUNT", "karpenter")


KPO_NPN_IP_COUNT = _as_int(_env("KPO_NPN_IP_COUNT", "16"), "KPO_NPN_IP_COUNT")

KPO_CPU_PER_OCPU_FACTOR = _as_float(
    _env("KPO_CPU_PER_OCPU_FACTOR", "2.0"), "KPO_CPU_PER_OCPU_FACTOR"
)


KPO_NATIVE_ALLOWED_SHAPES_ENV = _csv_list(_env("KPO_NATIVE_ALLOWED_SHAPES", ""))

KPO_NATIVE_CAPACITY_TYPE_VALUES_ENV = _normalize_capacity_values(
    _csv_list(_env("KPO_NATIVE_CAPACITY_TYPE_VALUES", "on-demand"))
)

KPO_NATIVE_CPU_LIMIT_RAW = _env("KPO_NATIVE_NODEPOOL_CPU_LIMIT", "").strip()

KPO_NATIVE_MEM_LIMIT_GI_RAW = _env("KPO_NATIVE_NODEPOOL_MEMORY_LIMIT_GI", "").strip()

CA_NATIVE_WORKER_POOLS_JSON = _env("CA_NATIVE_WORKER_POOLS_JSON", "").strip()

BENCHMARK_PRICEBOOK_PATH = _env("BENCHMARK_PRICEBOOK_PATH", "").strip()


CONTROL_PLANE_ENDPOINT_MODE = (
    _env("CONTROL_PLANE_ENDPOINT_MODE", "public").strip().lower()
)

CONTROL_PLANE_ALLOWED_CIDRS_RAW = _env("CONTROL_PLANE_ALLOWED_CIDRS", "").strip()


if BENCHMARK_MODE not in {"parity", "native"}:

    raise RuntimeError(f"BENCHMARK_MODE must be parity|native. Got: {BENCHMARK_MODE}")

if TARGET_CLUSTER not in {"ca", "kpo", "both"}:

    raise RuntimeError(f"TARGET_CLUSTER must be ca|kpo|both. Got: {TARGET_CLUSTER}")

if TARGET_SCENARIOS not in {"all", "burst", "steady", "scale_down"}:

    raise RuntimeError(
        f"TARGET_SCENARIOS must be all|burst|steady|scale_down. Got: {TARGET_SCENARIOS}"
    )

if IMAGE_MODE not in {"pin", "filter"}:

    raise RuntimeError(f"IMAGE_MODE must be pin|filter. Got: {IMAGE_MODE}")


# Keep Cell 4 self-contained: validate/sanitize filter inputs even if Cell 3 was not rerun.
kpo_image_os_version_filter = _env("KPO_IMAGE_OS_VERSION_FILTER", "8").strip() or "8"
if IMAGE_MODE == "filter":
    if not re.fullmatch(r"[0-9]+(\.[0-9]+)?", kpo_image_os_version_filter):
        raise RuntimeError(
            "KPO_IMAGE_OS_VERSION_FILTER must be numeric (for example 8 or 8.10) when IMAGE_MODE=filter. "
            f"Got: {kpo_image_os_version_filter}"
        )


if not re.search(r"v?1\.(3[1-9]|[4-9][0-9])", K8S_VERSION):

    raise RuntimeError(
        f"Kubernetes version must be >= 1.31 for this benchmark plan. Got: {K8S_VERSION}"
    )

if KPO_NPN_IP_COUNT < 1 or KPO_NPN_IP_COUNT > 256:

    raise RuntimeError(
        f"KPO_NPN_IP_COUNT must be within [1,256]. Got: {KPO_NPN_IP_COUNT}"
    )

if (KPO_NPN_IP_COUNT & (KPO_NPN_IP_COUNT - 1)) != 0:

    raise RuntimeError(
        f"KPO_NPN_IP_COUNT must be a power of two. Got: {KPO_NPN_IP_COUNT}"
    )

if KPO_CPU_PER_OCPU_FACTOR <= 0:

    raise RuntimeError(
        f"KPO_CPU_PER_OCPU_FACTOR must be > 0. Got: {KPO_CPU_PER_OCPU_FACTOR}"
    )

if CONTROL_PLANE_ENDPOINT_MODE not in {"public", "private"}:

    raise RuntimeError(
        "CONTROL_PLANE_ENDPOINT_MODE must be public|private. "
        f"Got: {CONTROL_PLANE_ENDPOINT_MODE}"
    )


CONTROL_PLANE_IS_PUBLIC = CONTROL_PLANE_ENDPOINT_MODE == "public"

ASSIGN_PUBLIC_IP_TO_CONTROL_PLANE = CONTROL_PLANE_ENDPOINT_MODE == "public"


if CONTROL_PLANE_ENDPOINT_MODE == "public":

    if CONTROL_PLANE_ALLOWED_CIDRS_RAW:

        CONTROL_PLANE_ALLOWED_CIDRS = _csv_list(CONTROL_PLANE_ALLOWED_CIDRS_RAW)

    else:

        CONTROL_PLANE_ALLOWED_CIDRS = ["0.0.0.0/0"]

    if not CONTROL_PLANE_ALLOWED_CIDRS:

        raise RuntimeError(
            "Public control plane mode requires non-empty CONTROL_PLANE_ALLOWED_CIDRS."
        )

else:

    CONTROL_PLANE_ALLOWED_CIDRS = _csv_list(CONTROL_PLANE_ALLOWED_CIDRS_RAW)


# Make later Terraform subprocesses honor selected OCI config/profile.

os.environ["OCI_CLI_CONFIG_FILE"] = OCI_CONFIG_FILE

os.environ["OCI_CONFIG_FILE_PROFILE"] = OCI_PROFILE

os.environ["TF_VAR_config_file_profile"] = OCI_PROFILE


ca_baseline = _load_pool("CA_BASELINE_POOL", autoscaled=False, require_image=True)

ca_autoscaled = _load_pool("CA_AUTOSCALED_POOL", autoscaled=True, require_image=True)

kpo_baseline = _load_pool("KPO_BASELINE_POOL", autoscaled=False, require_image=True)

kpo_autoscaled = _load_pool(
    "KPO_AUTOSCALED_POOL",
    autoscaled=True,
    require_image=(IMAGE_MODE == "pin"),
)


# Fairness checks are strict only in parity mode.

if BENCHMARK_MODE == "parity" and TARGET_CLUSTER == "both":

    if ca_autoscaled["shape"] != kpo_autoscaled["shape"]:

        raise RuntimeError(
            "Fairness gate (parity): CA and KPO autoscaled shapes must match in TARGET_CLUSTER=both"
        )

    if (
        ca_autoscaled["min_size"] != kpo_autoscaled["min_size"]
        or ca_autoscaled["max_size"] != kpo_autoscaled["max_size"]
    ):

        raise RuntimeError(
            "Fairness gate (parity): CA and KPO autoscaled min/max must match in TARGET_CLUSTER=both"
        )

    if _is_flex(ca_autoscaled["shape"]) and _is_flex(kpo_autoscaled["shape"]):

        if (
            abs(
                (ca_autoscaled["flex_ocpus"] or 0) - (kpo_autoscaled["flex_ocpus"] or 0)
            )
            > 1e-9
        ):

            raise RuntimeError(
                "Fairness gate (parity): CA and KPO autoscaled flex OCPU must match in TARGET_CLUSTER=both"
            )

        if (
            abs(
                (ca_autoscaled["flex_memory_gb"] or 0)
                - (kpo_autoscaled["flex_memory_gb"] or 0)
            )
            > 1e-9
        ):

            raise RuntimeError(
                "Fairness gate (parity): CA and KPO autoscaled flex memory must match in TARGET_CLUSTER=both"
            )


# KPO NodePool requirements are mode-aware:

# - parity: single-shape constrained path

# - native: multi-shape path from KPO_NATIVE_ALLOWED_SHAPES

if BENCHMARK_MODE == "native":

    kpo_allowed_shapes = list(KPO_NATIVE_ALLOWED_SHAPES_ENV) or [
        kpo_autoscaled["shape"]
    ]

    if kpo_autoscaled["shape"] not in kpo_allowed_shapes:

        kpo_allowed_shapes.append(kpo_autoscaled["shape"])

    kpo_capacity_types = KPO_NATIVE_CAPACITY_TYPE_VALUES_ENV

else:

    kpo_allowed_shapes = [kpo_autoscaled["shape"]]

    kpo_capacity_types = ["on-demand"]


if not kpo_allowed_shapes:

    raise RuntimeError("KPO allowed shapes resolved to an empty list.")


# Native manual limits override auto-derived limits.

manual_cpu_limit = None

manual_mem_limit_gi = None

if BENCHMARK_MODE == "native":

    if KPO_NATIVE_CPU_LIMIT_RAW:

        manual_cpu_limit = _as_float(
            KPO_NATIVE_CPU_LIMIT_RAW, "KPO_NATIVE_NODEPOOL_CPU_LIMIT"
        )

        if manual_cpu_limit <= 0:

            raise RuntimeError("KPO_NATIVE_NODEPOOL_CPU_LIMIT must be > 0")

    if KPO_NATIVE_MEM_LIMIT_GI_RAW:

        manual_mem_limit_gi = _as_float(
            KPO_NATIVE_MEM_LIMIT_GI_RAW, "KPO_NATIVE_NODEPOOL_MEMORY_LIMIT_GI"
        )

        if manual_mem_limit_gi <= 0:

            raise RuntimeError("KPO_NATIVE_NODEPOOL_MEMORY_LIMIT_GI must be > 0")


if (manual_cpu_limit is None) ^ (manual_mem_limit_gi is None):

    raise RuntimeError(
        "Manual KPO native limits require both CPU and memory Gi values together."
    )


# ---------------------------

# Paths (from Cell 2 SSOT; no repo-root assumptions)

# ---------------------------

PROJECT_ROOT = Path(_require("PROJECT_ROOT")).expanduser().resolve()

TF_STACK_ROOT = Path(_require("TF_STACK_ROOT")).expanduser().resolve()

module_src = Path(_require("TERRAFORM_OCI_OKE_MODULE_SOURCE")).expanduser().resolve()


if not module_src.exists() or not module_src.is_dir():

    raise RuntimeError(f"Invalid TERRAFORM_OCI_OKE_MODULE_SOURCE: {module_src}")

for marker in ["versions.tf", "module-cluster.tf", "module-workers.tf", "modules"]:

    if not (module_src / marker).exists():

        raise RuntimeError(
            "TERRAFORM_OCI_OKE_MODULE_SOURCE is not a valid terraform-oci-oke-main root. "
            f"Missing: {marker} in {module_src}"
        )


tf_root = TF_STACK_ROOT

ca_dir = tf_root / "ca"

kpo_dir = tf_root / "kpo"


for d in [tf_root, ca_dir, kpo_dir]:

    d.mkdir(parents=True, exist_ok=True)


ca_module_src_rel = os.path.relpath(module_src, ca_dir)

kpo_module_src_rel = os.path.relpath(module_src, kpo_dir)


# ---------------------------

# Common Terraform file templates

# ---------------------------

versions_tf = """terraform {

  required_version = ">= 1.3.0"



  required_providers {

    oci = {

      source  = "oracle/oci"

      version = ">= 7.30.0"

    }

  }

}

"""


providers_tf = """variable "benchmark" {

  type = any

}



provider "oci" {

  config_file_profile = var.benchmark.config_file_profile

  tenancy_ocid        = var.benchmark.tenancy_id

  region              = var.benchmark.region

}



provider "oci" {

  alias               = "home"

  config_file_profile = var.benchmark.config_file_profile

  tenancy_ocid        = var.benchmark.tenancy_id

  region              = var.benchmark.home_region

}

"""


def _main_tf(module_source_rel: str) -> str:

    return f"""module "oke" {{

  source = "{module_source_rel}"



  providers = {{

    oci      = oci

    oci.home = oci.home

  }}



  tenancy_id             = var.benchmark.tenancy_id

  compartment_id         = var.benchmark.compartment_id

  worker_compartment_id  = var.benchmark.worker_compartment_id

  network_compartment_id = var.benchmark.network_compartment_id



  region      = var.benchmark.region

  home_region = var.benchmark.home_region



  create_iam_resources         = var.benchmark.create_iam_resources

  create_iam_autoscaler_policy = var.benchmark.create_iam_autoscaler_policy

  create_iam_worker_policy     = var.benchmark.create_iam_worker_policy

  create_iam_operator_policy   = var.benchmark.create_iam_operator_policy

  create_iam_kms_policy        = var.benchmark.create_iam_kms_policy



  create_vcn    = var.benchmark.create_vcn

  vcn_name      = var.benchmark.vcn_name

  vcn_dns_label = var.benchmark.vcn_dns_label



  create_cluster     = var.benchmark.create_cluster

  cluster_type       = var.benchmark.cluster_type

  cluster_name       = var.benchmark.cluster_name

  kubernetes_version = var.benchmark.kubernetes_version



  cni_type                          = var.benchmark.cni_type

  control_plane_is_public           = var.benchmark.control_plane_is_public

  assign_public_ip_to_control_plane = var.benchmark.assign_public_ip_to_control_plane

  control_plane_allowed_cidrs       = var.benchmark.control_plane_allowed_cidrs



  create_bastion  = var.benchmark.create_bastion

  create_operator = var.benchmark.create_operator



  output_detail = var.benchmark.output_detail



  worker_pool_mode = var.benchmark.worker_pool_mode

  worker_pool_size = var.benchmark.worker_pool_size

  worker_pools     = var.benchmark.worker_pools



  cluster_addons           = var.benchmark.cluster_addons

  cluster_addons_to_remove = var.benchmark.cluster_addons_to_remove

}}

"""


outputs_tf = """output "cluster_id" {

  value = module.oke.cluster_id

}



output "cluster_endpoints" {

  value = module.oke.cluster_endpoints

}



output "apiserver_private_host" {

  value = module.oke.apiserver_private_host

}



output "worker_pool_ids" {

  value = module.oke.worker_pool_ids

}



output "worker_pools" {

  value = module.oke.worker_pools

}



output "vcn_id" {

  value = module.oke.vcn_id

}



output "control_plane_subnet_id" {

  value = module.oke.control_plane_subnet_id

}



output "worker_subnet_id" {

  value = module.oke.worker_subnet_id

}



output "pod_subnet_id" {

  value = module.oke.pod_subnet_id

}



output "int_lb_subnet_id" {

  value = module.oke.int_lb_subnet_id

}



output "pub_lb_subnet_id" {

  value = module.oke.pub_lb_subnet_id

}



output "policy_statements" {

  value = module.oke.policy_statements

}



output "dynamic_group_ids" {

  value = module.oke.dynamic_group_ids

}

"""


# ---------------------------

# Build stack data: CA

# ---------------------------

run_seed = _safe_name(RUN_ID, max_len=24)

ca_cluster_name = _safe_name(f"ca-{run_seed}", max_len=32)

kpo_cluster_name = _safe_name(f"kpo-{run_seed}", max_len=32)


ca_worker_pools_default = {
    "ca-baseline": _mk_worker_pool_entry(
        "ca-baseline",
        ca_baseline,
        autoscaled=False,
        allow_autoscaler=False,
        autoscale=False,
        extra_labels={
            "benchmark.oracle.com/role": "baseline",
            "benchmark.oracle.com/autoscaler": "ca",
        },
    ),
    "ca-benchmark-autoscaled": _mk_worker_pool_entry(
        "ca-benchmark-autoscaled",
        ca_autoscaled,
        autoscaled=True,
        allow_autoscaler=True,
        autoscale=True,
        extra_labels={
            "benchmark.oracle.com/role": "benchmark",
            "benchmark.oracle.com/autoscaler": "ca",
        },
    ),
}


if BENCHMARK_MODE == "native" and CA_NATIVE_WORKER_POOLS_JSON:

    ca_worker_pools = _apply_ca_native_worker_pools(
        ca_worker_pools_default, CA_NATIVE_WORKER_POOLS_JSON
    )

else:

    ca_worker_pools = ca_worker_pools_default


ca_discovery = (
    f"compartmentId:{COMPARTMENT_ID},"
    f"nodepoolTags:cluster_autoscaler=managed,"
    f"min:{ca_autoscaled['min_size']},max:{ca_autoscaled['max_size']}"
)


baseline_selector_json = json.dumps({"benchmark.oracle.com/role": "baseline"})


ca_cluster_addons = {
    "ClusterAutoscaler": {
        "remove_addon_resources_on_delete": True,
        "override_existing": True,
        "configurations": [
            {"key": "authType", "value": "workload"},
            {"key": "numOfReplicas", "value": "2"},
            {"key": "nodeGroupAutoDiscovery", "value": ca_discovery},
            {"key": "scaleDownEnabled", "value": "true"},
            {"key": "maxNodeProvisionTime", "value": "25m"},
            {"key": "scanInterval", "value": "10s"},
            {"key": "unremovableNodeRecheckTimeout", "value": "5m"},
            {"key": "nodeSelectors", "value": baseline_selector_json},
        ],
    },
    "CoreDNS": {
        "remove_addon_resources_on_delete": True,
        "override_existing": True,
        "configurations": [
            {"key": "nodeSelectors", "value": baseline_selector_json},
        ],
    },
}


skip_metrics_dep = _as_bool(
    _env("METRICS_SERVER_SKIP_ADDON_DEPENDENCIES_CHECK", "false"), False
)


if ENABLE_METRICS_SERVER:

    if not skip_metrics_dep:

        ca_cluster_addons["CertManager"] = {
            "remove_addon_resources_on_delete": True,
            "override_existing": True,
            "configurations": [
                {"key": "numOfReplicas", "value": "2"},
            ],
        }

    ms_cfg = []

    if skip_metrics_dep:

        ms_cfg.append({"key": "skipAddonDependenciesCheck", "value": "true"})

    ca_cluster_addons["KubernetesMetricsServer"] = {
        "remove_addon_resources_on_delete": True,
        "override_existing": True,
        "configurations": ms_cfg,
    }


ca_benchmark = {
    "oci_config_file": OCI_CONFIG_FILE,
    "config_file_profile": OCI_PROFILE,
    "tenancy_id": TENANCY_ID,
    "compartment_id": COMPARTMENT_ID,
    "worker_compartment_id": COMPARTMENT_ID,
    "network_compartment_id": COMPARTMENT_ID,
    "region": REGION,
    "home_region": HOME_REGION,
    "create_iam_resources": True,
    "create_iam_autoscaler_policy": "never",
    "create_iam_worker_policy": "never",
    "create_iam_operator_policy": "never",
    "create_iam_kms_policy": "never",
    "create_vcn": True,
    "vcn_name": f"{ca_cluster_name}-vcn",
    "vcn_dns_label": _dns_label(ca_cluster_name),
    "create_cluster": True,
    "cluster_type": "enhanced",
    "cluster_name": ca_cluster_name,
    "kubernetes_version": K8S_VERSION,
    "cni_type": "npn",
    "control_plane_is_public": CONTROL_PLANE_IS_PUBLIC,
    "assign_public_ip_to_control_plane": ASSIGN_PUBLIC_IP_TO_CONTROL_PLANE,
    "control_plane_allowed_cidrs": CONTROL_PLANE_ALLOWED_CIDRS,
    "create_bastion": False,
    "create_operator": False,
    "output_detail": True,
    "worker_pool_mode": "node-pool",
    "worker_pool_size": 0,
    "worker_pools": ca_worker_pools,
    "cluster_addons": ca_cluster_addons,
    "cluster_addons_to_remove": {},
}


# ---------------------------

# Build stack data: KPO

# ---------------------------

kpo_worker_pools = {
    "kpo-baseline": _mk_worker_pool_entry(
        "kpo-baseline",
        kpo_baseline,
        autoscaled=False,
        allow_autoscaler=False,
        autoscale=False,
        extra_labels={
            "benchmark.oracle.com/role": "baseline",
            "benchmark.oracle.com/autoscaler": "kpo",
        },
    ),
}


kpo_cluster_addons = {
    "CoreDNS": {
        "remove_addon_resources_on_delete": True,
        "override_existing": True,
        "configurations": [
            {"key": "nodeSelectors", "value": baseline_selector_json},
        ],
    }
}

if ENABLE_METRICS_SERVER:

    if not skip_metrics_dep:

        kpo_cluster_addons["CertManager"] = {
            "remove_addon_resources_on_delete": True,
            "override_existing": True,
            "configurations": [
                {"key": "numOfReplicas", "value": "2"},
            ],
        }

    ms_cfg = []

    if skip_metrics_dep:

        ms_cfg.append({"key": "skipAddonDependenciesCheck", "value": "true"})

    kpo_cluster_addons["KubernetesMetricsServer"] = {
        "remove_addon_resources_on_delete": True,
        "override_existing": True,
        "configurations": ms_cfg,
    }


kpo_benchmark = {
    "oci_config_file": OCI_CONFIG_FILE,
    "config_file_profile": OCI_PROFILE,
    "tenancy_id": TENANCY_ID,
    "compartment_id": COMPARTMENT_ID,
    "worker_compartment_id": COMPARTMENT_ID,
    "network_compartment_id": COMPARTMENT_ID,
    "region": REGION,
    "home_region": HOME_REGION,
    "create_iam_resources": True,
    "create_iam_autoscaler_policy": "never",
    "create_iam_worker_policy": "never",
    "create_iam_operator_policy": "never",
    "create_iam_kms_policy": "never",
    "create_vcn": True,
    "vcn_name": f"{kpo_cluster_name}-vcn",
    "vcn_dns_label": _dns_label(kpo_cluster_name),
    "create_cluster": True,
    "cluster_type": "enhanced",
    "cluster_name": kpo_cluster_name,
    "kubernetes_version": K8S_VERSION,
    "cni_type": "npn",
    "control_plane_is_public": CONTROL_PLANE_IS_PUBLIC,
    "assign_public_ip_to_control_plane": ASSIGN_PUBLIC_IP_TO_CONTROL_PLANE,
    "control_plane_allowed_cidrs": CONTROL_PLANE_ALLOWED_CIDRS,
    "create_bastion": False,
    "create_operator": False,
    "output_detail": True,
    "worker_pool_mode": "node-pool",
    "worker_pool_size": 0,
    "worker_pools": kpo_worker_pools,
    "cluster_addons": kpo_cluster_addons,
    "cluster_addons_to_remove": {},
}


# ---------------------------

# Write Terraform stacks

# ---------------------------

for stack_dir, module_rel, benchmark in [
    (ca_dir, ca_module_src_rel, ca_benchmark),
    (kpo_dir, kpo_module_src_rel, kpo_benchmark),
]:

    _write_text(stack_dir / "versions.tf", versions_tf)

    _write_text(stack_dir / "providers.tf", providers_tf)

    _write_text(stack_dir / "main.tf", _main_tf(module_rel))

    _write_text(stack_dir / "outputs.tf", outputs_tf)

    _write_json(stack_dir / "terraform.auto.tfvars.json", {"benchmark": benchmark})


# ---------------------------

# KPO template artifacts for Cell 8

# ---------------------------

kpo_values_lines = [
    "settings:",
    f'  clusterCompartmentId: "{COMPARTMENT_ID}"',
    f'  vcnCompartmentId: "{COMPARTMENT_ID}"',
    '  apiserverEndpoint: "__APISERVER_ENDPOINT__"',
    "  ociVcnIpNative: true",
    "  ipFamilies:",
    "    - IPv4",
]

if _is_flex(kpo_autoscaled["shape"]):

    kpo_values_lines.extend(
        [
            "  flexibleShapeConfigs:",
            f"    - ocpus: {_fmt_num(kpo_autoscaled['flex_ocpus'])}",
            f"      memoryInGbs: {_fmt_num(kpo_autoscaled['flex_memory_gb'])}",
        ]
    )

kpo_values_yaml = "\n".join(kpo_values_lines) + "\n"


if IMAGE_MODE == "filter":

    image_filter_compartment = _env("KPO_IMAGE_FILTER_COMPARTMENT_ID", "").strip()

    filter_block = [
        "        imageFilter:",
        '          osFilter: "Oracle Linux"',
        f'          osVersionFilter: "{kpo_image_os_version_filter}"',
    ]

    if image_filter_compartment:

        filter_block.append(f'          compartmentId: "{image_filter_compartment}"')

    image_config_block = "\n".join(filter_block)

else:

    if not kpo_autoscaled["image_id"]:

        raise RuntimeError("IMAGE_MODE=pin requires KPO_AUTOSCALED_POOL_IMAGE_ID")

    image_config_block = f"        imageId: \"{kpo_autoscaled['image_id']}\""


shape_cfg = ""

if _is_flex(kpo_autoscaled["shape"]):

    shape_cfg = f"""

  shapeConfigs:

    - ocpus: {_fmt_num(kpo_autoscaled['flex_ocpus'])}

      memoryInGbs: {_fmt_num(kpo_autoscaled['flex_memory_gb'])}

""".rstrip(
        "\n"
    )


kpo_ocinodeclass_yaml = f"""apiVersion: oci.oraclecloud.com/v1beta1

kind: OCINodeClass

metadata:

  name: benchmark-ocinodeclass

spec:{shape_cfg}

  volumeConfig:

    bootVolumeConfig:

      imageConfig:

        imageType: OKEImage

{image_config_block}

  networkConfig:

    primaryVnicConfig:

      subnetConfig:

        subnetId: "__WORKER_SUBNET_ID__"

    secondaryVnicConfigs:

      - subnetConfig:

          subnetId: "__POD_SUBNET_ID__"

        ipCount: {KPO_NPN_IP_COUNT}

""".replace(
    "spec:\n  volumeConfig",
    (
        "spec:\n  volumeConfig"
        if not shape_cfg
        else "spec:" + shape_cfg + "\n  volumeConfig"
    ),
)


# NodePool limits:

# - native manual limits win

# - else for flex autoscaled pools, derive from max_size * shape config

# - non-flex with no manual limits leaves limits unset

nodepool_limits = ""

if manual_cpu_limit is not None and manual_mem_limit_gi is not None:

    cpu_limit_txt = _fmt_num(manual_cpu_limit)

    mem_limit_txt = _fmt_num(manual_mem_limit_gi)

    nodepool_limits = f"""

  limits:

    cpu: "{cpu_limit_txt}"

    memory: "{mem_limit_txt}Gi"

""".rstrip(
        "\n"
    )

elif _is_flex(kpo_autoscaled["shape"]):

    cpu_limit = (
        kpo_autoscaled["max_size"]
        * kpo_autoscaled["flex_ocpus"]
        * KPO_CPU_PER_OCPU_FACTOR
    )

    mem_limit = kpo_autoscaled["max_size"] * kpo_autoscaled["flex_memory_gb"]

    nodepool_limits = f"""

  limits:

    cpu: "{_fmt_num(cpu_limit)}"

    memory: "{_fmt_num(mem_limit)}Gi"

""".rstrip(
        "\n"
    )


capacity_values_yaml = json.dumps(kpo_capacity_types)

shape_values_yaml = json.dumps(kpo_allowed_shapes)


kpo_nodepool_yaml = f"""apiVersion: karpenter.sh/v1

kind: NodePool

metadata:

  name: benchmark-nodepool

  annotations:

    benchmark.oracle.com/min-size: "{kpo_autoscaled['min_size']}"

    benchmark.oracle.com/max-size: "{kpo_autoscaled['max_size']}"

spec:

  template:

    metadata:

      labels:

        benchmark.oracle.com/role: "benchmark"

        benchmark.oracle.com/autoscaler: "kpo"

    spec:

      nodeClassRef:

        group: oci.oraclecloud.com

        kind: OCINodeClass

        name: benchmark-ocinodeclass

      requirements:

        - key: karpenter.sh/capacity-type

          operator: In

          values: {capacity_values_yaml}

        - key: oci.oraclecloud.com/instance-shape

          operator: In

          values: {shape_values_yaml}

  disruption:

    consolidationPolicy: WhenEmpty

    consolidateAfter: 5m{nodepool_limits}

"""


_write_text(kpo_dir / "manifests" / "kpo-values.yaml.tmpl", kpo_values_yaml)

_write_text(kpo_dir / "manifests" / "ocinodeclass.yaml.tmpl", kpo_ocinodeclass_yaml)

_write_text(kpo_dir / "manifests" / "nodepool.yaml.tmpl", kpo_nodepool_yaml)


# ---------------------------

# IAM policy templates

# ---------------------------

ca_policy = f"""# CA workload identity policy template (fill __CA_CLUSTER_OCID__)

Allow any-user to manage cluster-node-pools in compartment id {COMPARTMENT_ID} where all {{

  request.principal.type = 'workload',

  request.principal.namespace = '{CA_NAMESPACE}',

  request.principal.service_account = '{CA_SERVICE_ACCOUNT}',

  request.principal.cluster_id = '__CA_CLUSTER_OCID__'

}}

Allow any-user to manage instance-family in compartment id {COMPARTMENT_ID} where all {{

  request.principal.type = 'workload',

  request.principal.namespace = '{CA_NAMESPACE}',

  request.principal.service_account = '{CA_SERVICE_ACCOUNT}',

  request.principal.cluster_id = '__CA_CLUSTER_OCID__'

}}

Allow any-user to use subnets in compartment id {COMPARTMENT_ID} where all {{

  request.principal.type = 'workload',

  request.principal.namespace = '{CA_NAMESPACE}',

  request.principal.service_account = '{CA_SERVICE_ACCOUNT}',

  request.principal.cluster_id = '__CA_CLUSTER_OCID__'

}}

Allow any-user to read virtual-network-family in compartment id {COMPARTMENT_ID} where all {{

  request.principal.type = 'workload',

  request.principal.namespace = '{CA_NAMESPACE}',

  request.principal.service_account = '{CA_SERVICE_ACCOUNT}',

  request.principal.cluster_id = '__CA_CLUSTER_OCID__'

}}

Allow any-user to use vnics in compartment id {COMPARTMENT_ID} where all {{

  request.principal.type = 'workload',

  request.principal.namespace = '{CA_NAMESPACE}',

  request.principal.service_account = '{CA_SERVICE_ACCOUNT}',

  request.principal.cluster_id = '__CA_CLUSTER_OCID__'

}}

Allow any-user to inspect compartments in tenancy where all {{

  request.principal.type = 'workload',

  request.principal.namespace = '{CA_NAMESPACE}',

  request.principal.service_account = '{CA_SERVICE_ACCOUNT}',

  request.principal.cluster_id = '__CA_CLUSTER_OCID__'

}}

"""


kpo_policy = f"""# KPO workload identity policy template (fill __KPO_CLUSTER_OCID__)

Allow any-user to manage instance-family in compartment id {COMPARTMENT_ID} where all {{

  request.principal.type = 'workload',

  request.principal.namespace = '{KPO_NAMESPACE}',

  request.principal.service_account = '{KPO_SERVICE_ACCOUNT}',

  request.principal.cluster_id = '__KPO_CLUSTER_OCID__'

}}

Allow any-user to manage volumes in compartment id {COMPARTMENT_ID} where all {{

  request.principal.type = 'workload',

  request.principal.namespace = '{KPO_NAMESPACE}',

  request.principal.service_account = '{KPO_SERVICE_ACCOUNT}',

  request.principal.cluster_id = '__KPO_CLUSTER_OCID__'

}}

Allow any-user to manage volume-attachments in compartment id {COMPARTMENT_ID} where all {{

  request.principal.type = 'workload',

  request.principal.namespace = '{KPO_NAMESPACE}',

  request.principal.service_account = '{KPO_SERVICE_ACCOUNT}',

  request.principal.cluster_id = '__KPO_CLUSTER_OCID__'

}}

Allow any-user to manage virtual-network-family in compartment id {COMPARTMENT_ID} where all {{

  request.principal.type = 'workload',

  request.principal.namespace = '{KPO_NAMESPACE}',

  request.principal.service_account = '{KPO_SERVICE_ACCOUNT}',

  request.principal.cluster_id = '__KPO_CLUSTER_OCID__'

}}

Allow any-user to inspect compartments in tenancy where all {{

  request.principal.type = 'workload',

  request.principal.namespace = '{KPO_NAMESPACE}',

  request.principal.service_account = '{KPO_SERVICE_ACCOUNT}',

  request.principal.cluster_id = '__KPO_CLUSTER_OCID__'

}}

"""


if IMAGE_MODE == "filter" and _env("KPO_IMAGE_FILTER_COMPARTMENT_ID", "").strip():

    img_comp = _env("KPO_IMAGE_FILTER_COMPARTMENT_ID").strip()

    kpo_policy += f"""

Allow any-user to read instance-images in compartment id {img_comp} where all {{

  request.principal.type = 'workload',

  request.principal.namespace = '{KPO_NAMESPACE}',

  request.principal.service_account = '{KPO_SERVICE_ACCOUNT}',

  request.principal.cluster_id = '__KPO_CLUSTER_OCID__'

}}

"""


kpo_nodes_join = f"""# KPO node registration template

# Dynamic group rule:

# ALL {{instance.compartment.id = '{COMPARTMENT_ID}'}}



# Policy:

Allow dynamic-group __KPO_NODES_DYNAMIC_GROUP_NAME__ to {{CLUSTER_JOIN}} in compartment id {COMPARTMENT_ID}

where {{ target.cluster.id = '__KPO_CLUSTER_OCID__' }}

"""


telemetry_policy = """# Telemetry policy template

# Attach to the principal/group that will query oci_oke metrics:

Allow group __TELEMETRY_GROUP_NAME__ to read metrics in tenancy

"""


_write_text(ca_dir / "iam" / "ca_workload_identity.policy.tmpl", ca_policy)

_write_text(kpo_dir / "iam" / "kpo_workload_identity.policy.tmpl", kpo_policy)

_write_text(kpo_dir / "iam" / "kpo_nodes_cluster_join.policy.tmpl", kpo_nodes_join)

_write_text(tf_root / "iam" / "telemetry.policy.tmpl", telemetry_policy)


# ---------------------------

# Stack metadata

# ---------------------------

generation_summary = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
    "project_root": str(PROJECT_ROOT),
    "tf_root": str(tf_root),
    "module_source": str(module_src),
    "benchmark_mode": BENCHMARK_MODE,
    "target_cluster": TARGET_CLUSTER,
    "target_scenarios": TARGET_SCENARIOS,
    "repeats": REPEATS,
    "image_mode": IMAGE_MODE,
    "kpo_image_os_version_filter": (
        kpo_image_os_version_filter if IMAGE_MODE == "filter" else ""
    ),
    "enable_metrics_server": ENABLE_METRICS_SERVER,
    "control_plane_endpoint_mode": CONTROL_PLANE_ENDPOINT_MODE,
    "control_plane_is_public": CONTROL_PLANE_IS_PUBLIC,
    "assign_public_ip_to_control_plane": ASSIGN_PUBLIC_IP_TO_CONTROL_PLANE,
    "control_plane_allowed_cidrs": CONTROL_PLANE_ALLOWED_CIDRS,
    "region": REGION,
    "home_region": HOME_REGION,
    "kubernetes_version": K8S_VERSION,
    "kpo_cpu_per_ocpu_factor": KPO_CPU_PER_OCPU_FACTOR,
    "kpo_nodepool_capacity_types": kpo_capacity_types,
    "kpo_nodepool_allowed_shapes": kpo_allowed_shapes,
    "kpo_manual_limits_applied": bool(
        manual_cpu_limit is not None and manual_mem_limit_gi is not None
    ),
    "ca_native_worker_pools_json_set": bool(CA_NATIVE_WORKER_POOLS_JSON),
    "benchmark_pricebook_path": BENCHMARK_PRICEBOOK_PATH,
    "system_addons_mapped_to_baseline": True,
    "ca_stack_dir": str(ca_dir),
    "kpo_stack_dir": str(kpo_dir),
}

_write_json(tf_root / "generation-summary.json", generation_summary)


# ---------------------------

# Print summary

# ---------------------------

generated_files = [
    ca_dir / "versions.tf",
    ca_dir / "providers.tf",
    ca_dir / "main.tf",
    ca_dir / "outputs.tf",
    ca_dir / "terraform.auto.tfvars.json",
    ca_dir / "iam" / "ca_workload_identity.policy.tmpl",
    kpo_dir / "versions.tf",
    kpo_dir / "providers.tf",
    kpo_dir / "main.tf",
    kpo_dir / "outputs.tf",
    kpo_dir / "terraform.auto.tfvars.json",
    kpo_dir / "manifests" / "kpo-values.yaml.tmpl",
    kpo_dir / "manifests" / "ocinodeclass.yaml.tmpl",
    kpo_dir / "manifests" / "nodepool.yaml.tmpl",
    kpo_dir / "iam" / "kpo_workload_identity.policy.tmpl",
    kpo_dir / "iam" / "kpo_nodes_cluster_join.policy.tmpl",
    tf_root / "iam" / "telemetry.policy.tmpl",
    tf_root / "generation-summary.json",
]


print("Cell 4 complete: Terraform stacks generated for both CA and KPO.")

print(f"- Benchmark mode: {BENCHMARK_MODE}")

print(f"- CA stack:  {ca_dir}")

print(f"- KPO stack: {kpo_dir}")

print(f"- Module source: {module_src}")

print(f"- Project root: {PROJECT_ROOT}")

print(f"- TF root: {tf_root}")

print("\nGenerated files:")

for f in generated_files:

    print(f"- {f}")


print("\nKey contract checks baked into generated config:")

print("- create_vcn=true, create_cluster=true, cluster_type=enhanced")

print("- create_iam_resources=true (strict self-contained benchmark mode)")

print(
    f"- control plane endpoint mode: {CONTROL_PLANE_ENDPOINT_MODE} "
    f"(control_plane_is_public={CONTROL_PLANE_IS_PUBLIC}, "
    f"assign_public_ip_to_control_plane={ASSIGN_PUBLIC_IP_TO_CONTROL_PLANE})"
)

print(f"- control_plane_allowed_cidrs={CONTROL_PLANE_ALLOWED_CIDRS}")

print(f"- KPO NodePool cpu/OCPU conversion factor: {KPO_CPU_PER_OCPU_FACTOR}")

print(f"- KPO NodePool capacity types: {kpo_capacity_types}")

print(f"- KPO NodePool allowed shapes: {kpo_allowed_shapes}")

if manual_cpu_limit is not None:

    print(
        f"- KPO NodePool manual limits applied: cpu={_fmt_num(manual_cpu_limit)}, "
        f"memory={_fmt_num(manual_mem_limit_gi)}Gi"
    )

else:

    print("- KPO NodePool limits: auto (flex-derived when applicable)")

print(
    "- CA uses ClusterAutoscaler add-on with authType=workload and nodeGroupAutoDiscovery"
)

print(
    "- CoreDNS and CA add-on are pinned to baseline nodes for clean autoscaled-pool scale-down"
)

print(
    "- Terraform autoscaled CA pool uses autoscale=true + ignore_initial_pool_size=true"
)

print("- KPO stack includes Helm values + OCINodeClass + NodePool templates")

print(
    "- OCI provider blocks use supported args: config_file_profile + tenancy_ocid + region"
)

if BENCHMARK_MODE == "parity":

    print("- Fairness gate: strict parity checks enforced for TARGET_CLUSTER=both")

else:

    print(
        "- Fairness gate: native mode relaxes cross-target shape/flex/minmax equality"
    )

if ENABLE_METRICS_SERVER:

    if skip_metrics_dep:

        print(
            "- Metrics Server add-on enabled with skipAddonDependenciesCheck=true "
            "(standalone cert-manager path)"
        )

    else:

        print("- Metrics Server add-on enabled with CertManager add-on dependency")


print("\nNext: Run Cell 5")

In [ ]:
# Cell 5 — Preflight Gates (blocking): config + doc/version + OCI checks + terraform checks
#
# Hardened for seamless reruns:
# - Uses SSOT paths from Cell 2
# - Validates Cell 4 artifacts and mode consistency
# - Keeps all blocking gates
# - Uses resolved TERRAFORM_BIN path
# - Runs terraform plan with -detailed-exitcode (no saved tfplan dependency)
# - Adds retry logic for transient init/plan failures
# - Persists preflight artifact for diagnostics

import os
import re
import json
import subprocess
import shutil
import time
from pathlib import Path
from datetime import datetime, timezone

import oci


def _env(k, d=""):
    return os.environ.get(k, d)


def _require_env(k):
    v = _env(k, "").strip()
    if not v:
        raise RuntimeError(f"Missing required environment variable: {k}")
    return v


def _k8_minor(version: str) -> str:
    m = re.search(r"v?(\d+)\.(\d+)", version)
    if not m:
        return ""
    return f"{m.group(1)}.{m.group(2)}"


def _k8_ge(version: str, maj: int, minr: int) -> bool:
    m = re.search(r"v?(\d+)\.(\d+)", version)
    if not m:
        return False
    return (int(m.group(1)), int(m.group(2))) >= (maj, minr)


def _load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def _write_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def _run(cmd, cwd=None, env=None):
    print(f"\n$ {' '.join(cmd)}")
    p = subprocess.run(cmd, cwd=cwd, env=env, text=True, capture_output=True)
    out = p.stdout or ""
    err = p.stderr or ""
    if out:
        print(out)
    if err:
        print(err)
    return p.returncode, out, err


def _run_with_retries(
    cmd,
    cwd=None,
    env=None,
    retries=1,
    retry_delay_seconds=5.0,
    retry_if=None,
    success_rcs=None,
):
    allowed = {0} if success_rcs is None else {int(x) for x in success_rcs}
    attempts = max(1, int(retries))
    last = (1, "", "")
    for attempt in range(1, attempts + 1):
        rc, out, err = _run(cmd, cwd=cwd, env=env)
        last = (rc, out, err)
        if rc in allowed:
            return last

        combined = f"{out}\n{err}"
        should_retry = bool(retry_if and retry_if(combined)) and attempt < attempts
        if should_retry:
            print(
                f"Retrying in {retry_delay_seconds}s (attempt {attempt + 1}/{attempts}) due to transient error..."
            )
            time.sleep(max(0.0, float(retry_delay_seconds)))
            continue
        return last
    return last


def _semver_tuple(v):
    m = re.search(r"(\d+)\.(\d+)\.(\d+)", str(v or ""))
    if not m:
        return None
    return (int(m.group(1)), int(m.group(2)), int(m.group(3)))


def _semver_ge(a, b):
    av = _semver_tuple(a)
    bv = _semver_tuple(b)
    if av is None or bv is None:
        return False
    return av >= bv


def _terraform_version(terraform_bin):
    try:
        p = subprocess.run(
            [terraform_bin, "version", "-json"],
            text=True,
            capture_output=True,
            check=False,
        )
        if p.returncode == 0 and p.stdout:
            obj = json.loads(p.stdout)
            v = str(obj.get("terraform_version", "")).strip()
            if v:
                return v
    except Exception:
        pass

    try:
        p = subprocess.run(
            [terraform_bin, "version"],
            text=True,
            capture_output=True,
            check=False,
        )
        out = p.stdout or p.stderr or ""
        m = re.search(r"Terraform v(\d+\.\d+\.\d+)", out)
        return m.group(1) if m else ""
    except Exception:
        return ""


def _is_transient_init_error(text):
    t = str(text or "").lower()
    markers = [
        "requested url returned error: 500",
        "remote: internal server error",
        "error: rpc failed",
        "expected flush after ref listing",
        "timed out",
        "i/o timeout",
        "tls handshake timeout",
        "context deadline exceeded",
        "temporary failure",
        "temporarily unavailable",
        "connection reset",
        "connection refused",
        "could not download module",
        "eof",
    ]
    return any(m in t for m in markers)


def _is_transient_tf_error(text):
    t = str(text or "").lower()
    markers = [
        "timed out",
        "i/o timeout",
        "tls handshake timeout",
        "context deadline exceeded",
        "temporary failure",
        "temporarily unavailable",
        "connection reset",
        "connection refused",
        "service unavailable",
        "too many requests",
        "request was throttled",
        "throttl",
        "eof",
        "http response status code 429",
        "http response status code 500",
        "http response status code 502",
        "http response status code 503",
        "http response status code 504",
    ]
    return any(m in t for m in markers)


def _is_oci_provider_type_mismatch(text):
    t = str(text or "").lower()
    return "provider type mismatch" in t and "hashicorp/oci" in t and "oracle/oci" in t


def _bool(x):
    return str(x).strip().lower() in {"1", "true", "yes", "y", "on"}


def _to_int(v, d=None):
    try:
        return int(str(v).strip())
    except Exception:
        return d


def _to_float(v, d=None):
    try:
        return float(str(v).strip())
    except Exception:
        return d


def _assert(cond, ok_msg, fail_msg, oks, errs):
    if cond:
        oks.append(ok_msg)
    else:
        errs.append(fail_msg)


def _contains_addon(addons: dict, addon_name: str) -> bool:
    return isinstance(addons, dict) and addon_name in addons


def _addon_cfg_map(addon_obj):
    cfgs = addon_obj.get("configurations", []) if isinstance(addon_obj, dict) else []
    out = {}
    for c in cfgs:
        if isinstance(c, dict) and "key" in c:
            out[c["key"]] = c.get("value", "")
    return out


def _get_autoscaled_pool(worker_pools: dict):
    if not isinstance(worker_pools, dict):
        return None
    for _, v in worker_pools.items():
        if isinstance(v, dict) and v.get("autoscale") is True:
            return v
    return None


PROJECT_ROOT = Path(_require_env("PROJECT_ROOT")).expanduser().resolve()
tf_root = Path(_require_env("TF_STACK_ROOT")).expanduser().resolve()
module_src = (
    Path(_require_env("TERRAFORM_OCI_OKE_MODULE_SOURCE")).expanduser().resolve()
)

ca_dir = tf_root / "ca"
kpo_dir = tf_root / "kpo"

target = _env("TARGET_CLUSTER", "both").strip().lower()
if target not in {"ca", "kpo", "both"}:
    raise RuntimeError(f"TARGET_CLUSTER must be ca|kpo|both, got: {target}")

targets = ["ca", "kpo"] if target == "both" else [target]

run_id = _require_env("RUN_ID")
output_dir = Path(_require_env("OUTPUT_DIR")).expanduser().resolve()
artifact_dir = output_dir / run_id
artifact_dir.mkdir(parents=True, exist_ok=True)

print("=== Cell 5 Preflight ===")
print(f"Project root: {PROJECT_ROOT}")
print(f"TF root: {tf_root}")
print(f"Module source: {module_src}")
print(f"Selected target(s): {targets}")

terraform_bin = _env("TERRAFORM_BIN", "terraform").strip() or "terraform"
terraform_bin_path = shutil.which(terraform_bin)
if not terraform_bin_path:
    raise RuntimeError(
        f"Terraform binary not found: {terraform_bin}. Set TERRAFORM_BIN to a valid executable path."
    )

terraform_version = _terraform_version(terraform_bin_path)
print(f"Terraform bin: {terraform_bin_path}")
print(f"Terraform version: {terraform_version or 'unknown'}")

if not module_src.exists() or not module_src.is_dir():
    raise RuntimeError(f"Invalid TERRAFORM_OCI_OKE_MODULE_SOURCE: {module_src}")

for marker in ["versions.tf", "module-cluster.tf", "module-workers.tf", "modules"]:
    if not (module_src / marker).exists():
        raise RuntimeError(
            "TERRAFORM_OCI_OKE_MODULE_SOURCE is not a valid terraform-oci-oke-main root. "
            f"Missing: {marker} in {module_src}"
        )

oci_config_file = _require_env("OCI_CONFIG_FILE")
oci_profile = _require_env("OCI_PROFILE")
os.environ["OCI_CLI_CONFIG_FILE"] = oci_config_file
os.environ["OCI_CONFIG_FILE_PROFILE"] = oci_profile
os.environ["TF_VAR_config_file_profile"] = oci_profile

cmd_env = os.environ.copy()
cmd_env["OCI_CLI_CONFIG_FILE"] = oci_config_file
cmd_env["OCI_CONFIG_FILE_PROFILE"] = oci_profile
cmd_env["TF_VAR_config_file_profile"] = oci_profile
cmd_env["TF_VAR_oci_config_file"] = oci_config_file

gen_summary_path = tf_root / "generation-summary.json"
if not gen_summary_path.exists():
    raise RuntimeError(
        f"Missing required file: {gen_summary_path}. Run Cell 4 before Cell 5."
    )
_generation_summary = _load_json(gen_summary_path)

required_files = [
    ca_dir / "main.tf",
    ca_dir / "providers.tf",
    ca_dir / "terraform.auto.tfvars.json",
    kpo_dir / "main.tf",
    kpo_dir / "providers.tf",
    kpo_dir / "terraform.auto.tfvars.json",
    kpo_dir / "manifests" / "kpo-values.yaml.tmpl",
    kpo_dir / "manifests" / "ocinodeclass.yaml.tmpl",
    kpo_dir / "manifests" / "nodepool.yaml.tmpl",
    kpo_dir / "iam" / "kpo_workload_identity.policy.tmpl",
    kpo_dir / "iam" / "kpo_nodes_cluster_join.policy.tmpl",
    ca_dir / "iam" / "ca_workload_identity.policy.tmpl",
    tf_root / "iam" / "telemetry.policy.tmpl",
]
for f in required_files:
    if not f.exists():
        raise RuntimeError(f"Required file missing: {f}")

if _generation_summary.get("tf_root") and str(tf_root) != str(
    _generation_summary.get("tf_root")
):
    raise RuntimeError(
        "TF_STACK_ROOT mismatch with tf/generation-summary.json. "
        f"TF_STACK_ROOT={tf_root}, summary.tf_root={_generation_summary.get('tf_root')}"
    )

ca_bm = _load_json(ca_dir / "terraform.auto.tfvars.json")["benchmark"]
kpo_bm = _load_json(kpo_dir / "terraform.auto.tfvars.json")["benchmark"]

benchmark_mode = _env("BENCHMARK_MODE", "parity").strip().lower()
image_mode = _env("IMAGE_MODE", "pin").strip().lower()
if benchmark_mode not in {"parity", "native"}:
    raise RuntimeError(f"BENCHMARK_MODE must be parity|native, got: {benchmark_mode}")
if image_mode not in {"pin", "filter"}:
    raise RuntimeError(f"IMAGE_MODE must be pin|filter, got: {image_mode}")

supported_addon_versions = {
    "1.32": {
        "ClusterAutoscaler": "1.32.5",
        "CertManager": "1.17.1",
        "KubernetesMetricsServer": "0.7.2",
    },
    "1.33": {
        "ClusterAutoscaler": "1.33.3",
        "CertManager": "1.17.1",
        "KubernetesMetricsServer": "0.7.2",
    },
    "1.34": {
        "ClusterAutoscaler": "1.34.2",
        "CertManager": "1.17.1",
        "KubernetesMetricsServer": "0.7.2",
    },
}

oks = []
errs = []

k8 = _require_env("KUBERNETES_VERSION")
bench_region = _require_env("BENCHMARK_REGION")
bench_compartment = _require_env("BENCHMARK_COMPARTMENT_OCID")
home_region = _require_env("HOME_REGION")
tenancy_ocid = _require_env("TENANCY_OCID")
enable_metrics = _bool(_env("ENABLE_METRICS_SERVER", "false"))

tf_min_version = _env("TERRAFORM_MIN_VERSION", "1.6.0").strip() or "1.6.0"

_assert(
    bool(terraform_version),
    f"Terraform version detected ({terraform_version}).",
    "Unable to detect Terraform version from TERRAFORM_BIN. Use a valid Terraform CLI binary.",
    oks,
    errs,
)

_assert(
    bool(terraform_version) and _semver_ge(terraform_version, tf_min_version),
    f"Terraform version gate passed ({terraform_version} >= {tf_min_version}).",
    f"Terraform version gate failed: detected {terraform_version or 'unknown'} at {terraform_bin_path}. Install Terraform >= {tf_min_version} and set TERRAFORM_BIN if needed.",
    oks,
    errs,
)

if terraform_version == "1.5.7":
    errs.append(
        "Detected Terraform 1.5.7. This environment consistently resolves OCI provider aliases as hashicorp/oci and fails with oracle/oci module contracts. Use a newer Terraform binary (Homebrew core is pinned to 1.5.7) and set TERRAFORM_BIN."
    )

_assert(
    _generation_summary.get("benchmark_mode") == benchmark_mode,
    f"Generation summary benchmark_mode matches SSOT ({benchmark_mode}).",
    "generation-summary benchmark_mode mismatch vs current BENCHMARK_MODE.",
    oks,
    errs,
)
_assert(
    _generation_summary.get("image_mode") == image_mode,
    f"Generation summary image_mode matches SSOT ({image_mode}).",
    "generation-summary image_mode mismatch vs current IMAGE_MODE.",
    oks,
    errs,
)

_assert(
    _k8_ge(k8, 1, 31),
    f"Kubernetes version gate passed ({k8} >= 1.31).",
    f"Kubernetes version gate failed: {k8} must be >= 1.31.",
    oks,
    errs,
)

k8_minor = _k8_minor(k8)
_assert(
    k8_minor in supported_addon_versions,
    f"Addon support table found for Kubernetes {k8_minor}.",
    f"No addon support mapping for Kubernetes {k8_minor}. Update supported_addon_versions map from docs.",
    oks,
    errs,
)

for name, bm in [("ca", ca_bm), ("kpo", kpo_bm)]:
    _assert(
        bm.get("create_vcn") is True,
        f"{name}: create_vcn=true",
        f"{name}: create_vcn must be true",
        oks,
        errs,
    )
    _assert(
        bm.get("create_cluster") is True,
        f"{name}: create_cluster=true",
        f"{name}: create_cluster must be true",
        oks,
        errs,
    )
    _assert(
        str(bm.get("cluster_type", "")).lower() == "enhanced",
        f"{name}: cluster_type=enhanced",
        f"{name}: cluster_type must be enhanced",
        oks,
        errs,
    )
    _assert(
        str(bm.get("cni_type", "")).lower() == "npn",
        f"{name}: cni_type=npn (VCN-native).",
        f"{name}: cni_type must be npn for this workflow.",
        oks,
        errs,
    )
    _assert(
        str(bm.get("kubernetes_version", "")) == k8,
        f"{name}: kubernetes_version matches SSOT ({k8})",
        f"{name}: kubernetes_version mismatch vs SSOT",
        oks,
        errs,
    )
    _assert(
        bm.get("create_iam_resources") is True,
        f"{name}: create_iam_resources=true",
        f"{name}: create_iam_resources must be true in benchmark mode",
        oks,
        errs,
    )
    _assert(
        bool(home_region),
        f"{name}: HOME_REGION present for IAM operations",
        f"{name}: HOME_REGION missing",
        oks,
        errs,
    )

if target == "both":
    ca_auto = _get_autoscaled_pool(ca_bm.get("worker_pools", {}))
    kpo_auto = _get_autoscaled_pool(kpo_bm.get("worker_pools", {}))

    _assert(
        isinstance(ca_auto, dict),
        "ca: autoscaled pool located for fairness gate checks.",
        "ca: unable to locate autoscaled worker pool for fairness checks.",
        oks,
        errs,
    )

    if benchmark_mode == "parity":
        if isinstance(kpo_auto, dict):
            oks.append(
                "kpo: autoscaled pool located for fairness gate checks (terraform worker pool source)."
            )
            kpo_shape = str(kpo_auto.get("shape", "")).strip()
            kpo_min = _to_int(kpo_auto.get("min_size"), None)
            kpo_max = _to_int(kpo_auto.get("max_size"), None)
            kpo_flex_ocpus = _to_float(kpo_auto.get("flex_ocpus"), None)
            kpo_flex_memory = _to_float(kpo_auto.get("flex_memory_gb"), None)
        else:
            kpo_shape = _env("KPO_AUTOSCALED_POOL_SHAPE", "").strip()
            kpo_min = _to_int(_env("KPO_AUTOSCALED_POOL_MIN_SIZE", ""), None)
            kpo_max = _to_int(_env("KPO_AUTOSCALED_POOL_MAX_SIZE", ""), None)
            kpo_flex_ocpus = _to_float(_env("KPO_AUTOSCALED_POOL_FLEX_OCPUS", ""), None)
            kpo_flex_memory = _to_float(
                _env("KPO_AUTOSCALED_POOL_FLEX_MEMORY_GB", ""), None
            )

            _assert(
                bool(kpo_shape),
                "kpo: autoscaled profile shape resolved from SSOT env for fairness gate checks.",
                "kpo: unable to resolve KPO autoscaled profile shape for parity fairness checks.",
                oks,
                errs,
            )
            _assert(
                kpo_min is not None and kpo_max is not None,
                "kpo: autoscaled profile min/max resolved from SSOT env for fairness gate checks.",
                "kpo: unable to resolve KPO autoscaled profile min/max for parity fairness checks.",
                oks,
                errs,
            )

        if isinstance(ca_auto, dict):
            ca_shape = str(ca_auto.get("shape", "")).strip()
            ca_min = _to_int(ca_auto.get("min_size"), None)
            ca_max = _to_int(ca_auto.get("max_size"), None)
            ca_flex_ocpus = _to_float(ca_auto.get("flex_ocpus"), None)
            ca_flex_memory = _to_float(ca_auto.get("flex_memory_gb"), None)

            _assert(
                ca_shape == kpo_shape,
                "Parity gate: CA/KPO autoscaled shapes match.",
                "Parity gate failed: CA/KPO autoscaled shapes must match when TARGET_CLUSTER=both.",
                oks,
                errs,
            )
            _assert(
                ca_min is not None
                and ca_max is not None
                and kpo_min is not None
                and kpo_max is not None
                and ca_min == kpo_min
                and ca_max == kpo_max,
                "Parity gate: CA/KPO autoscaled min/max match.",
                "Parity gate failed: CA/KPO autoscaled min/max must match when TARGET_CLUSTER=both.",
                oks,
                errs,
            )

            ca_flex = str(ca_shape).endswith(".Flex")
            kpo_flex = str(kpo_shape).endswith(".Flex")
            if ca_flex and kpo_flex:
                _assert(
                    ca_flex_ocpus is not None
                    and kpo_flex_ocpus is not None
                    and abs(ca_flex_ocpus - kpo_flex_ocpus) < 1e-9,
                    "Parity gate: CA/KPO autoscaled flex OCPU match.",
                    "Parity gate failed: CA/KPO autoscaled flex OCPU must match.",
                    oks,
                    errs,
                )
                _assert(
                    ca_flex_memory is not None
                    and kpo_flex_memory is not None
                    and abs(ca_flex_memory - kpo_flex_memory) < 1e-9,
                    "Parity gate: CA/KPO autoscaled flex memory match.",
                    "Parity gate failed: CA/KPO autoscaled flex memory must match.",
                    oks,
                    errs,
                )
    else:
        if isinstance(kpo_auto, dict):
            oks.append(
                "kpo: autoscaled pool located (terraform worker pool source available, native mode)."
            )
        else:
            oks.append(
                "kpo: autoscaled worker pool not required in native mode (KPO scales via NodePool/NodeClaim templates)."
            )
        oks.append(
            "Native mode fairness gate: cross-target autoscaled shape/flex/minmax equality is intentionally relaxed."
        )

for name, p in [("ca", ca_dir / "providers.tf"), ("kpo", kpo_dir / "providers.tf")]:
    txt = p.read_text(encoding="utf-8")
    _assert(
        "config_file_profile" in txt,
        f"{name}: provider uses config_file_profile.",
        f"{name}: provider block missing config_file_profile.",
        oks,
        errs,
    )

    has_unsupported_config_file = re.search(r"(?m)^\s*config_file\s*=", txt) is not None
    _assert(
        not has_unsupported_config_file,
        f"{name}: provider does not use unsupported config_file arg.",
        f"{name}: provider block still contains unsupported config_file arg.",
        oks,
        errs,
    )

if "ca" in targets:
    ca_addons = ca_bm.get("cluster_addons", {})
    wp = ca_bm.get("worker_pools", {})

    _assert(
        _contains_addon(ca_addons, "ClusterAutoscaler"),
        "ca: ClusterAutoscaler add-on configured.",
        "ca: ClusterAutoscaler add-on missing.",
        oks,
        errs,
    )

    if _contains_addon(ca_addons, "ClusterAutoscaler"):
        ca_cfg = _addon_cfg_map(ca_addons["ClusterAutoscaler"])
        _assert(
            ca_cfg.get("authType") == "workload",
            "ca: authType=workload",
            "ca: ClusterAutoscaler authType must be workload",
            oks,
            errs,
        )

        replicas = (
            int(ca_cfg.get("numOfReplicas", "0"))
            if str(ca_cfg.get("numOfReplicas", "")).isdigit()
            else 0
        )
        _assert(
            replicas >= 2,
            f"ca: numOfReplicas={replicas} (>=2)",
            f"ca: numOfReplicas must be >=2 (got {replicas})",
            oks,
            errs,
        )

        has_nodes = "nodes" in ca_cfg and str(ca_cfg.get("nodes", "")).strip() != ""
        has_ngad = (
            "nodeGroupAutoDiscovery" in ca_cfg
            and str(ca_cfg.get("nodeGroupAutoDiscovery", "")).strip() != ""
        )
        _assert(
            has_nodes != has_ngad,
            "ca: exactly one of nodes/nodeGroupAutoDiscovery is set.",
            "ca: must set exactly one of nodes or nodeGroupAutoDiscovery (not both/none).",
            oks,
            errs,
        )

        if has_ngad:
            _assert(
                _k8_ge(k8, 1, 32),
                f"ca: nodeGroupAutoDiscovery gate passed for Kubernetes {k8}.",
                "ca: nodeGroupAutoDiscovery requires supported CA addon versions; enforce Kubernetes >=1.32 in this workflow.",
                oks,
                errs,
            )

    autoscaled_pools = [
        v for _, v in wp.items() if isinstance(v, dict) and v.get("autoscale") is True
    ]
    _assert(
        len(autoscaled_pools) >= 1,
        "ca: autoscaled worker pool exists.",
        "ca: no autoscaled worker pool found (autoscale=true).",
        oks,
        errs,
    )

    for idx, p in enumerate(autoscaled_pools):
        _assert(
            p.get("ignore_initial_pool_size") is True,
            f"ca: autoscaled pool #{idx+1} has ignore_initial_pool_size=true.",
            f"ca: autoscaled pool #{idx+1} must set ignore_initial_pool_size=true.",
            oks,
            errs,
        )

    if enable_metrics:
        _assert(
            _contains_addon(ca_addons, "KubernetesMetricsServer"),
            "ca: KubernetesMetricsServer add-on present.",
            "ca: ENABLE_METRICS_SERVER=true but KubernetesMetricsServer add-on missing.",
            oks,
            errs,
        )
        if _contains_addon(ca_addons, "KubernetesMetricsServer"):
            ms_cfg = _addon_cfg_map(ca_addons["KubernetesMetricsServer"])
            has_cert = _contains_addon(ca_addons, "CertManager")
            skip_dep = (
                str(ms_cfg.get("skipAddonDependenciesCheck", "false")).lower() == "true"
            )
            _assert(
                has_cert or skip_dep,
                "ca: Metrics Server dependency gate passed (CertManager present or skipAddonDependenciesCheck=true).",
                "ca: Metrics Server dependency gate failed (need CertManager add-on or skipAddonDependenciesCheck=true).",
                oks,
                errs,
            )

if "kpo" in targets:
    kpo_addons = kpo_bm.get("cluster_addons", {})

    _assert(
        (kpo_dir / "manifests" / "kpo-values.yaml.tmpl").exists(),
        "kpo: kpo-values.yaml.tmpl present.",
        "kpo: missing kpo-values.yaml.tmpl",
        oks,
        errs,
    )
    _assert(
        (kpo_dir / "manifests" / "ocinodeclass.yaml.tmpl").exists(),
        "kpo: ocinodeclass.yaml.tmpl present.",
        "kpo: missing ocinodeclass.yaml.tmpl",
        oks,
        errs,
    )
    _assert(
        (kpo_dir / "manifests" / "nodepool.yaml.tmpl").exists(),
        "kpo: nodepool.yaml.tmpl present.",
        "kpo: missing nodepool.yaml.tmpl",
        oks,
        errs,
    )
    _assert(
        (kpo_dir / "iam" / "kpo_workload_identity.policy.tmpl").exists(),
        "kpo: workload identity policy template present.",
        "kpo: missing kpo_workload_identity.policy.tmpl",
        oks,
        errs,
    )
    _assert(
        (kpo_dir / "iam" / "kpo_nodes_cluster_join.policy.tmpl").exists(),
        "kpo: cluster join policy template present.",
        "kpo: missing kpo_nodes_cluster_join.policy.tmpl",
        oks,
        errs,
    )

    kpo_values_text = (kpo_dir / "manifests" / "kpo-values.yaml.tmpl").read_text(
        encoding="utf-8"
    )
    oci_nc_text = (kpo_dir / "manifests" / "ocinodeclass.yaml.tmpl").read_text(
        encoding="utf-8"
    )

    has_image_filter = (
        re.search(r"(?m)^\s*imageFilter:\s*$", oci_nc_text or "") is not None
    )
    has_image_id = (
        re.search(r"(?m)^\s*imageId:\s*\".+\"\s*$", oci_nc_text or "") is not None
    )

    if image_mode == "filter":
        _assert(
            has_image_filter and not has_image_id,
            "kpo: IMAGE_MODE=filter uses imageFilter (without imageId).",
            "kpo: IMAGE_MODE=filter must render imageFilter and must not render imageId.",
            oks,
            errs,
        )

        osver_match = re.search(
            r"(?m)^\s*osVersionFilter:\s*\"([^\"]+)\"", oci_nc_text or ""
        )
        osver = osver_match.group(1).strip() if osver_match else ""
        _assert(
            bool(re.fullmatch(r"[0-9]+(\.[0-9]+)?", osver)),
            f"kpo: osVersionFilter is numeric ({osver}).",
            "kpo: osVersionFilter must be numeric (for example 8 or 8.10) in filter mode.",
            oks,
            errs,
        )

        filter_compartment = _env("KPO_IMAGE_FILTER_COMPARTMENT_ID", "").strip()
        if filter_compartment:
            _assert(
                f'compartmentId: "{filter_compartment}"' in oci_nc_text,
                "kpo: imageFilter compartmentId rendered for compartment-scoped filter mode.",
                "kpo: IMAGE_MODE=filter with KPO_IMAGE_FILTER_COMPARTMENT_ID requires compartmentId in OCINodeClass imageFilter.",
                oks,
                errs,
            )
    else:
        _assert(
            has_image_id and not has_image_filter,
            "kpo: IMAGE_MODE=pin uses imageId (without imageFilter).",
            "kpo: IMAGE_MODE=pin must render imageId and must not render imageFilter.",
            oks,
            errs,
        )

    _assert(
        "__APISERVER_ENDPOINT__" in kpo_values_text,
        "kpo: apiserver endpoint placeholder present.",
        "kpo: kpo-values.yaml.tmpl must include __APISERVER_ENDPOINT__ placeholder.",
        oks,
        errs,
    )
    _assert(
        "ociVcnIpNative: true" in kpo_values_text,
        "kpo: Helm values set ociVcnIpNative=true.",
        "kpo: kpo-values.yaml.tmpl must set ociVcnIpNative=true for VCN-native.",
        oks,
        errs,
    )
    _assert(
        "__WORKER_SUBNET_ID__" in oci_nc_text,
        "kpo: worker subnet placeholder present.",
        "kpo: ocinodeclass.yaml.tmpl must include __WORKER_SUBNET_ID__ placeholder.",
        oks,
        errs,
    )
    _assert(
        "__POD_SUBNET_ID__" in oci_nc_text,
        "kpo: pod subnet placeholder present.",
        "kpo: ocinodeclass.yaml.tmpl must include __POD_SUBNET_ID__ placeholder.",
        oks,
        errs,
    )
    _assert(
        "secondaryVnicConfigs" in oci_nc_text,
        "kpo: OCINodeClass has secondaryVnicConfigs for NPN.",
        "kpo: ocinodeclass.yaml.tmpl must include secondaryVnicConfigs for VCN-native.",
        oks,
        errs,
    )

    ip_count_match = re.search(r"ipCount:\s*(\d+)", oci_nc_text)
    if ip_count_match:
        ip_count = int(ip_count_match.group(1))
        _assert(
            (ip_count & (ip_count - 1)) == 0,
            f"kpo: ipCount={ip_count} is power-of-two.",
            f"kpo: ipCount must be power-of-two, got {ip_count}.",
            oks,
            errs,
        )
        _assert(
            ip_count <= 16,
            f"kpo: ipCount={ip_count} (safe default <=16).",
            f"kpo: ipCount should be <=16 unless CNI add-on >=3.2.0 is verified.",
            oks,
            errs,
        )
    else:
        errs.append("kpo: ipCount not found in ocinodeclass.yaml.tmpl")

    kpo_policy_text = (kpo_dir / "iam" / "kpo_workload_identity.policy.tmpl").read_text(
        encoding="utf-8"
    )
    kpo_join_text = (kpo_dir / "iam" / "kpo_nodes_cluster_join.policy.tmpl").read_text(
        encoding="utf-8"
    )

    _assert(
        "request.principal.type = 'workload'" in kpo_policy_text,
        "kpo: workload identity policy uses workload principal conditions.",
        "kpo: workload identity policy missing workload condition block.",
        oks,
        errs,
    )
    _assert(
        "Allow dynamic-group __KPO_NODES_DYNAMIC_GROUP_NAME__ to {CLUSTER_JOIN}"
        in kpo_join_text,
        "kpo: CLUSTER_JOIN statement present.",
        "kpo: cluster join policy missing CLUSTER_JOIN statement.",
        oks,
        errs,
    )
    _assert(
        "target.cluster.id = '__KPO_CLUSTER_OCID__'" in kpo_join_text,
        "kpo: CLUSTER_JOIN policy scoped to target.cluster.id.",
        "kpo: cluster join policy must scope CLUSTER_JOIN by target.cluster.id.",
        oks,
        errs,
    )

    if enable_metrics:
        _assert(
            _contains_addon(kpo_addons, "KubernetesMetricsServer"),
            "kpo: KubernetesMetricsServer add-on present.",
            "kpo: ENABLE_METRICS_SERVER=true but KubernetesMetricsServer add-on missing.",
            oks,
            errs,
        )
        if _contains_addon(kpo_addons, "KubernetesMetricsServer"):
            ms_cfg = _addon_cfg_map(kpo_addons["KubernetesMetricsServer"])
            has_cert = _contains_addon(kpo_addons, "CertManager")
            skip_dep = (
                str(ms_cfg.get("skipAddonDependenciesCheck", "false")).lower() == "true"
            )
            _assert(
                has_cert or skip_dep,
                "kpo: Metrics Server dependency gate passed (CertManager present or skipAddonDependenciesCheck=true).",
                "kpo: Metrics Server dependency gate failed (need CertManager add-on or skipAddonDependenciesCheck=true).",
                oks,
                errs,
            )

try:
    cfg = oci.config.from_file(
        file_location=oci_config_file,
        profile_name=oci_profile,
    )
    cfg["region"] = bench_region

    idc = oci.identity.IdentityClient(cfg)
    ce = oci.container_engine.ContainerEngineClient(cfg)
    compute = oci.core.ComputeClient(cfg)
    limits = oci.limits.LimitsClient(cfg)

    subs = oci.pagination.list_call_get_all_results(
        idc.list_region_subscriptions, tenancy_ocid
    ).data
    subscribed_regions = {s.region_name for s in subs}
    _assert(
        bench_region in subscribed_regions,
        f"OCI: benchmark region {bench_region} is subscribed.",
        f"OCI: benchmark region {bench_region} is not subscribed in tenancy.",
        oks,
        errs,
    )

    try:
        ce_opts = ce.get_cluster_options(cluster_option_id="all").data
        ce_versions = set(getattr(ce_opts, "kubernetes_versions", []) or [])
        _assert(
            k8 in ce_versions,
            f"OCI CE: Kubernetes version {k8} available in region {bench_region}.",
            f"OCI CE: Kubernetes version {k8} not available in region {bench_region}.",
            oks,
            errs,
        )
    except Exception as e:
        errs.append(f"OCI CE options check failed: {e}")

    required_shapes = set()
    if "ca" in targets:
        for p in ca_bm.get("worker_pools", {}).values():
            if isinstance(p, dict) and p.get("shape"):
                required_shapes.add(p["shape"])
    if "kpo" in targets:
        for p in kpo_bm.get("worker_pools", {}).values():
            if isinstance(p, dict) and p.get("shape"):
                required_shapes.add(p["shape"])

    try:
        shape_objs = oci.pagination.list_call_get_all_results(
            compute.list_shapes, bench_compartment
        ).data
        available_shapes = {s.shape for s in shape_objs if getattr(s, "shape", None)}
        for shp in sorted(required_shapes):
            _assert(
                shp in available_shapes,
                f"OCI Compute: required shape available: {shp}",
                f"OCI Compute: required shape not available in region/compartment: {shp}",
                oks,
                errs,
            )
    except Exception as e:
        errs.append(f"OCI Compute shape availability check failed: {e}")

    try:
        svc_resp = oci.pagination.list_call_get_all_results(
            limits.list_services, tenancy_ocid
        ).data
        svc_names = {
            (getattr(s, "name", None) or getattr(s, "service_name", None) or "").lower()
            for s in svc_resp
        }
        _assert(
            "compute" in svc_names,
            "OCI Limits: compute service visible at tenancy scope.",
            "OCI Limits: compute service not returned by list_services.",
            oks,
            errs,
        )

        lv = oci.pagination.list_call_get_all_results(
            limits.list_limit_values,
            tenancy_ocid,
            service_name="compute",
            scope_type="REGION",
        ).data
        _assert(
            len(lv) > 0,
            "OCI Limits: compute limit values retrieved at tenancy scope.",
            "OCI Limits: no compute limit values returned at tenancy scope.",
            oks,
            errs,
        )
    except Exception as e:
        errs.append(f"OCI Limits check failed (tenancy-scoped APIs): {e}")

except Exception as e:
    errs.append(f"OCI client initialization/auth check failed: {e}")

if k8_minor in supported_addon_versions:
    if "ca" in targets:
        _assert(
            "ClusterAutoscaler" in supported_addon_versions[k8_minor],
            f"Doc gate: ClusterAutoscaler supported for Kubernetes {k8_minor}.",
            f"Doc gate: ClusterAutoscaler support not found for Kubernetes {k8_minor}.",
            oks,
            errs,
        )

    if enable_metrics:
        _assert(
            "KubernetesMetricsServer" in supported_addon_versions[k8_minor],
            f"Doc gate: KubernetesMetricsServer supported for Kubernetes {k8_minor}.",
            f"Doc gate: KubernetesMetricsServer support not found for Kubernetes {k8_minor}.",
            oks,
            errs,
        )
        _assert(
            "CertManager" in supported_addon_versions[k8_minor],
            f"Doc gate: CertManager supported for Kubernetes {k8_minor}.",
            f"Doc gate: CertManager support not found for Kubernetes {k8_minor}.",
            oks,
            errs,
        )

print("\n=== Preflight Gate Results (before Terraform commands) ===")
for x in oks:
    print(f"[PASS] {x}")

if errs:
    for x in errs:
        print(f"[FAIL] {x}")
    raise RuntimeError(
        f"Preflight failed with {len(errs)} issue(s) before Terraform checks."
    )

tf_errs = []
tf_results = {}

tf_init_upgrade = _bool(_env("TF_INIT_UPGRADE", "false"))
tf_init_retries = max(1, _to_int(_env("TF_INIT_RETRIES", "3"), 3) or 3)
tf_init_retry_seconds = _to_float(_env("TF_INIT_RETRY_SECONDS", "8"), 8.0) or 8.0
tf_plan_retries = max(1, _to_int(_env("TF_PLAN_RETRIES", "2"), 2) or 2)
tf_plan_retry_seconds = _to_float(_env("TF_PLAN_RETRY_SECONDS", "8"), 8.0) or 8.0

print(f"Terraform init -upgrade enabled: {str(tf_init_upgrade).lower()}")
print(
    f"Terraform init retry policy: retries={tf_init_retries}, delay_seconds={tf_init_retry_seconds}"
)
print(
    f"Terraform plan retry policy: retries={tf_plan_retries}, delay_seconds={tf_plan_retry_seconds}"
)
print("Terraform plan mode: -detailed-exitcode (no persisted tfplan artifact)")

for t in targets:
    stack_dir = ca_dir if t == "ca" else kpo_dir
    print(f"\n=== Terraform checks for target: {t} ({stack_dir}) ===")
    tf_results[t] = {"stack_dir": str(stack_dir), "steps": []}

    init_cmd = [
        terraform_bin_path,
        f"-chdir={stack_dir}",
        "init",
        "-input=false",
        "-no-color",
    ]
    if tf_init_upgrade:
        init_cmd.insert(3, "-upgrade")

    rc, out, err = _run_with_retries(
        init_cmd,
        env=cmd_env,
        retries=tf_init_retries,
        retry_delay_seconds=tf_init_retry_seconds,
        retry_if=_is_transient_init_error,
    )
    tf_results[t]["steps"].append({"name": "init", "rc": rc})
    if rc != 0:
        combined = f"{out}\n{err}"
        if _is_oci_provider_type_mismatch(combined):
            tf_errs.append(
                f"{t}: terraform init failed due to OCI provider source mismatch (hashicorp/oci vs oracle/oci). "
                f"Detected Terraform {terraform_version or 'unknown'} at {terraform_bin_path}. "
                "Install a newer Terraform CLI, set TERRAFORM_BIN, rerun Cell 4, then rerun Cell 5."
            )
        else:
            tf_errs.append(f"{t}: command failed ({' '.join(init_cmd)})")
        continue

    basic_steps = [
        ("fmt", [terraform_bin_path, f"-chdir={stack_dir}", "fmt", "-recursive"]),
        (
            "fmt-check",
            [terraform_bin_path, f"-chdir={stack_dir}", "fmt", "-check", "-recursive"],
        ),
        (
            "validate",
            [terraform_bin_path, f"-chdir={stack_dir}", "validate", "-no-color"],
        ),
    ]

    failed = False
    for step_name, cmd in basic_steps:
        rc, _, _ = _run(cmd, env=cmd_env)
        tf_results[t]["steps"].append({"name": step_name, "rc": rc})
        if rc != 0:
            tf_errs.append(f"{t}: command failed ({' '.join(cmd)})")
            failed = True
            break
    if failed:
        continue

    plan_cmd = [
        terraform_bin_path,
        f"-chdir={stack_dir}",
        "plan",
        "-input=false",
        "-no-color",
        "-lock-timeout=5m",
        "-detailed-exitcode",
    ]

    rc, out, err = _run_with_retries(
        plan_cmd,
        env=cmd_env,
        retries=tf_plan_retries,
        retry_delay_seconds=tf_plan_retry_seconds,
        retry_if=_is_transient_tf_error,
        success_rcs={0, 2},
    )

    plan_status = "unknown"
    if rc == 0:
        plan_status = "no_changes"
    elif rc == 2:
        plan_status = "changes_present"

    tf_results[t]["steps"].append({"name": "plan", "rc": rc, "status": plan_status})

    plan_log_path = artifact_dir / "preflight" / f"{t}_terraform_plan.log"
    _write_json(
        artifact_dir / "preflight" / f"{t}_terraform_plan_meta.json",
        {
            "target": t,
            "generated_at_utc": datetime.now(timezone.utc)
            .isoformat()
            .replace("+00:00", "Z"),
            "rc": rc,
            "status": plan_status,
            "stack_dir": str(stack_dir),
            "command": plan_cmd,
            "log_path": str(plan_log_path),
        },
    )
    plan_log_path.parent.mkdir(parents=True, exist_ok=True)
    plan_log_path.write_text(
        (out or "") + ("\n" if out and err else "") + (err or ""), encoding="utf-8"
    )

    if rc not in {0, 2}:
        tf_errs.append(f"{t}: command failed ({' '.join(plan_cmd)})")

if tf_errs:
    print("\n=== Terraform Preflight Failures ===")
    for e in tf_errs:
        print(f"[FAIL] {e}")

    preflight_path = artifact_dir / "preflight" / "cell5_preflight.json"
    _write_json(
        preflight_path,
        {
            "generated_at_utc": datetime.now(timezone.utc)
            .isoformat()
            .replace("+00:00", "Z"),
            "run_id": run_id,
            "project_root": str(PROJECT_ROOT),
            "tf_root": str(tf_root),
            "selected_targets": targets,
            "terraform_bin": terraform_bin_path,
            "terraform_version": terraform_version,
            "benchmark_mode": benchmark_mode,
            "image_mode": image_mode,
            "passes": oks,
            "failures": errs + tf_errs,
            "terraform_results": tf_results,
        },
    )
    raise RuntimeError(f"Cell 5 failed with {len(tf_errs)} Terraform error(s).")

preflight_path = artifact_dir / "preflight" / "cell5_preflight.json"
_write_json(
    preflight_path,
    {
        "generated_at_utc": datetime.now(timezone.utc)
        .isoformat()
        .replace("+00:00", "Z"),
        "run_id": run_id,
        "project_root": str(PROJECT_ROOT),
        "tf_root": str(tf_root),
        "selected_targets": targets,
        "terraform_bin": terraform_bin_path,
        "terraform_version": terraform_version,
        "benchmark_mode": benchmark_mode,
        "image_mode": image_mode,
        "passes": oks,
        "failures": [],
        "terraform_results": tf_results,
    },
)

print(f"\nPreflight artifact: {preflight_path}")
print("\nCell 5 complete: all preflight gates passed.")
print("Next: Run Cell 6.")

In [ ]:
# Cell 6 — Image discovery and resolution validation
# Primary source: OKE node pool options
# Fallback source: Compute image listing (only when needed)
# Default mode: IMAGE_MODE=pin (reproducible image IDs)

import os
import re
import json
from pathlib import Path
from datetime import datetime, timezone

import oci


# -----------------------------
# Helpers
# -----------------------------
def _env(k, d=""):
    return os.environ.get(k, d)


def _require_env(k):
    v = _env(k, "").strip()
    if not v:
        raise RuntimeError(f"Missing required environment variable: {k}")
    return v


def _load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def _safe_write_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def _k8_minor(version: str) -> str:
    m = re.search(r"v?(\d+)\.(\d+)", str(version))
    if not m:
        raise RuntimeError(f"Invalid Kubernetes version: {version}")
    return f"{m.group(1)}.{m.group(2)}"


def _list_all(fn, *args, **kwargs):
    return oci.pagination.list_call_get_all_results(fn, *args, **kwargs).data


def _minor_in_text(minor: str, text: str) -> bool:
    if not minor or not text:
        return False
    return re.search(rf"(^|[^0-9]){re.escape(minor)}([^0-9]|$)", text) is not None


def _normalize_token(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", (s or "").lower())


def _display_has_k8(display_name: str, minor: str) -> bool:
    s = (display_name or "").lower()
    return ("oke" in s) and _minor_in_text(minor, s)


def _looks_like_ocid(v: str, kind: str = "") -> bool:
    if not v:
        return False
    s = v.strip()
    if kind:
        return s.startswith(f"ocid1.{kind}.")
    return s.startswith("ocid1.")


def _parse_kpo_required_shapes(nodepool_text: str):
    m = re.search(
        r"(?ms)key:\s*oci\.oraclecloud\.com/instance-shape.*?values:\s*\[(.*?)\]",
        nodepool_text or "",
    )
    if m:
        raw = m.group(1)
        shapes = []
        for part in raw.split(","):
            s = part.strip().strip('"').strip("'")
            if s:
                shapes.append(s)
        if shapes:
            return shapes

    shapes = re.findall(r"\b(?:VM|BM)\.[A-Za-z0-9_.-]+\b", nodepool_text or "")
    dedup = []
    seen = set()
    for s in shapes:
        if s not in seen:
            seen.add(s)
            dedup.append(s)
    return dedup


def _extract_yaml_scalar(text: str, key: str):
    m = re.search(rf'(?m)^\s*{re.escape(key)}:\s*"?([^"\n#]+)"?\s*$', text or "")
    return m.group(1).strip() if m else ""


def _parse_ocinodeclass_image_config(ocinodeclass_text: str):
    image_id = _extract_yaml_scalar(ocinodeclass_text, "imageId")
    if image_id:
        return {
            "mode": "id",
            "image_id": image_id,
            "os_filter": "",
            "os_version_filter": "",
            "compartment_id": "",
        }

    has_filter = (
        re.search(r"(?m)^\s*imageFilter:\s*$", ocinodeclass_text or "") is not None
    )
    if has_filter:
        return {
            "mode": "filter",
            "image_id": "",
            "os_filter": _extract_yaml_scalar(ocinodeclass_text, "osFilter"),
            "os_version_filter": _extract_yaml_scalar(
                ocinodeclass_text, "osVersionFilter"
            ),
            "compartment_id": _extract_yaml_scalar(ocinodeclass_text, "compartmentId"),
        }

    return {
        "mode": "unknown",
        "image_id": "",
        "os_filter": "",
        "os_version_filter": "",
        "compartment_id": "",
    }


def _collect_worker_pool_usages(stack_name: str, bm: dict):
    usages = []
    pools = bm.get("worker_pools", {})
    for pool_name, pool_cfg in pools.items():
        if not isinstance(pool_cfg, dict):
            continue
        image_id = (pool_cfg.get("image_id") or "").strip()
        shape = (pool_cfg.get("shape") or "").strip()
        if image_id and shape:
            usages.append(
                {
                    "stack": stack_name,
                    "pool": pool_name,
                    "shape": shape,
                    "required_shapes": [shape],
                    "image_id": image_id,
                    "autoscale": bool(pool_cfg.get("autoscale", False)),
                    "source": "worker_pool",
                }
            )
    return usages


def _policy_allows_read_instance_images(policy_text: str, compartment_id: str):
    text = policy_text or ""
    if re.search(
        r"Allow\s+any-user\s+to\s+read\s+instance-images\s+in\s+tenancy",
        text,
        flags=re.IGNORECASE,
    ):
        return True
    if compartment_id:
        return (
            re.search(
                rf"Allow\s+any-user\s+to\s+read\s+instance-images\s+in\s+compartment\s+id\s+{re.escape(compartment_id)}\b",
                text,
                flags=re.IGNORECASE,
            )
            is not None
        )
    return False


def _has_oke_derivation_evidence(
    img_obj, k8_version: str, k8_minor: str, get_image_obj
):
    display_name = getattr(img_obj, "display_name", "") or ""
    freeform_tags = getattr(img_obj, "freeform_tags", {}) or {}
    base_image_id = getattr(img_obj, "base_image_id", "") or ""

    evidences = []

    tag_k8 = str(freeform_tags.get("k8s_version", "")).strip()
    if tag_k8 in {k8_version, k8_minor} or _minor_in_text(k8_minor, tag_k8):
        evidences.append("freeform_tags.k8s_version")

    if _display_has_k8(display_name, k8_minor):
        evidences.append("display_name")

    if base_image_id:
        try:
            parent = get_image_obj(base_image_id)
            parent_name = getattr(parent, "display_name", "") or ""
            if _display_has_k8(parent_name, k8_minor):
                evidences.append("base_image.display_name")
        except Exception:
            pass

    return (len(evidences) > 0), evidences


# -----------------------------
# Context (SSOT)
# -----------------------------
PROJECT_ROOT = Path(_require_env("PROJECT_ROOT")).expanduser().resolve()
TF_STACK_ROOT = Path(_require_env("TF_STACK_ROOT")).expanduser().resolve()
MODULE_SOURCE = (
    Path(_require_env("TERRAFORM_OCI_OKE_MODULE_SOURCE")).expanduser().resolve()
)

target_cluster = _env("TARGET_CLUSTER", "both").strip().lower()
if target_cluster not in {"ca", "kpo", "both"}:
    raise RuntimeError(f"TARGET_CLUSTER must be ca|kpo|both, got: {target_cluster}")
selected_targets = ["ca", "kpo"] if target_cluster == "both" else [target_cluster]

image_mode = _env("IMAGE_MODE", "pin").strip().lower()
if image_mode not in {"pin", "filter"}:
    raise RuntimeError(f"IMAGE_MODE must be pin|filter, got: {image_mode}")
benchmark_mode = _env("BENCHMARK_MODE", "parity").strip().lower()
if benchmark_mode not in {"parity", "native"}:
    raise RuntimeError(f"BENCHMARK_MODE must be parity|native, got: {benchmark_mode}")

k8_version = _require_env("KUBERNETES_VERSION")
k8_minor = _k8_minor(k8_version)

region = _require_env("BENCHMARK_REGION")
profile = _require_env("OCI_PROFILE")
config_file = _require_env("OCI_CONFIG_FILE")
run_id = _require_env("RUN_ID")
output_dir = Path(_require_env("OUTPUT_DIR")).expanduser().resolve()

print("=== Cell 6 Image Discovery ===")
print(f"Project root: {PROJECT_ROOT}")
print(f"TF root: {TF_STACK_ROOT}")
print(f"Module source: {MODULE_SOURCE}")
print(f"Selected target(s): {selected_targets}")
print(f"IMAGE_MODE: {image_mode}")
print(f"BENCHMARK_MODE: {benchmark_mode}")
print(f"Kubernetes: {k8_version} ({k8_minor})")
print(f"Region: {region}")

if not MODULE_SOURCE.exists() or not MODULE_SOURCE.is_dir():
    raise RuntimeError(f"Invalid TERRAFORM_OCI_OKE_MODULE_SOURCE: {MODULE_SOURCE}")
for marker in ["versions.tf", "module-cluster.tf", "module-workers.tf", "modules"]:
    if not (MODULE_SOURCE / marker).exists():
        raise RuntimeError(
            "TERRAFORM_OCI_OKE_MODULE_SOURCE is not a valid terraform-oci-oke-main root. "
            f"Missing: {marker} in {MODULE_SOURCE}"
        )

gen_summary_path = TF_STACK_ROOT / "generation-summary.json"
if not gen_summary_path.exists():
    raise RuntimeError(f"Missing required file: {gen_summary_path}. Run Cell 4 first.")
generation_summary = _load_json(gen_summary_path)
if generation_summary.get("tf_root") and str(TF_STACK_ROOT) != str(
    generation_summary.get("tf_root")
):
    raise RuntimeError(
        "TF_STACK_ROOT mismatch with tf/generation-summary.json. "
        f"TF_STACK_ROOT={TF_STACK_ROOT}, summary.tf_root={generation_summary.get('tf_root')}"
    )
if (
    generation_summary.get("benchmark_mode")
    and generation_summary.get("benchmark_mode") != benchmark_mode
):
    raise RuntimeError(
        "BENCHMARK_MODE mismatch with tf/generation-summary.json. "
        f"BENCHMARK_MODE={benchmark_mode}, summary.benchmark_mode={generation_summary.get('benchmark_mode')}"
    )
if (
    generation_summary.get("image_mode")
    and generation_summary.get("image_mode") != image_mode
):
    raise RuntimeError(
        "IMAGE_MODE mismatch with tf/generation-summary.json. "
        f"IMAGE_MODE={image_mode}, summary.image_mode={generation_summary.get('image_mode')}"
    )

ca_dir = TF_STACK_ROOT / "ca"
kpo_dir = TF_STACK_ROOT / "kpo"
ca_tfvars = ca_dir / "terraform.auto.tfvars.json"
kpo_tfvars = kpo_dir / "terraform.auto.tfvars.json"

if "ca" in selected_targets and not ca_tfvars.exists():
    raise RuntimeError(
        f"Missing generated tfvars for target ca: {ca_tfvars}. Run Cell 4 first."
    )
if "kpo" in selected_targets and not kpo_tfvars.exists():
    raise RuntimeError(
        f"Missing generated tfvars for target kpo: {kpo_tfvars}. Run Cell 4 first."
    )

kpo_nodepool_tmpl = kpo_dir / "manifests" / "nodepool.yaml.tmpl"
kpo_ocinodeclass_tmpl = kpo_dir / "manifests" / "ocinodeclass.yaml.tmpl"
kpo_wi_policy_tmpl = kpo_dir / "iam" / "kpo_workload_identity.policy.tmpl"
if "kpo" in selected_targets:
    for p in [kpo_nodepool_tmpl, kpo_ocinodeclass_tmpl, kpo_wi_policy_tmpl]:
        if not p.exists():
            raise RuntimeError(f"Missing required KPO artifact for Cell 6: {p}")

os.environ["OCI_CLI_CONFIG_FILE"] = config_file
os.environ["OCI_CONFIG_FILE_PROFILE"] = profile
cfg = oci.config.from_file(file_location=config_file, profile_name=profile)
cfg["region"] = region

ce = oci.container_engine.ContainerEngineClient(cfg)
compute = oci.core.ComputeClient(cfg)

benchmarks = {}
if "ca" in selected_targets:
    benchmarks["ca"] = _load_json(ca_tfvars).get("benchmark", {})
if "kpo" in selected_targets:
    benchmarks["kpo"] = _load_json(kpo_tfvars).get("benchmark", {})

usages = []
for stack_name, bm in benchmarks.items():
    usages.extend(_collect_worker_pool_usages(stack_name, bm))

errors = []
warnings = []

kpo_shape_requirements = []
kpo_nodeclass_cfg = {}
if "kpo" in selected_targets:
    nodepool_text = kpo_nodepool_tmpl.read_text(encoding="utf-8")
    ocinodeclass_text = kpo_ocinodeclass_tmpl.read_text(encoding="utf-8")
    kpo_shape_requirements = _parse_kpo_required_shapes(nodepool_text)
    if not kpo_shape_requirements:
        errors.append("kpo: unable to resolve required shapes from nodepool.yaml.tmpl.")
    kpo_nodeclass_cfg = _parse_ocinodeclass_image_config(ocinodeclass_text)

    if image_mode == "pin":
        if kpo_nodeclass_cfg.get("mode") != "id":
            errors.append(
                "kpo: IMAGE_MODE=pin requires OCINodeClass imageId path, but imageId is not present."
            )
        else:
            image_id = (kpo_nodeclass_cfg.get("image_id") or "").strip()
            if not _looks_like_ocid(image_id, "image"):
                errors.append(
                    f"kpo: invalid OCINodeClass imageId for pin mode: {image_id or '<empty>'}"
                )
            elif kpo_shape_requirements:
                usages.append(
                    {
                        "stack": "kpo",
                        "pool": "benchmark-nodepool",
                        "shape": kpo_shape_requirements[0],
                        "required_shapes": list(kpo_shape_requirements),
                        "image_id": image_id,
                        "autoscale": True,
                        "source": "ocinodeclass",
                    }
                )
    else:
        if kpo_nodeclass_cfg.get("mode") != "filter":
            errors.append(
                "kpo: IMAGE_MODE=filter requires OCINodeClass imageFilter path, but imageFilter is not present."
            )
        os_filter = (kpo_nodeclass_cfg.get("os_filter") or "").strip()
        if "ubuntu" in os_filter.lower():
            errors.append(
                "kpo: IMAGE_MODE=filter with Ubuntu is not supported; use pinned imageId path."
            )

if not usages and not ("kpo" in selected_targets and image_mode == "filter"):
    raise RuntimeError("No image usages found in selected targets.")

# -----------------------------
# Primary source: OKE node pool options
# -----------------------------
oke_sources_by_id = {}
try:
    np_opts = ce.get_node_pool_options(node_pool_option_id="all").data
    for src in list(getattr(np_opts, "sources", []) or []):
        image_id = getattr(src, "image_id", None)
        if not image_id:
            continue
        oke_sources_by_id[image_id] = {
            "source_name": getattr(src, "source_name", "") or "",
            "source_type": getattr(src, "source_type", "") or "",
            "image_id": image_id,
        }
except Exception as e:
    raise RuntimeError(
        f"Failed reading OKE node pool options (primary image source): {e}"
    )

# -----------------------------
# Cached OCI lookups
# -----------------------------
image_cache = {}
compat_cache = {}


def _get_image_obj(image_id: str):
    if image_id in image_cache:
        return image_cache[image_id]
    obj = compute.get_image(image_id).data
    image_cache[image_id] = obj
    return obj


def _get_compat_shapes(image_id: str):
    if image_id in compat_cache:
        return compat_cache[image_id]
    entries = _list_all(
        compute.list_image_shape_compatibility_entries, image_id=image_id
    )
    shapes = {getattr(x, "shape", "") for x in entries if getattr(x, "shape", "")}
    compat_cache[image_id] = shapes
    return shapes


def _list_images_filtered(compartment_id: str, os_filter: str, os_version: str):
    kwargs = {"compartment_id": compartment_id, "lifecycle_state": "AVAILABLE"}
    if os_filter:
        kwargs["operating_system"] = os_filter
    if os_version:
        kwargs["operating_system_version"] = os_version
    try:
        return _list_all(compute.list_images, **kwargs)
    except TypeError:
        kwargs.pop("operating_system", None)
        kwargs.pop("operating_system_version", None)
        return _list_all(compute.list_images, **kwargs)


def _os_filter_match(os_filter: str, source_name: str, image_os_name: str):
    if not os_filter:
        return True
    want = _normalize_token(os_filter)
    return (want in _normalize_token(source_name)) or (
        want in _normalize_token(image_os_name)
    )


def _os_version_match(os_version_filter: str, source_name: str, image_os_version: str):
    if not os_version_filter:
        return True
    v = str(os_version_filter).strip()
    return (v in (source_name or "")) or str(image_os_version or "").startswith(v)


# -----------------------------
# Validate explicit image IDs
# -----------------------------
usage_by_image = {}
for u in usages:
    usage_by_image.setdefault(u["image_id"], []).append(u)

results = []

for image_id, image_usages in usage_by_image.items():
    discovered_in_oke_source = image_id in oke_sources_by_id
    source_name = oke_sources_by_id.get(image_id, {}).get("source_name", "")
    minor_match_primary = _minor_in_text(k8_minor, source_name)

    try:
        img = _get_image_obj(image_id)
    except Exception as e:
        errors.append(f"{image_id}: compute.get_image failed: {e}")
        continue

    display_name = getattr(img, "display_name", "") or ""
    base_image_id = getattr(img, "base_image_id", "") or ""
    freeform_tags = getattr(img, "freeform_tags", {}) or {}
    os_name = getattr(img, "operating_system", "") or ""
    os_version = getattr(img, "operating_system_version", "") or ""

    try:
        compat_shapes = _get_compat_shapes(image_id)
    except Exception as e:
        compat_shapes = set()
        errors.append(f"{image_id}: failed to read image shape compatibility: {e}")

    required_shapes = []
    for u in image_usages:
        for s in u.get("required_shapes") or [u.get("shape")]:
            if s and s not in required_shapes:
                required_shapes.append(s)

    for shape in required_shapes:
        if shape not in compat_shapes:
            errors.append(
                f"{image_id}: shape incompatibility; required shape {shape} not in compatibility list."
            )

    has_kpo_usage = any(u.get("stack") == "kpo" for u in image_usages)
    evidence = []
    if has_kpo_usage and not discovered_in_oke_source:
        ok, evidence = _has_oke_derivation_evidence(
            img, k8_version, k8_minor, _get_image_obj
        )
        if not ok:
            errors.append(
                f"{image_id}: KPO image is not in OKE node pool sources and has no k8s_version/base-image evidence for Kubernetes {k8_minor}."
            )

    if image_mode == "filter" and has_kpo_usage:
        is_ubuntu = ("ubuntu" in display_name.lower()) or ("ubuntu" in os_name.lower())
        if is_ubuntu:
            errors.append(
                f"{image_id}: IMAGE_MODE=filter with Ubuntu for KPO is not supported; use pinned imageId path."
            )

    results.append(
        {
            "image_id": image_id,
            "discovered_in_oke_nodepool_options": discovered_in_oke_source,
            "primary_source_name": source_name,
            "k8_minor_match_in_primary_source": minor_match_primary,
            "display_name": display_name,
            "operating_system": os_name,
            "operating_system_version": os_version,
            "base_image_id": base_image_id,
            "freeform_tags": freeform_tags,
            "required_shapes": required_shapes,
            "shape_compatibility_count": len(compat_shapes),
            "oke_derivation_evidence": evidence,
            "usages": image_usages,
        }
    )

# -----------------------------
# KPO filter-mode candidate resolution
# -----------------------------
kpo_filter_resolution = {
    "applies": False,
    "source": "",
    "required_shapes": [],
    "os_filter": "",
    "os_version_filter": "",
    "compartment_id": "",
    "compartment_scope_mode": "",
    "default_resolution_path": None,
    "candidate_image_ids": [],
    "candidate_count": 0,
    "fallback_used": False,
    "policy_check_passed": None,
    "policy_check_reason": "",
}

if "kpo" in selected_targets and image_mode == "filter":
    kpo_filter_resolution["applies"] = True
    kpo_filter_resolution["required_shapes"] = list(kpo_shape_requirements)

    os_filter = (kpo_nodeclass_cfg.get("os_filter") or "").strip()
    os_version_filter = (kpo_nodeclass_cfg.get("os_version_filter") or "").strip()

    nodeclass_compartment_id = (kpo_nodeclass_cfg.get("compartment_id") or "").strip()
    env_compartment_id = _env("KPO_IMAGE_FILTER_COMPARTMENT_ID", "").strip()
    compartment_id = nodeclass_compartment_id or env_compartment_id
    explicit_compartment_scope = bool(compartment_id)

    kpo_filter_resolution["os_filter"] = os_filter
    kpo_filter_resolution["os_version_filter"] = os_version_filter
    kpo_filter_resolution["compartment_id"] = compartment_id
    kpo_filter_resolution["compartment_scope_mode"] = (
        "explicit" if explicit_compartment_scope else "default"
    )
    kpo_filter_resolution["default_resolution_path"] = not explicit_compartment_scope

    if explicit_compartment_scope:
        policy_text = kpo_wi_policy_tmpl.read_text(encoding="utf-8")
        policy_ok = _policy_allows_read_instance_images(policy_text, compartment_id)
        kpo_filter_resolution["policy_check_passed"] = policy_ok
        kpo_filter_resolution["policy_check_reason"] = "required-explicit-compartment"
        if not policy_ok:
            errors.append(
                "kpo: imageFilter uses explicit compartment-scoped images but workload identity policy template "
                "does not grant 'read instance-images' for the selected compartment."
            )
    else:
        kpo_filter_resolution["policy_check_passed"] = True
        kpo_filter_resolution["policy_check_reason"] = "not-required-default-resolution"

    # Primary candidate source: OKE node pool options
    primary_candidates = []
    for iid, src in oke_sources_by_id.items():
        sname = src.get("source_name", "")
        if k8_minor and not _minor_in_text(k8_minor, sname):
            continue

        try:
            img_obj = _get_image_obj(iid)
            img_os_name = getattr(img_obj, "operating_system", "") or ""
            img_os_version = getattr(img_obj, "operating_system_version", "") or ""
        except Exception:
            continue

        if not _os_filter_match(os_filter, sname, img_os_name):
            continue
        if not _os_version_match(os_version_filter, sname, img_os_version):
            continue
        if "ubuntu" in (img_os_name or "").lower():
            continue

        try:
            compat_shapes = _get_compat_shapes(iid)
        except Exception:
            continue
        if any(s not in compat_shapes for s in kpo_shape_requirements):
            continue

        primary_candidates.append(iid)

    primary_candidates = sorted(set(primary_candidates))
    candidates = list(primary_candidates)

    if candidates:
        kpo_filter_resolution["source"] = "oke-node-pool-options"
        kpo_filter_resolution["fallback_used"] = False
        if explicit_compartment_scope:
            warnings.append(
                "kpo: filter candidates were derived from OKE node pool options; runtime selection "
                "is still subject to NodeClass compartment constraints."
            )
        else:
            warnings.append(
                "kpo: filter candidates were derived from OKE node pool options using default image resolution path "
                "(no explicit imageFilter compartmentId)."
            )
    elif explicit_compartment_scope:
        kpo_filter_resolution["source"] = "compute-list-images"
        kpo_filter_resolution["fallback_used"] = True
        try:
            listed = _list_images_filtered(compartment_id, os_filter, os_version_filter)
        except Exception as e:
            listed = []
            errors.append(
                f"kpo: failed listing images in compartment {compartment_id} for filter mode: {e}"
            )

        fallback_candidates = []
        for img in listed:
            iid = getattr(img, "id", "") or ""
            if not iid:
                continue
            try:
                compat_shapes = _get_compat_shapes(iid)
            except Exception:
                continue
            if any(s not in compat_shapes for s in kpo_shape_requirements):
                continue

            ok, _ = _has_oke_derivation_evidence(
                img, k8_version, k8_minor, _get_image_obj
            )
            if not ok:
                continue

            display_name = getattr(img, "display_name", "") or ""
            os_name = getattr(img, "operating_system", "") or ""
            if "ubuntu" in display_name.lower() or "ubuntu" in os_name.lower():
                continue

            fallback_candidates.append(iid)

        candidates = sorted(set(fallback_candidates))
    else:
        kpo_filter_resolution["source"] = "oke-node-pool-options"
        kpo_filter_resolution["fallback_used"] = False
        errors.append(
            "kpo: IMAGE_MODE=filter resolved zero primary candidates from OKE node pool options. "
            "Compute fallback was skipped because no explicit imageFilter compartmentId was provided. "
            "Set KPO_IMAGE_FILTER_COMPARTMENT_ID (or NodeClass imageFilter.compartmentId) to enable compartment-scoped fallback validation."
        )

    kpo_filter_resolution["candidate_image_ids"] = candidates[:200]
    kpo_filter_resolution["candidate_count"] = len(candidates)

    if not candidates:
        errors.append(
            "kpo: IMAGE_MODE=filter resolved zero candidate images after applying "
            "Kubernetes/shape/OS/derivation filters."
        )
    else:
        os.environ["KPO_FILTER_CANDIDATE_IMAGE_COUNT"] = str(len(candidates))
        if len(candidates) == 1:
            os.environ["RESOLVED_KPO_BENCHMARK_NODEPOOL_IMAGE_ID"] = candidates[0]
        else:
            warnings.append(
                f"kpo: filter mode resolved {len(candidates)} candidate images (drift mode)."
            )

# -----------------------------
# Persist resolved image state
# -----------------------------
resolved_env_keys = []
for u in usages:
    env_key = (
        f"RESOLVED_{u['stack'].upper()}_{u['pool'].upper().replace('-', '_')}_IMAGE_ID"
    )
    os.environ[env_key] = u["image_id"]
    resolved_env_keys.append(env_key)

resolution_payload = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
    "run_id": run_id,
    "region": region,
    "kubernetes_version": k8_version,
    "k8_minor": k8_minor,
    "target_cluster": target_cluster,
    "selected_targets": selected_targets,
    "benchmark_mode": benchmark_mode,
    "image_mode": image_mode,
    "drift_mode": image_mode == "filter",
    "project_root": str(PROJECT_ROOT),
    "tf_root": str(TF_STACK_ROOT),
    "module_source": str(MODULE_SOURCE),
    "oke_primary_source_count": len(oke_sources_by_id),
    "validated_image_count": len(results),
    "resolved_image_env_keys": sorted(set(resolved_env_keys)),
    "kpo_filter_resolution": kpo_filter_resolution,
    "results": results,
    "warnings": warnings,
    "errors": errors,
}

artifact_dir = output_dir / run_id
artifact_dir.mkdir(parents=True, exist_ok=True)
resolution_path = artifact_dir / "image_selection.json"
_safe_write_json(resolution_path, resolution_payload)

print("\nCell 6 image discovery summary:")
print(f"- OKE primary source images discovered: {len(oke_sources_by_id)}")
print(f"- Explicit image IDs validated: {len(results)}")
print(f"- Artifact: {resolution_path}")
if image_mode == "pin":
    print("- IMAGE_MODE=pin: explicit image IDs retained for reproducibility.")
else:
    print("- IMAGE_MODE=filter: drift-mode validation path executed.")

if kpo_filter_resolution.get("applies"):
    print(
        f"- KPO filter candidates: {kpo_filter_resolution.get('candidate_count', 0)} "
        f"(source={kpo_filter_resolution.get('source')}, fallback_used={kpo_filter_resolution.get('fallback_used')})"
    )

if warnings:
    print("\nImage validation warnings:")
    for w in warnings:
        print(f"- {w}")

if errors:
    print("\nImage validation issues:")
    for e in errors:
        print(f"- {e}")
    raise RuntimeError(f"Cell 6 failed with {len(errors)} image validation issue(s).")

print("\nCell 6 complete: image discovery/selection validation passed.")
print("Next: Run Cell 7.")

In [ ]:
# Cell 7 — Provision selected target(s): resilient terraform apply + output capture + kubeconfig refs
#
# Reliability goals:
# - No dependency on stale persisted tfplan artifacts
# - Fresh per-target plan/apply with retry/backoff for transient failures
# - Resume-aware behavior: preserve successful targets and retry only failed targets
# - Rich per-target diagnostics persisted to artifact files

import os
import re
import json
import time
import shutil
import subprocess
from pathlib import Path
from datetime import datetime, timezone


def _env(k, d=""):
    return os.environ.get(k, d)


def _require_env(k):
    v = _env(k, "").strip()
    if not v:
        raise RuntimeError(f"Missing required env var: {k}")
    return v


def _bool(x):
    return str(x).strip().lower() in {"1", "true", "yes", "y", "on"}


def _to_int(v, d=None):
    try:
        return int(str(v).strip())
    except Exception:
        return d


def _to_float(v, d=None):
    try:
        return float(str(v).strip())
    except Exception:
        return d


def _load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def _write_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def _tail(text: str, max_lines: int = 120):
    lines = (text or "").splitlines()
    if len(lines) <= max_lines:
        return "\n".join(lines)
    return "\n".join(lines[-max_lines:])


def _run_capture(cmd, cwd=None, env=None):
    print(f"\n$ {' '.join(cmd)}")
    p = subprocess.run(cmd, cwd=cwd, env=env, text=True, capture_output=True)
    out = p.stdout or ""
    err = p.stderr or ""
    if out:
        print(out)
    if err:
        print(err)
    return p.returncode, out, err


def _run_with_retries(
    cmd,
    cwd=None,
    env=None,
    retries=1,
    retry_delay_seconds=5.0,
    retry_if=None,
    success_rcs=None,
):
    allowed = {0} if success_rcs is None else {int(x) for x in success_rcs}
    attempts = max(1, int(retries))
    last = (1, "", "", 1)
    for attempt in range(1, attempts + 1):
        rc, out, err = _run_capture(cmd, cwd=cwd, env=env)
        last = (rc, out, err, attempt)
        if rc in allowed:
            return last

        combined = f"{out}\n{err}"
        should_retry = bool(retry_if and retry_if(combined)) and attempt < attempts
        if should_retry:
            print(
                f"Retrying in {retry_delay_seconds}s (attempt {attempt + 1}/{attempts}) due to transient error..."
            )
            time.sleep(max(0.0, float(retry_delay_seconds)))
            continue
        return last
    return last


def _is_transient_tf_error(text):
    t = str(text or "").lower()
    markers = [
        "timed out",
        "i/o timeout",
        "tls handshake timeout",
        "context deadline exceeded",
        "temporary failure",
        "temporarily unavailable",
        "connection reset",
        "connection refused",
        "service unavailable",
        "too many requests",
        "request was throttled",
        "throttl",
        "eof",
        "http response status code 429",
        "http response status code 500",
        "http response status code 502",
        "http response status code 503",
        "http response status code 504",
        "failed to acquire lock",
    ]
    return any(m in t for m in markers)


def _is_transient_oci_cli_error(text):
    t = str(text or "").lower()
    markers = [
        "too many requests",
        "request was throttled",
        "throttl",
        "service unavailable",
        "internal server error",
        "timed out",
        "i/o timeout",
        "connection reset",
        "connection refused",
        "context deadline exceeded",
        "eof",
    ]
    return any(m in t for m in markers)


def _run_json_with_retries(
    cmd,
    cwd=None,
    env=None,
    retries=1,
    retry_delay_seconds=5.0,
    retry_if=None,
):
    rc, out, err, attempts_used = _run_with_retries(
        cmd,
        cwd=cwd,
        env=env,
        retries=retries,
        retry_delay_seconds=retry_delay_seconds,
        retry_if=retry_if,
        success_rcs={0},
    )
    if rc != 0:
        return rc, None, out, err, attempts_used
    try:
        return rc, json.loads(out or "{}"), out, err, attempts_used
    except Exception as e:
        return 1, None, out, f"{err}\nJSON parse error: {e}", attempts_used


def _tf_out_val(tf_output_obj, name, default=None):
    x = tf_output_obj.get(name, {})
    if isinstance(x, dict) and "value" in x:
        return x["value"]
    return default


def _normalize_endpoint(endpoint: str) -> str:
    if not endpoint:
        return ""
    e = endpoint.strip()
    if not e:
        return ""
    if not e.startswith("http://") and not e.startswith("https://"):
        e = f"https://{e}"
    return e


def _extract_endpoint(cluster_endpoints, apiserver_private_host, prefer="private"):
    endpoint = ""

    if isinstance(cluster_endpoints, dict) and prefer == "private":
        for k in ("private_endpoint", "privateEndpoint", "private", "endpoint_private"):
            v = cluster_endpoints.get(k)
            if isinstance(v, str) and v.strip():
                endpoint = v.strip()
                break

    if isinstance(cluster_endpoints, dict) and prefer == "public":
        for k in ("public_endpoint", "publicEndpoint", "public", "endpoint_public"):
            v = cluster_endpoints.get(k)
            if isinstance(v, str) and v.strip():
                endpoint = v.strip()
                break

    if (
        not endpoint
        and prefer == "private"
        and isinstance(apiserver_private_host, str)
        and apiserver_private_host.strip()
    ):
        host = apiserver_private_host.strip()
        if host.startswith("http://") or host.startswith("https://"):
            endpoint = host
        else:
            endpoint = f"https://{host}:6443"

    return _normalize_endpoint(endpoint)


def _kubeconfig_server(kubeconfig_path: Path) -> str:
    text = kubeconfig_path.read_text(encoding="utf-8")
    for ln in text.splitlines():
        m = re.match(r"^\s*server:\s*(\S+)\s*$", ln)
        if m:
            return m.group(1).strip()
    return ""


def _module_markers_ok(module_root: Path):
    markers = ["versions.tf", "module-cluster.tf", "module-workers.tf", "modules"]
    missing = [m for m in markers if not (module_root / m).exists()]
    return (len(missing) == 0), missing


def _collect_image_usages(image_selection: dict):
    out = {}
    for r in image_selection.get("results", []) or []:
        iid = (r.get("image_id") or "").strip()
        if not iid:
            continue
        for u in r.get("usages", []) or []:
            stack = (u.get("stack") or "").strip()
            pool = (u.get("pool") or "").strip()
            if not stack or not pool:
                continue
            out.setdefault(stack, {})[pool] = iid
    return out


def _cmd_failure(step, cmd, rc, out, err):
    combined = f"{out}\n{err}".strip()
    return {
        "step": step,
        "command": cmd,
        "rc": rc,
        "message": f"{step} failed (exit {rc})",
        "stdout_tail": _tail(out),
        "stderr_tail": _tail(err),
        "combined_tail": _tail(combined),
    }


def _export_target_env(target: str, record: dict):
    t = target.upper()

    def _set(name, value):
        if value is None:
            value = ""
        os.environ[name] = str(value)

    _set(f"{t}_CLUSTER_ID", record.get("cluster_id", ""))
    _set(f"{t}_KUBECONFIG", record.get("kubeconfig_path", ""))
    _set(f"{t}_KUBECONFIG_SERVER", record.get("kubeconfig_server", ""))
    _set(f"{t}_APISERVER_ENDPOINT", record.get("apiserver_endpoint", ""))
    _set(
        f"{t}_APISERVER_PRIVATE_ENDPOINT", record.get("apiserver_private_endpoint", "")
    )
    _set(f"{t}_APISERVER_PUBLIC_ENDPOINT", record.get("apiserver_public_endpoint", ""))
    _set(f"{t}_WORKER_SUBNET_ID", record.get("worker_subnet_id", ""))
    _set(f"{t}_POD_SUBNET_ID", record.get("pod_subnet_id", ""))
    _set(f"{t}_VCN_ID", record.get("vcn_id", ""))


# ---------------------------
# Context
# ---------------------------
PROJECT_ROOT = Path(_require_env("PROJECT_ROOT")).expanduser().resolve()
TF_STACK_ROOT = Path(_require_env("TF_STACK_ROOT")).expanduser().resolve()
MODULE_SOURCE = (
    Path(_require_env("TERRAFORM_OCI_OKE_MODULE_SOURCE")).expanduser().resolve()
)

ok_module, missing_markers = _module_markers_ok(MODULE_SOURCE)
if not ok_module:
    raise RuntimeError(
        "TERRAFORM_OCI_OKE_MODULE_SOURCE is not a valid terraform-oci-oke-main root. "
        f"Missing: {missing_markers} in {MODULE_SOURCE}"
    )

gen_summary_path = TF_STACK_ROOT / "generation-summary.json"
if not gen_summary_path.exists():
    raise RuntimeError(f"Missing required file: {gen_summary_path}. Run Cell 4 first.")
generation_summary = _load_json(gen_summary_path)
if generation_summary.get("tf_root") and str(TF_STACK_ROOT) != str(
    generation_summary.get("tf_root")
):
    raise RuntimeError(
        "TF_STACK_ROOT mismatch with tf/generation-summary.json. "
        f"TF_STACK_ROOT={TF_STACK_ROOT}, summary.tf_root={generation_summary.get('tf_root')}"
    )

target_cluster = _env("TARGET_CLUSTER", "both").strip().lower()
if target_cluster not in {"ca", "kpo", "both"}:
    raise RuntimeError(f"TARGET_CLUSTER must be ca|kpo|both, got: {target_cluster}")
selected_targets = ["ca", "kpo"] if target_cluster == "both" else [target_cluster]

run_id = _require_env("RUN_ID")
output_dir = Path(_require_env("OUTPUT_DIR")).expanduser().resolve()
artifact_dir = output_dir / run_id
artifact_dir.mkdir(parents=True, exist_ok=True)
provision_dir = artifact_dir / "provision"
provision_dir.mkdir(parents=True, exist_ok=True)
kubeconfig_dir = artifact_dir / "kubeconfigs"
kubeconfig_dir.mkdir(parents=True, exist_ok=True)

region = _require_env("BENCHMARK_REGION")
oci_profile = _require_env("OCI_PROFILE")
oci_config_file = _require_env("OCI_CONFIG_FILE")
benchmark_mode = _env("BENCHMARK_MODE", "").strip().lower()
image_mode = _env("IMAGE_MODE", "pin").strip().lower() or "pin"

if (
    generation_summary.get("image_mode")
    and generation_summary.get("image_mode") != image_mode
):
    raise RuntimeError(
        "IMAGE_MODE mismatch with generation-summary.json. "
        f"current={image_mode}, generation_summary={generation_summary.get('image_mode')}. "
        "Rerun Cell 4+ with consistent settings."
    )
if (
    benchmark_mode
    and generation_summary.get("benchmark_mode")
    and generation_summary.get("benchmark_mode") != benchmark_mode
):
    raise RuntimeError(
        "BENCHMARK_MODE mismatch with generation-summary.json. "
        f"current={benchmark_mode}, generation_summary={generation_summary.get('benchmark_mode')}. "
        "Rerun Cell 4+ with consistent settings."
    )

kubeconfig_endpoint_mode = _env("KUBECONFIG_ENDPOINT_MODE", "auto").strip().lower()
if kubeconfig_endpoint_mode not in {"auto", "public", "private"}:
    raise RuntimeError(
        f"KUBECONFIG_ENDPOINT_MODE must be auto|public|private, got: {kubeconfig_endpoint_mode}"
    )

resume_enabled = _bool(_env("PROVISION_RESUME", "true"))

terraform_bin = _env("TERRAFORM_BIN", "terraform").strip() or "terraform"
terraform_exec = shutil.which(terraform_bin)
if not terraform_exec:
    raise RuntimeError(
        f"Terraform binary not found in PATH: {terraform_bin}. "
        "Set TERRAFORM_BIN to an installed terraform binary path/name."
    )

oci_exec = shutil.which("oci")
if not oci_exec:
    raise RuntimeError("OCI CLI binary not found in PATH: oci")

cmd_env = os.environ.copy()
cmd_env["OCI_CLI_CONFIG_FILE"] = oci_config_file
cmd_env["OCI_CONFIG_FILE_PROFILE"] = oci_profile
cmd_env["TF_VAR_config_file_profile"] = oci_profile
cmd_env["TF_VAR_oci_config_file"] = oci_config_file

image_selection_path = artifact_dir / "image_selection.json"
if not image_selection_path.exists():
    raise RuntimeError(
        f"Missing image selection artifact from Cell 6: {image_selection_path}"
    )
image_selection = _load_json(image_selection_path)
if image_selection.get("errors"):
    raise RuntimeError(
        "Cell 6 image_selection.json contains errors; rerun/fix Cell 6 before provisioning. "
        f"errors={len(image_selection.get('errors', []))}"
    )

warnings = []
image_targets = image_selection.get("selected_targets") or []
if image_targets:
    if not set(selected_targets).issubset(set(image_targets)):
        raise RuntimeError(
            "Cell 6 target set does not cover current TARGET_CLUSTER. "
            f"image_selection.selected_targets={image_targets}, current={selected_targets}. "
            "Rerun Cell 6 after changing target selection."
        )
    if sorted(set(image_targets)) != sorted(set(selected_targets)):
        warnings.append(
            "Cell 6 selected targets differ from current run; proceeding because current targets are a subset."
        )

tf_init_upgrade = _bool(_env("TF_INIT_UPGRADE", "false"))
tf_init_retries = max(1, _to_int(_env("TF_INIT_RETRIES", "2"), 2) or 2)
tf_init_retry_seconds = _to_float(_env("TF_INIT_RETRY_SECONDS", "8"), 8.0) or 8.0
tf_plan_retries = max(1, _to_int(_env("TF_PLAN_RETRIES", "2"), 2) or 2)
tf_plan_retry_seconds = _to_float(_env("TF_PLAN_RETRY_SECONDS", "8"), 8.0) or 8.0
tf_apply_retries = max(1, _to_int(_env("TF_APPLY_RETRIES", "2"), 2) or 2)
tf_apply_retry_seconds = _to_float(_env("TF_APPLY_RETRY_SECONDS", "12"), 12.0) or 12.0
oci_kubeconfig_retries = max(1, _to_int(_env("OCI_KUBECONFIG_RETRIES", "2"), 2) or 2)
oci_kubeconfig_retry_seconds = (
    _to_float(_env("OCI_KUBECONFIG_RETRY_SECONDS", "8"), 8.0) or 8.0
)

print("=== Cell 7 Provision ===")
print(f"Project root: {PROJECT_ROOT}")
print(f"TF root: {TF_STACK_ROOT}")
print(f"Module source: {MODULE_SOURCE}")
print(f"Selected target(s): {selected_targets}")
print(f"Terraform bin: {terraform_exec}")
print(f"Run artifact dir: {artifact_dir}")
print(f"Resume enabled: {str(resume_enabled).lower()}")
print(f"Terraform init -upgrade enabled: {str(tf_init_upgrade).lower()}")
print(
    f"Retry policy: init={tf_init_retries}/{tf_init_retry_seconds}s, plan={tf_plan_retries}/{tf_plan_retry_seconds}s, apply={tf_apply_retries}/{tf_apply_retry_seconds}s, kubeconfig={oci_kubeconfig_retries}/{oci_kubeconfig_retry_seconds}s"
)

provision_path = artifact_dir / "provisioning.json"
existing_summary = {}
existing_targets = {}
existing_capacity = {}
if provision_path.exists():
    try:
        existing_summary = _load_json(provision_path)
        existing_targets = (
            existing_summary.get("targets", {})
            if isinstance(existing_summary.get("targets", {}), dict)
            else {}
        )
        existing_capacity = (
            existing_summary.get("capacity_selection", {})
            if isinstance(existing_summary.get("capacity_selection", {}), dict)
            else {}
        )
    except Exception:
        existing_summary = {}
        existing_targets = {}
        existing_capacity = {}

target_results = {}
capacity_selection = {}
resolved_images_by_usage = _collect_image_usages(image_selection)


def _persist_summary():
    successful_targets = [
        t
        for t in selected_targets
        if target_results.get(t, {}).get("status") == "success"
    ]
    failed_targets = [
        t
        for t in selected_targets
        if target_results.get(t, {}).get("status") == "failed"
    ]
    status = (
        "success"
        if not failed_targets
        else ("partial_failure" if successful_targets else "failed")
    )

    summary = {
        "generated_at_utc": datetime.now(timezone.utc)
        .isoformat()
        .replace("+00:00", "Z"),
        "run_id": run_id,
        "region": region,
        "target_cluster": target_cluster,
        "selected_targets": selected_targets,
        "kubeconfig_endpoint_mode": kubeconfig_endpoint_mode,
        "benchmark_mode": benchmark_mode,
        "image_mode": image_mode,
        "resume_enabled": resume_enabled,
        "status": status,
        "successful_targets": successful_targets,
        "failed_targets": failed_targets,
        "image_selection_artifact": str(image_selection_path),
        "resolved_images": {
            "validated_image_count": image_selection.get("validated_image_count", 0),
            "oke_primary_source_count": image_selection.get(
                "oke_primary_source_count", 0
            ),
            "resolved_by_usage": resolved_images_by_usage,
            "kpo_filter_resolution": image_selection.get("kpo_filter_resolution", {}),
        },
        "capacity_selection": capacity_selection,
        "cluster_ocids": {
            t: target_results.get(t, {}).get("cluster_id", "")
            for t in selected_targets
            if target_results.get(t, {}).get("status") == "success"
        },
        "node_pool_ids": {
            t: target_results.get(t, {}).get("worker_pool_ids", {})
            for t in selected_targets
            if target_results.get(t, {}).get("status") == "success"
        },
        "kubeconfig_refs": {
            t: {
                "kubeconfig": target_results.get(t, {}).get("kubeconfig_path", ""),
                "kubeconfig_endpoint": target_results.get(t, {}).get(
                    "kubeconfig_endpoint", ""
                ),
                "kubeconfig_server": target_results.get(t, {}).get(
                    "kubeconfig_server", ""
                ),
            }
            for t in selected_targets
            if target_results.get(t, {}).get("status") == "success"
        },
        "targets": target_results,
        "warnings": warnings,
    }
    _write_json(provision_path, summary)
    os.environ["PROVISIONING_ARTIFACT"] = str(provision_path)


for t in selected_targets:
    existing_target = (
        existing_targets.get(t, {})
        if isinstance(existing_targets.get(t, {}), dict)
        else {}
    )
    existing_kubeconfig = (
        Path(existing_target.get("kubeconfig_path", ""))
        if existing_target.get("kubeconfig_path")
        else None
    )
    can_resume_skip = (
        resume_enabled
        and existing_target.get("status") == "success"
        and bool(existing_target.get("cluster_id"))
        and existing_kubeconfig is not None
        and existing_kubeconfig.exists()
    )

    if can_resume_skip:
        print(
            f"\n=== Target {t}: resume skip (already successful in provisioning artifact) ==="
        )
        r = dict(existing_target)
        r["resumed"] = True
        r["resume_action"] = "skipped"
        target_results[t] = r
        if t in existing_capacity:
            capacity_selection[t] = existing_capacity[t]
        _export_target_env(t, r)
        _persist_summary()
        continue

    stack_dir = TF_STACK_ROOT / t
    tfvars_path = stack_dir / "terraform.auto.tfvars.json"
    if not stack_dir.exists():
        target_results[t] = {
            "status": "failed",
            "stack_dir": str(stack_dir),
            "failure": {
                "step": "precheck",
                "message": f"Stack dir not found: {stack_dir}",
            },
        }
        _persist_summary()
        continue
    if not tfvars_path.exists():
        target_results[t] = {
            "status": "failed",
            "stack_dir": str(stack_dir),
            "failure": {
                "step": "precheck",
                "message": f"Missing tfvars: {tfvars_path}",
            },
        }
        _persist_summary()
        continue

    print(f"\n=== Applying target: {t} ({stack_dir}) ===")
    target_step_records = []
    failure = None
    tf_output_artifact = artifact_dir / f"{t}_terraform_output.json"
    plan_log_path = provision_dir / f"{t}_terraform_plan.log"
    plan_meta_path = provision_dir / f"{t}_terraform_plan_meta.json"
    apply_log_path = provision_dir / f"{t}_terraform_apply.log"
    apply_meta_path = provision_dir / f"{t}_terraform_apply_meta.json"
    init_log_path = provision_dir / f"{t}_terraform_init.log"
    init_meta_path = provision_dir / f"{t}_terraform_init_meta.json"

    try:
        init_cmd = [
            terraform_exec,
            f"-chdir={stack_dir}",
            "init",
            "-input=false",
            "-no-color",
        ]
        if tf_init_upgrade:
            init_cmd.insert(3, "-upgrade")

        rc, out, err, init_attempts = _run_with_retries(
            init_cmd,
            env=cmd_env,
            retries=tf_init_retries,
            retry_delay_seconds=tf_init_retry_seconds,
            retry_if=_is_transient_tf_error,
            success_rcs={0},
        )
        init_log_path.write_text(
            (out or "") + ("\n" if out and err else "") + (err or ""), encoding="utf-8"
        )
        _write_json(
            init_meta_path,
            {
                "target": t,
                "generated_at_utc": datetime.now(timezone.utc)
                .isoformat()
                .replace("+00:00", "Z"),
                "command": init_cmd,
                "rc": rc,
                "attempts_used": init_attempts,
                "log_path": str(init_log_path),
            },
        )
        target_step_records.append(
            {"name": "init", "rc": rc, "attempts_used": init_attempts}
        )
        if rc != 0:
            failure = _cmd_failure("terraform_init", init_cmd, rc, out, err)

        plan_rc = None
        plan_status = "not_run"
        plan_attempts = 0
        if not failure:
            plan_cmd = [
                terraform_exec,
                f"-chdir={stack_dir}",
                "plan",
                "-input=false",
                "-no-color",
                "-lock-timeout=5m",
                "-detailed-exitcode",
            ]
            rc, out, err, plan_attempts = _run_with_retries(
                plan_cmd,
                env=cmd_env,
                retries=tf_plan_retries,
                retry_delay_seconds=tf_plan_retry_seconds,
                retry_if=_is_transient_tf_error,
                success_rcs={0, 2},
            )
            plan_log_path.write_text(
                (out or "") + ("\n" if out and err else "") + (err or ""),
                encoding="utf-8",
            )
            plan_rc = rc
            if rc == 0:
                plan_status = "no_changes"
            elif rc == 2:
                plan_status = "changes_present"
            else:
                plan_status = "failed"
            _write_json(
                plan_meta_path,
                {
                    "target": t,
                    "generated_at_utc": datetime.now(timezone.utc)
                    .isoformat()
                    .replace("+00:00", "Z"),
                    "command": plan_cmd,
                    "rc": rc,
                    "status": plan_status,
                    "attempts_used": plan_attempts,
                    "log_path": str(plan_log_path),
                },
            )
            target_step_records.append(
                {
                    "name": "plan",
                    "rc": rc,
                    "status": plan_status,
                    "attempts_used": plan_attempts,
                }
            )
            if rc not in {0, 2}:
                failure = _cmd_failure("terraform_plan", plan_cmd, rc, out, err)

        apply_rc = None
        apply_status = "not_run"
        apply_attempts = 0
        if not failure:
            apply_cmd = [
                terraform_exec,
                f"-chdir={stack_dir}",
                "apply",
                "-input=false",
                "-no-color",
                "-lock-timeout=5m",
                "-auto-approve",
            ]
            if plan_rc == 0:
                apply_status = "skipped_no_changes"
                _write_json(
                    apply_meta_path,
                    {
                        "target": t,
                        "generated_at_utc": datetime.now(timezone.utc)
                        .isoformat()
                        .replace("+00:00", "Z"),
                        "command": apply_cmd,
                        "rc": 0,
                        "status": apply_status,
                        "attempts_used": 0,
                        "log_path": str(apply_log_path),
                    },
                )
                apply_log_path.write_text("", encoding="utf-8")
                target_step_records.append(
                    {
                        "name": "apply",
                        "rc": 0,
                        "status": apply_status,
                        "attempts_used": 0,
                    }
                )
            else:
                rc, out, err, apply_attempts = _run_with_retries(
                    apply_cmd,
                    env=cmd_env,
                    retries=tf_apply_retries,
                    retry_delay_seconds=tf_apply_retry_seconds,
                    retry_if=_is_transient_tf_error,
                    success_rcs={0},
                )
                apply_log_path.write_text(
                    (out or "") + ("\n" if out and err else "") + (err or ""),
                    encoding="utf-8",
                )
                apply_rc = rc
                apply_status = "applied" if rc == 0 else "failed"
                _write_json(
                    apply_meta_path,
                    {
                        "target": t,
                        "generated_at_utc": datetime.now(timezone.utc)
                        .isoformat()
                        .replace("+00:00", "Z"),
                        "command": apply_cmd,
                        "rc": rc,
                        "status": apply_status,
                        "attempts_used": apply_attempts,
                        "log_path": str(apply_log_path),
                    },
                )
                target_step_records.append(
                    {
                        "name": "apply",
                        "rc": rc,
                        "status": apply_status,
                        "attempts_used": apply_attempts,
                    }
                )
                if rc != 0:
                    failure = _cmd_failure("terraform_apply", apply_cmd, rc, out, err)

        tf_outputs = {}
        output_attempts = 0
        if not failure:
            output_cmd = [terraform_exec, f"-chdir={stack_dir}", "output", "-json"]
            rc, out_obj, out_raw, err_raw, output_attempts = _run_json_with_retries(
                output_cmd,
                env=cmd_env,
                retries=tf_plan_retries,
                retry_delay_seconds=tf_plan_retry_seconds,
                retry_if=_is_transient_tf_error,
            )
            target_step_records.append(
                {"name": "output", "rc": rc, "attempts_used": output_attempts}
            )
            if rc != 0 or not isinstance(out_obj, dict):
                failure = _cmd_failure(
                    "terraform_output_json",
                    output_cmd,
                    rc,
                    out_raw or "",
                    err_raw or "",
                )
            else:
                tf_outputs = out_obj
                _write_json(tf_output_artifact, tf_outputs)

        if failure:
            target_results[t] = {
                "status": "failed",
                "resumed": False,
                "stack_dir": str(stack_dir),
                "steps": target_step_records,
                "failure": failure,
                "init_log_path": str(init_log_path),
                "init_meta_path": str(init_meta_path),
                "plan_log_path": str(plan_log_path),
                "plan_meta_path": str(plan_meta_path),
                "apply_log_path": str(apply_log_path),
                "apply_meta_path": str(apply_meta_path),
            }
            _persist_summary()
            continue

        cluster_id = _tf_out_val(tf_outputs, "cluster_id", "")
        if not cluster_id:
            target_results[t] = {
                "status": "failed",
                "resumed": False,
                "stack_dir": str(stack_dir),
                "steps": target_step_records,
                "failure": {
                    "step": "cluster_id",
                    "message": "terraform output missing cluster_id",
                },
                "terraform_output_artifact": str(tf_output_artifact),
            }
            _persist_summary()
            continue

        cluster_endpoints = _tf_out_val(tf_outputs, "cluster_endpoints", {})
        apiserver_private_host = _tf_out_val(tf_outputs, "apiserver_private_host", "")
        worker_pool_ids = _tf_out_val(tf_outputs, "worker_pool_ids", {})
        worker_pools = _tf_out_val(tf_outputs, "worker_pools", {})
        worker_subnet_id = _tf_out_val(tf_outputs, "worker_subnet_id", "")
        pod_subnet_id = _tf_out_val(tf_outputs, "pod_subnet_id", "")
        control_plane_subnet_id = _tf_out_val(tf_outputs, "control_plane_subnet_id", "")
        int_lb_subnet_id = _tf_out_val(tf_outputs, "int_lb_subnet_id", "")
        pub_lb_subnet_id = _tf_out_val(tf_outputs, "pub_lb_subnet_id", "")
        vcn_id = _tf_out_val(tf_outputs, "vcn_id", "")

        apiserver_private_endpoint = _extract_endpoint(
            cluster_endpoints, apiserver_private_host, prefer="private"
        )
        apiserver_public_endpoint = _extract_endpoint(
            cluster_endpoints, apiserver_private_host, prefer="public"
        )

        if kubeconfig_endpoint_mode == "public":
            kube_endpoint_flag = "PUBLIC_ENDPOINT"
        elif kubeconfig_endpoint_mode == "private":
            kube_endpoint_flag = "PRIVATE_ENDPOINT"
        else:
            kube_endpoint_flag = (
                "PUBLIC_ENDPOINT" if apiserver_public_endpoint else "PRIVATE_ENDPOINT"
            )

        kubeconfig_mode_label = (
            "public" if kube_endpoint_flag == "PUBLIC_ENDPOINT" else "private"
        )
        apiserver_endpoint = (
            (apiserver_public_endpoint or apiserver_private_endpoint)
            if kubeconfig_mode_label == "public"
            else (apiserver_private_endpoint or apiserver_public_endpoint)
        )

        kubeconfig_path = kubeconfig_dir / f"{t}-{kubeconfig_mode_label}.kubeconfig"
        kubeconfig_cmd = [
            oci_exec,
            "--config-file",
            oci_config_file,
            "--profile",
            oci_profile,
            "--region",
            region,
            "ce",
            "cluster",
            "create-kubeconfig",
            "--cluster-id",
            cluster_id,
            "--file",
            str(kubeconfig_path),
            "--token-version",
            "2.0.0",
            "--kube-endpoint",
            kube_endpoint_flag,
            "--overwrite",
        ]
        rc, out, err, kubeconfig_attempts = _run_with_retries(
            kubeconfig_cmd,
            env=cmd_env,
            retries=oci_kubeconfig_retries,
            retry_delay_seconds=oci_kubeconfig_retry_seconds,
            retry_if=_is_transient_oci_cli_error,
            success_rcs={0},
        )
        target_step_records.append(
            {
                "name": "create_kubeconfig",
                "rc": rc,
                "attempts_used": kubeconfig_attempts,
            }
        )
        if rc != 0:
            target_results[t] = {
                "status": "failed",
                "resumed": False,
                "stack_dir": str(stack_dir),
                "steps": target_step_records,
                "failure": _cmd_failure(
                    "oci_create_kubeconfig", kubeconfig_cmd, rc, out, err
                ),
                "terraform_output_artifact": str(tf_output_artifact),
            }
            _persist_summary()
            continue

        kubeconfig_server = _kubeconfig_server(kubeconfig_path)

        bm = _load_json(tfvars_path).get("benchmark", {})
        pools = bm.get("worker_pools", {})
        pool_capacity = {}
        for pool_name, pool_cfg in pools.items():
            if not isinstance(pool_cfg, dict):
                continue
            pool_capacity[pool_name] = {
                "shape": pool_cfg.get("shape"),
                "image_id": pool_cfg.get("image_id"),
                "size": pool_cfg.get("size"),
                "min_size": pool_cfg.get("min_size"),
                "max_size": pool_cfg.get("max_size"),
                "ocpus": pool_cfg.get("ocpus"),
                "memory": pool_cfg.get("memory"),
                "autoscale": pool_cfg.get("autoscale"),
                "allow_autoscaler": pool_cfg.get("allow_autoscaler"),
                "ignore_initial_pool_size": pool_cfg.get("ignore_initial_pool_size"),
            }
        capacity_selection[t] = {
            "kubernetes_version": bm.get("kubernetes_version"),
            "cluster_name": bm.get("cluster_name"),
            "worker_pools": pool_capacity,
        }

        target_results[t] = {
            "status": "success",
            "resumed": False,
            "resume_action": "applied",
            "stack_dir": str(stack_dir),
            "steps": target_step_records,
            "plan_status": plan_status,
            "apply_status": apply_status,
            "cluster_id": cluster_id,
            "kubernetes_version": bm.get("kubernetes_version"),
            "compartment_id": bm.get("compartment_id"),
            "worker_compartment_id": bm.get("worker_compartment_id"),
            "network_compartment_id": bm.get("network_compartment_id"),
            "cluster_endpoints": cluster_endpoints,
            "apiserver_private_host": apiserver_private_host,
            "apiserver_private_endpoint": apiserver_private_endpoint,
            "apiserver_public_endpoint": apiserver_public_endpoint,
            "apiserver_endpoint": apiserver_endpoint,
            "kubeconfig_endpoint": kubeconfig_mode_label,
            "kubeconfig_server": kubeconfig_server,
            "kubeconfig": str(kubeconfig_path),
            "kubeconfig_path": str(kubeconfig_path),
            "worker_pool_ids": worker_pool_ids,
            "worker_pools": worker_pools,
            "resolved_images": resolved_images_by_usage.get(t, {}),
            "vcn_id": vcn_id,
            "control_plane_subnet_id": control_plane_subnet_id,
            "worker_subnet_id": worker_subnet_id,
            "pod_subnet_id": pod_subnet_id,
            "int_lb_subnet_id": int_lb_subnet_id,
            "pub_lb_subnet_id": pub_lb_subnet_id,
            "terraform_output_artifact": str(tf_output_artifact),
            "init_log_path": str(init_log_path),
            "init_meta_path": str(init_meta_path),
            "plan_log_path": str(plan_log_path),
            "plan_meta_path": str(plan_meta_path),
            "apply_log_path": str(apply_log_path),
            "apply_meta_path": str(apply_meta_path),
        }

        _export_target_env(t, target_results[t])
        _persist_summary()

    except Exception as e:
        target_results[t] = {
            "status": "failed",
            "resumed": False,
            "stack_dir": str(stack_dir),
            "steps": target_step_records,
            "failure": {"step": "exception", "message": str(e)},
        }
        _persist_summary()

_persist_summary()

failed_targets = [
    t for t in selected_targets if target_results.get(t, {}).get("status") == "failed"
]
successful_targets = [
    t for t in selected_targets if target_results.get(t, {}).get("status") == "success"
]

print("\nCell 7 provisioning summary:")
for t in selected_targets:
    r = target_results.get(t, {})
    status = r.get("status", "unknown")
    if status == "success":
        print(f"- {t}: status=success cluster_id={r.get('cluster_id', '')}")
        print(f"  kubeconfig={r.get('kubeconfig_path', '')}")
        print(f"  kubeconfig_endpoint={r.get('kubeconfig_endpoint', '')}")
        print(f"  kubeconfig_server={r.get('kubeconfig_server', '')}")
        print(f"  apiserver_endpoint={r.get('apiserver_endpoint', '')}")
    else:
        f = r.get("failure", {})
        print(
            f"- {t}: status=failed step={f.get('step', '')} message={f.get('message', '')}"
        )

if warnings:
    print("\nProvisioning warnings:")
    for w in warnings:
        print(f"- {w}")

print(f"- Provisioning artifact: {provision_path}")

if failed_targets:
    raise RuntimeError(
        "Cell 7 failed for target(s): "
        + ", ".join(failed_targets)
        + f". Successful target(s): {', '.join(successful_targets) if successful_targets else 'none'}. "
        + f"Review diagnostics in {provision_path} and rerun Cell 7 (resume is enabled by default)."
    )

print("\nCell 7 complete: selected target(s) provisioned and outputs persisted.")
print("Next: Run Cell 8.")

In [ ]:
# Cell 8 — Post-Provision Autoscaler Setup and Health (CA + KPO, with IAM bootstrap)
import json
import ipaddress
import os
import re
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse
import oci


# ---------------------------
# Helpers
# ---------------------------
def _env(k, d=""):
    return os.environ.get(k, d)


def _env_nonempty(k, d=""):
    v = os.environ.get(k, "")
    if v is None:
        return d
    s = str(v).strip()
    return s if s else d


def _require_env(k):
    v = _env(k, "").strip()
    if not v:
        raise RuntimeError(f"Missing required env var: {k}")
    return v


def _bool(v, default=False):
    if v is None:
        return default
    return str(v).strip().lower() in {"1", "true", "yes", "y", "on"}


def _to_int(v, d):
    try:
        return int(str(v).strip())
    except Exception:
        return d


def _to_float(v, d):
    try:
        return float(str(v).strip())
    except Exception:
        return d


def _load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def _write_json(path: Path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def _find_project_root(start: Path) -> Path:
    explicit = _env_nonempty("PROJECT_ROOT", "").strip()
    if explicit:
        p = Path(os.path.expanduser(explicit)).resolve()
        if (p / "tf").exists() and (
            p / "Cluster_Autoscaler_Vs_Karpenter_On_OKE.ipynb"
        ).exists():
            return p
        raise RuntimeError(
            f"PROJECT_ROOT is set but invalid: {p}. "
            "Expected tf/ and Cluster_Autoscaler_Vs_Karpenter_On_OKE.ipynb."
        )
    p = start.resolve()
    for c in [p] + list(p.parents):
        if (c / "tf").exists() and (
            c / "Cluster_Autoscaler_Vs_Karpenter_On_OKE.ipynb"
        ).exists():
            return c
        nested = c / "OCIK8sClusterAutoscalerVsKarpenter"
        if (nested / "tf").exists() and (
            nested / "Cluster_Autoscaler_Vs_Karpenter_On_OKE.ipynb"
        ).exists():
            return nested.resolve()
    raise RuntimeError("Could not resolve project root. Set PROJECT_ROOT.")


def _shell_join(cmd):
    return " ".join(cmd)


def _run_capture(cmd, cwd=None, env=None, print_output=True):
    print(f"\n$ {_shell_join(cmd)}")
    p = subprocess.run(cmd, cwd=cwd, env=env, text=True, capture_output=True)
    out = p.stdout or ""
    err = p.stderr or ""
    if print_output:
        if out:
            print(out)
        if err:
            print(err)
    return p.returncode, out, err


def _run_json_quiet(cmd):
    rc, out, err = _run_capture(cmd, print_output=False)
    if rc != 0:
        raise RuntimeError(
            f"Command failed rc={rc}: {_shell_join(cmd)} :: {(err or out).strip()}"
        )
    try:
        return json.loads(out or "{}")
    except Exception as e:
        raise RuntimeError(f"JSON parse failed for {_shell_join(cmd)}: {e}")


def _is_transient_error(text):
    t = str(text or "").lower()
    markers = [
        "timed out",
        "timeout",
        "i/o timeout",
        "tls handshake timeout",
        "temporary failure",
        "temporarily unavailable",
        "connection reset",
        "connection refused",
        "context deadline exceeded",
        "service unavailable",
        "too many requests",
        "rate limit",
        "429",
        "502 bad gateway",
        "503 service unavailable",
        "504 gateway timeout",
        "eof",
    ]
    return any(m in t for m in markers)


def _run_with_retries(
    cmd, retries=1, delay_seconds=5.0, retry_if=None, success_rcs=None
):
    allowed = {0} if success_rcs is None else {int(x) for x in success_rcs}
    attempts = max(1, int(retries))
    last = (1, "", "", 1)
    for attempt in range(1, attempts + 1):
        rc, out, err = _run_capture(cmd)
        last = (rc, out, err, attempt)
        if rc in allowed:
            return last
        should_retry = (
            bool(retry_if and retry_if(f"{out}\n{err}")) and attempt < attempts
        )
        if should_retry:
            print(f"Retrying in {delay_seconds}s (attempt {attempt + 1}/{attempts})...")
            time.sleep(max(0.0, float(delay_seconds)))
            continue
        return last
    return last


def _run_or_raise(cmd, retries=1, delay_seconds=5.0, retry_if=None, success_rcs=None):
    rc, out, err, attempts = _run_with_retries(
        cmd=cmd,
        retries=retries,
        delay_seconds=delay_seconds,
        retry_if=retry_if,
        success_rcs=success_rcs,
    )
    allowed = {0} if success_rcs is None else {int(x) for x in success_rcs}
    if rc not in allowed:
        raise RuntimeError(
            f"Command failed after {attempts} attempt(s), rc={rc}: {_shell_join(cmd)}"
        )
    return out, err, attempts


def _run_json_or_raise(cmd, retries=1, delay_seconds=5.0, retry_if=None):
    out, err, attempts = _run_or_raise(
        cmd=cmd,
        retries=retries,
        delay_seconds=delay_seconds,
        retry_if=retry_if,
    )
    try:
        return json.loads(out or "{}"), attempts, out, err
    except Exception as e:
        raise RuntimeError(
            f"Failed to parse JSON after {attempts} attempt(s): {_shell_join(cmd)} ; {e}"
        )


def _sanitize_name(name: str, max_len: int = 120) -> str:
    out = re.sub(r"[^A-Za-z0-9_.-]+", "-", name).strip("-")
    if len(out) > max_len:
        out = out[:max_len]
    return out or "bench-policy"


def _render_template(path: Path, replacements: dict):
    text = path.read_text(encoding="utf-8")
    for k, v in replacements.items():
        text = text.replace(k, v)
    return text


def _policy_statements_from_text(text: str):
    acc = []
    for raw in text.splitlines():
        s = raw.strip()
        if not s or s.startswith("#"):
            continue
        acc.append(s)
    blob = "\n".join(acc)
    blocks = []
    cur = []
    depth = 0
    for ln in blob.splitlines():
        if ln.startswith("Allow ") and not cur:
            cur = [ln]
            depth += ln.count("{") - ln.count("}")
            if depth <= 0:
                blocks.append(" ".join(cur))
                cur = []
                depth = 0
            continue
        if cur:
            cur.append(ln)
            depth += ln.count("{") - ln.count("}")
            if depth <= 0:
                blocks.append(" ".join(cur))
                cur = []
                depth = 0
    return [re.sub(r"\s+", " ", b).strip() for b in blocks if b.strip()]


def _rewrite_workload_principal_targets(
    policy_text: str, namespace: str, service_account: str
) -> str:
    out = policy_text
    out = re.sub(
        r"request\.principal\.namespace\s*=\s*'[^']*'",
        f"request.principal.namespace = '{namespace}'",
        out,
    )
    out = re.sub(
        r"request\.principal\.service_account\s*=\s*'[^']*'",
        f"request.principal.service_account = '{service_account}'",
        out,
    )
    return out


def _normalize_statements_for_policy_compartment(
    statements: list, policy_compartment_id: str
):
    normalized = [s.strip() for s in statements if s and s.strip()]
    if not normalized:
        return normalized, 0
    if policy_compartment_id.startswith("ocid1.tenancy."):
        return normalized, 0
    target = f"in compartment id {policy_compartment_id}"
    out = []
    rewrites = 0
    for stmt in normalized:
        updated = re.sub(r"\bin tenancy\b", target, stmt)
        if updated != stmt:
            rewrites += 1
        out.append(updated)
    return out, rewrites


def _read_kubeconfig_server(kubeconfig_path: str) -> str:
    text = Path(kubeconfig_path).read_text(encoding="utf-8")
    for ln in text.splitlines():
        m = re.match(r"^\s*server:\s*(\S+)\s*$", ln)
        if m:
            return m.group(1).strip()
    return ""


def _extract_apiserver_host(endpoint: str) -> str:
    if not endpoint:
        return ""
    s = endpoint.strip()
    if s.startswith(("http://", "https://")):
        return (urlparse(s).hostname or "").strip()
    if ":" in s:
        return s.split(":", 1)[0].strip()
    return s


def _resolve_kpo_apiserver_endpoint(
    target_info: dict, source: str, kubeconfig_path: str
):
    src = (source or "private").strip().lower()
    allowed = {"provisioned", "selected", "auto", "public", "private", "kubeconfig"}
    if src not in allowed:
        raise RuntimeError(
            "KPO_APISERVER_ENDPOINT_SOURCE must be one of: "
            "provisioned|selected|auto|public|private|kubeconfig"
        )
    selected_ep = str(target_info.get("apiserver_endpoint", "") or "").strip()
    public_ep = str(target_info.get("apiserver_public_endpoint", "") or "").strip()
    private_ep = str(target_info.get("apiserver_private_endpoint", "") or "").strip()
    kubeconfig_ep = str(_read_kubeconfig_server(kubeconfig_path) or "").strip()

    def _is_private_endpoint(ep: str) -> bool:
        host = _extract_apiserver_host(ep)
        if not host:
            return False
        try:
            return ipaddress.ip_address(host).is_private
        except ValueError:
            return False

    if src == "public":
        for label, ep in [
            ("public", public_ep),
            ("provisioned", selected_ep),
            ("kubeconfig", kubeconfig_ep),
        ]:
            ep = str(ep or "").strip()
            if ep:
                return ep, label
        raise RuntimeError(
            "KPO_APISERVER_ENDPOINT_SOURCE=public requested, but no public endpoint was found."
        )
    if src == "kubeconfig":
        if not kubeconfig_ep:
            raise RuntimeError(
                "KPO_APISERVER_ENDPOINT_SOURCE=kubeconfig requested, but kubeconfig server is empty."
            )
        return kubeconfig_ep, "kubeconfig"
    for label, ep in [
        ("private", private_ep),
        ("provisioned", selected_ep),
        ("kubeconfig", kubeconfig_ep),
    ]:
        ep = str(ep or "").strip()
        if ep and _is_private_endpoint(ep):
            return ep, label
    raise RuntimeError(
        "Could not resolve a private API endpoint for KPO worker bootstrap. "
        "Expected apiserver_private_endpoint/apiserver_endpoint/kubeconfig server to resolve to RFC1918 private IP. "
        f"apiserver_private_endpoint='{private_ep}', "
        f"apiserver_endpoint='{selected_ep}', "
        f"kubeconfig_server='{kubeconfig_ep}', "
        f"apiserver_public_endpoint='{public_ep}'."
    )


def _version_tuple(v: str):
    m = re.search(r"(\d+)\.(\d+)\.(\d+)", str(v or ""))
    if not m:
        return None
    return int(m.group(1)), int(m.group(2)), int(m.group(3))


def _instance_ocid_from_provider_id(provider_id: str):
    if not provider_id:
        return ""
    s = provider_id.strip()
    if s.startswith("oci://"):
        s = s[len("oci://") :]
    parts = [p for p in s.split("/") if p]
    if not parts:
        return ""
    if parts[-1].startswith("ocid1.instance."):
        return parts[-1]
    for p in parts:
        if p.startswith("ocid1.instance."):
            return p
    return ""


def _ensure_dynamic_group(
    identity_client,
    compartment_id: str,
    dg_name: str,
    matching_rule: str,
    description: str,
):
    matching_rule = (matching_rule or "").strip()
    if not matching_rule:
        raise RuntimeError("KPO dynamic group matching rule is empty.")
    existing = None
    page = None
    while True:
        resp = identity_client.list_dynamic_groups(
            compartment_id=compartment_id, page=page
        )
        for dg in resp.data:
            if dg.name == dg_name:
                existing = dg
                break
        if existing or not resp.has_next_page:
            break
        page = resp.next_page
    if existing is None:
        details = oci.identity.models.CreateDynamicGroupDetails(
            compartment_id=compartment_id,
            name=dg_name,
            description=description,
            matching_rule=matching_rule,
        )
        created = identity_client.create_dynamic_group(details).data
        return {
            "action": "created",
            "id": created.id,
            "name": created.name,
            "matching_rule": created.matching_rule,
        }
    if (existing.matching_rule or "").strip() != matching_rule or (
        existing.description or ""
    ) != description:
        upd = oci.identity.models.UpdateDynamicGroupDetails(
            description=description, matching_rule=matching_rule
        )
        updated = identity_client.update_dynamic_group(existing.id, upd).data
        return {
            "action": "updated",
            "id": updated.id,
            "name": updated.name,
            "matching_rule": updated.matching_rule,
        }
    return {
        "action": "unchanged",
        "id": existing.id,
        "name": existing.name,
        "matching_rule": existing.matching_rule,
    }


def _ensure_policy(
    identity_client,
    compartment_id: str,
    policy_name: str,
    statements: list,
    description: str,
):
    normalized = [s.strip() for s in statements if s and s.strip()]
    if not normalized:
        raise RuntimeError(f"Policy {policy_name} has no statements after rendering.")
    existing = None
    page = None
    while True:
        resp = identity_client.list_policies(compartment_id=compartment_id, page=page)
        for p in resp.data:
            if p.name == policy_name:
                existing = p
                break
        if existing or not resp.has_next_page:
            break
        page = resp.next_page
    if existing is None:
        details = oci.identity.models.CreatePolicyDetails(
            compartment_id=compartment_id,
            name=policy_name,
            description=description,
            statements=normalized,
        )
        created = identity_client.create_policy(details).data
        return {
            "action": "created",
            "id": created.id,
            "name": created.name,
            "statements": normalized,
        }
    current = [s.strip() for s in (existing.statements or [])]
    if current != normalized or (existing.description or "") != description:
        upd = oci.identity.models.UpdatePolicyDetails(
            description=description, statements=normalized
        )
        updated = identity_client.update_policy(existing.id, upd).data
        return {
            "action": "updated",
            "id": updated.id,
            "name": updated.name,
            "statements": normalized,
        }
    return {
        "action": "unchanged",
        "id": existing.id,
        "name": existing.name,
        "statements": normalized,
    }


def _get_cluster_addon_names(
    oci_config_file: str,
    oci_profile: str,
    region: str,
    cluster_id: str,
    retries: int,
    delay_seconds: float,
):
    try:
        parsed, _, _, _ = _run_json_or_raise(
            [
                "oci",
                "--config-file",
                oci_config_file,
                "--profile",
                oci_profile,
                "--region",
                region,
                "ce",
                "cluster",
                "list-addons",
                "--cluster-id",
                cluster_id,
                "--all",
                "--output",
                "json",
            ],
            retries=retries,
            delay_seconds=delay_seconds,
            retry_if=_is_transient_error,
        )
        names = set()
        for item in parsed.get("data", []) or []:
            if isinstance(item, dict):
                for k in ("name", "addon-name", "addon_name"):
                    v = item.get(k)
                    if isinstance(v, str) and v.strip():
                        names.add(v.strip())
        if names:
            return names
    except RuntimeError:
        pass
    return set()


def _resolve_kpo_chart_ref(project_root: Path, explicit_ref: str):
    if explicit_ref:
        exp = explicit_ref.strip()
        p = Path(os.path.expanduser(exp))
        if p.exists():
            return str(p.resolve())
        if exp.startswith(("oci://", "http://", "https://")):
            return exp
        if "/" in exp:
            return exp
        raise RuntimeError(f"KPO_HELM_CHART not resolvable: {exp}")
    for c in [
        project_root / "docs" / "karpenter-provider-oci" / "chart",
        project_root / "references" / "karpenter-provider-oci-main" / "chart",
        project_root / "vendor" / "karpenter-provider-oci" / "chart",
        project_root / "vendor" / "karpenter-provider-oci-main" / "chart",
    ]:
        if c.exists():
            return str(c.resolve())
    raise RuntimeError(
        "Unable to resolve KPO Helm chart source. Set KPO_HELM_CHART "
        "or place chart under docs/karpenter-provider-oci/chart."
    )


def _k8s_api_reachability_check(
    kubeconfig: str, timeout_seconds: int, retries: int, delay_seconds: float
):
    rc, out, err, attempts = _run_with_retries(
        [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "get",
            "--raw",
            "/version",
            f"--request-timeout={timeout_seconds}s",
        ],
        retries=retries,
        delay_seconds=delay_seconds,
        retry_if=_is_transient_error,
    )
    if rc == 0:
        return True, "", attempts
    txt = (err or out or "").strip()
    if len(txt) > 600:
        txt = txt[-600:]
    return False, txt or f"kubectl exited with {rc}", attempts


def _detect_vcn_native_cni_version(kubeconfig: str, retries: int, delay_seconds: float):
    obj, attempts, _, _ = _run_json_or_raise(
        [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "-n",
            "kube-system",
            "get",
            "ds",
            "-l",
            "app=vcn-native-ip-cni",
            "-o",
            "json",
        ],
        retries=retries,
        delay_seconds=delay_seconds,
        retry_if=_is_transient_error,
    )
    items = obj.get("items", []) or []
    if not items:
        raise RuntimeError("VCN-native CNI daemonset not found.")
    containers = (
        items[0]
        .get("spec", {})
        .get("template", {})
        .get("spec", {})
        .get("containers", [])
    )
    if not containers:
        raise RuntimeError("VCN-native CNI daemonset has no containers.")
    image = containers[0].get("image", "")
    m = re.search(r":([0-9]+\.[0-9]+\.[0-9]+)", image)
    if not m:
        raise RuntimeError(f"Could not parse CNI version from image: {image}")
    return m.group(1), attempts


def _parse_ip_count(ocinodeclass_text: str):
    m = re.search(r"(?m)^\s*ipCount:\s*(\d+)\s*$", ocinodeclass_text)
    if not m:
        raise RuntimeError("Could not find ipCount in rendered OCINodeClass manifest.")
    return int(m.group(1))


def _is_power_of_two(n: int):
    return n > 0 and (n & (n - 1)) == 0


def _discover_kpo_nsg_ids_from_baseline_nodes(
    kubeconfig: str,
    worker_subnet_id: str,
    pod_subnet_id: str,
    oci_config_file: str,
    oci_profile: str,
    region: str,
    kubectl_retries: int,
    oci_retries: int,
    retry_delay_seconds: float,
):
    nodes_obj, _, _, _ = _run_json_or_raise(
        [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "get",
            "nodes",
            "-l",
            "benchmark.oracle.com/autoscaler=kpo,benchmark.oracle.com/role=baseline",
            "-o",
            "json",
        ],
        retries=kubectl_retries,
        delay_seconds=retry_delay_seconds,
        retry_if=_is_transient_error,
    )
    nodes = nodes_obj.get("items", []) or []
    if not nodes:
        raise RuntimeError("No baseline KPO nodes found for NSG discovery.")
    worker_nsg_id = ""
    pod_nsg_id = ""
    for node in nodes:
        provider_id = ((node.get("spec", {}) or {}).get("providerID") or "").strip()
        instance_id = _instance_ocid_from_provider_id(provider_id)
        if not instance_id:
            continue
        vnics_obj, _, _, _ = _run_json_or_raise(
            [
                "oci",
                "--config-file",
                oci_config_file,
                "--profile",
                oci_profile,
                "--region",
                region,
                "compute",
                "instance",
                "list-vnics",
                "--instance-id",
                instance_id,
                "--all",
                "--output",
                "json",
            ],
            retries=oci_retries,
            delay_seconds=retry_delay_seconds,
            retry_if=_is_transient_error,
        )
        for vnic in vnics_obj.get("data", []) or []:
            subnet = (vnic.get("subnet-id") or "").strip()
            nsg_ids = vnic.get("nsg-ids", []) or []
            if subnet == worker_subnet_id and nsg_ids and not worker_nsg_id:
                worker_nsg_id = nsg_ids[0]
            if subnet == pod_subnet_id and nsg_ids and not pod_nsg_id:
                pod_nsg_id = nsg_ids[0]
        if worker_nsg_id and pod_nsg_id:
            break
    if not worker_nsg_id or not pod_nsg_id:
        raise RuntimeError(
            "Failed to discover worker/pod NSG IDs from baseline nodes. "
            f"worker_nsg_id={worker_nsg_id or '<missing>'}, pod_nsg_id={pod_nsg_id or '<missing>'}"
        )
    return worker_nsg_id, pod_nsg_id


def _inject_nsgs_into_ocinodeclass(
    ocinodeclass_text: str,
    worker_subnet_id: str,
    pod_subnet_id: str,
    worker_nsg_id: str,
    pod_nsg_id: str,
):
    def _norm_subnet(v: str):
        return (v or "").split("#", 1)[0].strip().strip('"').strip("'").strip()

    has_worker_nsg = worker_nsg_id in ocinodeclass_text
    has_pod_nsg = pod_nsg_id in ocinodeclass_text
    injected_worker = has_worker_nsg
    injected_pod = has_pod_nsg
    out = []
    for line in ocinodeclass_text.splitlines():
        out.append(line)
        m = re.match(r"^(\s*)subnetId:\s*(.+?)\s*$", line)
        if not m:
            continue
        subnet_indent = m.group(1)
        subnet_val = _norm_subnet(m.group(2))
        sibling_indent = subnet_indent[:-2] if len(subnet_indent) >= 2 else ""
        list_indent = sibling_indent + "  "
        if subnet_val == worker_subnet_id and not injected_worker:
            out.append(f"{sibling_indent}networkSecurityGroupConfigs:")
            out.append(f'{list_indent}- networkSecurityGroupId: "{worker_nsg_id}"')
            injected_worker = True
            continue
        if subnet_val == pod_subnet_id and not injected_pod:
            out.append(f"{sibling_indent}networkSecurityGroupConfigs:")
            out.append(f'{list_indent}- networkSecurityGroupId: "{pod_nsg_id}"')
            injected_pod = True
            continue
    if not injected_worker or not injected_pod:
        raise RuntimeError(
            "Failed to inject NSG blocks into OCINodeClass manifest. "
            f"worker_injected={injected_worker}, pod_injected={injected_pod}."
        )
    return "\n".join(out) + "\n"


def _extract_nodepool_limits(nodepool_text: str):
    cpu_match = re.search(r'(?m)^\s*cpu:\s*"?([^"\n]+)"?\s*$', nodepool_text)
    mem_match = re.search(r'(?m)^\s*memory:\s*"?([^"\n]+)"?\s*$', nodepool_text)
    return (
        cpu_match.group(1).strip() if cpu_match else "",
        mem_match.group(1).strip() if mem_match else "",
    )


def _node_ready(node_obj: dict) -> bool:
    for c in (node_obj.get("status", {}) or {}).get("conditions", []) or []:
        if c.get("type") == "Ready":
            return str(c.get("status", "")).strip().lower() == "true"
    return False


def _list_pool_nodes(kubeconfig: str, pool_name: str):
    obj = _run_json_quiet(
        [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "get",
            "nodes",
            "-l",
            f"oke.oraclecloud.com/pool.name={pool_name}",
            "-o",
            "json",
        ]
    )
    out = []
    for n in obj.get("items", []) or []:
        md = n.get("metadata", {}) or {}
        spec = n.get("spec", {}) or {}
        out.append(
            {
                "name": md.get("name", ""),
                "ready": _node_ready(n),
                "provider_id": spec.get("providerID", ""),
                "instance_id": _instance_ocid_from_provider_id(
                    spec.get("providerID", "")
                ),
            }
        )
    return out


def _resolve_kpo_baseline_pool_meta(provisioning_obj: dict):
    kpo_target = ((provisioning_obj or {}).get("targets", {}) or {}).get(
        "kpo", {}
    ) or {}
    worker_pool_ids = kpo_target.get("worker_pool_ids", {}) or {}
    if not worker_pool_ids:
        raise RuntimeError("KPO worker_pool_ids missing in provisioning artifact.")
    pool_name = (
        "kpo-baseline"
        if "kpo-baseline" in worker_pool_ids
        else sorted(worker_pool_ids.keys())[0]
    )
    pool_id = worker_pool_ids.get(pool_name, "")
    if not pool_id:
        raise RuntimeError("KPO baseline pool ID not found.")
    desired = 1
    try:
        desired = int(
            (
                ((provisioning_obj or {}).get("capacity_selection", {}) or {})
                .get("kpo", {})
                .get("worker_pools", {})
                .get(pool_name, {})
                .get("size", 1)
            )
        )
    except Exception:
        desired = 1
    if desired <= 0:
        desired = 1
    return {"pool_name": pool_name, "pool_id": pool_id, "desired_size": desired}


def _collect_namespace_events(kubeconfig: str, namespace: str):
    obj = _run_json_quiet(
        [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "-n",
            namespace,
            "get",
            "events",
            "-o",
            "json",
        ]
    )
    out = []
    for e in obj.get("items", []) or []:
        inv = e.get("involvedObject", {}) or {}
        out.append(
            {
                "namespace": namespace,
                "reason": e.get("reason", ""),
                "message": e.get("message", "") or "",
                "object_kind": inv.get("kind", ""),
                "object_name": inv.get("name", ""),
                "last_timestamp": e.get("lastTimestamp", "")
                or e.get("eventTime", "")
                or "",
            }
        )
    return out


def _collect_kpo_rollout_diagnostics(kubeconfig: str, namespace: str):
    deploy_obj = _run_json_quiet(
        [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "-n",
            namespace,
            "get",
            "deploy",
            "-l",
            "app.kubernetes.io/name=karpenter",
            "-o",
            "json",
        ]
    )
    deploy_items = deploy_obj.get("items", []) or []
    deployment = deploy_items[0] if deploy_items else {}
    desired = int((deployment.get("spec", {}) or {}).get("replicas", 0) or 0)
    available = int(
        (deployment.get("status", {}) or {}).get("availableReplicas", 0) or 0
    )
    pods_obj = _run_json_quiet(
        [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "-n",
            namespace,
            "get",
            "pods",
            "-l",
            "app.kubernetes.io/name=karpenter",
            "-o",
            "json",
        ]
    )
    pods = []
    pods_by_name = {}
    for p in pods_obj.get("items", []) or []:
        md = p.get("metadata", {}) or {}
        st = p.get("status", {}) or {}
        spec = p.get("spec", {}) or {}
        cs = st.get("containerStatuses") or []
        waiting_reason = ""
        if cs and isinstance(cs[0], dict):
            waiting_reason = (
                (cs[0].get("state", {}) or {}).get("waiting", {}) or {}
            ).get("reason", "")
        rec = {
            "name": md.get("name", ""),
            "phase": st.get("phase", ""),
            "node": spec.get("nodeName", ""),
            "pod_ip": st.get("podIP", ""),
            "waiting_reason": waiting_reason,
            "ready": all((c.get("ready", False) for c in cs)) if cs else False,
        }
        pods.append(rec)
        pods_by_name[rec["name"]] = rec
    events = _collect_namespace_events(kubeconfig, namespace)
    events.extend(_collect_namespace_events(kubeconfig, "kube-system"))
    cni_nodes = set()
    cni_events = []
    for ev in events:
        msg = str(ev.get("message", "")).lower()
        if (
            "failed to create pod sandbox" in msg
            and "oci-ipvlan" in msg
            and ("error configuring vnic" in msg or "link not found" in msg)
        ):
            cni_events.append(ev)
            node = pods_by_name.get(ev.get("object_name", ""), {}).get("node", "")
            if node:
                cni_nodes.add(node)
    if not cni_nodes and cni_events:
        kspods = _run_json_quiet(
            [
                "kubectl",
                "--kubeconfig",
                kubeconfig,
                "-n",
                "kube-system",
                "get",
                "pods",
                "-o",
                "json",
            ]
        )
        ks_map = {
            ((p.get("metadata", {}) or {}).get("name", "")): (
                (p.get("spec", {}) or {}).get("nodeName", "")
            )
            for p in kspods.get("items", []) or []
        }
        for ev in cni_events:
            if ev.get("namespace") == "kube-system":
                n = ks_map.get(ev.get("object_name", ""), "")
                if n:
                    cni_nodes.add(n)
    return {
        "desired_replicas": desired,
        "available_replicas": available,
        "pods": pods,
        "cni_sandbox_failure_nodes": sorted([n for n in cni_nodes if n]),
        "cni_sandbox_events": cni_events[-40:],
    }


def _delete_and_replace_pool_node(
    node_pool_id: str,
    instance_id: str,
    oci_config_file: str,
    oci_profile: str,
    region: str,
    oci_retries: int,
    retry_delay_seconds: float,
    wait_seconds: int,
):
    _run_or_raise(
        [
            "oci",
            "--config-file",
            oci_config_file,
            "--profile",
            oci_profile,
            "--region",
            region,
            "ce",
            "node-pool",
            "delete-node",
            "--node-pool-id",
            node_pool_id,
            "--node-id",
            instance_id,
            "--is-decrement-size",
            "false",
            "--override-eviction-grace-duration",
            "PT0M",
            "--is-force-deletion-after-override-grace-duration",
            "true",
            "--force",
            "--wait-for-state",
            "SUCCEEDED",
            "--max-wait-seconds",
            str(wait_seconds),
        ],
        retries=oci_retries,
        delay_seconds=retry_delay_seconds,
        retry_if=_is_transient_error,
    )


def _wait_for_pool_nodes_ready(
    kubeconfig: str, pool_name: str, desired_ready: int, timeout_seconds: int
):
    start = time.time()
    last = []
    while True:
        last = _list_pool_nodes(kubeconfig, pool_name)
        ready_count = sum(1 for n in last if n.get("ready"))
        if ready_count >= desired_ready:
            return {"timeout": False, "ready_count": ready_count, "nodes": last}
        if (time.time() - start) > timeout_seconds:
            return {"timeout": True, "ready_count": ready_count, "nodes": last}
        time.sleep(20)


# ---------------------------
# Context
# ---------------------------
project_root = _find_project_root(Path.cwd())
tf_root = Path(
    os.path.expanduser(
        _env_nonempty("TF_STACK_ROOT", str((project_root / "tf").resolve()))
    )
).resolve()
if not tf_root.exists():
    raise RuntimeError(f"TF_STACK_ROOT does not exist: {tf_root}")
run_id = _require_env("RUN_ID")
output_dir = Path(os.path.expanduser(_require_env("OUTPUT_DIR"))).resolve()
artifact_dir = output_dir / run_id
provisioning_path = artifact_dir / "provisioning.json"
if not provisioning_path.exists():
    raise RuntimeError(
        f"Missing provisioning artifact from Cell 7: {provisioning_path}"
    )
provisioning = _load_json(provisioning_path)
target_cluster = _env_nonempty("TARGET_CLUSTER", "both").strip().lower()
if target_cluster not in {"ca", "kpo", "both"}:
    raise RuntimeError(f"TARGET_CLUSTER must be ca|kpo|both, got: {target_cluster}")
selected_targets = ["ca", "kpo"] if target_cluster == "both" else [target_cluster]
for t in selected_targets:
    ti = (provisioning.get("targets", {}) or {}).get(t)
    if not isinstance(ti, dict):
        raise RuntimeError(f"provisioning.json missing target section: {t}")
    if ti.get("status") != "success":
        raise RuntimeError(
            f"Target {t} is not successful in provisioning.json. "
            f"status={ti.get('status')}, failed_targets={provisioning.get('failed_targets', [])}"
        )
region = _require_env("BENCHMARK_REGION")
home_region = _require_env("HOME_REGION")
oci_profile = _require_env("OCI_PROFILE")
oci_config_file = _require_env("OCI_CONFIG_FILE")
compartment_id = _require_env("BENCHMARK_COMPARTMENT_OCID")
iam_policy_compartment_id = _require_env("IAM_POLICY_COMPARTMENT_OCID")
tenancy_ocid = _require_env("TENANCY_OCID")
ca_namespace = _env_nonempty("CA_NAMESPACE", "kube-system")
ca_service_account = _env_nonempty("CA_SERVICE_ACCOUNT", "cluster-autoscaler")
kpo_namespace = _env_nonempty("KPO_NAMESPACE", "karpenter")
kpo_service_account = _env_nonempty("KPO_SERVICE_ACCOUNT", "karpenter")
kpo_release_name = _env_nonempty("KPO_HELM_RELEASE", "karpenter").strip()
kpo_endpoint_source = (
    _env_nonempty("KPO_APISERVER_ENDPOINT_SOURCE", "private").strip().lower()
)
allow_api_unreachable = _bool(
    _env_nonempty("HEALTHCHECK_ALLOW_API_UNREACHABLE", "false")
)
k8s_api_check_timeout_seconds = _to_int(
    _env_nonempty("K8S_API_CHECK_TIMEOUT_SECONDS", "15"), 15
)
default_retries = max(1, _to_int(_env_nonempty("CELL8_CMD_RETRIES", "3"), 3))
retry_delay_seconds = max(
    0.0, _to_float(_env_nonempty("CELL8_RETRY_DELAY_SECONDS", "8"), 8.0)
)
kubectl_retries = max(
    1,
    _to_int(
        _env_nonempty("CELL8_KUBECTL_RETRIES", str(default_retries)), default_retries
    ),
)
helm_retries = max(
    1,
    _to_int(_env_nonempty("CELL8_HELM_RETRIES", str(default_retries)), default_retries),
)
oci_retries = max(
    1,
    _to_int(_env_nonempty("CELL8_OCI_RETRIES", str(default_retries)), default_retries),
)
kpo_helm_replica_count = max(
    1, _to_int(_env_nonempty("KPO_HELM_REPLICA_COUNT", "2"), 2)
)
kpo_min_available_replicas = max(
    1, _to_int(_env_nonempty("KPO_MIN_AVAILABLE_REPLICAS", "1"), 1)
)
kpo_allow_degraded_rollout = _bool(_env_nonempty("KPO_ALLOW_DEGRADED_ROLLOUT", "false"))
kpo_auto_repair_node_cni = _bool(_env_nonempty("KPO_AUTO_REPAIR_NODE_CNI", "true"))
kpo_rollout_timeout_seconds = max(
    60, _to_int(_env_nonempty("KPO_ROLLOUT_TIMEOUT_SECONDS", "600"), 600)
)
kpo_rollout_retry_timeout_seconds = max(
    120, _to_int(_env_nonempty("KPO_ROLLOUT_RETRY_TIMEOUT_SECONDS", "900"), 900)
)
kpo_node_repair_wait_seconds = max(
    600, _to_int(_env_nonempty("KPO_NODE_REPAIR_WAIT_SECONDS", "1800"), 1800)
)
kpo_pool_ready_wait_seconds = max(
    300, _to_int(_env_nonempty("KPO_POOL_READY_WAIT_SECONDS", "1800"), 1800)
)
kpo_chart_ref = ""
if "kpo" in selected_targets:
    kpo_chart_ref = _resolve_kpo_chart_ref(
        project_root, _env_nonempty("KPO_HELM_CHART", "").strip()
    )
print("=== Cell 8 Post-Provisioning ===")
print(f"Project root: {project_root}")
print(f"TF root: {tf_root}")
print(f"Artifact dir: {artifact_dir}")
print(f"Selected target(s): {selected_targets}")
print(f"KPO endpoint source: {kpo_endpoint_source}")
if kpo_chart_ref:
    print(f"KPO chart source: {kpo_chart_ref}")
print(
    f"Retry policy: default={default_retries}, kubectl={kubectl_retries}, "
    f"helm={helm_retries}, oci={oci_retries}, delay_seconds={retry_delay_seconds}"
)
if "kpo" in selected_targets:
    print(
        f"KPO rollout policy: replicas={kpo_helm_replica_count}, "
        f"min_available={kpo_min_available_replicas}, "
        f"allow_degraded={str(kpo_allow_degraded_rollout).lower()}, "
        f"auto_repair_node_cni={str(kpo_auto_repair_node_cni).lower()}"
    )
os.environ["OCI_CLI_CONFIG_FILE"] = oci_config_file
os.environ["OCI_CONFIG_FILE_PROFILE"] = oci_profile
cfg_home = oci.config.from_file(file_location=oci_config_file, profile_name=oci_profile)
cfg_home["region"] = home_region
identity = oci.identity.IdentityClient(cfg_home)
summary = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
    "run_id": run_id,
    "selected_targets": selected_targets,
    "iam": {},
    "health": {},
    "k8s_connectivity": {},
    "applied_manifests": {},
    "warnings": [],
}
# ---------------------------
# Connectivity preflight
# ---------------------------
unreachable_targets = []
for t in selected_targets:
    tgt = provisioning["targets"][t]
    kubeconfig_path = tgt["kubeconfig_path"]
    reachable, err_text, attempts = _k8s_api_reachability_check(
        kubeconfig=kubeconfig_path,
        timeout_seconds=k8s_api_check_timeout_seconds,
        retries=kubectl_retries,
        delay_seconds=retry_delay_seconds,
    )
    summary["k8s_connectivity"][t] = {
        "kubeconfig_path": kubeconfig_path,
        "kubeconfig_endpoint": tgt.get("kubeconfig_endpoint", ""),
        "kubeconfig_server": _read_kubeconfig_server(kubeconfig_path),
        "apiserver_endpoint": tgt.get("apiserver_endpoint", ""),
        "apiserver_public_endpoint": tgt.get("apiserver_public_endpoint", ""),
        "apiserver_private_endpoint": tgt.get("apiserver_private_endpoint", ""),
        "reachable": reachable,
        "error": err_text,
        "attempts_used": attempts,
    }
    if not reachable:
        unreachable_targets.append(t)
if unreachable_targets and not allow_api_unreachable:
    details = []
    for t in unreachable_targets:
        c = summary["k8s_connectivity"][t]
        details.append(
            f"{t}: kubeconfig_endpoint={c.get('kubeconfig_endpoint','')}, "
            f"kubeconfig_server={c.get('kubeconfig_server','')}, "
            f"apiserver_endpoint={c.get('apiserver_endpoint','')}, "
            f"public={c.get('apiserver_public_endpoint','')}, "
            f"private={c.get('apiserver_private_endpoint','')}, "
            f"error={c.get('error','')}"
        )
    raise RuntimeError(
        "Kubernetes API unreachable for selected target(s). "
        "Set up endpoint network access or set HEALTHCHECK_ALLOW_API_UNREACHABLE=true.\n"
        + "\n".join(details)
    )
# ---------------------------
# IAM bootstrap
# ---------------------------
if "ca" in selected_targets:
    ca_cluster_id = provisioning["targets"]["ca"]["cluster_id"]
    ca_policy_name = _sanitize_name(
        _env_nonempty("CA_WORKLOAD_POLICY_NAME", f"ca-workload-{run_id[:16]}")
    )
    ca_template = tf_root / "ca" / "iam" / "ca_workload_identity.policy.tmpl"
    if not ca_template.exists():
        raise RuntimeError(f"Missing CA IAM template: {ca_template}")
    ca_rendered = _render_template(ca_template, {"__CA_CLUSTER_OCID__": ca_cluster_id})
    ca_rendered = _rewrite_workload_principal_targets(
        ca_rendered, ca_namespace, ca_service_account
    )
    ca_statements_raw = _policy_statements_from_text(ca_rendered)
    ca_statements, rewrites = _normalize_statements_for_policy_compartment(
        ca_statements_raw, iam_policy_compartment_id
    )
    if rewrites:
        summary["warnings"].append(
            f"CA policy: rewrote {rewrites} tenancy-scoped statement(s) to policy compartment scope."
        )
    summary["iam"]["ca_workload_policy"] = _ensure_policy(
        identity,
        iam_policy_compartment_id,
        ca_policy_name,
        ca_statements,
        f"CA workload identity policy for run {run_id}",
    )
if "kpo" in selected_targets:
    kpo_cluster_id = provisioning["targets"]["kpo"]["cluster_id"]
    dg_name = _sanitize_name(
        _env_nonempty("KPO_NODES_DYNAMIC_GROUP_NAME", f"kpo-nodes-{run_id[:16]}")
    )
    join_policy_name = _sanitize_name(
        _env_nonempty("KPO_NODES_POLICY_NAME", f"kpo-cluster-join-{run_id[:16]}")
    )
    workload_policy_name = _sanitize_name(
        _env_nonempty("KPO_WORKLOAD_POLICY_NAME", f"kpo-workload-{run_id[:16]}")
    )
    dg_rule = _env_nonempty(
        "KPO_NODES_DYNAMIC_GROUP_RULE",
        f"ALL {{instance.compartment.id = '{compartment_id}'}}",
    )
    summary["iam"]["kpo_nodes_dynamic_group"] = _ensure_dynamic_group(
        identity, tenancy_ocid, dg_name, dg_rule, f"KPO launched nodes for run {run_id}"
    )
    join_template = tf_root / "kpo" / "iam" / "kpo_nodes_cluster_join.policy.tmpl"
    workload_template = tf_root / "kpo" / "iam" / "kpo_workload_identity.policy.tmpl"
    if not join_template.exists():
        raise RuntimeError(f"Missing KPO join policy template: {join_template}")
    if not workload_template.exists():
        raise RuntimeError(f"Missing KPO workload policy template: {workload_template}")
    join_rendered = _render_template(
        join_template,
        {
            "__KPO_NODES_DYNAMIC_GROUP_NAME__": dg_name,
            "__KPO_CLUSTER_OCID__": kpo_cluster_id,
        },
    )
    join_raw = _policy_statements_from_text(join_rendered)
    join_statements, join_rewrites = _normalize_statements_for_policy_compartment(
        join_raw, iam_policy_compartment_id
    )
    if join_rewrites:
        summary["warnings"].append(
            f"KPO join policy: rewrote {join_rewrites} tenancy-scoped statement(s) to policy compartment scope."
        )
    summary["iam"]["kpo_nodes_join_policy"] = _ensure_policy(
        identity,
        iam_policy_compartment_id,
        join_policy_name,
        join_statements,
        f"KPO node CLUSTER_JOIN policy for run {run_id}",
    )
    workload_rendered = _render_template(
        workload_template, {"__KPO_CLUSTER_OCID__": kpo_cluster_id}
    )
    workload_rendered = _rewrite_workload_principal_targets(
        workload_rendered, kpo_namespace, kpo_service_account
    )
    workload_raw = _policy_statements_from_text(workload_rendered)
    workload_statements, workload_rewrites = (
        _normalize_statements_for_policy_compartment(
            workload_raw, iam_policy_compartment_id
        )
    )
    if workload_rewrites:
        summary["warnings"].append(
            f"KPO workload policy: rewrote {workload_rewrites} tenancy-scoped statement(s) to policy compartment scope."
        )
    summary["iam"]["kpo_workload_policy"] = _ensure_policy(
        identity,
        iam_policy_compartment_id,
        workload_policy_name,
        workload_statements,
        f"KPO workload identity policy for run {run_id}",
    )
# ---------------------------
# CA health
# ---------------------------
if "ca" in selected_targets:
    ca_target = provisioning["targets"]["ca"]
    addon_names = _get_cluster_addon_names(
        oci_config_file,
        oci_profile,
        region,
        ca_target["cluster_id"],
        oci_retries,
        retry_delay_seconds,
    )
    if "ClusterAutoscaler" not in addon_names:
        raise RuntimeError("CA health gate failed: ClusterAutoscaler add-on not found.")
    if "ca" in unreachable_targets:
        summary["health"]["ca"] = {
            "k8s_api_reachable": False,
            "cluster_autoscaler_addon_present": True,
            "detected_cluster_addons": sorted(addon_names),
            "kubernetes_health_check_skipped": True,
            "skip_reason": "API unreachable.",
        }
    else:
        dep_obj, _, _, _ = _run_json_or_raise(
            [
                "kubectl",
                "--kubeconfig",
                ca_target["kubeconfig_path"],
                "-n",
                "kube-system",
                "get",
                "deploy",
                "-o",
                "json",
            ],
            retries=kubectl_retries,
            delay_seconds=retry_delay_seconds,
            retry_if=_is_transient_error,
        )
        ca_deps = [
            i
            for i in dep_obj.get("items", []) or []
            if "cluster-autoscaler" in ((i.get("metadata", {}) or {}).get("name", ""))
        ]
        if not ca_deps:
            raise RuntimeError(
                "CA health gate failed: no cluster-autoscaler deployment found."
            )
        available = int(ca_deps[0].get("status", {}).get("availableReplicas", 0) or 0)
        if available < 1:
            raise RuntimeError(
                "CA health gate failed: cluster-autoscaler has no available replicas."
            )
        summary["health"]["ca"] = {
            "k8s_api_reachable": True,
            "cluster_autoscaler_addon_present": True,
            "detected_cluster_addons": sorted(addon_names),
            "cluster_autoscaler_available_replicas": available,
        }
# ---------------------------
# KPO install + health + CRDs/resources
# ---------------------------
if "kpo" in selected_targets:
    kpo_target = provisioning["targets"]["kpo"]
    kpo_kubeconfig = kpo_target["kubeconfig_path"]
    if "kpo" in unreachable_targets:
        summary["health"]["kpo"] = {
            "k8s_api_reachable": False,
            "kpo_install_skipped": True,
            "skip_reason": "API unreachable.",
        }
    else:
        values_template = tf_root / "kpo" / "manifests" / "kpo-values.yaml.tmpl"
        ocinodeclass_template = tf_root / "kpo" / "manifests" / "ocinodeclass.yaml.tmpl"
        nodepool_template = tf_root / "kpo" / "manifests" / "nodepool.yaml.tmpl"
        for p in [values_template, ocinodeclass_template, nodepool_template]:
            if not p.exists():
                raise RuntimeError(f"Missing required KPO template: {p}")
        endpoint, endpoint_source_used = _resolve_kpo_apiserver_endpoint(
            kpo_target, kpo_endpoint_source, kpo_kubeconfig
        )
        api_host = _extract_apiserver_host(endpoint)
        if not api_host:
            raise RuntimeError("KPO path: APISERVER endpoint host resolved empty.")
        values_text = values_template.read_text(encoding="utf-8").replace(
            "__APISERVER_ENDPOINT__", api_host
        )
        if "__APISERVER_ENDPOINT__" in values_text:
            raise RuntimeError(
                "KPO values render failed: unresolved __APISERVER_ENDPOINT__ placeholder."
            )
        values_text = (
            values_text.rstrip() + f"\n\nreplicaCount: {kpo_helm_replica_count}\n"
        )
        rendered_dir = artifact_dir / "rendered" / "kpo"
        rendered_dir.mkdir(parents=True, exist_ok=True)
        values_path = rendered_dir / "kpo-values.yaml"
        values_path.write_text(values_text, encoding="utf-8")
        _run_or_raise(
            [
                "helm",
                "upgrade",
                "--install",
                kpo_release_name,
                kpo_chart_ref,
                "--namespace",
                kpo_namespace,
                "--create-namespace",
                "--kubeconfig",
                kpo_kubeconfig,
                "-f",
                str(values_path),
            ],
            retries=helm_retries,
            delay_seconds=retry_delay_seconds,
            retry_if=_is_transient_error,
        )
        rollout_cmd = [
            "kubectl",
            "--kubeconfig",
            kpo_kubeconfig,
            "-n",
            kpo_namespace,
            "rollout",
            "status",
            "deployment",
            "-l",
            "app.kubernetes.io/name=karpenter",
            f"--timeout={kpo_rollout_timeout_seconds}s",
        ]
        rollout_rc, rollout_out, rollout_err, rollout_attempts = _run_with_retries(
            rollout_cmd,
            retries=kubectl_retries,
            delay_seconds=retry_delay_seconds,
            retry_if=_is_transient_error,
        )
        rollout_diag = _collect_kpo_rollout_diagnostics(kpo_kubeconfig, kpo_namespace)
        summary["health"]["kpo_rollout_initial"] = {
            "rc": rollout_rc,
            "attempts_used": rollout_attempts,
            "output_tail": "\n".join(
                (rollout_out or rollout_err or "").splitlines()[-30:]
            ),
            "desired_replicas": rollout_diag.get("desired_replicas", 0),
            "available_replicas": rollout_diag.get("available_replicas", 0),
            "pods": rollout_diag.get("pods", []),
            "cni_sandbox_failure_nodes": rollout_diag.get(
                "cni_sandbox_failure_nodes", []
            ),
            "cni_sandbox_events": rollout_diag.get("cni_sandbox_events", []),
        }
        if (
            rollout_rc != 0
            and kpo_auto_repair_node_cni
            and rollout_diag.get("cni_sandbox_failure_nodes")
        ):
            pool_meta = _resolve_kpo_baseline_pool_meta(provisioning)
            pool_name = pool_meta["pool_name"]
            pool_id = pool_meta["pool_id"]
            desired_pool_size = int(pool_meta["desired_size"])
            pool_nodes = _list_pool_nodes(kpo_kubeconfig, pool_name)
            by_name = {n["name"]: n for n in pool_nodes}
            repaired_nodes = []
            for node_name in rollout_diag.get("cni_sandbox_failure_nodes", []):
                node_info = by_name.get(node_name, {})
                instance_id = node_info.get("instance_id", "")
                if not instance_id:
                    continue
                print(
                    f"Auto-repair: deleting node {node_name} ({instance_id}) from pool {pool_name}"
                )
                _delete_and_replace_pool_node(
                    pool_id,
                    instance_id,
                    oci_config_file,
                    oci_profile,
                    region,
                    oci_retries,
                    retry_delay_seconds,
                    kpo_node_repair_wait_seconds,
                )
                repaired_nodes.append(
                    {"node_name": node_name, "instance_id": instance_id}
                )
            wait_res = _wait_for_pool_nodes_ready(
                kpo_kubeconfig,
                pool_name,
                desired_pool_size,
                kpo_pool_ready_wait_seconds,
            )
            summary["health"]["kpo_rollout_repair"] = {
                "auto_repair_attempted": True,
                "pool_name": pool_name,
                "pool_id": pool_id,
                "desired_pool_size": desired_pool_size,
                "repaired_nodes": repaired_nodes,
                "pool_ready_timeout": wait_res.get("timeout", False),
                "pool_ready_count": wait_res.get("ready_count", 0),
                "pool_nodes_after_repair": wait_res.get("nodes", []),
            }
            rollout_cmd_retry = [
                "kubectl",
                "--kubeconfig",
                kpo_kubeconfig,
                "-n",
                kpo_namespace,
                "rollout",
                "status",
                "deployment",
                "-l",
                "app.kubernetes.io/name=karpenter",
                f"--timeout={kpo_rollout_retry_timeout_seconds}s",
            ]
            rollout_rc, rollout_out, rollout_err, rollout_attempts2 = _run_with_retries(
                rollout_cmd_retry,
                retries=kubectl_retries,
                delay_seconds=retry_delay_seconds,
                retry_if=_is_transient_error,
            )
            rollout_diag = _collect_kpo_rollout_diagnostics(
                kpo_kubeconfig, kpo_namespace
            )
            summary["health"]["kpo_rollout_after_repair"] = {
                "rc": rollout_rc,
                "attempts_used": rollout_attempts2,
                "output_tail": "\n".join(
                    (rollout_out or rollout_err or "").splitlines()[-30:]
                ),
                "desired_replicas": rollout_diag.get("desired_replicas", 0),
                "available_replicas": rollout_diag.get("available_replicas", 0),
                "pods": rollout_diag.get("pods", []),
                "cni_sandbox_failure_nodes": rollout_diag.get(
                    "cni_sandbox_failure_nodes", []
                ),
                "cni_sandbox_events": rollout_diag.get("cni_sandbox_events", []),
            }
        if rollout_rc != 0:
            available = int(rollout_diag.get("available_replicas", 0))
            desired = int(rollout_diag.get("desired_replicas", 0))
            if kpo_allow_degraded_rollout and available >= kpo_min_available_replicas:
                summary["warnings"].append(
                    "KPO rollout timed out but minimum availability gate passed. "
                    f"available={available}, desired={desired}, min_required={kpo_min_available_replicas}"
                )
            else:
                raise RuntimeError(
                    "KPO rollout failed. "
                    f"available={available}, desired={desired}, "
                    f"detected_cni_nodes={rollout_diag.get('cni_sandbox_failure_nodes', [])}. "
                    "Inspect kpo_rollout_* diagnostics in post_provisioning.json."
                )
        required_crds = [
            "nodepools.karpenter.sh",
            "nodeclaims.karpenter.sh",
            "ocinodeclasses.oci.oraclecloud.com",
        ]
        for crd in required_crds:
            _run_or_raise(
                [
                    "kubectl",
                    "--kubeconfig",
                    kpo_kubeconfig,
                    "get",
                    "crd",
                    crd,
                    "-o",
                    "name",
                ],
                retries=kubectl_retries,
                delay_seconds=retry_delay_seconds,
                retry_if=_is_transient_error,
            )
        worker_subnet_id = (kpo_target.get("worker_subnet_id") or "").strip()
        pod_subnet_id = (kpo_target.get("pod_subnet_id") or "").strip()
        if not worker_subnet_id:
            raise RuntimeError(
                "KPO path: worker_subnet_id missing from Cell 7 outputs."
            )
        if not pod_subnet_id:
            raise RuntimeError("KPO path: pod_subnet_id missing from Cell 7 outputs.")
        worker_nsg_id, pod_nsg_id = _discover_kpo_nsg_ids_from_baseline_nodes(
            kubeconfig=kpo_kubeconfig,
            worker_subnet_id=worker_subnet_id,
            pod_subnet_id=pod_subnet_id,
            oci_config_file=oci_config_file,
            oci_profile=oci_profile,
            region=region,
            kubectl_retries=kubectl_retries,
            oci_retries=oci_retries,
            retry_delay_seconds=retry_delay_seconds,
        )
        ocinodeclass_text = _render_template(
            ocinodeclass_template,
            {
                "__WORKER_SUBNET_ID__": worker_subnet_id,
                "__POD_SUBNET_ID__": pod_subnet_id,
            },
        )
        if (
            "__WORKER_SUBNET_ID__" in ocinodeclass_text
            or "__POD_SUBNET_ID__" in ocinodeclass_text
        ):
            raise RuntimeError(
                "KPO OCINodeClass render failed: unresolved subnet placeholders."
            )
        ocinodeclass_text = _inject_nsgs_into_ocinodeclass(
            ocinodeclass_text,
            worker_subnet_id,
            pod_subnet_id,
            worker_nsg_id,
            pod_nsg_id,
        )
        nodepool_text = nodepool_template.read_text(encoding="utf-8")
        cpu_limit, memory_limit = _extract_nodepool_limits(nodepool_text)
        if not cpu_limit or not memory_limit:
            raise RuntimeError(
                "NodePool limits gate failed: cpu and memory limits must be set."
            )
        ip_count = _parse_ip_count(ocinodeclass_text)
        if not _is_power_of_two(ip_count):
            raise RuntimeError(
                f"KPO VCN-native gate failed: ipCount must be power-of-two, got {ip_count}."
            )
        cni_ver, cni_attempts = _detect_vcn_native_cni_version(
            kpo_kubeconfig, kubectl_retries, retry_delay_seconds
        )
        cni_tuple = _version_tuple(cni_ver)
        if cni_tuple is None:
            raise RuntimeError(
                f"KPO VCN-native gate failed: could not parse CNI version ({cni_ver})."
            )
        if cni_tuple < (3, 0, 0):
            raise RuntimeError(
                f"KPO VCN-native gate failed: add-on must be >=3.0.0 (got {cni_ver})."
            )
        if cni_tuple < (3, 2, 0):
            summary["warnings"].append(
                f"KPO CNI {cni_ver} below recommended 3.2.0; enforcing ipCount<=16."
            )
            if ip_count > 16:
                raise RuntimeError(
                    f"KPO VCN-native gate failed: CNI {cni_ver} requires ipCount<=16 (got {ip_count})."
                )
        ocinodeclass_path = rendered_dir / "ocinodeclass.yaml"
        nodepool_path = rendered_dir / "nodepool.yaml"
        ocinodeclass_path.write_text(ocinodeclass_text, encoding="utf-8")
        nodepool_path.write_text(nodepool_text, encoding="utf-8")
        _run_or_raise(
            [
                "kubectl",
                "--kubeconfig",
                kpo_kubeconfig,
                "apply",
                "-f",
                str(ocinodeclass_path),
            ],
            retries=kubectl_retries,
            delay_seconds=retry_delay_seconds,
            retry_if=_is_transient_error,
        )
        _run_or_raise(
            [
                "kubectl",
                "--kubeconfig",
                kpo_kubeconfig,
                "apply",
                "-f",
                str(nodepool_path),
            ],
            retries=kubectl_retries,
            delay_seconds=retry_delay_seconds,
            retry_if=_is_transient_error,
        )
        _run_or_raise(
            [
                "kubectl",
                "--kubeconfig",
                kpo_kubeconfig,
                "get",
                "ocinodeclass",
                "benchmark-ocinodeclass",
                "-o",
                "yaml",
            ],
            retries=kubectl_retries,
            delay_seconds=retry_delay_seconds,
            retry_if=_is_transient_error,
        )
        _run_or_raise(
            [
                "kubectl",
                "--kubeconfig",
                kpo_kubeconfig,
                "get",
                "nodepool",
                "benchmark-nodepool",
                "-o",
                "yaml",
            ],
            retries=kubectl_retries,
            delay_seconds=retry_delay_seconds,
            retry_if=_is_transient_error,
        )
        nodeclaims_obj, _, _, _ = _run_json_or_raise(
            [
                "kubectl",
                "--kubeconfig",
                kpo_kubeconfig,
                "get",
                "nodeclaims",
                "-A",
                "-o",
                "json",
            ],
            retries=kubectl_retries,
            delay_seconds=retry_delay_seconds,
            retry_if=_is_transient_error,
        )
        nodeclaims = nodeclaims_obj.get("items", []) or []
        nodeclaim_names = [
            f"{(i.get('metadata', {}) or {}).get('namespace','')}/{(i.get('metadata', {}) or {}).get('name','')}"
            for i in nodeclaims
            if isinstance(i, dict)
        ]
        summary["health"]["kpo"] = {
            "k8s_api_reachable": True,
            "controller_namespace": kpo_namespace,
            "helm_release_name": kpo_release_name,
            "helm_chart_source": kpo_chart_ref,
            "helm_replica_count_applied": kpo_helm_replica_count,
            "rollout_min_available_replicas": kpo_min_available_replicas,
            "rollout_allow_degraded": kpo_allow_degraded_rollout,
            "rollout_auto_repair_node_cni": kpo_auto_repair_node_cni,
            "apiserver_endpoint_source": endpoint_source_used,
            "apiserver_endpoint_selected": endpoint,
            "apiserver_endpoint_host_applied": api_host,
            "required_crds_present": required_crds,
            "vcn_native_cni_version": cni_ver,
            "vcn_native_cni_detection_attempts": cni_attempts,
            "ip_count": ip_count,
            "ip_count_power_of_two": True,
            "vcn_native_gate_passed": True,
            "worker_nsg_id": worker_nsg_id,
            "pod_nsg_id": pod_nsg_id,
            "nodepool_limits_applied": {"cpu": cpu_limit, "memory": memory_limit},
            "nodeclaim_count_diagnostic": len(nodeclaims),
            "nodeclaim_names_diagnostic": nodeclaim_names[:50],
        }
        summary["applied_manifests"]["kpo"] = {
            "helm_values": str(values_path),
            "ocinodeclass": str(ocinodeclass_path),
            "nodepool": str(nodepool_path),
        }
# ---------------------------
# Persist
# ---------------------------
post_provision_path = artifact_dir / "post_provisioning.json"
_write_json(post_provision_path, summary)
os.environ["POST_PROVISIONING_ARTIFACT"] = str(post_provision_path)
print("\nCell 8 summary:")
print(json.dumps(summary, indent=2))
print(f"\nCell 8 complete. Artifact: {post_provision_path}")
print("Next: Run Cell 9.")

In [ ]:
# Cell 9 — Deploy Benchmark Workload (identical profile for CA and KPO)

import os
import re
import json
import time
import subprocess
from pathlib import Path
from datetime import datetime, timezone


def _env(k, d=""):
    return os.environ.get(k, d)


def _require(k):
    v = _env(k, "").strip()
    if not v:
        raise RuntimeError(f"Missing required env var: {k}")
    return v


def _as_int(v, default):
    try:
        return int(str(v).strip())
    except Exception:
        return default


def _run(cmd, capture=False, check=True):
    print(f"\n$ {' '.join(cmd)}")
    if capture:
        p = subprocess.run(cmd, text=True, capture_output=True)
        if p.stdout:
            print(p.stdout)
        if p.stderr:
            print(p.stderr)
        if check and p.returncode != 0:
            raise RuntimeError(f"Command failed (exit {p.returncode}): {' '.join(cmd)}")
        return p.stdout
    rc = subprocess.run(cmd, text=True).returncode
    if check and rc != 0:
        raise RuntimeError(f"Command failed (exit {rc}): {' '.join(cmd)}")
    return ""


def _run_json(cmd):
    out = _run(cmd, capture=True)
    try:
        return json.loads(out or "{}")
    except Exception as e:
        tail = (out or "")[-2000:]
        raise RuntimeError(
            f"Failed to parse JSON from command: {' '.join(cmd)}; error={e}; output_tail={tail}"
        )


def _write_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def _load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def _require_file(path: Path, label: str):
    if not path.exists():
        raise RuntimeError(f"Missing {label}: {path}")


def _extract_nodepool_name_from_yaml(yaml_text: str) -> str:
    m = re.search(
        r"(?ms)^kind:\s*NodePool\s*$.*?^metadata:\s*$.*?^\s*name:\s*([^\s#]+)\s*$",
        yaml_text,
    )
    if not m:
        return ""
    return m.group(1).strip().strip("'").strip('"')


def _resolve_kpo_nodepool_name(post_provisioning: dict):
    explicit = _env("KPO_WORKLOAD_NODEPOOL", "").strip()
    if explicit:
        return explicit, "env:KPO_WORKLOAD_NODEPOOL"

    nodepool_manifest = (
        ((post_provisioning.get("applied_manifests") or {}).get("kpo") or {}).get(
            "nodepool"
        )
        or ""
    ).strip()
    if nodepool_manifest:
        p = Path(nodepool_manifest)
        if p.exists():
            name = _extract_nodepool_name_from_yaml(p.read_text(encoding="utf-8"))
            if name:
                return name, f"post_provisioning.applied_manifests.kpo.nodepool:{p}"

    return "benchmark-nodepool", "default:fallback"


def _deployment_pod_snapshot(kubeconfig: str, namespace: str, deploy_name: str):
    dep = _run_json(
        [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "-n",
            namespace,
            "get",
            "deploy",
            deploy_name,
            "-o",
            "json",
        ]
    )
    pods_obj = _run_json(
        [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "-n",
            namespace,
            "get",
            "pods",
            "-l",
            f"app.kubernetes.io/name={deploy_name}",
            "-o",
            "json",
        ]
    )

    pod_items = pods_obj.get("items", []) or []
    phase_counts = {}
    pod_brief = []
    for p in pod_items:
        status = p.get("status", {}) or {}
        phase = status.get("phase", "Unknown")
        phase_counts[phase] = int(phase_counts.get(phase, 0)) + 1

        cstats = status.get("containerStatuses", []) or []
        ready = bool(cstats) and all(bool(cs.get("ready")) for cs in cstats)

        waiting_reason = ""
        for cs in cstats:
            st = cs.get("state", {}) or {}
            waiting = st.get("waiting") or {}
            if waiting.get("reason"):
                waiting_reason = waiting.get("reason")
                break

        pod_brief.append(
            {
                "name": (p.get("metadata", {}) or {}).get("name", ""),
                "phase": phase,
                "ready": ready,
                "node": status.get("nodeName", ""),
                "pod_ip": status.get("podIP", ""),
                "waiting_reason": waiting_reason,
            }
        )

    dep_spec = dep.get("spec", {}) or {}
    dep_status = dep.get("status", {}) or {}

    return {
        "deployment": {
            "spec_replicas": int(dep_spec.get("replicas", 0) or 0),
            "available_replicas": int(dep_status.get("availableReplicas", 0) or 0),
            "ready_replicas": int(dep_status.get("readyReplicas", 0) or 0),
            "updated_replicas": int(dep_status.get("updatedReplicas", 0) or 0),
            "unavailable_replicas": int(dep_status.get("unavailableReplicas", 0) or 0),
        },
        "pods": {
            "total": len(pod_items),
            "phase_counts": phase_counts,
            "items": pod_brief,
        },
    }


def _wait_for_podset(
    kubeconfig: str,
    namespace: str,
    deploy_name: str,
    expected_replicas: int,
    timeout_seconds: int,
    poll_seconds: int,
):
    # For base replica floor 0, deployment existence is the only required gate.
    if expected_replicas == 0:
        snap = _deployment_pod_snapshot(kubeconfig, namespace, deploy_name)
        return True, 0, snap

    start = time.time()
    last = {}
    while True:
        snap = _deployment_pod_snapshot(kubeconfig, namespace, deploy_name)
        last = snap

        dep_spec = int(snap["deployment"]["spec_replicas"])
        pod_total = int(snap["pods"]["total"])

        if dep_spec == expected_replicas and pod_total >= expected_replicas:
            return True, int(time.time() - start), snap

        elapsed = int(time.time() - start)
        if elapsed >= timeout_seconds:
            return False, elapsed, last

        time.sleep(poll_seconds)


run_id = _require("RUN_ID")
output_dir = Path(os.path.expanduser(_require("OUTPUT_DIR"))).resolve()
artifact_dir = output_dir / run_id

provisioning_path = artifact_dir / "provisioning.json"
post_provisioning_path = artifact_dir / "post_provisioning.json"

_require_file(provisioning_path, "provisioning artifact from Cell 7")
provisioning = _load_json(provisioning_path)

post_provisioning = {}
if post_provisioning_path.exists():
    post_provisioning = _load_json(post_provisioning_path)

target_cluster = _env("TARGET_CLUSTER", "both").strip().lower()
if target_cluster not in {"ca", "kpo", "both"}:
    raise RuntimeError(f"TARGET_CLUSTER must be ca|kpo|both, got: {target_cluster}")

requested_targets = ["ca", "kpo"] if target_cluster == "both" else [target_cluster]
provisioned_targets = provisioning.get("selected_targets") or sorted(
    (provisioning.get("targets") or {}).keys()
)

missing_targets = [t for t in requested_targets if t not in provisioned_targets]
if missing_targets:
    raise RuntimeError(
        f"Requested target(s) not provisioned in this run: {missing_targets}. "
        f"Provisioned targets: {provisioned_targets}."
    )

selected_targets = requested_targets
target_scenarios = _env("TARGET_SCENARIOS", "all").strip().lower()

workload_name = _env("WORKLOAD_NAME", "benchmark-load").strip()
if not workload_name:
    raise RuntimeError("WORKLOAD_NAME must not be empty")

workload_base_replicas = _as_int(_env("WORKLOAD_BASE_REPLICAS", "0"), 0)
if workload_base_replicas < 0:
    raise RuntimeError("WORKLOAD_BASE_REPLICAS must be >= 0")

workload_image = _env("WORKLOAD_IMAGE", "registry.k8s.io/pause:3.9").strip()
workload_cpu_request = _env("WORKLOAD_CPU_REQUEST", "500m").strip()
workload_mem_request = _env("WORKLOAD_MEM_REQUEST", "512Mi").strip()
workload_cpu_limit = _env("WORKLOAD_CPU_LIMIT", workload_cpu_request).strip()
workload_mem_limit = _env("WORKLOAD_MEM_LIMIT", workload_mem_request).strip()

for n, v in [
    ("WORKLOAD_IMAGE", workload_image),
    ("WORKLOAD_CPU_REQUEST", workload_cpu_request),
    ("WORKLOAD_MEM_REQUEST", workload_mem_request),
    ("WORKLOAD_CPU_LIMIT", workload_cpu_limit),
    ("WORKLOAD_MEM_LIMIT", workload_mem_limit),
]:
    if not v:
        raise RuntimeError(f"{n} must not be empty")

ns_prefix = _env("WORKLOAD_NAMESPACE_PREFIX", "benchmark").strip() or "benchmark"
workload_namespace_ca = (
    _env("WORKLOAD_NAMESPACE_CA", f"{ns_prefix}-ca").strip() or f"{ns_prefix}-ca"
)
workload_namespace_kpo = (
    _env("WORKLOAD_NAMESPACE_KPO", f"{ns_prefix}-kpo").strip() or f"{ns_prefix}-kpo"
)

podset_wait_seconds = _as_int(
    _env("CELL9_PODSET_WAIT_SECONDS", _env("SCENARIO_PENDING_TIMEOUT_SECONDS", "300")),
    300,
)
podset_poll_seconds = _as_int(
    _env("CELL9_PODSET_POLL_SECONDS", _env("SCENARIO_SCALE_UP_POLL_SECONDS", "2")),
    2,
)
if podset_wait_seconds < 0:
    raise RuntimeError("CELL9_PODSET_WAIT_SECONDS must be >= 0")
if podset_poll_seconds < 1:
    raise RuntimeError("CELL9_PODSET_POLL_SECONDS must be >= 1")

kpo_workload_nodepool, kpo_nodepool_source = _resolve_kpo_nodepool_name(
    post_provisioning
)

render_dir = artifact_dir / "rendered" / "workload"
render_dir.mkdir(parents=True, exist_ok=True)

summary = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
    "run_id": run_id,
    "target_cluster": target_cluster,
    "selected_targets": selected_targets,
    "target_scenarios": target_scenarios,
    "workload_profile": {
        "name": workload_name,
        "image": workload_image,
        "base_replicas": workload_base_replicas,
        "resources": {
            "requests": {"cpu": workload_cpu_request, "memory": workload_mem_request},
            "limits": {"cpu": workload_cpu_limit, "memory": workload_mem_limit},
        },
    },
    "cell9_policy": {
        "podset_wait_seconds": podset_wait_seconds,
        "podset_poll_seconds": podset_poll_seconds,
        "kpo_nodepool_source": kpo_nodepool_source,
    },
    "targets": {},
}

print("=== Cell 9 Workload Deploy ===")
print(f"run_id: {run_id}")
print(f"selected_targets: {selected_targets}")
print(f"base_replicas: {workload_base_replicas}")
print(f"image: {workload_image}")
print(f"kpo_nodepool_selector: {kpo_workload_nodepool} ({kpo_nodepool_source})")
print(
    f"podset_wait_seconds: {podset_wait_seconds}, poll_seconds: {podset_poll_seconds}"
)

for t in selected_targets:
    if t not in (provisioning.get("targets") or {}):
        raise RuntimeError(f"Target {t} missing in provisioning.json targets map")

    tgt = provisioning["targets"][t]
    kubeconfig = (tgt.get("kubeconfig_path") or "").strip()
    if not kubeconfig:
        raise RuntimeError(f"{t}: missing kubeconfig_path in provisioning.json")
    if not Path(kubeconfig).exists():
        raise RuntimeError(f"{t}: kubeconfig file not found: {kubeconfig}")

    # Connectivity preflight for target cluster.
    version_raw = _run(
        [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "get",
            "--raw",
            "/version",
            "--request-timeout=15s",
        ],
        capture=True,
    )
    k8s_git_version = ""
    try:
        k8s_git_version = (json.loads(version_raw or "{}") or {}).get("gitVersion", "")
    except Exception:
        k8s_git_version = ""

    if t == "kpo":
        # Validate selected KPO nodepool exists before binding workload selector.
        try:
            _run(
                [
                    "kubectl",
                    "--kubeconfig",
                    kubeconfig,
                    "get",
                    "nodepool",
                    kpo_workload_nodepool,
                    "-o",
                    "name",
                ],
                capture=True,
            )
        except Exception:
            existing = _run(
                [
                    "kubectl",
                    "--kubeconfig",
                    kubeconfig,
                    "get",
                    "nodepool",
                    "-o",
                    "name",
                ],
                capture=True,
                check=False,
            ).strip()
            raise RuntimeError(
                f"kpo: workload nodepool not found: {kpo_workload_nodepool}. "
                f"Set KPO_WORKLOAD_NODEPOOL or verify Cell 8 outputs.\n"
                f"Existing NodePools:\n{existing or '<none>'}"
            )
        selector_items = [("karpenter.sh/nodepool", kpo_workload_nodepool)]
        namespace = workload_namespace_kpo
    else:
        selector_items = [
            ("benchmark.oracle.com/autoscaler", "ca"),
            ("benchmark.oracle.com/role", "benchmark"),
        ]
        namespace = workload_namespace_ca

    selector_dict = {k: v for k, v in selector_items}
    node_selector_yaml = "\n".join([f'        {k}: "{v}"' for k, v in selector_items])

    manifest = f"""apiVersion: v1
kind: Namespace
metadata:
  name: {namespace}
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: {workload_name}
  namespace: {namespace}
  labels:
    app.kubernetes.io/name: {workload_name}
    benchmark.oracle.com/autoscaler: "{t}"
spec:
  replicas: {workload_base_replicas}
  selector:
    matchLabels:
      app.kubernetes.io/name: {workload_name}
  template:
    metadata:
      labels:
        app.kubernetes.io/name: {workload_name}
        benchmark.oracle.com/autoscaler: "{t}"
    spec:
      nodeSelector:
{node_selector_yaml}
      containers:
      - name: {workload_name}
        image: {workload_image}
        imagePullPolicy: IfNotPresent
        resources:
          requests:
            cpu: "{workload_cpu_request}"
            memory: "{workload_mem_request}"
          limits:
            cpu: "{workload_cpu_limit}"
            memory: "{workload_mem_limit}"
"""

    manifest_path = render_dir / f"{t}-workload.yaml"
    manifest_path.write_text(manifest, encoding="utf-8")

    _run(["kubectl", "--kubeconfig", kubeconfig, "apply", "-f", str(manifest_path)])

    ok, elapsed, snap = _wait_for_podset(
        kubeconfig=kubeconfig,
        namespace=namespace,
        deploy_name=workload_name,
        expected_replicas=workload_base_replicas,
        timeout_seconds=podset_wait_seconds,
        poll_seconds=podset_poll_seconds,
    )
    if not ok:
        raise RuntimeError(
            f"{t}: workload pod set did not materialize within {podset_wait_seconds}s. "
            f"Expected replicas={workload_base_replicas}; last_snapshot={json.dumps(snap, indent=2)}"
        )

    pod_counts = snap["pods"]["phase_counts"]
    pending = int(pod_counts.get("Pending", 0))
    running = int(pod_counts.get("Running", 0))

    summary["targets"][t] = {
        "namespace": namespace,
        "deployment": workload_name,
        "kubeconfig": kubeconfig,
        "manifest_path": str(manifest_path),
        "replicas_spec": int(snap["deployment"]["spec_replicas"]),
        "replicas_available": int(snap["deployment"]["available_replicas"]),
        "pod_count": int(snap["pods"]["total"]),
        "pod_pending": pending,
        "pod_running": running,
        "pod_phase_counts": pod_counts,
        "pods": snap["pods"]["items"],
        "node_selector": selector_dict,
        "k8s_api_git_version": k8s_git_version,
        "podset_wait": {
            "expected_replicas": workload_base_replicas,
            "elapsed_seconds": elapsed,
            "timeout_seconds": podset_wait_seconds,
            "poll_seconds": podset_poll_seconds,
            "podset_exists": True,
        },
    }

    os.environ[f"WORKLOAD_NAMESPACE_{t.upper()}"] = namespace
    os.environ[f"WORKLOAD_DEPLOYMENT_{t.upper()}"] = workload_name

if "kpo" in selected_targets:
    os.environ["WORKLOAD_NAMESPACE_KPO"] = summary["targets"]["kpo"]["namespace"]
    os.environ["WORKLOAD_DEPLOYMENT_KPO"] = workload_name
    os.environ["KPO_WORKLOAD_NODEPOOL"] = kpo_workload_nodepool

artifact_path = artifact_dir / "workload_deployments.json"
_write_json(artifact_path, summary)
os.environ["WORKLOAD_DEPLOYMENT_ARTIFACT"] = str(artifact_path)

print("\nCell 9 summary:")
print(json.dumps(summary, indent=2))
print(f"\nCell 9 complete. Artifact: {artifact_path}")

if workload_base_replicas == 0:
    print("Note: WORKLOAD_BASE_REPLICAS=0, so no NodeClaims are expected yet.")
    print("Cell 10 should drive scenario replicas and trigger autoscaler behavior.")
print("Next: Run Cell 10.")

In [ ]:
# Cell 10 — Burst + Scale-Down only (Primary request-based events only)

import os
import json
import time
import subprocess
from pathlib import Path
from datetime import datetime, timezone


def _env(k, d=""):
    return os.environ.get(k, d)


def _require(k):
    v = _env(k, "").strip()
    if not v:
        raise RuntimeError(f"Missing required env var: {k}")
    return v


def _bool(v, default=False):
    if v is None:
        return default
    return str(v).strip().lower() in {"1", "true", "yes", "y", "on"}


def _as_int(v, name):
    try:
        return int(str(v).strip())
    except Exception:
        raise RuntimeError(f"{name} must be an integer, got: {v}")


def _now_utc():
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def _run(cmd, capture=False, check=True):
    print(f"\n$ {' '.join(cmd)}")
    if capture:
        p = subprocess.run(cmd, text=True, capture_output=True)
        if p.stdout:
            print(p.stdout)
        if p.stderr:
            print(p.stderr)
        if check and p.returncode != 0:
            raise RuntimeError(f"Command failed (exit {p.returncode}): {' '.join(cmd)}")
        return p.stdout
    rc = subprocess.run(cmd, text=True).returncode
    if check and rc != 0:
        raise RuntimeError(f"Command failed (exit {rc}): {' '.join(cmd)}")
    return ""


def _run_json(cmd):
    out = _run(cmd, capture=True)
    try:
        return json.loads(out or "{}")
    except Exception as e:
        raise RuntimeError(
            f"Failed to parse JSON output from: {' '.join(cmd)} ; error={e}"
        )


def _write_json(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def _append_jsonl(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(obj, sort_keys=True) + "\n")


def _deployment_status(kubeconfig, ns, deploy):
    obj = _run_json(
        [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "-n",
            ns,
            "get",
            "deploy",
            deploy,
            "-o",
            "json",
        ]
    )
    spec = obj.get("spec", {}) or {}
    status = obj.get("status", {}) or {}
    return {
        "spec_replicas": int(spec.get("replicas", 0) or 0),
        "available_replicas": int(status.get("availableReplicas", 0) or 0),
        "ready_replicas": int(status.get("readyReplicas", 0) or 0),
        "updated_replicas": int(status.get("updatedReplicas", 0) or 0),
        "unavailable_replicas": int(status.get("unavailableReplicas", 0) or 0),
    }


def _pod_phase_counts(kubeconfig, ns, deploy):
    obj = _run_json(
        [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "-n",
            ns,
            "get",
            "pods",
            "-l",
            f"app.kubernetes.io/name={deploy}",
            "-o",
            "json",
        ]
    )
    counts = {"Pending": 0, "Running": 0, "Succeeded": 0, "Failed": 0, "Unknown": 0}
    items = obj.get("items", []) or []
    for it in items:
        phase = (it.get("status", {}) or {}).get("phase", "Unknown")
        if phase not in counts:
            phase = "Unknown"
        counts[phase] += 1
    counts["total"] = len(items)
    return counts


def _benchmark_nodes(kubeconfig, target, kpo_nodepool_name):
    if target == "kpo":
        label_selector = f"karpenter.sh/nodepool={kpo_nodepool_name}"
    else:
        label_selector = f"benchmark.oracle.com/autoscaler={target},benchmark.oracle.com/role=benchmark"

    obj = _run_json(
        [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "get",
            "nodes",
            "-l",
            label_selector,
            "-o",
            "json",
        ]
    )
    items = obj.get("items", []) or []
    ready = 0
    for n in items:
        for c in (n.get("status", {}) or {}).get("conditions", []) or []:
            if c.get("type") == "Ready" and c.get("status") == "True":
                ready += 1
                break
    return {"selector": label_selector, "total": len(items), "ready": ready}


def _nodeclaims_count(kubeconfig):
    obj = _run_json(
        ["kubectl", "--kubeconfig", kubeconfig, "get", "nodeclaims", "-A", "-o", "json"]
    )
    return len(obj.get("items", []) or [])


def _scale_deploy(kubeconfig, ns, deploy, replicas):
    _run(
        [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "-n",
            ns,
            "scale",
            "deploy",
            deploy,
            f"--replicas={replicas}",
        ]
    )


def _wait_loop(check_fn, timeout_s, poll_s):
    start = time.time()
    last = {}
    while time.time() - start <= timeout_s:
        ok, snapshot = check_fn()
        last = snapshot
        if ok:
            return True, int(time.time() - start), snapshot
        time.sleep(poll_s)
    return False, int(time.time() - start), last


def _execute_single_profile(
    profile_name,
    up_replicas,
    hold_seconds,
    down_replicas,
    repeats,
    poll_s,
    pending_timeout_s,  # retained for call compatibility with Cell 11; not used in primary-only flow
    ready_timeout_s,
    scale_down_timeout_s,
    selected_targets,
    workload,
    kpo_nodepool_name,
    expected_floor,
    require_kpo_claims_floor,
    artifact_dir,
    run_id,
    precheck_timeout_s=900,
    poll_up_s=None,
    poll_down_s=None,
):
    poll_up_s = int(poll_s if poll_up_s is None else poll_up_s)
    poll_down_s = int(poll_s if poll_down_s is None else poll_down_s)

    events_jsonl = artifact_dir / f"scenario_events_{profile_name}.jsonl"
    if events_jsonl.exists():
        events_jsonl.unlink()

    summary = {
        "generated_at_utc": _now_utc(),
        "run_id": run_id,
        "profile": profile_name,
        "selected_targets": selected_targets,
        "repeats": repeats,
        "event_contract_version": "v1",
        "legacy_compatibility_enabled": False,
        "config": {
            "up_replicas": up_replicas,
            "down_replicas": down_replicas,
            "hold_seconds": hold_seconds,
            "poll_seconds": poll_s,
            "poll_up_seconds": poll_up_s,
            "poll_down_seconds": poll_down_s,
            "pending_timeout_seconds": pending_timeout_s,
            "ready_timeout_seconds": ready_timeout_s,
            "scale_down_timeout_seconds": scale_down_timeout_s,
            "precheck_timeout_seconds": precheck_timeout_s,
            "kpo_nodepool_selector": kpo_nodepool_name,
            "expected_floor": expected_floor,
            "require_kpo_claims_floor": require_kpo_claims_floor,
        },
        "runs": [],
    }

    def _emit(ev):
        _append_jsonl(events_jsonl, ev)
        print(
            f"[event] {ev['target']} r{ev['repeat']} {profile_name} -> {ev['event']} @ {ev['time_utc']}"
        )

    def _snapshot(kubeconfig, ns, deploy, target):
        dep = _deployment_status(kubeconfig, ns, deploy)
        pods = _pod_phase_counts(kubeconfig, ns, deploy)
        nodes = _benchmark_nodes(kubeconfig, target, kpo_nodepool_name)
        nodeclaims = _nodeclaims_count(kubeconfig) if target == "kpo" else None
        return {
            "deployment": dep,
            "pods": pods,
            "benchmark_nodes": nodes,
            "nodeclaims": nodeclaims,
        }

    def _workload_at_floor(snap, desired_replicas):
        dep = snap["deployment"]
        pods = snap["pods"]
        return (
            dep["spec_replicas"] == desired_replicas
            and dep["available_replicas"] <= desired_replicas
            and pods["total"] == desired_replicas
        )

    def _capacity_at_floor(snap, target, floor):
        nodes = snap["benchmark_nodes"]
        nodeclaims = snap.get("nodeclaims")
        floor_ok = nodes["ready"] <= floor
        claims_ok = True
        if target == "kpo" and require_kpo_claims_floor:
            claims_ok = (nodeclaims or 0) <= floor
        return floor_ok and claims_ok

    def _both_at_floor(snap, target, floor, desired_replicas):
        return _workload_at_floor(snap, desired_replicas) and _capacity_at_floor(
            snap, target, floor
        )

    for repeat in range(1, repeats + 1):
        for t in selected_targets:
            tgt = workload.get("targets", {}).get(t)
            if not tgt:
                raise RuntimeError(
                    f"{t}: missing workload target in workload_deployments.json"
                )

            kubeconfig = tgt["kubeconfig"]
            ns = tgt["namespace"]
            deploy = tgt["deployment"]
            floor = int(expected_floor.get(t, 0))

            run_rec = {
                "target": t,
                "repeat": repeat,
                "profile": profile_name,
                "started_at_utc": _now_utc(),
                "up_replicas": up_replicas,
                "down_replicas": down_replicas,
                "hold_seconds": hold_seconds,
                "expected_floor": floor,
                "status": "running",
            }

            _emit(
                {
                    "run_id": run_id,
                    "target": t,
                    "repeat": repeat,
                    "scenario": profile_name,
                    "event": "precheck_start",
                    "time_utc": _now_utc(),
                    "details": {"expected_floor": floor},
                }
            )

            def _precheck():
                snap = _snapshot(kubeconfig, ns, deploy, t)
                return _both_at_floor(snap, t, floor, down_replicas), snap

            ok_pre, pre_elapsed, pre_snap = _wait_loop(
                _precheck, precheck_timeout_s, poll_up_s
            )
            if not ok_pre:
                _emit(
                    {
                        "run_id": run_id,
                        "target": t,
                        "repeat": repeat,
                        "scenario": profile_name,
                        "event": "precheck_failed",
                        "time_utc": _now_utc(),
                        "details": {"elapsed_seconds": pre_elapsed, **pre_snap},
                    }
                )
                run_rec["status"] = "failed_precheck_floor"
                run_rec["failed_stage"] = "precheck"
                run_rec["ended_at_utc"] = _now_utc()
                summary["runs"].append(run_rec)
                _emit(
                    {
                        "run_id": run_id,
                        "target": t,
                        "repeat": repeat,
                        "scenario": profile_name,
                        "event": "scenario_end",
                        "time_utc": _now_utc(),
                        "details": {"status": run_rec["status"]},
                    }
                )
                continue

            baseline_nodes = pre_snap["benchmark_nodes"]
            baseline_running = int(pre_snap["pods"].get("Running", 0) or 0)
            run_rec["baseline_benchmark_nodes"] = baseline_nodes
            run_rec["baseline_running_pods"] = baseline_running

            _emit(
                {
                    "run_id": run_id,
                    "target": t,
                    "repeat": repeat,
                    "scenario": profile_name,
                    "event": "scenario_start",
                    "time_utc": _now_utc(),
                    "details": {
                        "up_replicas": up_replicas,
                        "down_replicas": down_replicas,
                        "baseline_benchmark_nodes_ready": baseline_nodes["ready"],
                        "baseline_running_pods": baseline_running,
                        "benchmark_node_selector": baseline_nodes["selector"],
                        "expected_floor": floor,
                    },
                }
            )

            _emit(
                {
                    "run_id": run_id,
                    "target": t,
                    "repeat": repeat,
                    "scenario": profile_name,
                    "event": "scale_up_requested",
                    "time_utc": _now_utc(),
                    "details": {
                        "from_replicas": down_replicas,
                        "to_replicas": up_replicas,
                    },
                }
            )
            up_start = time.time()
            _scale_deploy(kubeconfig, ns, deploy, up_replicas)

            first_new_node_ready_emitted = False
            first_pod_running_emitted = False
            all_pods_running_emitted = False
            last_up_snap = {}

            while time.time() - up_start <= ready_timeout_s:
                elapsed = int(time.time() - up_start)
                snap = _snapshot(kubeconfig, ns, deploy, t)
                last_up_snap = snap
                pods = snap["pods"]
                dep = snap["deployment"]
                nodes = snap["benchmark_nodes"]

                node_growth = int(nodes.get("ready", 0)) > int(
                    baseline_nodes.get("ready", 0)
                )

                if (not first_new_node_ready_emitted) and node_growth:
                    first_new_node_ready_emitted = True
                    _emit(
                        {
                            "run_id": run_id,
                            "target": t,
                            "repeat": repeat,
                            "scenario": profile_name,
                            "event": "first_new_node_ready",
                            "time_utc": _now_utc(),
                            "details": {"elapsed_seconds": elapsed, **snap},
                        }
                    )

                if (not first_pod_running_emitted) and int(
                    pods.get("Running", 0)
                ) > baseline_running:
                    first_pod_running_emitted = True
                    _emit(
                        {
                            "run_id": run_id,
                            "target": t,
                            "repeat": repeat,
                            "scenario": profile_name,
                            "event": "first_pod_running",
                            "time_utc": _now_utc(),
                            "details": {"elapsed_seconds": elapsed, **snap},
                        }
                    )

                all_running_now = (
                    int(dep.get("spec_replicas", 0)) == up_replicas
                    and int(dep.get("available_replicas", 0)) >= up_replicas
                    and int(pods.get("Running", 0)) >= up_replicas
                    and int(pods.get("Pending", 0)) == 0
                )
                if all_running_now:
                    all_pods_running_emitted = True
                    _emit(
                        {
                            "run_id": run_id,
                            "target": t,
                            "repeat": repeat,
                            "scenario": profile_name,
                            "event": "all_pods_running",
                            "time_utc": _now_utc(),
                            "details": {"elapsed_seconds": elapsed, **snap},
                        }
                    )
                    break

                time.sleep(poll_up_s)

            if not all_pods_running_emitted:
                run_rec["status"] = "failed_upscale_timeout"
                run_rec["failed_stage"] = "all_pods_running"
                _emit(
                    {
                        "run_id": run_id,
                        "target": t,
                        "repeat": repeat,
                        "scenario": profile_name,
                        "event": "scale_up_issue",
                        "time_utc": _now_utc(),
                        "details": {
                            "elapsed_seconds": int(time.time() - up_start),
                            "reason": "all_pods_running_timeout",
                            **last_up_snap,
                        },
                    }
                )

            if hold_seconds > 0:
                print(f"[hold] {t} r{repeat} {profile_name}: sleeping {hold_seconds}s")
                time.sleep(hold_seconds)

            _emit(
                {
                    "run_id": run_id,
                    "target": t,
                    "repeat": repeat,
                    "scenario": profile_name,
                    "event": "scale_down_requested",
                    "time_utc": _now_utc(),
                    "details": {
                        "from_replicas": up_replicas,
                        "to_replicas": down_replicas,
                    },
                }
            )
            down_start = time.time()
            _scale_deploy(kubeconfig, ns, deploy, down_replicas)

            workload_floor_reached = False
            capacity_floor_reached = False
            last_down_snap = {}

            while time.time() - down_start <= scale_down_timeout_s:
                elapsed = int(time.time() - down_start)
                snap = _snapshot(kubeconfig, ns, deploy, t)
                last_down_snap = snap

                workload_ok = _workload_at_floor(snap, down_replicas)
                capacity_ok = _capacity_at_floor(snap, t, floor)

                if workload_ok and not workload_floor_reached:
                    workload_floor_reached = True
                    _emit(
                        {
                            "run_id": run_id,
                            "target": t,
                            "repeat": repeat,
                            "scenario": profile_name,
                            "event": "workload_floor_reached",
                            "time_utc": _now_utc(),
                            "details": {"elapsed_seconds": elapsed, **snap},
                        }
                    )

                if capacity_ok and not capacity_floor_reached:
                    capacity_floor_reached = True
                    _emit(
                        {
                            "run_id": run_id,
                            "target": t,
                            "repeat": repeat,
                            "scenario": profile_name,
                            "event": "capacity_floor_reached",
                            "time_utc": _now_utc(),
                            "details": {"elapsed_seconds": elapsed, **snap},
                        }
                    )

                if workload_floor_reached and capacity_floor_reached:
                    if run_rec["status"] == "running":
                        run_rec["status"] = "success"
                    break

                time.sleep(poll_down_s)

            if not (workload_floor_reached and capacity_floor_reached):
                _emit(
                    {
                        "run_id": run_id,
                        "target": t,
                        "repeat": repeat,
                        "scenario": profile_name,
                        "event": "scale_down_issue",
                        "time_utc": _now_utc(),
                        "details": {
                            "elapsed_seconds": int(time.time() - down_start),
                            "workload_floor_reached": workload_floor_reached,
                            "capacity_floor_reached": capacity_floor_reached,
                            **last_down_snap,
                        },
                    }
                )
                if run_rec["status"] == "running":
                    run_rec["status"] = "failed_scale_down_timeout"

            run_rec["ended_at_utc"] = _now_utc()
            summary["runs"].append(run_rec)

            _emit(
                {
                    "run_id": run_id,
                    "target": t,
                    "repeat": repeat,
                    "scenario": profile_name,
                    "event": "scenario_end",
                    "time_utc": _now_utc(),
                    "details": {"status": run_rec["status"]},
                }
            )

    summary_path = artifact_dir / f"scenario_execution_{profile_name}_summary.json"
    _write_json(summary_path, summary)

    os.environ[f"SCENARIO_EVENTS_{profile_name.upper()}_ARTIFACT"] = str(events_jsonl)
    os.environ[f"SCENARIO_EXECUTION_{profile_name.upper()}_SUMMARY_ARTIFACT"] = str(
        summary_path
    )

    print(f"\nCell 10 ({profile_name}) summary:")
    print(
        json.dumps(
            {
                "run_id": run_id,
                "profile": profile_name,
                "selected_targets": selected_targets,
                "repeats": repeats,
                "event_contract_version": "v1",
                "legacy_compatibility_enabled": False,
                "event_ledger": str(events_jsonl),
                "summary_json": str(summary_path),
                "run_count": len(summary["runs"]),
            },
            indent=2,
        )
    )
    print(f"\nCell 10 ({profile_name}) complete.")
    print(f"- Event ledger: {events_jsonl}")
    print(f"- Summary: {summary_path}")

    return events_jsonl, summary_path


run_id = _require("RUN_ID")
output_dir = Path(_require("OUTPUT_DIR"))
artifact_dir = output_dir / run_id

workload_path = artifact_dir / "workload_deployments.json"
if not workload_path.exists():
    raise RuntimeError(
        f"Missing workload deployment artifact from Cell 9: {workload_path}"
    )
workload = json.loads(workload_path.read_text(encoding="utf-8"))

target_cluster = _env("TARGET_CLUSTER", "both").strip().lower()
if target_cluster not in {"ca", "kpo", "both"}:
    raise RuntimeError(f"TARGET_CLUSTER must be ca|kpo|both, got: {target_cluster}")
selected_targets = ["ca", "kpo"] if target_cluster == "both" else [target_cluster]

workload_targets = workload.get("targets", {}) or {}
missing_workload_targets = [t for t in selected_targets if t not in workload_targets]
if missing_workload_targets:
    raise RuntimeError(
        f"Selected target(s) missing in Cell 9 workload artifact: {missing_workload_targets}. "
        f"Available targets: {sorted(workload_targets.keys())}"
    )

target_scenarios = _env("TARGET_SCENARIOS", "all").strip().lower()
if target_scenarios not in {"all", "burst", "steady", "scale_down"}:
    raise RuntimeError(
        f"TARGET_SCENARIOS must be all|burst|steady|scale_down, got: {target_scenarios}"
    )

repeats = _as_int(_env("REPEATS", "3"), "REPEATS")
poll_s = _as_int(_env("SCENARIO_POLL_SECONDS", "10"), "SCENARIO_POLL_SECONDS")
poll_up_s = _as_int(
    _env("SCENARIO_SCALE_UP_POLL_SECONDS", str(poll_s)),
    "SCENARIO_SCALE_UP_POLL_SECONDS",
)
poll_down_s = _as_int(
    _env("SCENARIO_SCALE_DOWN_POLL_SECONDS", str(poll_s)),
    "SCENARIO_SCALE_DOWN_POLL_SECONDS",
)
pending_timeout_s = _as_int(
    _env("SCENARIO_PENDING_TIMEOUT_SECONDS", "300"),
    "SCENARIO_PENDING_TIMEOUT_SECONDS",
)
ready_timeout_s = _as_int(
    _env("SCENARIO_READY_TIMEOUT_SECONDS", "1800"),
    "SCENARIO_READY_TIMEOUT_SECONDS",
)
scale_down_timeout_s = _as_int(
    _env("SCENARIO_SCALE_DOWN_TIMEOUT_SECONDS", "1800"),
    "SCENARIO_SCALE_DOWN_TIMEOUT_SECONDS",
)
precheck_timeout_s = _as_int(
    _env("SCENARIO_PRECHECK_TIMEOUT_SECONDS", "900"),
    "SCENARIO_PRECHECK_TIMEOUT_SECONDS",
)

kpo_nodepool_name = _env("KPO_WORKLOAD_NODEPOOL", "benchmark-nodepool").strip()
if "kpo" in workload.get("targets", {}):
    sel = workload["targets"]["kpo"].get("node_selector") or {}
    kpo_nodepool_name = str(sel.get("karpenter.sh/nodepool", kpo_nodepool_name)).strip()
if not kpo_nodepool_name:
    raise RuntimeError("KPO nodepool selector cannot be empty")

ca_floor = _as_int(
    _env("CA_EXPECTED_BENCHMARK_NODE_FLOOR", _env("CA_AUTOSCALED_POOL_MIN_SIZE", "0")),
    "CA_EXPECTED_BENCHMARK_NODE_FLOOR",
)
kpo_floor = _as_int(
    _env(
        "KPO_EXPECTED_BENCHMARK_NODE_FLOOR", _env("KPO_AUTOSCALED_POOL_MIN_SIZE", "0")
    ),
    "KPO_EXPECTED_BENCHMARK_NODE_FLOOR",
)
expected_floor = {"ca": ca_floor, "kpo": kpo_floor}
require_kpo_claims_floor = _bool(_env("REQUIRE_KPO_NODECLAIMS_AT_FLOOR", "true"))

burst_replicas = _as_int(
    _env("SCENARIO_BURST_REPLICAS", "20"), "SCENARIO_BURST_REPLICAS"
)
burst_hold_seconds = _as_int(
    _env("SCENARIO_BURST_HOLD_SECONDS", "120"), "SCENARIO_BURST_HOLD_SECONDS"
)

if repeats < 1:
    raise RuntimeError("REPEATS must be >= 1")
if burst_replicas < 1:
    raise RuntimeError("SCENARIO_BURST_REPLICAS must be >= 1")
if burst_hold_seconds < 0:
    raise RuntimeError("SCENARIO_BURST_HOLD_SECONDS must be >= 0")
if poll_s < 1 or poll_up_s < 1 or poll_down_s < 1:
    raise RuntimeError("Poll seconds must be >= 1")

print("=== Cell 10 Burst + Scale-Down ===")
print(f"run_id: {run_id}")
print(f"targets: {selected_targets}")
print(f"target_scenarios: {target_scenarios}")
print(f"repeats: {repeats}")
print(f"kpo_nodepool_selector: {kpo_nodepool_name}")
print(f"expected_floor: {expected_floor}")
print(f"precheck_timeout_seconds: {precheck_timeout_s}")
print(f"poll_seconds(default/up/down): {poll_s}/{poll_up_s}/{poll_down_s}")

if target_scenarios not in {"all", "burst", "scale_down"}:
    print(
        "Skipping burst execution because TARGET_SCENARIOS does not include burst semantics."
    )
    print("Next: Run Cell 11 for Steady + Scale-Down.")
else:
    _execute_single_profile(
        profile_name="burst",
        up_replicas=burst_replicas,
        hold_seconds=burst_hold_seconds,
        down_replicas=0,
        repeats=repeats,
        poll_s=poll_s,
        pending_timeout_s=pending_timeout_s,
        ready_timeout_s=ready_timeout_s,
        scale_down_timeout_s=scale_down_timeout_s,
        selected_targets=selected_targets,
        workload=workload,
        kpo_nodepool_name=kpo_nodepool_name,
        expected_floor=expected_floor,
        require_kpo_claims_floor=require_kpo_claims_floor,
        artifact_dir=artifact_dir,
        run_id=run_id,
        precheck_timeout_s=precheck_timeout_s,
        poll_up_s=poll_up_s,
        poll_down_s=poll_down_s,
    )
    print("Next: Run Cell 11 for Steady + Scale-Down.")

In [ ]:
# Cell 11 — Steady + Scale-Down only (Primary request-based events only)

import json
from pathlib import Path

required_helpers = ["_execute_single_profile", "_env", "_require", "_bool"]
missing_helpers = [h for h in required_helpers if h not in globals()]
if missing_helpers:
    raise RuntimeError(
        f"Cell 10 helpers not loaded: {missing_helpers}. Run Cell 10 first in this kernel session."
    )


def _to_int(v, name):
    try:
        return int(str(v).strip())
    except Exception:
        raise RuntimeError(f"{name} must be an integer, got: {v}")


run_id = _require("RUN_ID")
output_dir = Path(_require("OUTPUT_DIR"))
artifact_dir = output_dir / run_id

workload_path = artifact_dir / "workload_deployments.json"
if not workload_path.exists():
    raise RuntimeError(
        f"Missing workload deployment artifact from Cell 9: {workload_path}"
    )
workload = json.loads(workload_path.read_text(encoding="utf-8"))

target_cluster = _env("TARGET_CLUSTER", "both").strip().lower()
if target_cluster not in {"ca", "kpo", "both"}:
    raise RuntimeError(f"TARGET_CLUSTER must be ca|kpo|both, got: {target_cluster}")
selected_targets = ["ca", "kpo"] if target_cluster == "both" else [target_cluster]

workload_targets = workload.get("targets", {}) or {}
missing_workload_targets = [t for t in selected_targets if t not in workload_targets]
if missing_workload_targets:
    raise RuntimeError(
        f"Selected target(s) missing in Cell 9 workload artifact: {missing_workload_targets}. "
        f"Available targets: {sorted(workload_targets.keys())}"
    )

target_scenarios = _env("TARGET_SCENARIOS", "all").strip().lower()
if target_scenarios not in {"all", "burst", "steady", "scale_down"}:
    raise RuntimeError(
        f"TARGET_SCENARIOS must be all|burst|steady|scale_down, got: {target_scenarios}"
    )

repeats = _to_int(_env("REPEATS", "3"), "REPEATS")
poll_s = _to_int(_env("SCENARIO_POLL_SECONDS", "10"), "SCENARIO_POLL_SECONDS")
poll_up_s = _to_int(
    _env("SCENARIO_SCALE_UP_POLL_SECONDS", str(poll_s)),
    "SCENARIO_SCALE_UP_POLL_SECONDS",
)
poll_down_s = _to_int(
    _env("SCENARIO_SCALE_DOWN_POLL_SECONDS", str(poll_s)),
    "SCENARIO_SCALE_DOWN_POLL_SECONDS",
)
pending_timeout_s = _to_int(
    _env("SCENARIO_PENDING_TIMEOUT_SECONDS", "300"),
    "SCENARIO_PENDING_TIMEOUT_SECONDS",
)
ready_timeout_s = _to_int(
    _env("SCENARIO_READY_TIMEOUT_SECONDS", "1800"),
    "SCENARIO_READY_TIMEOUT_SECONDS",
)
scale_down_timeout_s = _to_int(
    _env("SCENARIO_SCALE_DOWN_TIMEOUT_SECONDS", "1800"),
    "SCENARIO_SCALE_DOWN_TIMEOUT_SECONDS",
)
precheck_timeout_s = _to_int(
    _env("SCENARIO_PRECHECK_TIMEOUT_SECONDS", "900"),
    "SCENARIO_PRECHECK_TIMEOUT_SECONDS",
)

kpo_nodepool_name = _env("KPO_WORKLOAD_NODEPOOL", "benchmark-nodepool").strip()
if "kpo" in workload_targets:
    sel = workload_targets["kpo"].get("node_selector") or {}
    kpo_nodepool_name = str(sel.get("karpenter.sh/nodepool", kpo_nodepool_name)).strip()
if not kpo_nodepool_name:
    raise RuntimeError("KPO nodepool selector cannot be empty")

ca_floor = _to_int(
    _env("CA_EXPECTED_BENCHMARK_NODE_FLOOR", _env("CA_AUTOSCALED_POOL_MIN_SIZE", "0")),
    "CA_EXPECTED_BENCHMARK_NODE_FLOOR",
)
kpo_floor = _to_int(
    _env(
        "KPO_EXPECTED_BENCHMARK_NODE_FLOOR", _env("KPO_AUTOSCALED_POOL_MIN_SIZE", "0")
    ),
    "KPO_EXPECTED_BENCHMARK_NODE_FLOOR",
)
expected_floor = {"ca": ca_floor, "kpo": kpo_floor}
require_kpo_claims_floor = _bool(_env("REQUIRE_KPO_NODECLAIMS_AT_FLOOR", "true"))

steady_replicas = _to_int(
    _env("SCENARIO_STEADY_REPLICAS", "8"), "SCENARIO_STEADY_REPLICAS"
)
steady_hold_seconds = _to_int(
    _env("SCENARIO_STEADY_HOLD_SECONDS", "300"), "SCENARIO_STEADY_HOLD_SECONDS"
)

if repeats < 1:
    raise RuntimeError("REPEATS must be >= 1")
if steady_replicas < 1:
    raise RuntimeError("SCENARIO_STEADY_REPLICAS must be >= 1")
if steady_hold_seconds < 0:
    raise RuntimeError("SCENARIO_STEADY_HOLD_SECONDS must be >= 0")
if poll_s < 1 or poll_up_s < 1 or poll_down_s < 1:
    raise RuntimeError("Poll seconds must be >= 1")

print("=== Cell 11 Steady + Scale-Down ===")
print(f"run_id: {run_id}")
print(f"targets: {selected_targets}")
print(f"target_scenarios: {target_scenarios}")
print(f"repeats: {repeats}")
print(f"kpo_nodepool_selector: {kpo_nodepool_name}")
print(f"expected_floor: {expected_floor}")
print(f"precheck_timeout_seconds: {precheck_timeout_s}")
print(f"poll_seconds(default/up/down): {poll_s}/{poll_up_s}/{poll_down_s}")

if target_scenarios not in {"all", "steady", "scale_down"}:
    print(
        "Skipping steady execution because TARGET_SCENARIOS does not include steady semantics."
    )
    print("Next: Run Cell 12 to gather telemetry.")
else:
    _execute_single_profile(
        profile_name="steady",
        up_replicas=steady_replicas,
        hold_seconds=steady_hold_seconds,
        down_replicas=0,
        repeats=repeats,
        poll_s=poll_s,
        pending_timeout_s=pending_timeout_s,
        ready_timeout_s=ready_timeout_s,
        scale_down_timeout_s=scale_down_timeout_s,
        selected_targets=selected_targets,
        workload=workload,
        kpo_nodepool_name=kpo_nodepool_name,
        expected_floor=expected_floor,
        require_kpo_claims_floor=require_kpo_claims_floor,
        artifact_dir=artifact_dir,
        run_id=run_id,
        precheck_timeout_s=precheck_timeout_s,
        poll_up_s=poll_up_s,
        poll_down_s=poll_down_s,
    )
    print("Next: Run Cell 12 to gather telemetry.")

In [ ]:
# Cell 12 — Custom Run Queue Builder (UI-only; execution moved to Cell 13)

import os
import re
import json
from pathlib import Path
from datetime import datetime, timezone

import ipywidgets as widgets
from IPython.display import display, HTML


# -----------------------------
# Helpers
# -----------------------------
def _now_utc():
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def _env_local(name, default=None):
    fn = globals().get("_env")
    if callable(fn):
        try:
            return fn(name, default)
        except Exception:
            pass
    return os.environ.get(name, default)


def _require_local(name):
    fn = globals().get("_require")
    if callable(fn):
        try:
            return fn(name)
        except Exception:
            pass
    v = os.environ.get(name, "")
    if not str(v).strip():
        raise RuntimeError(f"Missing required environment variable: {name}")
    return v


def _to_int(v, name):
    try:
        return int(str(v).strip())
    except Exception:
        raise RuntimeError(f"{name} must be an integer, got: {v}")


def _load_json(path: Path, default=None):
    if not path.exists():
        return default
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def _write_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, sort_keys=True)
        f.write("\n")


def _append_line(path: Path, line: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(f"{_now_utc()} {line}\n")


def _next_queue_id(items):
    max_n = 0
    for it in items:
        qid = str((it or {}).get("queue_id", ""))
        m = re.match(r"^q(\d+)$", qid)
        if m:
            max_n = max(max_n, int(m.group(1)))
    return f"q{max_n + 1:04d}"


def _normalize_queue_item(it):
    if not isinstance(it, dict):
        return None
    try:
        replicas = int(it.get("replicas"))
        max_nodes = int(it.get("max_node_count"))
    except Exception:
        return None
    if replicas < 1 or max_nodes < 1:
        return None
    return {
        "queue_id": str(it.get("queue_id") or ""),
        "replicas": replicas,
        "max_node_count": max_nodes,
        "created_at_utc": str(it.get("created_at_utc") or _now_utc()),
        "status": str(it.get("status") or "queued"),
        "source": str(it.get("source") or "cell12-ui"),
    }


# -----------------------------
# Runtime context
# -----------------------------
run_id = _require_local("RUN_ID")
output_dir = Path(_require_local("OUTPUT_DIR"))
artifact_dir = output_dir / run_id
artifact_dir.mkdir(parents=True, exist_ok=True)

workload = _load_json(artifact_dir / "workload_deployments.json", {}) or {}
if not workload:
    raise RuntimeError("Missing workload_deployments.json. Run Cell 9 first.")

provisioning = _load_json(artifact_dir / "provisioning.json", {}) or {}

target_cluster = str(_env_local("TARGET_CLUSTER", "both")).strip().lower()
if target_cluster not in {"ca", "kpo", "both"}:
    raise RuntimeError(f"TARGET_CLUSTER must be ca|kpo|both, got: {target_cluster}")
selected_targets = ["ca", "kpo"] if target_cluster == "both" else [target_cluster]

workload_targets = workload.get("targets", {}) or {}
missing_targets = [t for t in selected_targets if t not in workload_targets]
if missing_targets:
    raise RuntimeError(
        f"Selected target(s) missing in workload artifact: {missing_targets}. "
        f"Available: {sorted(workload_targets.keys())}"
    )

# Derive default max nodes from provisioning first, fallback env
default_max_nodes = None
try:
    ca_capacity = (provisioning.get("capacity_selection") or {}).get("ca") or {}
    ca_pool = (ca_capacity.get("worker_pools") or {}).get(
        "ca-benchmark-autoscaled"
    ) or {}
    if ca_pool.get("max_size") is not None:
        default_max_nodes = int(ca_pool["max_size"])
except Exception:
    default_max_nodes = None

if default_max_nodes is None:
    default_max_nodes = _to_int(
        _env_local("CA_AUTOSCALED_POOL_MAX_SIZE", "10"), "CA_AUTOSCALED_POOL_MAX_SIZE"
    )
default_max_nodes = max(1, min(1000, int(default_max_nodes)))

# Queue/index artifacts
custom_index_path = artifact_dir / "scenario_custom_runs_index.json"
ui_log_path = artifact_dir / "scenario_custom_queue_ui.log"

existing = _load_json(custom_index_path, default={}) or {}
if not isinstance(existing, dict):
    existing = {}

execution_history = []
if isinstance(existing.get("execution_history"), list):
    execution_history.extend(existing["execution_history"])
if isinstance(existing.get("entries"), list):
    execution_history.extend(existing["entries"])

queue = []
if isinstance(existing.get("queue"), list):
    for raw in existing["queue"]:
        norm = _normalize_queue_item(raw)
        if norm:
            if not norm["queue_id"]:
                norm["queue_id"] = _next_queue_id(queue)
            queue.append(norm)

index_doc = {
    "run_id": run_id,
    "schema_version": "v2-custom-queue",
    "generated_at_utc": _now_utc(),
    "selected_targets": selected_targets,
    "defaults": {
        "max_nodes": default_max_nodes,
    },
    "queue": queue,
    "execution_history": execution_history,
    # compatibility mirror for old readers; kept equal to execution_history
    "entries": execution_history,
}


def _persist_index():
    index_doc["generated_at_utc"] = _now_utc()
    index_doc["run_id"] = run_id
    index_doc["selected_targets"] = selected_targets
    index_doc["defaults"] = {"max_nodes": default_max_nodes}
    index_doc["queue"] = queue
    index_doc["execution_history"] = execution_history
    index_doc["entries"] = execution_history
    _write_json(custom_index_path, index_doc)
    os.environ["CUSTOM_SCENARIO_INDEX_ARTIFACT"] = str(custom_index_path)


def _validate_inputs(replicas, max_nodes):
    issues = []
    notes = []

    if replicas < 1:
        issues.append("Replicas must be >= 1.")
    if replicas > 5000:
        issues.append("Replicas must be <= 5000.")
    if max_nodes < 1:
        issues.append("Max nodes must be >= 1.")
    if max_nodes > 1000:
        issues.append("Max nodes must be <= 1000.")

    if max_nodes > 0:
        ratio = replicas / float(max_nodes)
        notes.append(f"Pods per max-node ratio: {ratio:.2f}")
        if ratio > 300:
            notes.append(
                "High pods/node ratio; verify node shape capacity before execution."
            )

    return (len(issues) == 0), issues, notes


def _log(msg):
    _append_line(ui_log_path, msg)
    with log_out:
        print(msg)


# -----------------------------
# UI
# -----------------------------
display(
    HTML(
        """
<style>
  .c12-wrap { max-width: 1200px; margin: 0 auto; }
  .c12-section {
    border: 1px solid #dfe3eb;
    background: #f7f9fc;
    padding: 12px;
    margin: 8px 0;
    border-left: 4px solid #99aac7;
  }
  .ok-strip {
    background: #e8f4fd;
    color: #0b5394;
    padding: 6px 10px;
    border: 1px solid #a4c2f4;
    border-radius: 6px;
    font-size: 13px;
    margin: 0 0 6px 0;
  }
  .warn-strip {
    background: #fff8e1;
    color: #6b4f00;
    padding: 6px 10px;
    border: 1px solid #f0d98c;
    border-radius: 6px;
    font-size: 13px;
    margin: 0 0 6px 0;
  }
  .err-strip {
    background: #fdecea;
    color: #ba1a1a;
    padding: 6px 10px;
    border: 1px solid #f5c2c7;
    border-radius: 6px;
    font-size: 13px;
    margin: 0 0 6px 0;
  }
</style>
"""
    )
)

header_html = widgets.HTML(
    value=(
        "<div class='ok-strip'><b>Cell 12: Custom Queue Builder</b> "
        "(UI only). This cell does <b>not</b> execute workloads.</div>"
    )
)

hint_html = widgets.HTML(
    value=(
        "<div class='warn-strip'>Provide only two values per case: "
        "<b>Replicas</b> and <b>Max Node Count</b>. Add multiple runs "
        "(for example 50, 75, 100, 150), then execute in Cell 13.</div>"
    )
)

status_html = widgets.HTML()
preview_html = widgets.HTML()
queue_html = widgets.HTML()
summary_html = widgets.HTML()

replicas_slider = widgets.IntSlider(
    value=max(
        1,
        _to_int(
            _env_local("SCENARIO_STEADY_REPLICAS", "8"), "SCENARIO_STEADY_REPLICAS"
        ),
    ),
    min=1,
    max=5000,
    step=1,
    description="Replicas",
    readout=False,
    continuous_update=False,
    style={"description_width": "90px"},
    layout=widgets.Layout(width="560px"),
)
replicas_in = widgets.BoundedIntText(
    value=replicas_slider.value,
    min=1,
    max=5000,
    step=1,
    layout=widgets.Layout(width="120px"),
)
widgets.link((replicas_slider, "value"), (replicas_in, "value"))

max_nodes_slider = widgets.IntSlider(
    value=max(1, int(default_max_nodes)),
    min=1,
    max=1000,
    step=1,
    description="Max Nodes",
    readout=False,
    continuous_update=False,
    style={"description_width": "90px"},
    layout=widgets.Layout(width="560px"),
)
max_nodes_in = widgets.BoundedIntText(
    value=max_nodes_slider.value,
    min=1,
    max=1000,
    step=1,
    layout=widgets.Layout(width="120px"),
)
widgets.link((max_nodes_slider, "value"), (max_nodes_in, "value"))

add_btn = widgets.Button(description="Add Run", button_style="primary", icon="plus")
remove_last_btn = widgets.Button(
    description="Remove Last", button_style="", icon="undo"
)
clear_btn = widgets.Button(
    description="Clear Queue", button_style="warning", icon="trash"
)
reload_btn = widgets.Button(
    description="Reload From Disk", button_style="", icon="refresh"
)

log_out = widgets.Output(
    layout={
        "border": "1px solid #ddd",
        "padding": "8px",
        "max_height": "320px",
        "overflow_y": "auto",
    }
)


def _render_queue():
    if not queue:
        queue_html.value = "<i>No queued custom runs.</i>"
        return

    rows = []
    for i, q in enumerate(queue, start=1):
        rows.append(
            "<tr>"
            f"<td style='padding:4px 10px'>{i}</td>"
            f"<td style='padding:4px 10px'>{q['queue_id']}</td>"
            f"<td style='padding:4px 10px'>{q['replicas']}</td>"
            f"<td style='padding:4px 10px'>{q['max_node_count']}</td>"
            f"<td style='padding:4px 10px'>{q['created_at_utc']}</td>"
            "</tr>"
        )

    queue_html.value = (
        "<table style='border-collapse:collapse;'>"
        "<thead><tr>"
        "<th style='padding:4px 10px;text-align:left;'>#</th>"
        "<th style='padding:4px 10px;text-align:left;'>Queue ID</th>"
        "<th style='padding:4px 10px;text-align:left;'>Replicas</th>"
        "<th style='padding:4px 10px;text-align:left;'>Max Nodes</th>"
        "<th style='padding:4px 10px;text-align:left;'>Created (UTC)</th>"
        "</tr></thead>"
        f"<tbody>{''.join(rows)}</tbody></table>"
    )


def _render_summary():
    count = len(queue)
    total_replicas = sum(int(q["replicas"]) for q in queue) if queue else 0
    max_nodes_peak = max((int(q["max_node_count"]) for q in queue), default=0)
    summary_html.value = (
        "<div class='ok-strip'>"
        f"Queued runs: <b>{count}</b> | "
        f"Total queued replicas: <b>{total_replicas}</b> | "
        f"Peak requested max_nodes: <b>{max_nodes_peak}</b> | "
        f"History records: <b>{len(execution_history)}</b>"
        "</div>"
    )


def _refresh_validation(_=None):
    replicas = int(replicas_in.value)
    max_nodes = int(max_nodes_in.value)

    ok, issues, notes = _validate_inputs(replicas, max_nodes)
    if ok:
        status_html.value = (
            "<div class='ok-strip'>Input is valid. Add run to queue.</div>"
        )
    else:
        status_html.value = "<div class='err-strip'>" + "<br>".join(issues) + "</div>"

    lines = [
        f"Current input: replicas=<b>{replicas}</b>, max_nodes=<b>{max_nodes}</b>",
        f"Targets: <b>{', '.join(selected_targets)}</b>",
    ]
    lines.extend(notes)
    preview_html.value = "<div class='warn-strip'>" + "<br>".join(lines) + "</div>"

    add_btn.disabled = not ok
    remove_last_btn.disabled = len(queue) == 0
    clear_btn.disabled = len(queue) == 0


def _on_add(_):
    replicas = int(replicas_in.value)
    max_nodes = int(max_nodes_in.value)
    ok, issues, _ = _validate_inputs(replicas, max_nodes)
    if not ok:
        _log("[validation error] " + "; ".join(issues))
        _refresh_validation()
        return

    item = {
        "queue_id": _next_queue_id(queue),
        "replicas": replicas,
        "max_node_count": max_nodes,
        "created_at_utc": _now_utc(),
        "status": "queued",
        "source": "cell12-ui",
    }
    queue.append(item)
    _persist_index()
    _render_queue()
    _render_summary()
    _refresh_validation()
    _log(f"[queued] {item['queue_id']} replicas={replicas}, max_nodes={max_nodes}")


def _on_remove_last(_):
    if not queue:
        return
    item = queue.pop()
    _persist_index()
    _render_queue()
    _render_summary()
    _refresh_validation()
    _log(
        f"[queue] removed {item.get('queue_id', '?')} replicas={item.get('replicas')} max_nodes={item.get('max_node_count')}"
    )


def _on_clear(_):
    removed = len(queue)
    queue.clear()
    _persist_index()
    _render_queue()
    _render_summary()
    _refresh_validation()
    _log(f"[queue] cleared {removed} queued run(s)")


def _on_reload(_):
    loaded = _load_json(custom_index_path, default={}) or {}
    if not isinstance(loaded, dict):
        _log("[reload] invalid index JSON on disk; keeping in-memory queue")
        return

    queue.clear()
    if isinstance(loaded.get("queue"), list):
        for raw in loaded["queue"]:
            norm = _normalize_queue_item(raw)
            if norm:
                if not norm["queue_id"]:
                    norm["queue_id"] = _next_queue_id(queue)
                queue.append(norm)

    _render_queue()
    _render_summary()
    _refresh_validation()
    _log(f"[reload] loaded {len(queue)} queued run(s) from disk")


add_btn.on_click(_on_add)
remove_last_btn.on_click(_on_remove_last)
clear_btn.on_click(_on_clear)
reload_btn.on_click(_on_reload)

replicas_slider.observe(_refresh_validation, names="value")
replicas_in.observe(_refresh_validation, names="value")
max_nodes_slider.observe(_refresh_validation, names="value")
max_nodes_in.observe(_refresh_validation, names="value")

_persist_index()
_render_queue()
_render_summary()
_refresh_validation()

display(
    widgets.VBox(
        [
            widgets.HTML("<div class='c12-wrap'>"),
            header_html,
            hint_html,
            status_html,
            widgets.HBox(
                [replicas_slider, replicas_in], layout=widgets.Layout(gap="12px")
            ),
            widgets.HBox(
                [max_nodes_slider, max_nodes_in], layout=widgets.Layout(gap="12px")
            ),
            preview_html,
            widgets.HBox(
                [add_btn, remove_last_btn, clear_btn, reload_btn],
                layout=widgets.Layout(gap="8px"),
            ),
            widgets.HTML("<div class='c12-section'><b>Queued Runs</b></div>"),
            summary_html,
            queue_html,
            widgets.HTML(
                "<div class='c12-section'><b>UI Log (also persisted to file)</b></div>"
            ),
            log_out,
            widgets.HTML("</div>"),
        ]
    )
)

print("Cell 12 loaded (queue builder only).")
print(f"run_id: {run_id}")
print(f"targets: {selected_targets}")
print(f"default max_nodes: {default_max_nodes}")
print(f"queue/index artifact: {custom_index_path}")
print(f"ui log artifact: {ui_log_path}")
print("Next: run Cell 13 to execute queued custom runs with live notebook output.")

In [ ]:
# Cell 13 — Execute Custom Run Queue (live output + durable logs)

import os
import re
import json
import math
import time
import threading
from pathlib import Path
from datetime import datetime, timezone


# -----------------------------
# Required helpers from earlier cells
# -----------------------------
required_helpers = ["_execute_single_profile", "_run", "_run_json"]
missing_helpers = [h for h in required_helpers if h not in globals()]
if missing_helpers:
    raise RuntimeError(
        f"Missing required helpers from prior cells: {missing_helpers}. "
        "Run Cells 10 and 11 first in this kernel."
    )


# -----------------------------
# Local utility helpers
# -----------------------------
def _env_local(name, default=None):
    fn = globals().get("_env")
    if callable(fn):
        try:
            return fn(name, default)
        except Exception:
            pass
    return os.environ.get(name, default)


def _require_local(name):
    fn = globals().get("_require")
    if callable(fn):
        try:
            return fn(name)
        except Exception:
            pass
    v = os.environ.get(name, "")
    if not str(v).strip():
        raise RuntimeError(f"Missing required environment variable: {name}")
    return v


def _bool_local(v):
    return str(v).strip().lower() in {"1", "true", "yes", "y", "on"}


def _now_utc():
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def _to_int(v, name):
    try:
        return int(str(v).strip())
    except Exception:
        raise RuntimeError(f"{name} must be an integer, got: {v}")


def _load_json(path: Path, default=None):
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding="utf-8"))


def _write_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def _append_log_line(path: Path, msg: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    line = f"{_now_utc()} {msg}"
    print(line, flush=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(line + "\n")


def _safe_profile_name(replicas, max_nodes, queue_id):
    ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    q = re.sub(r"[^a-zA-Z0-9_-]", "-", str(queue_id or "q"))
    return f"custom-r{int(replicas)}-n{int(max_nodes)}-{q}-{ts}"


def _parse_cpu_quantity(v):
    if v is None:
        return None
    s = str(v).strip().lower()
    if not s:
        return None
    try:
        if s.endswith("m"):
            return float(s[:-1]) / 1000.0
        return float(s)
    except Exception:
        return None


def _parse_mem_gib(v):
    if v is None:
        return None
    s = str(v).strip().lower()
    if not s:
        return None
    try:
        if s.endswith("gib"):
            return float(s[:-3])
        if s.endswith("gi"):
            return float(s[:-2])
        if s.endswith("gb"):
            return float(s[:-2]) * (1000.0 / 1024.0)
        if s.endswith("mib"):
            return float(s[:-3]) / 1024.0
        if s.endswith("mi"):
            return float(s[:-2]) / 1024.0
        return float(s)
    except Exception:
        return None


def _fmt_cpu(v):
    if v is None:
        return None
    if abs(v - round(v)) < 1e-9:
        return str(int(round(v)))
    return f"{v:.3f}".rstrip("0").rstrip(".")


def _fmt_mem_gi(v):
    if v is None:
        return None
    if abs(v - round(v)) < 1e-9:
        return f"{int(round(v))}Gi"
    return f"{v:.3f}".rstrip("0").rstrip(".") + "Gi"


def _estimate_required_nodes(replicas, pod_cpu_req, node_cpu):
    if pod_cpu_req is None or node_cpu is None or pod_cpu_req <= 0 or node_cpu <= 0:
        return None
    pods_per_node = int(node_cpu // pod_cpu_req)
    if pods_per_node <= 0:
        return None
    return int(math.ceil(float(replicas) / float(pods_per_node)))


def _start_heartbeat(label, log_path: Path, interval_s=15):
    stop = threading.Event()
    started = time.time()

    def _run_hb():
        while not stop.wait(interval_s):
            elapsed = int(time.time() - started)
            _append_log_line(log_path, f"[heartbeat] {label} elapsed_seconds={elapsed}")

    t = threading.Thread(target=_run_hb, daemon=True)
    t.start()
    return stop, t


# -----------------------------
# kubectl patch helpers
# -----------------------------
def _kubectl_patch_merge(kubeconfig, args, patch_obj):
    cmd = [
        "kubectl",
        "--kubeconfig",
        kubeconfig,
        *args,
        "--type",
        "merge",
        "-p",
        json.dumps(patch_obj, separators=(",", ":")),
    ]
    _run(cmd)


def _kubectl_patch_json(kubeconfig, args, patch_ops):
    cmd = [
        "kubectl",
        "--kubeconfig",
        kubeconfig,
        *args,
        "--type",
        "json",
        "-p",
        json.dumps(patch_ops, separators=(",", ":")),
    ]
    _run(cmd)


def _get_ca_autodiscovery_state(
    kubeconfig, namespace="kube-system", deploy="cluster-autoscaler"
):
    dep = _run_json(
        [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "-n",
            namespace,
            "get",
            "deploy",
            deploy,
            "-o",
            "json",
        ]
    )

    containers = (
        ((dep.get("spec") or {}).get("template") or {}).get("spec") or {}
    ).get("containers") or []
    for c_idx, c in enumerate(containers):
        cmd = c.get("command") or []
        for a_idx, arg in enumerate(cmd):
            if isinstance(arg, str) and arg.startswith("--node-group-auto-discovery="):
                m = re.search(r",max:(\d+)", arg)
                max_nodes = int(m.group(1)) if m else None
                return {
                    "namespace": namespace,
                    "deployment": deploy,
                    "container_name": c.get("name"),
                    "container_index": c_idx,
                    "command": cmd,
                    "arg_index": a_idx,
                    "arg_value": arg,
                    "max_nodes": max_nodes,
                }

    raise RuntimeError("CA deployment arg '--node-group-auto-discovery=...' not found.")


def _set_ca_max_nodes(
    kubeconfig, new_max_nodes, namespace="kube-system", deploy="cluster-autoscaler"
):
    state = _get_ca_autodiscovery_state(kubeconfig, namespace=namespace, deploy=deploy)
    old_arg = state["arg_value"]

    if re.search(r",max:\d+", old_arg):
        new_arg = re.sub(r"(,max:)\d+", rf"\g<1>{int(new_max_nodes)}", old_arg)
    else:
        new_arg = old_arg + f",max:{int(new_max_nodes)}"

    new_cmd = list(state["command"])
    new_cmd[state["arg_index"]] = new_arg

    c_idx = int(state["container_index"])
    patch_ops = [
        {
            "op": "replace",
            "path": f"/spec/template/spec/containers/{c_idx}/command",
            "value": new_cmd,
        }
    ]

    _kubectl_patch_json(
        kubeconfig,
        ["-n", namespace, "patch", "deployment", deploy],
        patch_ops,
    )

    _run(
        [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "-n",
            namespace,
            "rollout",
            "status",
            f"deploy/{deploy}",
            "--timeout=600s",
        ]
    )

    return {
        "before": state,
        "after": {
            "max_nodes": int(new_max_nodes),
            "arg_value": new_arg,
        },
    }


def _restore_ca_autodiscovery(kubeconfig, snapshot):
    before = (snapshot or {}).get("before") or {}
    namespace = before.get("namespace") or "kube-system"
    deploy = before.get("deployment") or "cluster-autoscaler"
    command = before.get("command")
    c_idx = before.get("container_index")

    if not isinstance(command, list) or c_idx is None:
        return

    patch_ops = [
        {
            "op": "replace",
            "path": f"/spec/template/spec/containers/{int(c_idx)}/command",
            "value": command,
        }
    ]

    _kubectl_patch_json(
        kubeconfig,
        ["-n", namespace, "patch", "deployment", deploy],
        patch_ops,
    )

    _run(
        [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "-n",
            namespace,
            "rollout",
            "status",
            f"deploy/{deploy}",
            "--timeout=600s",
        ]
    )


def _get_kpo_nodepool_state(kubeconfig, nodepool_name):
    obj = _run_json(
        [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "get",
            "nodepool",
            nodepool_name,
            "-o",
            "json",
        ]
    )

    md = obj.get("metadata") or {}
    spec = obj.get("spec") or {}
    ann = md.get("annotations") or {}
    limits = spec.get("limits") or {}

    max_size = _to_int(
        ann.get("benchmark.oracle.com/max-size", "0"), "kpo annotation max-size"
    )
    cpu_limit = _parse_cpu_quantity(limits.get("cpu"))
    mem_limit_gi = _parse_mem_gib(limits.get("memory"))

    if max_size <= 0 or cpu_limit is None or mem_limit_gi is None:
        raise RuntimeError(
            "Unable to derive KPO per-node limits from NodePool "
            "(need max-size annotation and cpu/memory limits)."
        )

    return {
        "name": nodepool_name,
        "annotations": ann,
        "limits": limits,
        "max_size": max_size,
        "cpu_limit": cpu_limit,
        "memory_limit_gi": mem_limit_gi,
        "cpu_per_node": cpu_limit / max_size,
        "memory_per_node_gi": mem_limit_gi / max_size,
    }


def _set_kpo_max_nodes(kubeconfig, nodepool_name, new_max_nodes):
    before = _get_kpo_nodepool_state(kubeconfig, nodepool_name)

    cpu_new = before["cpu_per_node"] * int(new_max_nodes)
    mem_new_gi = before["memory_per_node_gi"] * int(new_max_nodes)

    patch = {
        "metadata": {
            "annotations": {"benchmark.oracle.com/max-size": str(int(new_max_nodes))}
        },
        "spec": {
            "limits": {
                "cpu": _fmt_cpu(cpu_new),
                "memory": _fmt_mem_gi(mem_new_gi),
            }
        },
    }

    _kubectl_patch_merge(
        kubeconfig,
        ["patch", "nodepool", nodepool_name],
        patch,
    )

    after = _get_kpo_nodepool_state(kubeconfig, nodepool_name)
    return {"before": before, "after": after}


def _restore_kpo_nodepool(kubeconfig, snapshot):
    before = (snapshot or {}).get("before") or {}
    nodepool_name = before.get("name")
    if not nodepool_name:
        return

    ann = dict(before.get("annotations") or {})
    lim = dict(before.get("limits") or {})

    patch = {
        "metadata": {"annotations": ann},
        "spec": {"limits": lim},
    }

    _kubectl_patch_merge(
        kubeconfig,
        ["patch", "nodepool", nodepool_name],
        patch,
    )


# -----------------------------
# Runtime context and config
# -----------------------------
run_id = _require_local("RUN_ID")
output_dir = Path(_require_local("OUTPUT_DIR")).expanduser().resolve()
artifact_dir = output_dir / run_id

if not artifact_dir.exists():
    raise RuntimeError(f"Artifact directory not found: {artifact_dir}")

workload = _load_json(artifact_dir / "workload_deployments.json", {})
if not workload:
    raise RuntimeError("Missing workload_deployments.json. Run Cell 9 first.")

provisioning = _load_json(artifact_dir / "provisioning.json", {}) or {}
generation_summary = _load_json(Path("tf") / "generation-summary.json", {}) or {}

target_cluster = str(_env_local("TARGET_CLUSTER", "both")).strip().lower()
if target_cluster not in {"ca", "kpo", "both"}:
    raise RuntimeError(f"TARGET_CLUSTER must be ca|kpo|both, got: {target_cluster}")
selected_targets = ["ca", "kpo"] if target_cluster == "both" else [target_cluster]

workload_targets = workload.get("targets", {}) or {}
missing_targets = [t for t in selected_targets if t not in workload_targets]
if missing_targets:
    raise RuntimeError(
        f"Selected target(s) missing in workload artifact: {missing_targets}. "
        f"Available targets: {sorted(workload_targets.keys())}"
    )

ca_kubeconfig = str((workload_targets.get("ca") or {}).get("kubeconfig") or "").strip()
kpo_kubeconfig = str(
    (workload_targets.get("kpo") or {}).get("kubeconfig") or ""
).strip()

if "ca" in selected_targets and not ca_kubeconfig:
    raise RuntimeError("CA kubeconfig missing from workload artifact.")
if "kpo" in selected_targets and not kpo_kubeconfig:
    raise RuntimeError("KPO kubeconfig missing from workload artifact.")

kpo_nodepool_name = str(
    _env_local("KPO_WORKLOAD_NODEPOOL", "benchmark-nodepool")
).strip()
if "kpo" in workload_targets:
    sel = workload_targets["kpo"].get("node_selector") or {}
    kpo_nodepool_name = str(sel.get("karpenter.sh/nodepool", kpo_nodepool_name)).strip()
if "kpo" in selected_targets and not kpo_nodepool_name:
    raise RuntimeError("KPO nodepool selector cannot be empty.")

repeats_default = _to_int(_env_local("CUSTOM_REPEATS", "1"), "CUSTOM_REPEATS")
poll_s = _to_int(_env_local("SCENARIO_POLL_SECONDS", "10"), "SCENARIO_POLL_SECONDS")
poll_up_s = _to_int(
    _env_local("SCENARIO_SCALE_UP_POLL_SECONDS", str(poll_s)),
    "SCENARIO_SCALE_UP_POLL_SECONDS",
)
poll_down_s = _to_int(
    _env_local("SCENARIO_SCALE_DOWN_POLL_SECONDS", str(poll_s)),
    "SCENARIO_SCALE_DOWN_POLL_SECONDS",
)
pending_timeout_s = _to_int(
    _env_local("SCENARIO_PENDING_TIMEOUT_SECONDS", "300"),
    "SCENARIO_PENDING_TIMEOUT_SECONDS",
)
ready_timeout_s = _to_int(
    _env_local("SCENARIO_READY_TIMEOUT_SECONDS", "1800"),
    "SCENARIO_READY_TIMEOUT_SECONDS",
)
scale_down_timeout_s = _to_int(
    _env_local("SCENARIO_SCALE_DOWN_TIMEOUT_SECONDS", "1800"),
    "SCENARIO_SCALE_DOWN_TIMEOUT_SECONDS",
)
precheck_timeout_s = _to_int(
    _env_local("SCENARIO_PRECHECK_TIMEOUT_SECONDS", "900"),
    "SCENARIO_PRECHECK_TIMEOUT_SECONDS",
)
custom_hold_seconds = _to_int(
    _env_local(
        "CUSTOM_HOLD_SECONDS", _env_local("SCENARIO_STEADY_HOLD_SECONDS", "300")
    ),
    "CUSTOM_HOLD_SECONDS",
)
custom_down_replicas = _to_int(
    _env_local("CUSTOM_DOWN_REPLICAS", "0"), "CUSTOM_DOWN_REPLICAS"
)
require_kpo_claims_floor = _bool_local(
    _env_local("REQUIRE_KPO_NODECLAIMS_AT_FLOOR", "true")
)

ca_floor = _to_int(
    _env_local(
        "CA_EXPECTED_BENCHMARK_NODE_FLOOR",
        _env_local("CA_AUTOSCALED_POOL_MIN_SIZE", "0"),
    ),
    "CA_EXPECTED_BENCHMARK_NODE_FLOOR",
)
kpo_floor = _to_int(
    _env_local(
        "KPO_EXPECTED_BENCHMARK_NODE_FLOOR",
        _env_local("KPO_AUTOSCALED_POOL_MIN_SIZE", "0"),
    ),
    "KPO_EXPECTED_BENCHMARK_NODE_FLOOR",
)
expected_floor = {"ca": ca_floor, "kpo": kpo_floor}

fail_fast = _bool_local(_env_local("CUSTOM_FAIL_FAST", "true"))
heartbeat_interval_s = _to_int(
    _env_local("CUSTOM_HEARTBEAT_SECONDS", "15"), "CUSTOM_HEARTBEAT_SECONDS"
)

custom_index_path = artifact_dir / "scenario_custom_runs_index.json"
if not custom_index_path.exists():
    raise RuntimeError(
        f"Missing custom queue index: {custom_index_path}. "
        "Run Cell 12 first to create and populate queue."
    )

exec_log_path = artifact_dir / "scenario_custom_execution.log"

index_doc = _load_json(custom_index_path, {})
if not isinstance(index_doc, dict):
    raise RuntimeError(f"Invalid JSON structure in {custom_index_path}")

queue = index_doc.get("queue")
if not isinstance(queue, list):
    queue = []

history = index_doc.get("execution_history")
if not isinstance(history, list):
    history = index_doc.get("entries")
if not isinstance(history, list):
    history = []


def _persist_index():
    index_doc["schema_version"] = "v2-custom-queue"
    index_doc["run_id"] = run_id
    index_doc["selected_targets"] = selected_targets
    index_doc["generated_at_utc"] = _now_utc()
    index_doc["queue"] = queue
    index_doc["execution_history"] = history
    index_doc["entries"] = history  # compatibility mirror
    _write_json(custom_index_path, index_doc)
    os.environ["CUSTOM_SCENARIO_INDEX_ARTIFACT"] = str(custom_index_path)


def _validate_queue_item(item):
    if not isinstance(item, dict):
        raise RuntimeError(f"Queue item must be object, got: {type(item).__name__}")
    replicas = _to_int(item.get("replicas"), "queue.replicas")
    max_nodes = _to_int(item.get("max_node_count"), "queue.max_node_count")
    if replicas < 1:
        raise RuntimeError("queue.replicas must be >= 1")
    if max_nodes < 1:
        raise RuntimeError("queue.max_node_count must be >= 1")
    if max_nodes > 1000:
        raise RuntimeError("queue.max_node_count > 1000 blocked by safety guard")
    qid = str(item.get("queue_id") or "").strip() or f"q{(len(history) + 1):04d}"
    return qid, replicas, max_nodes


def _apply_overrides(max_nodes):
    snap = {"applied_at_utc": _now_utc(), "targets": {}}

    if "ca" in selected_targets:
        snap["targets"]["ca"] = _set_ca_max_nodes(ca_kubeconfig, max_nodes)

    if "kpo" in selected_targets:
        snap["targets"]["kpo"] = _set_kpo_max_nodes(
            kpo_kubeconfig, kpo_nodepool_name, max_nodes
        )

    return snap


def _restore_overrides(snapshot):
    errs = []
    targets = (snapshot or {}).get("targets") or {}

    if "kpo" in targets and "kpo" in selected_targets:
        try:
            _restore_kpo_nodepool(kpo_kubeconfig, targets["kpo"])
        except Exception as e:
            errs.append(f"kpo restore failed: {e}")

    if "ca" in targets and "ca" in selected_targets:
        try:
            _restore_ca_autodiscovery(ca_kubeconfig, targets["ca"])
        except Exception as e:
            errs.append(f"ca restore failed: {e}")

    if errs:
        raise RuntimeError("; ".join(errs))


def _warn_capacity_estimates(replicas, max_nodes):
    req_cpu = _parse_cpu_quantity(
        (
            ((workload.get("workload_profile") or {}).get("resources") or {}).get(
                "requests"
            )
            or {}
        ).get("cpu")
    )
    if req_cpu is None:
        return

    if "ca" in selected_targets:
        ca_pool = (
            ((provisioning.get("capacity_selection") or {}).get("ca") or {}).get(
                "worker_pools"
            )
            or {}
        ).get("ca-benchmark-autoscaled") or {}
        if not ca_pool:
            ca_pool = (
                ((provisioning.get("targets") or {}).get("ca") or {}).get(
                    "worker_pools"
                )
                or {}
            ).get("ca-benchmark-autoscaled") or {}
        ca_ocpus = _parse_cpu_quantity(ca_pool.get("ocpus"))
        cpu_factor = 2.0
        try:
            f = float(generation_summary.get("kpo_cpu_per_ocpu_factor"))
            if f > 0:
                cpu_factor = f
        except Exception:
            pass
        ca_node_cpu = (ca_ocpus * cpu_factor) if ca_ocpus is not None else None
        need = _estimate_required_nodes(replicas, req_cpu, ca_node_cpu)
        if need is not None and need > max_nodes:
            _append_log_line(
                exec_log_path,
                f"[warn] CA CPU estimate needs ~{need} nodes for replicas={replicas}, above max_nodes={max_nodes}",
            )

    if "kpo" in selected_targets:
        try:
            kpo_state = _get_kpo_nodepool_state(kpo_kubeconfig, kpo_nodepool_name)
            need = _estimate_required_nodes(
                replicas, req_cpu, kpo_state.get("cpu_per_node")
            )
            if need is not None and need > max_nodes:
                _append_log_line(
                    exec_log_path,
                    f"[warn] KPO CPU estimate needs ~{need} nodes for replicas={replicas}, above max_nodes={max_nodes}",
                )
        except Exception as e:
            _append_log_line(
                exec_log_path, f"[warn] Unable to compute KPO estimate: {e}"
            )


# -----------------------------
# Execute queue
# -----------------------------
os.environ["CUSTOM_SCENARIO_INDEX_ARTIFACT"] = str(custom_index_path)
os.environ["CUSTOM_SCENARIO_EXEC_LOG_ARTIFACT"] = str(exec_log_path)

print("=== Cell 13 Execute Custom Queue ===")
print(f"run_id: {run_id}")
print(f"artifact_dir: {artifact_dir}")
print(f"selected_targets: {selected_targets}")
print(f"queue_index: {custom_index_path}")
print(f"execution_log: {exec_log_path}")
print(f"queued_items: {len(queue)}")
print(f"fail_fast: {fail_fast}")
print(f"heartbeat_interval_seconds: {heartbeat_interval_s}")

if len(queue) == 0:
    _append_log_line(exec_log_path, "[info] queue is empty; nothing to execute.")
    print("Cell 13 complete: queue empty.")
else:
    _persist_index()

    processed = 0
    success = 0
    failed = 0

    _append_log_line(exec_log_path, "=== custom queue execution start ===")
    _append_log_line(
        exec_log_path,
        f"run_id={run_id} selected_targets={selected_targets} initial_queue={len(queue)}",
    )

    while queue:
        raw = queue.pop(0)
        _persist_index()

        started_at_utc = _now_utc()
        queue_id = None
        profile_name = None
        replicas = None
        max_nodes = None
        status = "failed"
        error = ""
        applied_snapshot = None
        events_path = None
        summary_path = None

        try:
            queue_id, replicas, max_nodes = _validate_queue_item(raw)
            profile_name = _safe_profile_name(replicas, max_nodes, queue_id)

            _append_log_line(
                exec_log_path,
                f"--- item start queue_id={queue_id} profile={profile_name} replicas={replicas} max_nodes={max_nodes}",
            )
            _warn_capacity_estimates(replicas, max_nodes)

            _append_log_line(exec_log_path, "[step] applying capacity overrides")
            applied_snapshot = _apply_overrides(max_nodes)

            _append_log_line(exec_log_path, "[step] executing profile")
            hb_stop, hb_thread = _start_heartbeat(
                profile_name, exec_log_path, interval_s=heartbeat_interval_s
            )
            try:
                events_path, summary_path = _execute_single_profile(
                    profile_name=profile_name,
                    up_replicas=replicas,
                    hold_seconds=custom_hold_seconds,
                    down_replicas=custom_down_replicas,
                    repeats=repeats_default,
                    poll_s=poll_s,
                    pending_timeout_s=pending_timeout_s,
                    ready_timeout_s=ready_timeout_s,
                    scale_down_timeout_s=scale_down_timeout_s,
                    selected_targets=selected_targets,
                    workload=workload,
                    kpo_nodepool_name=kpo_nodepool_name,
                    expected_floor=expected_floor,
                    require_kpo_claims_floor=require_kpo_claims_floor,
                    artifact_dir=artifact_dir,
                    run_id=run_id,
                    precheck_timeout_s=precheck_timeout_s,
                    poll_up_s=poll_up_s,
                    poll_down_s=poll_down_s,
                )
            finally:
                hb_stop.set()
                hb_thread.join(timeout=2)

            status = "success"
            _append_log_line(
                exec_log_path,
                f"[ok] profile complete queue_id={queue_id} profile={profile_name}",
            )

        except Exception as e:
            status = "failed"
            error = str(e)
            _append_log_line(
                exec_log_path,
                f"[error] queue_id={queue_id or 'unknown'} profile={profile_name or 'unknown'} error={error}",
            )

        finally:
            if applied_snapshot is not None:
                _append_log_line(exec_log_path, "[step] restoring capacity overrides")
                try:
                    _restore_overrides(applied_snapshot)
                    _append_log_line(exec_log_path, "[ok] restore complete")
                except Exception as restore_err:
                    if status == "success":
                        status = "failed"
                        error = f"restore_failed: {restore_err}"
                    else:
                        error = f"{error} | restore_failed: {restore_err}"
                    _append_log_line(
                        exec_log_path, f"[error] restore failed: {restore_err}"
                    )

        ended_at_utc = _now_utc()

        entry = {
            "run_id": run_id,
            "queue_id": queue_id,
            "profile": profile_name,
            "scenario_type": "custom",
            "replicas": replicas,
            "max_node_count": max_nodes,
            "selected_targets": selected_targets,
            "repeats": repeats_default,
            "hold_seconds": custom_hold_seconds,
            "down_replicas": custom_down_replicas,
            "started_at_utc": started_at_utc,
            "ended_at_utc": ended_at_utc,
            "status": status,
            "error": error,
            "event_contract_version": "v1",
            "source": "cell13-executor",
            "artifacts": {
                "events_jsonl": str(events_path) if events_path else "",
                "summary_json": str(summary_path) if summary_path else "",
                "executor_log": str(exec_log_path),
            },
            "capacity_override_snapshot": applied_snapshot or {},
        }

        history.append(entry)
        _persist_index()

        processed += 1
        if status == "success":
            success += 1
        else:
            failed += 1
            if fail_fast:
                _append_log_line(
                    exec_log_path,
                    f"[stop] fail_fast=true, halting queue after queue_id={queue_id}",
                )
                break

    _append_log_line(exec_log_path, "=== custom queue execution end ===")

    summary = {
        "run_id": run_id,
        "processed": processed,
        "success": success,
        "failed": failed,
        "remaining_queue_items": len(queue),
        "queue_index": str(custom_index_path),
        "execution_log": str(exec_log_path),
        "history_records": len(history),
    }

    os.environ["CUSTOM_SCENARIO_INDEX_ARTIFACT"] = str(custom_index_path)
    os.environ["CUSTOM_SCENARIO_EXEC_LOG_ARTIFACT"] = str(exec_log_path)

    print("\nCell 13 summary:")
    print(json.dumps(summary, indent=2))
    print(f"\nCell 13 complete. Queue index: {custom_index_path}")
    print(f"Execution log: {exec_log_path}")
    if len(queue) > 0:
        print("Some queue items remain. Re-run Cell 13 to continue.")
    else:
        print("Queue fully processed. Next: run telemetry/KPI/report cells.")

In [ ]:
# Cell 14 — Collect Telemetry (burst + steady + custom profiles)

import os
import re
import csv
import json
import subprocess
from pathlib import Path
from datetime import datetime, timezone, timedelta


def _env(k, d=""):
    return os.environ.get(k, d)


def _to_int(v, name):
    try:
        return int(str(v).strip())
    except Exception:
        raise RuntimeError(f"{name} must be an integer, got: {v}")


def _now_utc():
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def _parse_ts(ts):
    if not ts:
        return None
    s = str(ts).strip()
    if not s:
        return None
    if s.endswith("Z"):
        s = s[:-1] + "+00:00"
    try:
        return datetime.fromisoformat(s)
    except Exception:
        return None


def _to_iso_utc(ts):
    dt = _parse_ts(ts) if isinstance(ts, str) else ts
    if not dt:
        return None
    return dt.astimezone(timezone.utc).isoformat().replace("+00:00", "Z")


def _first_nonempty(*vals):
    for v in vals:
        if v is None:
            continue
        s = str(v).strip()
        if s:
            return s
    return ""


def _read_json(path: Path, default=None):
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding="utf-8"))


def _read_jsonl(path: Path):
    rows = []
    if not path.exists():
        return rows
    for line in path.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if not s:
            continue
        try:
            rows.append(json.loads(s))
        except Exception:
            continue
    return rows


def _write_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def _run(cmd, check=True):
    print(f"\n$ {' '.join(cmd)}")
    p = subprocess.run(cmd, text=True, capture_output=True)
    out = p.stdout or ""
    err = p.stderr or ""
    if out:
        print(out)
    if err:
        print(err)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed (exit {p.returncode}): {' '.join(cmd)}")
    return p.returncode, out, err


def _run_json(cmd, check=True):
    rc, out, err = _run(cmd, check=check)
    if rc != 0:
        return None, rc, err
    try:
        return json.loads(out or "{}"), rc, err
    except Exception as e:
        tail = (out or "")[-2000:]
        if check:
            raise RuntimeError(
                f"Failed to parse JSON output from: {' '.join(cmd)} ; error={e} ; output_tail={tail}"
            )
        return None, rc, f"json_parse_error={e}; output_tail={tail}"


def _candidate_output_roots():
    roots = []
    out_env = _env("OUTPUT_DIR", "").strip()
    if out_env:
        roots.append(Path(os.path.expanduser(out_env)).resolve())

    cwd = Path.cwd().resolve()
    roots.append((cwd / "OCIK8sClusterAutoscalerVsKarpenter" / "artifacts").resolve())
    roots.append((cwd / "artifacts").resolve())

    uniq = []
    seen = set()
    for r in roots:
        k = str(r)
        if k not in seen:
            seen.add(k)
            uniq.append(r)
    return uniq


def _is_valid_run_dir(run_dir: Path):
    if not run_dir.is_dir():
        return False
    must_have = [
        run_dir / "scenario_execution_burst_summary.json",
        run_dir / "scenario_execution_steady_summary.json",
        run_dir / "provisioning.json",
        run_dir / "workload_deployments.json",
    ]
    return all(p.exists() for p in must_have)


def _discover_run_context():
    run_id_env = _env("RUN_ID", "").strip()
    out_env = _env("OUTPUT_DIR", "").strip()
    roots = _candidate_output_roots()

    if run_id_env and out_env:
        out = Path(os.path.expanduser(out_env)).resolve()
        ad = out / run_id_env
        if _is_valid_run_dir(ad):
            return out, run_id_env, ad

    if run_id_env:
        for r in roots:
            ad = r / run_id_env
            if _is_valid_run_dir(ad):
                return r, run_id_env, ad

    if out_env:
        out = Path(os.path.expanduser(out_env)).resolve()
        if out.exists():
            runs = [p for p in out.glob("ca-vs-kpo-*") if _is_valid_run_dir(p)]
            if runs:
                runs.sort(key=lambda p: p.stat().st_mtime, reverse=True)
                ad = runs[0]
                return out, ad.name, ad

    candidates = []
    for r in roots:
        if not r.exists():
            continue
        for p in r.glob("ca-vs-kpo-*"):
            if _is_valid_run_dir(p):
                candidates.append((p.stat().st_mtime, r, p))

    if candidates:
        candidates.sort(key=lambda x: x[0], reverse=True)
        _, out, ad = candidates[0]
        return out, ad.name, ad

    raise RuntimeError(
        "Could not resolve run context for Cell 14. "
        "Run Cell 2 in this kernel, then rerun Cell 14."
    )


def _event_time_from_k8s_event(item):
    series = item.get("series") or {}
    ts = _first_nonempty(
        item.get("eventTime"),
        item.get("lastTimestamp"),
        series.get("lastObservedTime"),
        item.get("firstTimestamp"),
        ((item.get("metadata") or {}).get("creationTimestamp")),
    )
    return _to_iso_utc(ts)


def _extract_metric_points(
    resp_obj,
    target,
    cluster_id,
    metric_name,
    agg_name,
    query_text,
    namespace,
    resolution,
):
    rows = []
    count = 0
    data = resp_obj.get("data") if isinstance(resp_obj, dict) else []
    if not isinstance(data, list):
        data = []

    for series in data:
        if not isinstance(series, dict):
            continue

        dims = series.get("dimensions") or {}
        if not isinstance(dims, dict):
            dims = {}

        points = series.get("aggregated-datapoints")
        if points is None:
            points = series.get("aggregatedDatapoints")
        if not isinstance(points, list):
            points = []

        for dp in points:
            if not isinstance(dp, dict):
                continue
            ts = _to_iso_utc(dp.get("timestamp"))
            val = dp.get("value")
            try:
                fval = float(val)
            except Exception:
                fval = None

            rows.append(
                {
                    "run_id": run_id,
                    "target": target,
                    "cluster_id": cluster_id,
                    "metric_name": metric_name,
                    "aggregation": agg_name,
                    "timestamp": ts,
                    "value": fval,
                    "resolution": resolution,
                    "namespace": namespace,
                    "query_text": query_text,
                    "dimensions_json": json.dumps(dims, sort_keys=True),
                }
            )
            count += 1

    return rows, count


# Resolve run context
output_dir, run_id, artifact_dir = _discover_run_context()
os.environ["OUTPUT_DIR"] = str(output_dir)
os.environ["RUN_ID"] = run_id

# Inputs
provisioning = _read_json(artifact_dir / "provisioning.json", {}) or {}
workload = _read_json(artifact_dir / "workload_deployments.json", {}) or {}
workload_targets = workload.get("targets", {}) or {}

target_cluster = _env("TARGET_CLUSTER", "both").strip().lower()
if target_cluster not in {"ca", "kpo", "both"}:
    raise RuntimeError(f"TARGET_CLUSTER must be ca|kpo|both, got: {target_cluster}")
selected_targets = ["ca", "kpo"] if target_cluster == "both" else [target_cluster]

missing = [t for t in selected_targets if t not in workload_targets]
if missing:
    raise RuntimeError(
        f"Selected target(s) missing in workload_deployments.json: {missing}. "
        f"Available targets: {sorted(workload_targets.keys())}"
    )

summary_files = sorted(artifact_dir.glob("scenario_execution_*_summary.json"))
event_files = sorted(artifact_dir.glob("scenario_events_*.jsonl"))
if not summary_files and not event_files:
    raise RuntimeError(
        "No scenario summary/event files found. Run scenario cells first."
    )

# Build scenario time window from summaries + events
all_times = []
for sf in summary_files:
    s = _read_json(sf, {}) or {}
    for r in s.get("runs") or []:
        st = _parse_ts(r.get("started_at_utc"))
        en = _parse_ts(r.get("ended_at_utc"))
        if st:
            all_times.append(st)
        if en:
            all_times.append(en)

all_scenario_events = []
for ef in event_files:
    rows = _read_jsonl(ef)
    all_scenario_events.extend(rows)
    for e in rows:
        dt = _parse_ts(e.get("time_utc"))
        if dt:
            all_times.append(dt)

if not all_times:
    raise RuntimeError("Unable to derive scenario time window from summaries/events.")

window_start_dt = min(all_times)
window_end_dt = max(all_times)

query_pad_seconds = _to_int(
    _env("TELEMETRY_QUERY_PAD_SECONDS", "300"), "TELEMETRY_QUERY_PAD_SECONDS"
)
query_start_dt = window_start_dt - timedelta(seconds=query_pad_seconds)
query_end_dt = window_end_dt + timedelta(seconds=query_pad_seconds)

window_start_utc = _to_iso_utc(window_start_dt)
window_end_utc = _to_iso_utc(window_end_dt)
query_start_utc = _to_iso_utc(query_start_dt)
query_end_utc = _to_iso_utc(query_end_dt)

region = _first_nonempty(
    _env("OCI_REGION", ""),
    provisioning.get("region"),
)
if not region:
    raise RuntimeError(
        "Region could not be resolved from OCI_REGION or provisioning.json"
    )

oci_profile = _env("OCI_PROFILE", "DEFAULT").strip() or "DEFAULT"
oci_config_file = os.path.expanduser(
    _env("OCI_CONFIG_FILE", "~/.oci/config").strip() or "~/.oci/config"
)

cluster_ocids = provisioning.get("cluster_ocids") or {}
provision_targets = provisioning.get("targets") or {}


def _target_cluster_id(target):
    return _first_nonempty(
        cluster_ocids.get(target),
        ((provision_targets.get(target) or {}).get("cluster_id")),
    )


compartment_id = _first_nonempty(
    _env("BENCHMARK_COMPARTMENT_OCID", ""),
    ((provision_targets.get("ca") or {}).get("compartment_id")),
    ((provision_targets.get("kpo") or {}).get("compartment_id")),
    ((provision_targets.get("ca") or {}).get("network_compartment_id")),
)
if not compartment_id:
    raise RuntimeError("Compartment OCID could not be resolved for telemetry queries")

# Artifact paths
telemetry_dir = artifact_dir / "telemetry"
metrics_raw_dir = telemetry_dir / "oci_metrics_raw"
controller_logs_dir = telemetry_dir / "controller_logs"
kpo_snapshots_dir = telemetry_dir / "kpo_snapshots"

telemetry_dir.mkdir(parents=True, exist_ok=True)
metrics_raw_dir.mkdir(parents=True, exist_ok=True)
controller_logs_dir.mkdir(parents=True, exist_ok=True)
kpo_snapshots_dir.mkdir(parents=True, exist_ok=True)

metrics_csv_path = telemetry_dir / "oci_metrics.csv"
metrics_queries_path = telemetry_dir / "oci_metrics_queries.json"
k8s_events_jsonl_path = telemetry_dir / "k8s_events.jsonl"
scenario_event_samples_path = telemetry_dir / "scenario_event_samples.jsonl"
telemetry_summary_path = telemetry_dir / "telemetry_collection_summary.json"

print("=== Cell 14 Collect Telemetry ===")
print(f"run_id: {run_id}")
print(f"artifact_dir: {artifact_dir}")
print(f"selected_targets: {selected_targets}")
print(f"window_start_utc: {window_start_utc}")
print(f"window_end_utc: {window_end_utc}")
print(f"window_query_start_utc: {query_start_utc}")
print(f"window_query_end_utc: {query_end_utc}")
print(f"region: {region}")
print(f"oci_profile: {oci_profile}")
print(f"oci_config_file: {oci_config_file}")

metric_namespace = "oci_oke"
metric_resolution = _env("TELEMETRY_METRIC_RESOLUTION", "1m").strip() or "1m"
metric_specs = [
    {"metric": "UnschedulablePods", "dimension": "resourceId", "agg": "max"},
    {"metric": "NodeState", "dimension": "clusterId", "agg": "max"},
    {"metric": "KubernetesNodeCondition", "dimension": "clusterId", "agg": "max"},
    {"metric": "APIServerRequestCount", "dimension": "resourceId", "agg": "sum"},
    {"metric": "APIServerResponseCount", "dimension": "resourceId", "agg": "sum"},
]

metrics_rows = []
metrics_queries = []
metrics_datapoints_by_target_metric = {}
k8s_events_rows = []
errors = []
snapshot_files = []
controller_log_files = []

for target in selected_targets:
    cluster_id = _target_cluster_id(target)
    kubeconfig = _first_nonempty((workload_targets.get(target) or {}).get("kubeconfig"))

    if not cluster_id:
        errors.append(f"missing_cluster_id:{target}")
        continue
    if not kubeconfig:
        errors.append(f"missing_kubeconfig:{target}")
        continue

    # OCI Monitoring metrics
    for spec in metric_specs:
        metric = spec["metric"]
        dimension = spec["dimension"]
        agg = spec["agg"]
        query_text = f'{metric}[1m]{{{dimension} = "{cluster_id}"}}.{agg}()'

        cmd = [
            "oci",
            "--config-file",
            oci_config_file,
            "--profile",
            oci_profile,
            "--region",
            region,
            "monitoring",
            "metric-data",
            "summarize-metrics-data",
            "--compartment-id",
            compartment_id,
            "--namespace",
            metric_namespace,
            "--query-text",
            query_text,
            "--start-time",
            query_start_utc,
            "--end-time",
            query_end_utc,
            "--resolution",
            metric_resolution,
            "--output",
            "json",
        ]

        resp, rc, err = _run_json(cmd, check=False)
        raw_path = metrics_raw_dir / f"{target}_{metric}.json"
        _write_json(
            raw_path,
            {
                "target": target,
                "metric": metric,
                "query_text": query_text,
                "rc": rc,
                "error": err,
                "response": resp if resp is not None else {},
            },
        )

        qrec = {
            "target": target,
            "cluster_id": cluster_id,
            "metric": metric,
            "aggregation": agg,
            "dimension": dimension,
            "query_text": query_text,
            "namespace": metric_namespace,
            "start_time": query_start_utc,
            "end_time": query_end_utc,
            "resolution": metric_resolution,
            "raw_output": str(raw_path),
            "status": "success" if rc == 0 and resp is not None else "error",
            "error": err if rc != 0 else "",
            "datapoints": 0,
        }

        if rc == 0 and resp is not None:
            rows, dcount = _extract_metric_points(
                resp_obj=resp,
                target=target,
                cluster_id=cluster_id,
                metric_name=metric,
                agg_name=agg,
                query_text=query_text,
                namespace=metric_namespace,
                resolution=metric_resolution,
            )
            metrics_rows.extend(rows)
            qrec["datapoints"] = dcount
            metrics_datapoints_by_target_metric[f"{target}:{metric}"] = dcount
        else:
            errors.append(f"metrics_query_failed:{target}:{metric}")
            metrics_datapoints_by_target_metric[f"{target}:{metric}"] = 0

        metrics_queries.append(qrec)

    # Kubernetes events (raw + jsonl)
    events_cmd = [
        "kubectl",
        "--kubeconfig",
        kubeconfig,
        "get",
        "events",
        "-A",
        "-o",
        "json",
    ]
    events_obj, rc_ev, _ = _run_json(events_cmd, check=False)
    k8s_raw_path = telemetry_dir / f"k8s_events_{target}.json"
    _write_json(k8s_raw_path, events_obj if isinstance(events_obj, dict) else {})

    if rc_ev != 0 or events_obj is None:
        errors.append(f"k8s_events_failed:{target}")
    else:
        for it in events_obj.get("items") or []:
            ts = _event_time_from_k8s_event(it)
            dt = _parse_ts(ts)
            if dt and not (query_start_dt <= dt <= query_end_dt):
                continue

            involved = it.get("involvedObject") or {}
            meta = it.get("metadata") or {}
            k8s_events_rows.append(
                {
                    "run_id": run_id,
                    "target": target,
                    "time_utc": ts,
                    "namespace": meta.get("namespace"),
                    "type": it.get("type"),
                    "reason": it.get("reason"),
                    "message": it.get("message"),
                    "count": it.get("count"),
                    "involved_kind": involved.get("kind"),
                    "involved_name": involved.get("name"),
                }
            )

    # Controller logs
    if target == "ca":
        ns = _env("CA_NAMESPACE", "kube-system").strip() or "kube-system"
        dep_obj, rc_dep, _ = _run_json(
            [
                "kubectl",
                "--kubeconfig",
                kubeconfig,
                "-n",
                ns,
                "get",
                "deploy",
                "-o",
                "json",
            ],
            check=False,
        )
        dep_names = []
        if rc_dep == 0 and dep_obj:
            dep_names = [
                ((x.get("metadata") or {}).get("name"))
                for x in (dep_obj.get("items") or [])
                if ((x.get("metadata") or {}).get("name"))
            ]

        wanted = [
            d for d in ["cluster-autoscaler", "kube-dns-autoscaler"] if d in dep_names
        ]
        for dep in wanted:
            log_cmd = [
                "kubectl",
                "--kubeconfig",
                kubeconfig,
                "-n",
                ns,
                "logs",
                f"deploy/{dep}",
                "--all-containers=true",
                "--since-time",
                window_start_utc,
                "--timestamps=true",
            ]
            rc_l, out_l, err_l = _run(log_cmd, check=False)
            log_path = controller_logs_dir / f"{target}_{ns}_{dep}.log"
            log_path.write_text((out_l or "") + (err_l or ""), encoding="utf-8")
            controller_log_files.append(str(log_path))
            if rc_l != 0:
                errors.append(f"controller_log_failed:{target}:{dep}")

        # CA status configmap snapshot
        rc_cm, out_cm, err_cm = _run(
            [
                "kubectl",
                "--kubeconfig",
                kubeconfig,
                "-n",
                ns,
                "get",
                "configmap",
                "cluster-autoscaler-status",
                "-o",
                "yaml",
            ],
            check=False,
        )
        cm_path = controller_logs_dir / f"{target}_{ns}_cluster-autoscaler-status.yaml"
        cm_path.write_text((out_cm or "") + (err_cm or ""), encoding="utf-8")
        snapshot_files.append(str(cm_path))
        if rc_cm != 0:
            errors.append("cluster_autoscaler_status_configmap_missing_or_failed")

    if target == "kpo":
        ns = _env("KPO_NAMESPACE", "karpenter").strip() or "karpenter"
        dep_obj, rc_dep, _ = _run_json(
            [
                "kubectl",
                "--kubeconfig",
                kubeconfig,
                "-n",
                ns,
                "get",
                "deploy",
                "-o",
                "json",
            ],
            check=False,
        )
        dep_names = []
        if rc_dep == 0 and dep_obj:
            dep_names = [
                ((x.get("metadata") or {}).get("name"))
                for x in (dep_obj.get("items") or [])
                if ((x.get("metadata") or {}).get("name"))
            ]

        wanted = [d for d in dep_names if "karpenter" in d]
        for dep in wanted:
            log_cmd = [
                "kubectl",
                "--kubeconfig",
                kubeconfig,
                "-n",
                ns,
                "logs",
                f"deploy/{dep}",
                "--all-containers=true",
                "--since-time",
                window_start_utc,
                "--timestamps=true",
            ]
            rc_l, out_l, err_l = _run(log_cmd, check=False)
            log_path = controller_logs_dir / f"{target}_{ns}_{dep}.log"
            log_path.write_text((out_l or "") + (err_l or ""), encoding="utf-8")
            controller_log_files.append(str(log_path))
            if rc_l != 0:
                errors.append(f"controller_log_failed:{target}:{dep}")

        # KPO snapshots
        for resource, fname in [
            ("nodepool", "nodepool"),
            ("nodeclaims", "nodeclaims"),
            ("ocinodeclass", "ocinodeclass"),
        ]:
            obj, rc_s, err_s = _run_json(
                [
                    "kubectl",
                    "--kubeconfig",
                    kubeconfig,
                    "get",
                    resource,
                    "-A",
                    "-o",
                    "json",
                ],
                check=False,
            )
            sp = kpo_snapshots_dir / f"kpo_{fname}.json"
            _write_json(sp, obj if isinstance(obj, dict) else {})
            snapshot_files.append(str(sp))
            if rc_s != 0:
                errors.append(f"kpo_snapshot_failed:{resource}:{err_s}")

# Scenario event samples (for KPI trace support)
scenario_event_samples = []
for ef in event_files:
    rows = _read_jsonl(ef)
    src = ef.name
    for e in rows:
        d = e.get("details") or {}
        bn = d.get("benchmark_nodes") or {}
        pods = d.get("pods") or {}
        scenario_event_samples.append(
            {
                "run_id": run_id,
                "profile": e.get("scenario"),
                "scenario": e.get("scenario"),
                "target": e.get("target"),
                "repeat": e.get("repeat"),
                "event": e.get("event"),
                "time_utc": e.get("time_utc"),
                "benchmark_nodes_total": bn.get("total"),
                "benchmark_nodes_ready": bn.get("ready"),
                "nodeclaims": d.get("nodeclaims"),
                "pods_total": pods.get("total"),
                "pods_pending": pods.get("Pending"),
                "pods_running": pods.get("Running"),
                "source_file": src,
            }
        )

with scenario_event_samples_path.open("w", encoding="utf-8") as f:
    for row in scenario_event_samples:
        f.write(json.dumps(row, sort_keys=True) + "\n")

# Write metrics CSV
metrics_fields = [
    "run_id",
    "target",
    "cluster_id",
    "metric_name",
    "aggregation",
    "timestamp",
    "value",
    "resolution",
    "namespace",
    "query_text",
    "dimensions_json",
]
with metrics_csv_path.open("w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=metrics_fields)
    w.writeheader()
    for row in metrics_rows:
        w.writerow({k: row.get(k) for k in metrics_fields})

# Write queries + k8s event jsonl
_write_json(metrics_queries_path, metrics_queries)

with k8s_events_jsonl_path.open("w", encoding="utf-8") as f:
    for row in k8s_events_rows:
        f.write(json.dumps(row, sort_keys=True) + "\n")

# Summary
telemetry_summary = {
    "generated_at_utc": _now_utc(),
    "run_id": run_id,
    "selected_targets": selected_targets,
    "window": {
        "scenario_start_utc": window_start_utc,
        "scenario_end_utc": window_end_utc,
        "query_start_utc": query_start_utc,
        "query_end_utc": query_end_utc,
        "query_pad_seconds": query_pad_seconds,
    },
    "region": region,
    "oci_profile": oci_profile,
    "telemetry_backend": {
        "primary_metrics_namespace": metric_namespace,
        "metric_interval": "1m",
        "metric_resolution": metric_resolution,
        "note": "Metrics Server is optional support only and is not used as benchmark KPI backend.",
    },
    "scenario_inputs": {
        "summary_files": [str(p) for p in summary_files],
        "event_files": [str(p) for p in event_files],
    },
    "artifacts": {
        "metrics_csv": str(metrics_csv_path),
        "metrics_queries": str(metrics_queries_path),
        "metrics_raw_dir": str(metrics_raw_dir),
        "k8s_events_jsonl": str(k8s_events_jsonl_path),
        "controller_logs_dir": str(controller_logs_dir),
        "kpo_snapshots_dir": str(kpo_snapshots_dir),
        "scenario_event_samples_jsonl": str(scenario_event_samples_path),
    },
    "counts": {
        "metrics_rows": len(metrics_rows),
        "metrics_queries": len(metrics_queries),
        "k8s_events": len(k8s_events_rows),
        "controller_log_files": len(controller_log_files),
        "snapshot_files": len(snapshot_files),
        "scenario_event_samples": len(scenario_event_samples),
        "errors": len(errors),
    },
    "metrics_datapoints_by_target_metric": metrics_datapoints_by_target_metric,
    "errors": errors,
}

_write_json(telemetry_summary_path, telemetry_summary)

os.environ["TELEMETRY_DIR"] = str(telemetry_dir)
os.environ["TELEMETRY_SUMMARY_ARTIFACT"] = str(telemetry_summary_path)
os.environ["OCI_METRICS_CSV_ARTIFACT"] = str(metrics_csv_path)
os.environ["K8S_EVENTS_JSONL_ARTIFACT"] = str(k8s_events_jsonl_path)
os.environ["SCENARIO_EVENT_SAMPLES_ARTIFACT"] = str(scenario_event_samples_path)

print("\nCell 14 summary:")
print(json.dumps(telemetry_summary, indent=2))
print(f"\nCell 14 complete. Artifact: {telemetry_summary_path}")
print("Next: Run Cell 15.")

In [ ]:
# Cell 15 — KPI Computation (Primary-only, includes burst/steady/custom + telemetry)

import os
import re
import json
import csv
from pathlib import Path
from datetime import datetime, timezone
from statistics import mean
from collections import defaultdict


def _env(k, d=""):
    return os.environ.get(k, d)


def _now_utc():
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def _parse_ts(ts):
    if not ts:
        return None
    s = str(ts).strip()
    if not s:
        return None
    if s.endswith("Z"):
        s = s[:-1] + "+00:00"
    try:
        return datetime.fromisoformat(s)
    except Exception:
        return None


def _to_iso_utc(ts):
    dt = _parse_ts(ts) if isinstance(ts, str) else ts
    if not dt:
        return None
    return dt.astimezone(timezone.utc).isoformat().replace("+00:00", "Z")


def _to_seconds(a, b):
    da = _parse_ts(a)
    db = _parse_ts(b)
    if not da or not db:
        return None
    return float((db - da).total_seconds())


def _first_nonempty(*vals):
    for v in vals:
        if v is None:
            continue
        s = str(v).strip()
        if s:
            return s
    return ""


def _safe_int(v, default=None):
    try:
        return int(v)
    except Exception:
        return default


def _safe_float(v, default=None):
    try:
        return float(v)
    except Exception:
        return default


def _read_json(path: Path, default=None):
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding="utf-8"))


def _read_jsonl(path: Path):
    rows = []
    if not path.exists():
        return rows
    for line in path.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if not s:
            continue
        try:
            rows.append(json.loads(s))
        except Exception:
            continue
    return rows


def _percentile(values, p):
    if not values:
        return None
    vals = sorted(values)
    if len(vals) == 1:
        return float(vals[0])
    idx = (len(vals) - 1) * p
    lo = int(idx)
    hi = min(lo + 1, len(vals) - 1)
    frac = idx - lo
    return float(vals[lo] * (1 - frac) + vals[hi] * frac)


def _summary_stats(values):
    vals = [float(v) for v in values if v is not None]
    if not vals:
        return {
            "count": 0,
            "mean": None,
            "p50": None,
            "p90": None,
            "min": None,
            "max": None,
        }
    return {
        "count": len(vals),
        "mean": float(mean(vals)),
        "p50": _percentile(vals, 0.50),
        "p90": _percentile(vals, 0.90),
        "min": float(min(vals)),
        "max": float(max(vals)),
    }


def _parse_cpu_cores(v):
    if v is None:
        return None
    s = str(v).strip().lower()
    if not s:
        return None
    try:
        if s.endswith("m"):
            return float(s[:-1]) / 1000.0
        return float(s)
    except Exception:
        return None


def _parse_mem_gib(v):
    if v is None:
        return None
    s = str(v).strip().lower()
    if not s:
        return None
    units = [
        ("gib", 1.0),
        ("gi", 1.0),
        ("gb", 1000.0 / 1024.0),
        ("mib", 1.0 / 1024.0),
        ("mi", 1.0 / 1024.0),
        ("mb", 1.0 / 1024.0),
        ("kib", 1.0 / (1024.0 * 1024.0)),
        ("ki", 1.0 / (1024.0 * 1024.0)),
        ("b", 1.0 / (1024.0 * 1024.0 * 1024.0)),
    ]
    try:
        for u, f in units:
            if s.endswith(u):
                return float(s[: -len(u)].strip()) * f
        return float(s) / (1024.0 * 1024.0 * 1024.0)
    except Exception:
        return None


def _load_workload_request_shape(workload):
    prof = workload.get("workload_profile") or {}
    req = (prof.get("resources") or {}).get("requests") or {}
    return {
        "cpu_cores": _parse_cpu_cores(req.get("cpu")),
        "memory_gib": _parse_mem_gib(req.get("memory")),
    }


def _infer_ca_node_capacity(provisioning):
    ca = (provisioning.get("capacity_selection") or {}).get("ca") or {}
    pools = ca.get("worker_pools") or {}
    if not pools:
        pools = ((provisioning.get("targets") or {}).get("ca") or {}).get(
            "worker_pools"
        ) or {}

    for name, p in pools.items():
        if not isinstance(p, dict):
            continue
        if (
            bool(p.get("allow_autoscaler"))
            or bool(p.get("autoscale"))
            or "autoscaled" in str(name)
        ):
            mem = p.get("memory")
            mem_gi = _parse_mem_gib(f"{mem}Gi") if mem is not None else None
            return {
                "cpu_cores": _parse_cpu_cores(p.get("ocpus")),
                "memory_gib": mem_gi,
                "source": "provisioning.ca.worker_pools",
            }

    return {"cpu_cores": None, "memory_gib": None, "source": "unresolved"}


def _infer_kpo_node_capacity(artifact_dir: Path):
    yaml_path = artifact_dir / "rendered" / "kpo" / "ocinodeclass.yaml"
    if yaml_path.exists():
        txt = yaml_path.read_text(encoding="utf-8")
        m_cpu = re.search(r"\bocpus:\s*([0-9]+(?:\.[0-9]+)?)", txt)
        m_mem = re.search(r"\bmemoryInGbs:\s*([0-9]+(?:\.[0-9]+)?)", txt)
        cpu = float(m_cpu.group(1)) if m_cpu else None
        mem = float(m_mem.group(1)) if m_mem else None
        if cpu is not None or mem is not None:
            return {
                "cpu_cores": cpu,
                "memory_gib": mem,
                "source": "rendered/kpo/ocinodeclass.yaml",
            }

    return {"cpu_cores": None, "memory_gib": None, "source": "unresolved"}


def _estimate_node_minutes(points, start_ts, end_ts):
    st = _parse_ts(start_ts)
    en = _parse_ts(end_ts)
    if not st or not en or en <= st:
        return None

    samples = []
    for t, c in points:
        dt = _parse_ts(t)
        if dt is None:
            continue
        cv = _safe_int(c)
        if cv is None:
            continue
        if st <= dt <= en:
            samples.append((dt, cv))

    if not samples:
        return None

    samples.sort(key=lambda x: x[0])

    total_minutes = 0.0
    prev_t = st
    prev_c = samples[0][1]

    for t, c in samples:
        if t > prev_t:
            total_minutes += prev_c * ((t - prev_t).total_seconds() / 60.0)
        prev_t = t
        prev_c = c

    if en > prev_t:
        total_minutes += prev_c * ((en - prev_t).total_seconds() / 60.0)

    return float(total_minutes)


def _estimate_packing_efficiency(
    up_replicas, req_cpu, req_mem, peak_nodes, node_cpu, node_mem
):
    reps = _safe_int(up_replicas)
    nodes = _safe_int(peak_nodes)
    if reps is None or nodes is None or reps <= 0 or nodes <= 0:
        return None

    ratios = []
    if req_cpu is not None and node_cpu is not None and node_cpu > 0:
        ratios.append((reps * req_cpu) / (nodes * node_cpu))
    if req_mem is not None and node_mem is not None and node_mem > 0:
        ratios.append((reps * req_mem) / (nodes * node_mem))

    if not ratios:
        return None

    return float(min(max(ratios), 1.0))


def _candidate_output_roots():
    roots = []
    out_env = _env("OUTPUT_DIR", "").strip()
    if out_env:
        roots.append(Path(os.path.expanduser(out_env)).resolve())

    cwd = Path.cwd().resolve()
    roots.append((cwd / "OCIK8sClusterAutoscalerVsKarpenter" / "artifacts").resolve())
    roots.append((cwd / "artifacts").resolve())

    uniq = []
    seen = set()
    for r in roots:
        k = str(r)
        if k not in seen:
            seen.add(k)
            uniq.append(r)
    return uniq


def _is_valid_run_dir(run_dir: Path):
    if not run_dir.is_dir():
        return False
    must_have = [
        run_dir / "scenario_execution_burst_summary.json",
        run_dir / "scenario_execution_steady_summary.json",
        run_dir / "provisioning.json",
        run_dir / "workload_deployments.json",
        run_dir / "telemetry" / "telemetry_collection_summary.json",
        run_dir / "telemetry" / "oci_metrics.csv",
    ]
    return all(p.exists() for p in must_have)


def _discover_run_context():
    run_id_env = _env("RUN_ID", "").strip()
    out_env = _env("OUTPUT_DIR", "").strip()
    roots = _candidate_output_roots()

    if run_id_env and out_env:
        out = Path(os.path.expanduser(out_env)).resolve()
        ad = out / run_id_env
        if _is_valid_run_dir(ad):
            return out, run_id_env, ad

    if run_id_env:
        for r in roots:
            ad = r / run_id_env
            if _is_valid_run_dir(ad):
                return r, run_id_env, ad

    if out_env:
        out = Path(os.path.expanduser(out_env)).resolve()
        if out.exists():
            runs = [p for p in out.glob("ca-vs-kpo-*") if _is_valid_run_dir(p)]
            if runs:
                runs.sort(key=lambda p: p.stat().st_mtime, reverse=True)
                ad = runs[0]
                return out, ad.name, ad

    candidates = []
    for r in roots:
        if not r.exists():
            continue
        for p in r.glob("ca-vs-kpo-*"):
            if _is_valid_run_dir(p):
                candidates.append((p.stat().st_mtime, r, p))

    if candidates:
        candidates.sort(key=lambda x: x[0], reverse=True)
        _, out, ad = candidates[0]
        return out, ad.name, ad

    raise RuntimeError(
        "Could not resolve run context for Cell 15. "
        "Run Cell 14 telemetry in this kernel, then rerun Cell 15."
    )


def _profile_from_summary_name(path: Path):
    m = re.match(r"^scenario_execution_(.+)_summary\.json$", path.name)
    return m.group(1) if m else ""


def _profile_from_events_name(path: Path):
    m = re.match(r"^scenario_events_(.+)\.jsonl$", path.name)
    return m.group(1) if m else ""


def _scenario_type(profile: str):
    if profile == "burst":
        return "burst"
    if profile == "steady":
        return "steady"
    if profile.startswith("custom-"):
        return "custom"
    return "other"


def _parse_custom_profile(profile: str):
    m = re.match(
        r"^custom-r(?P<rep>\d+)-n(?P<max>\d+)-(?P<qid>[qi]\d+)-", profile or ""
    )
    if not m:
        return {"replicas": None, "max_node_count": None, "queue_id": None}
    return {
        "replicas": int(m.group("rep")),
        "max_node_count": int(m.group("max")),
        "queue_id": m.group("qid"),
    }


def _event_ts(event_obj):
    if not event_obj:
        return None
    return event_obj.get("time_utc")


def _controller_counts(log_path: Path):
    warning = 0
    error = 0
    info = 0
    patt_glog = re.compile(r"\s([IWEF])\d{4}\s")

    with log_path.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if '"level":"ERROR"' in line:
                error += 1
                continue
            if '"level":"WARN"' in line or '"level":"WARNING"' in line:
                warning += 1
                continue
            if '"level":"INFO"' in line:
                info += 1
                continue

            m = patt_glog.search(line)
            if m:
                lvl = m.group(1)
                if lvl == "W":
                    warning += 1
                elif lvl in {"E", "F"}:
                    error += 1
                elif lvl == "I":
                    info += 1

    return {"info": info, "warning": warning, "error": error}


def _aggregate(records, group_fields, metric_keys):
    grouped = {}
    for r in records:
        key = tuple(r.get(f) for f in group_fields)
        if key not in grouped:
            grouped[key] = {
                "records": [],
                "statuses": [],
                "primary_complete": [],
            }
        grouped[key]["records"].append(r)
        grouped[key]["statuses"].append(r.get("status"))
        grouped[key]["primary_complete"].append(bool(r.get("primary_kpi_complete")))

    out = []
    for key, data in sorted(
        grouped.items(),
        key=lambda kv: tuple("" if v is None else str(v) for v in kv[0]),
    ):
        base = {group_fields[i]: key[i] for i in range(len(group_fields))}
        base["run_count"] = len(data["records"])
        base["success_count"] = sum(1 for s in data["statuses"] if str(s) == "success")
        base["primary_complete_count"] = sum(1 for b in data["primary_complete"] if b)
        for m in metric_keys:
            base[m] = _summary_stats([x.get(m) for x in data["records"]])
        out.append(base)
    return out


# Resolve run context
output_dir, run_id, artifact_dir = _discover_run_context()
os.environ["OUTPUT_DIR"] = str(output_dir)
os.environ["RUN_ID"] = run_id

print("=== Cell 15 KPI Computation ===")
print(f"run_id: {run_id}")
print(f"artifact_dir: {artifact_dir}")

# Core inputs
workload = _read_json(artifact_dir / "workload_deployments.json", {}) or {}
provisioning = _read_json(artifact_dir / "provisioning.json", {}) or {}
generation_summary = _read_json(Path("tf") / "generation-summary.json", {}) or {}
telemetry_summary = (
    _read_json(artifact_dir / "telemetry" / "telemetry_collection_summary.json", {})
    or {}
)

benchmark_mode = _first_nonempty(
    generation_summary.get("benchmark_mode"),
    _env("BENCHMARK_MODE", "parity"),
).lower()
if benchmark_mode not in {"parity", "native"}:
    benchmark_mode = "parity"

workload_req = _load_workload_request_shape(workload)
node_capacity = {
    "ca": _infer_ca_node_capacity(provisioning),
    "kpo": _infer_kpo_node_capacity(artifact_dir),
}

# Scenario files (dynamic)
summary_files = sorted(artifact_dir.glob("scenario_execution_*_summary.json"))
event_files = sorted(artifact_dir.glob("scenario_events_*.jsonl"))

summary_by_profile = {}
summary_path_by_profile = {}
for p in summary_files:
    obj = _read_json(p, {}) or {}
    profile = _first_nonempty(obj.get("profile"), _profile_from_summary_name(p))
    if not profile:
        continue
    summary_by_profile[profile] = obj
    summary_path_by_profile[profile] = str(p)

events_by_profile = {}
events_path_by_profile = {}
all_events = []
for p in event_files:
    rows = _read_jsonl(p)
    profile = _profile_from_events_name(p)
    if not profile:
        continue
    events_by_profile[profile] = rows
    events_path_by_profile[profile] = str(p)
    all_events.extend(rows)

if not summary_by_profile and not events_by_profile:
    raise RuntimeError("No scenario execution/events found for KPI computation.")

# Optional custom history for failed attempts without event files
custom_index = _read_json(artifact_dir / "scenario_custom_runs_index.json", {}) or {}
history = custom_index.get("execution_history")
if not isinstance(history, list):
    history = custom_index.get("entries")
if not isinstance(history, list):
    history = []

history_by_profile = {}
for h in history:
    if not isinstance(h, dict):
        continue
    prof = str(h.get("profile") or "").strip()
    if not prof:
        continue
    prev = history_by_profile.get(prof)
    if prev is None:
        history_by_profile[prof] = h
    else:
        p_end = _first_nonempty(prev.get("ended_at_utc"), prev.get("started_at_utc"))
        h_end = _first_nonempty(h.get("ended_at_utc"), h.get("started_at_utc"))
        if (_parse_ts(h_end) or datetime.min.replace(tzinfo=timezone.utc)) >= (
            _parse_ts(p_end) or datetime.min.replace(tzinfo=timezone.utc)
        ):
            history_by_profile[prof] = h

# Build event index + traces
event_index = {}
run_trace = defaultdict(list)
for e in all_events:
    profile = _first_nonempty(e.get("scenario"))
    target = e.get("target")
    repeat = int(e.get("repeat", 0) or 0)
    ev = e.get("event")
    if not profile or not target or not ev:
        continue
    event_index[(profile, target, repeat, ev)] = e
    run_trace[(profile, target, repeat)].append(e)

# Prefer telemetry scenario trace samples if present
trace_samples_path = artifact_dir / "telemetry" / "scenario_event_samples.jsonl"
trace_samples = _read_jsonl(trace_samples_path)
if trace_samples:
    run_trace = defaultdict(list)
    for tr in trace_samples:
        profile = _first_nonempty(tr.get("profile"), tr.get("scenario"))
        target = tr.get("target")
        repeat = int(tr.get("repeat", 0) or 0)
        if not profile or not target:
            continue
        run_trace[(profile, target, repeat)].append(tr)

required_primary_events = [
    "scale_up_requested",
    "first_new_node_ready",
    "first_pod_running",
    "all_pods_running",
    "scale_down_requested",
    "workload_floor_reached",
    "capacity_floor_reached",
]

primary_kpi_keys = [
    "scale_up_to_first_new_node_ready_seconds",
    "scale_up_to_first_pod_running_seconds",
    "scale_up_to_all_pods_running_seconds",
    "scale_down_to_workload_floor_seconds",
    "scale_down_to_capacity_floor_seconds",
]

profiles = sorted(set(summary_by_profile.keys()) | set(events_by_profile.keys()))
run_records = []

for profile in profiles:
    summary = summary_by_profile.get(profile, {}) or {}
    config = summary.get("config") or {}
    scenario_kind = _scenario_type(profile)
    profile_custom = _parse_custom_profile(profile)
    h_entry = history_by_profile.get(profile, {}) or {}

    run_keys = set()
    for r in summary.get("runs", []) or []:
        t = r.get("target")
        rep = int(r.get("repeat", 0) or 0)
        if t:
            run_keys.add((t, rep))

    for e in events_by_profile.get(profile, []):
        t = e.get("target")
        rep = int(e.get("repeat", 0) or 0)
        if t:
            run_keys.add((t, rep))

    for target, repeat in sorted(run_keys):
        srun = None
        for r in summary.get("runs", []) or []:
            if r.get("target") == target and int(r.get("repeat", 0) or 0) == repeat:
                srun = r
                break

        def _ev(name):
            return event_index.get((profile, target, repeat, name))

        e_start = _ev("scenario_start")
        e_end = _ev("scenario_end")
        e_up = _ev("scale_up_requested")
        e_node = _ev("first_new_node_ready")
        e_pod = _ev("first_pod_running")
        e_all = _ev("all_pods_running")
        e_down = _ev("scale_down_requested")
        e_wf = _ev("workload_floor_reached")
        e_cf = _ev("capacity_floor_reached")

        missing_primary = [n for n in required_primary_events if _ev(n) is None]
        primary_complete = len(missing_primary) == 0

        status = _first_nonempty(
            (srun or {}).get("status"),
            ((_ev("scenario_end") or {}).get("details") or {}).get("status"),
            h_entry.get("status"),
            "unknown",
        )

        up_replicas = (
            (srun or {}).get("up_replicas")
            if (srun or {}).get("up_replicas") is not None
            else (
                config.get("up_replicas")
                if config.get("up_replicas") is not None
                else ((_ev("scenario_start") or {}).get("details") or {}).get(
                    "up_replicas"
                )
            )
        )
        down_replicas = (
            (srun or {}).get("down_replicas")
            if (srun or {}).get("down_replicas") is not None
            else (
                config.get("down_replicas")
                if config.get("down_replicas") is not None
                else ((_ev("scenario_start") or {}).get("details") or {}).get(
                    "down_replicas"
                )
            )
        )

        if up_replicas is None and profile_custom["replicas"] is not None:
            up_replicas = profile_custom["replicas"]

        k_scale_up_first_node = _to_seconds(_event_ts(e_up), _event_ts(e_node))
        k_scale_up_first_pod = _to_seconds(_event_ts(e_up), _event_ts(e_pod))
        k_scale_up_all = _to_seconds(_event_ts(e_up), _event_ts(e_all))
        k_scale_down_wf = _to_seconds(_event_ts(e_down), _event_ts(e_wf))
        k_scale_down_cf = _to_seconds(_event_ts(e_down), _event_ts(e_cf))

        scenario_start_utc = _to_iso_utc(
            _event_ts(e_start) or (srun or {}).get("started_at_utc")
        )
        scenario_end_utc = _to_iso_utc(
            _event_ts(e_end) or (srun or {}).get("ended_at_utc")
        )
        scenario_total = _to_seconds(scenario_start_utc, scenario_end_utc)

        trace = run_trace.get((profile, target, repeat), [])

        node_points = []
        nodeclaim_samples = 0
        for ev in trace:
            if "time_utc" in ev:
                tval = ev.get("time_utc")
                bn_total = ev.get("benchmark_nodes_total")
                bn_ready = ev.get("benchmark_nodes_ready")
                if bn_total is None:
                    bn_total = bn_ready
                if bn_total is not None:
                    node_points.append((tval, bn_total))
                nc = _safe_int(ev.get("nodeclaims"))
                if nc is not None:
                    nodeclaim_samples += 1
            else:
                det = ev.get("details") or {}
                bn = det.get("benchmark_nodes") or {}
                node_count = bn.get("total")
                if node_count is None:
                    node_count = bn.get("ready")
                if node_count is not None:
                    node_points.append((ev.get("time_utc"), node_count))
                nc = _safe_int(det.get("nodeclaims"))
                if nc is not None:
                    nodeclaim_samples += 1

        native_peak_nodes = None
        if node_points:
            vals = [_safe_int(c) for _, c in node_points]
            vals = [v for v in vals if v is not None]
            if vals:
                native_peak_nodes = max(vals)

        native_node_minutes = _estimate_node_minutes(
            node_points, scenario_start_utc, scenario_end_utc
        )
        cap = node_capacity.get(target) or {}
        native_packing_eff = _estimate_packing_efficiency(
            up_replicas,
            workload_req.get("cpu_cores"),
            workload_req.get("memory_gib"),
            native_peak_nodes,
            cap.get("cpu_cores"),
            cap.get("memory_gib"),
        )

        native_notes = []
        if not node_points:
            native_notes.append("no benchmark_nodes samples in trace")
        if native_node_minutes is None:
            native_notes.append("node-minutes unavailable due incomplete window/trace")
        if native_packing_eff is None:
            native_notes.append(
                "packing efficiency unavailable due missing capacity/workload inputs"
            )

        custom_queue_id = _first_nonempty(
            profile_custom["queue_id"], h_entry.get("queue_id")
        )
        custom_max_nodes = (
            profile_custom["max_node_count"]
            if profile_custom["max_node_count"] is not None
            else _safe_int(h_entry.get("max_node_count"))
        )

        run_records.append(
            {
                "run_id": run_id,
                "profile": profile,
                "scenario_type": scenario_kind,
                "target": target,
                "repeat": repeat,
                "status": status,
                "up_replicas": up_replicas,
                "down_replicas": down_replicas,
                "custom_queue_id": custom_queue_id,
                "custom_max_node_count": custom_max_nodes,
                "summary_source": summary_path_by_profile.get(profile, ""),
                "events_source": events_path_by_profile.get(profile, ""),
                "scenario_start_utc": scenario_start_utc,
                "scale_up_requested_utc": _to_iso_utc(_event_ts(e_up)),
                "first_new_node_ready_utc": _to_iso_utc(_event_ts(e_node)),
                "first_pod_running_utc": _to_iso_utc(_event_ts(e_pod)),
                "all_pods_running_utc": _to_iso_utc(_event_ts(e_all)),
                "scale_down_requested_utc": _to_iso_utc(_event_ts(e_down)),
                "workload_floor_reached_utc": _to_iso_utc(_event_ts(e_wf)),
                "capacity_floor_reached_utc": _to_iso_utc(_event_ts(e_cf)),
                "scenario_end_utc": scenario_end_utc,
                "scale_up_to_first_new_node_ready_seconds": k_scale_up_first_node,
                "scale_up_to_first_pod_running_seconds": k_scale_up_first_pod,
                "scale_up_to_all_pods_running_seconds": k_scale_up_all,
                "scale_down_to_workload_floor_seconds": k_scale_down_wf,
                "scale_down_to_capacity_floor_seconds": k_scale_down_cf,
                "scenario_total_seconds": scenario_total,
                "primary_kpi_complete": primary_complete,
                "native_peak_node_count": native_peak_nodes,
                "native_node_minutes_estimate": native_node_minutes,
                "native_packing_efficiency_estimate": native_packing_eff,
                "native_cost_estimate": None,
                "unschedulable_minutes_in_run": None,
                "unschedulable_max_pods_in_run": None,
                "data_quality": {
                    "missing_primary_events": missing_primary,
                    "primary_kpi_complete": primary_complete,
                    "trace_samples": len(trace),
                    "nodeclaim_samples": nodeclaim_samples,
                },
                "native_kpi_notes": native_notes
                + ["cost estimate unavailable: no pricebook input"],
            }
        )

profiles_with_records = {r["profile"] for r in run_records}
attempts_without_scenario_events = []
for h in history:
    if not isinstance(h, dict):
        continue
    profile = str(h.get("profile") or "").strip()
    if not profile:
        continue
    if profile in profiles_with_records:
        continue
    attempts_without_scenario_events.append(
        {
            "profile": profile,
            "status": h.get("status"),
            "queue_id": h.get("queue_id"),
            "replicas": h.get("replicas"),
            "max_node_count": h.get("max_node_count"),
            "error": h.get("error"),
            "started_at_utc": h.get("started_at_utc"),
            "ended_at_utc": h.get("ended_at_utc"),
        }
    )

# Telemetry-derived KPIs
metrics_csv_path = artifact_dir / "telemetry" / "oci_metrics.csv"
metrics_rows = 0
unsched_rows_by_target = defaultdict(list)

if not metrics_csv_path.exists():
    raise RuntimeError(
        f"Missing telemetry metrics CSV: {metrics_csv_path}. Run Cell 14 first."
    )

with metrics_csv_path.open("r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        metrics_rows += 1
        if row.get("metric_name") != "UnschedulablePods":
            continue
        target = row.get("target")
        dt = _parse_ts(row.get("timestamp"))
        val = _safe_float(row.get("value"), default=0.0)
        if target and dt:
            unsched_rows_by_target[target].append((dt, val))

for rec in run_records:
    st = _parse_ts(rec.get("scenario_start_utc"))
    en = _parse_ts(rec.get("scenario_end_utc"))
    if not st or not en or en <= st:
        continue

    rows = unsched_rows_by_target.get(rec.get("target"), [])
    pos_buckets = set()
    max_val = 0.0
    for dt, val in rows:
        if st <= dt <= en and val > 0:
            bucket = dt.replace(second=0, microsecond=0).isoformat()
            pos_buckets.add(bucket)
            if val > max_val:
                max_val = val

    rec["unschedulable_minutes_in_run"] = len(pos_buckets)
    rec["unschedulable_max_pods_in_run"] = max_val if len(pos_buckets) > 0 else 0.0

unsched_summary = []
for target, rows in sorted(unsched_rows_by_target.items()):
    positive = [(dt, v) for dt, v in rows if v > 0]
    unsched_summary.append(
        {
            "target": target,
            "datapoints_total": len(rows),
            "unschedulable_datapoints": len(positive),
            "unschedulable_duration_minutes": len(
                {dt.replace(second=0, microsecond=0).isoformat() for dt, _ in positive}
            ),
            "unschedulable_max_pods": max([v for _, v in positive], default=0.0),
        }
    )

# Controller severities
controller_dir = artifact_dir / "telemetry" / "controller_logs"
controller_by_target = {}
controller_by_file = []
if controller_dir.exists():
    for log_path in sorted(controller_dir.glob("*.log")):
        counts = _controller_counts(log_path)
        fname = log_path.name
        target = fname.split("_", 1)[0] if "_" in fname else "unknown"

        rec = {"file": str(log_path), "target": target, **counts}
        controller_by_file.append(rec)

        agg = controller_by_target.setdefault(
            target, {"info": 0, "warning": 0, "error": 0, "files": 0}
        )
        agg["info"] += counts["info"]
        agg["warning"] += counts["warning"]
        agg["error"] += counts["error"]
        agg["files"] += 1

# Kubernetes events severity from jsonl
k8s_events_jsonl_path = artifact_dir / "telemetry" / "k8s_events.jsonl"
k8s_event_counts = {}
if k8s_events_jsonl_path.exists():
    for row in _read_jsonl(k8s_events_jsonl_path):
        target = row.get("target")
        if not target:
            continue
        rec = k8s_event_counts.setdefault(
            target, {"total": 0, "warning": 0, "normal": 0, "other": 0}
        )
        rec["total"] += 1
        t = str(row.get("type") or "").strip().lower()
        if t == "warning":
            rec["warning"] += 1
        elif t == "normal":
            rec["normal"] += 1
        else:
            rec["other"] += 1

# NodeClaim diagnostics
nodeclaim_diag_map = {}
for (profile, target, repeat), rows in run_trace.items():
    key = (target, profile)
    rec = nodeclaim_diag_map.setdefault(
        key,
        {
            "target": target,
            "profile": profile,
            "scenario_type": _scenario_type(profile),
            "samples": 0,
            "nonzero_samples": 0,
            "minutes_with_nodeclaims": set(),
            "peak_nodeclaims": 0,
            "first_nonzero_ts": None,
            "last_nonzero_ts": None,
        },
    )

    for ev in rows:
        if "time_utc" in ev and "nodeclaims" in ev:
            ts = ev.get("time_utc")
            nc = _safe_int(ev.get("nodeclaims"))
        else:
            det = ev.get("details") or {}
            ts = ev.get("time_utc")
            nc = _safe_int(det.get("nodeclaims"))

        rec["samples"] += 1
        if nc is not None and nc > 0:
            rec["nonzero_samples"] += 1
            rec["peak_nodeclaims"] = max(rec["peak_nodeclaims"], nc)
            dt = _parse_ts(ts)
            if dt is not None:
                bucket = dt.replace(second=0, microsecond=0).isoformat()
                rec["minutes_with_nodeclaims"].add(bucket)
                iso = dt.astimezone(timezone.utc).isoformat().replace("+00:00", "Z")
                if rec["first_nonzero_ts"] is None or iso < rec["first_nonzero_ts"]:
                    rec["first_nonzero_ts"] = iso
                if rec["last_nonzero_ts"] is None or iso > rec["last_nonzero_ts"]:
                    rec["last_nonzero_ts"] = iso

nodeclaim_diagnostics = []
for _, v in sorted(nodeclaim_diag_map.items(), key=lambda kv: (kv[0][0], kv[0][1])):
    out = dict(v)
    out["minutes_with_nodeclaims"] = len(v["minutes_with_nodeclaims"])
    nodeclaim_diagnostics.append(out)

# Aggregates
secondary_metric_keys = [
    "native_peak_node_count",
    "native_node_minutes_estimate",
    "native_packing_efficiency_estimate",
    "unschedulable_minutes_in_run",
    "unschedulable_max_pods_in_run",
]

agg_by_target_profile = _aggregate(
    run_records,
    ["target", "profile", "scenario_type"],
    primary_kpi_keys + secondary_metric_keys,
)

agg_by_target_scenario_type = _aggregate(
    run_records,
    ["target", "scenario_type"],
    primary_kpi_keys + secondary_metric_keys,
)

custom_records = [r for r in run_records if r.get("scenario_type") == "custom"]
agg_by_target_custom_replicas = _aggregate(
    custom_records,
    ["target", "up_replicas", "custom_max_node_count"],
    primary_kpi_keys + secondary_metric_keys,
)

# Outputs
kpi_dir = artifact_dir / "kpi"
kpi_dir.mkdir(parents=True, exist_ok=True)

kpi_per_run_csv = kpi_dir / "kpi_per_run.csv"
kpi_summary_path = kpi_dir / "kpi_summary.json"

csv_fields = [
    "run_id",
    "profile",
    "scenario_type",
    "target",
    "repeat",
    "status",
    "up_replicas",
    "down_replicas",
    "custom_queue_id",
    "custom_max_node_count",
    "scenario_start_utc",
    "scale_up_requested_utc",
    "first_new_node_ready_utc",
    "first_pod_running_utc",
    "all_pods_running_utc",
    "scale_down_requested_utc",
    "workload_floor_reached_utc",
    "capacity_floor_reached_utc",
    "scenario_end_utc",
    "scale_up_to_first_new_node_ready_seconds",
    "scale_up_to_first_pod_running_seconds",
    "scale_up_to_all_pods_running_seconds",
    "scale_down_to_workload_floor_seconds",
    "scale_down_to_capacity_floor_seconds",
    "scenario_total_seconds",
    "primary_kpi_complete",
    "native_peak_node_count",
    "native_node_minutes_estimate",
    "native_packing_efficiency_estimate",
    "native_cost_estimate",
    "unschedulable_minutes_in_run",
    "unschedulable_max_pods_in_run",
]

with kpi_per_run_csv.open("w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=csv_fields)
    w.writeheader()
    for r in run_records:
        row = {k: r.get(k) for k in csv_fields}
        w.writerow(row)

pricebook_path = _first_nonempty(
    generation_summary.get("benchmark_pricebook_path"),
    _env("BENCHMARK_PRICEBOOK_PATH", ""),
)
pricebook_present = bool(pricebook_path)

kpi_summary = {
    "generated_at_utc": _now_utc(),
    "run_id": run_id,
    "kpi_schema_version": "v3-primary-only",
    "primary_event_contract_version": "v1",
    "benchmark_mode": benchmark_mode,
    "inputs": {
        "artifact_dir": str(artifact_dir),
        "summary_files": [str(p) for p in summary_files],
        "event_files": [str(p) for p in event_files],
        "custom_runs_index": str(artifact_dir / "scenario_custom_runs_index.json"),
        "telemetry_summary": str(
            artifact_dir / "telemetry" / "telemetry_collection_summary.json"
        ),
        "telemetry_metrics_csv": str(metrics_csv_path),
        "telemetry_k8s_events_jsonl": str(k8s_events_jsonl_path),
        "telemetry_trace_samples": str(trace_samples_path),
        "controller_logs_dir": str(controller_dir),
        "workload_request_shape": workload_req,
        "node_capacity_inference": node_capacity,
    },
    "per_run_kpis": run_records,
    "aggregates": {
        "by_target_profile": agg_by_target_profile,
        "by_target_scenario_type": agg_by_target_scenario_type,
        "by_target_custom_replicas": agg_by_target_custom_replicas,
        "unschedulable": unsched_summary,
        "controller_log_severity_by_target": controller_by_target,
        "controller_log_severity_by_file": controller_by_file,
        "k8s_event_counts": k8s_event_counts,
        "nodeclaim_diagnostics_by_target_profile": nodeclaim_diagnostics,
    },
    "native_mode_kpis": {
        "enabled": benchmark_mode == "native",
        "benchmark_mode": benchmark_mode,
        "pricebook_present": pricebook_present,
        "pricebook_path": pricebook_path if pricebook_present else None,
        "cost_estimate_supported": False,
        "cost_estimate_reason": "pricebook integration not configured",
    },
    "data_quality": {
        "total_profiles_discovered": len(profiles),
        "total_run_records": len(run_records),
        "primary_complete_runs": sum(
            1 for r in run_records if r.get("primary_kpi_complete")
        ),
        "runs_with_missing_primary_events": sum(
            1 for r in run_records if not r.get("primary_kpi_complete")
        ),
        "attempts_without_scenario_events": attempts_without_scenario_events,
        "metrics_rows": metrics_rows,
        "telemetry_present": bool(telemetry_summary),
        "telemetry_errors": (
            telemetry_summary.get("errors")
            if isinstance(telemetry_summary, dict)
            else []
        ),
    },
    "artifacts": {
        "kpi_per_run_csv": str(kpi_per_run_csv),
        "kpi_summary_json": str(kpi_summary_path),
    },
}

kpi_summary_path.write_text(
    json.dumps(kpi_summary, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)

os.environ["KPI_DIR"] = str(kpi_dir)
os.environ["KPI_PER_RUN_ARTIFACT"] = str(kpi_per_run_csv)
os.environ["KPI_SUMMARY_ARTIFACT"] = str(kpi_summary_path)

print("\nCell 15 summary:")
print(
    json.dumps(
        {
            "run_id": run_id,
            "schema": kpi_summary["kpi_schema_version"],
            "benchmark_mode": benchmark_mode,
            "profiles_discovered": len(profiles),
            "per_run_records": len(run_records),
            "primary_complete_runs": kpi_summary["data_quality"][
                "primary_complete_runs"
            ],
            "runs_with_missing_primary_events": kpi_summary["data_quality"][
                "runs_with_missing_primary_events"
            ],
            "attempts_without_scenario_events": len(attempts_without_scenario_events),
            "metrics_rows": metrics_rows,
            "kpi_summary": str(kpi_summary_path),
            "kpi_per_run_csv": str(kpi_per_run_csv),
        },
        indent=2,
    )
)
print(f"\nCell 15 complete. Artifact: {kpi_summary_path}")
print("Next: Run Cell 16.")

In [ ]:
# Cell 16 — Reporting (Primary-only, includes burst/steady/custom)

import os
import json
import csv
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict


def _env(k, d=""):
    return os.environ.get(k, d)


def _now_utc():
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def _first_nonempty(*vals):
    for v in vals:
        if v is None:
            continue
        s = str(v).strip()
        if s:
            return s
    return ""


def _parse_ts(ts):
    if not ts:
        return None
    s = str(ts).strip()
    if not s:
        return None
    if s.endswith("Z"):
        s = s[:-1] + "+00:00"
    try:
        return datetime.fromisoformat(s)
    except Exception:
        return None


def _fmt_ts(dt):
    if not dt:
        return ""
    return dt.astimezone(timezone.utc).isoformat().replace("+00:00", "Z")


def _read_json(path: Path, default=None):
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding="utf-8"))


def _write_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def _to_int(v, default=0):
    try:
        return int(v)
    except Exception:
        return default


def _to_float(v, default=None):
    try:
        return float(v)
    except Exception:
        return default


def _fmt_num(v, nd=3):
    if v is None:
        return "n/a"
    try:
        return f"{float(v):.{nd}f}"
    except Exception:
        return "n/a"


def _fmt_sec(v):
    if v is None:
        return "n/a"
    try:
        return f"{float(v):.3f}s"
    except Exception:
        return "n/a"


def _candidate_output_roots():
    roots = []
    out_env = _env("OUTPUT_DIR", "").strip()
    if out_env:
        roots.append(Path(os.path.expanduser(out_env)).resolve())

    cwd = Path.cwd().resolve()
    roots.append((cwd / "OCIK8sClusterAutoscalerVsKarpenter" / "artifacts").resolve())
    roots.append((cwd / "artifacts").resolve())

    uniq = []
    seen = set()
    for r in roots:
        k = str(r)
        if k not in seen:
            seen.add(k)
            uniq.append(r)
    return uniq


def _is_valid_run_dir(run_dir: Path):
    if not run_dir.is_dir():
        return False
    must_have = [
        run_dir / "kpi" / "kpi_summary.json",
        run_dir / "kpi" / "kpi_per_run.csv",
        run_dir / "telemetry" / "telemetry_collection_summary.json",
        run_dir / "provisioning.json",
    ]
    return all(p.exists() for p in must_have)


def _discover_run_context():
    run_id_env = _env("RUN_ID", "").strip()
    out_env = _env("OUTPUT_DIR", "").strip()
    roots = _candidate_output_roots()

    if run_id_env and out_env:
        out = Path(os.path.expanduser(out_env)).resolve()
        ad = out / run_id_env
        if _is_valid_run_dir(ad):
            return out, run_id_env, ad

    if run_id_env:
        for r in roots:
            ad = r / run_id_env
            if _is_valid_run_dir(ad):
                return r, run_id_env, ad

    if out_env:
        out = Path(os.path.expanduser(out_env)).resolve()
        if out.exists():
            runs = [p for p in out.glob("ca-vs-kpo-*") if _is_valid_run_dir(p)]
            if runs:
                runs.sort(key=lambda p: p.stat().st_mtime, reverse=True)
                ad = runs[0]
                return out, ad.name, ad

    candidates = []
    for r in roots:
        if not r.exists():
            continue
        for p in r.glob("ca-vs-kpo-*"):
            if _is_valid_run_dir(p):
                candidates.append((p.stat().st_mtime, r, p))

    if candidates:
        candidates.sort(key=lambda x: x[0], reverse=True)
        _, out, ad = candidates[0]
        return out, ad.name, ad

    raise RuntimeError("Could not resolve run context for Cell 16.")


def _scenario_type(profile: str):
    if profile == "burst":
        return "burst"
    if profile == "steady":
        return "steady"
    if str(profile).startswith("custom-"):
        return "custom"
    return "other"


def _delta(kpo_val, ca_val):
    if kpo_val is None or ca_val is None:
        return None
    return float(kpo_val) - float(ca_val)


# Resolve run context
output_dir, run_id, artifact_dir = _discover_run_context()
os.environ["OUTPUT_DIR"] = str(output_dir)
os.environ["RUN_ID"] = run_id

print("=== Cell 16 Reporting ===")
print(f"run_id: {run_id}")
print(f"artifact_dir: {artifact_dir}")

# Load inputs
kpi_summary = _read_json(artifact_dir / "kpi" / "kpi_summary.json", {}) or {}
telemetry_summary = (
    _read_json(artifact_dir / "telemetry" / "telemetry_collection_summary.json", {})
    or {}
)
provisioning = _read_json(artifact_dir / "provisioning.json", {}) or {}
post_prov = _read_json(artifact_dir / "post_provisioning.json", {}) or {}
workload = _read_json(artifact_dir / "workload_deployments.json", {}) or {}
custom_index = _read_json(artifact_dir / "scenario_custom_runs_index.json", {}) or {}
generation_summary = _read_json(Path("tf") / "generation-summary.json", {}) or {}

if not kpi_summary:
    raise RuntimeError("Missing kpi_summary.json. Run Cell 15 first.")

per_run = kpi_summary.get("per_run_kpis") or []
if not isinstance(per_run, list):
    per_run = []

agg_by_target_profile = (kpi_summary.get("aggregates") or {}).get(
    "by_target_profile"
) or []
agg_by_target_scenario_type = (kpi_summary.get("aggregates") or {}).get(
    "by_target_scenario_type"
) or []
agg_by_target_custom_replicas = (kpi_summary.get("aggregates") or {}).get(
    "by_target_custom_replicas"
) or []

benchmark_mode = _first_nonempty(
    kpi_summary.get("benchmark_mode"),
    generation_summary.get("benchmark_mode"),
    _env("BENCHMARK_MODE", "parity"),
).lower()
if benchmark_mode not in {"parity", "native"}:
    benchmark_mode = "parity"

kpi_schema_version = _first_nonempty(
    kpi_summary.get("kpi_schema_version"),
    "v3-primary-only",
)

selected_targets = sorted({r.get("target") for r in per_run if r.get("target")})
if not selected_targets:
    selected_targets = sorted((telemetry_summary.get("selected_targets") or []))
if not selected_targets:
    tc = _env("TARGET_CLUSTER", "both").strip().lower()
    selected_targets = ["ca", "kpo"] if tc == "both" else [tc]

# Timeline rows from per-run KPI records
timeline_rows = []
for r in per_run:
    row = {
        "run_id": run_id,
        "profile": r.get("profile"),
        "scenario_type": _first_nonempty(
            r.get("scenario_type"), _scenario_type(r.get("profile"))
        ),
        "target": r.get("target"),
        "repeat": _to_int(r.get("repeat"), 0),
        "status": r.get("status"),
        "primary_kpi_complete": bool(r.get("primary_kpi_complete")),
        "up_replicas": _to_int(r.get("up_replicas"), 0),
        "down_replicas": _to_int(r.get("down_replicas"), 0),
        "custom_queue_id": r.get("custom_queue_id"),
        "custom_max_node_count": r.get("custom_max_node_count"),
        "scenario_start_utc": r.get("scenario_start_utc"),
        "scale_up_requested_utc": r.get("scale_up_requested_utc"),
        "first_new_node_ready_utc": r.get("first_new_node_ready_utc"),
        "first_pod_running_utc": r.get("first_pod_running_utc"),
        "all_pods_running_utc": r.get("all_pods_running_utc"),
        "scale_down_requested_utc": r.get("scale_down_requested_utc"),
        "workload_floor_reached_utc": r.get("workload_floor_reached_utc"),
        "capacity_floor_reached_utc": r.get("capacity_floor_reached_utc"),
        "scenario_end_utc": r.get("scenario_end_utc"),
        "scale_up_to_first_new_node_ready_seconds": r.get(
            "scale_up_to_first_new_node_ready_seconds"
        ),
        "scale_up_to_first_pod_running_seconds": r.get(
            "scale_up_to_first_pod_running_seconds"
        ),
        "scale_up_to_all_pods_running_seconds": r.get(
            "scale_up_to_all_pods_running_seconds"
        ),
        "scale_down_to_workload_floor_seconds": r.get(
            "scale_down_to_workload_floor_seconds"
        ),
        "scale_down_to_capacity_floor_seconds": r.get(
            "scale_down_to_capacity_floor_seconds"
        ),
        "scenario_total_seconds": r.get("scenario_total_seconds"),
        "unschedulable_minutes_in_run": r.get("unschedulable_minutes_in_run"),
        "unschedulable_max_pods_in_run": r.get("unschedulable_max_pods_in_run"),
        "native_peak_node_count": r.get("native_peak_node_count"),
        "native_node_minutes_estimate": r.get("native_node_minutes_estimate"),
        "native_packing_efficiency_estimate": r.get(
            "native_packing_efficiency_estimate"
        ),
    }
    timeline_rows.append(row)


def _timeline_sort_key(x):
    st = _parse_ts(x.get("scenario_start_utc"))
    return (
        st or datetime.min.replace(tzinfo=timezone.utc),
        str(x.get("profile") or ""),
        str(x.get("target") or ""),
        int(x.get("repeat") or 0),
    )


timeline_rows.sort(key=_timeline_sort_key)

# Execution matrix from summary files (dynamic)
execution_matrix_map = {}
for sf in sorted(artifact_dir.glob("scenario_execution_*_summary.json")):
    s = _read_json(sf, {}) or {}
    profile = s.get("profile")
    if not profile:
        continue
    cfg = s.get("config") or {}
    for rr in s.get("runs", []) or []:
        target = rr.get("target")
        if not target:
            continue
        key = (profile, target)
        rec = execution_matrix_map.setdefault(
            key,
            {
                "profile": profile,
                "scenario_type": _scenario_type(profile),
                "target": target,
                "repeats": 0,
                "successes": 0,
                "failures": 0,
                "up_replicas": rr.get("up_replicas", cfg.get("up_replicas")),
                "down_replicas": rr.get("down_replicas", cfg.get("down_replicas")),
                "hold_seconds": cfg.get("hold_seconds"),
                "poll_seconds": cfg.get("poll_seconds"),
                "poll_up_seconds": cfg.get("poll_up_seconds"),
                "poll_down_seconds": cfg.get("poll_down_seconds"),
                "ready_timeout_seconds": cfg.get("ready_timeout_seconds"),
                "scale_down_timeout_seconds": cfg.get("scale_down_timeout_seconds"),
                "summary_file": str(sf),
            },
        )
        rec["repeats"] += 1
        if str(rr.get("status")) == "success":
            rec["successes"] += 1
        else:
            rec["failures"] += 1

execution_matrix = sorted(
    execution_matrix_map.values(),
    key=lambda x: (x["scenario_type"], x["profile"], x["target"]),
)

# High-level totals
pods_requested_total = sum(_to_int(r.get("up_replicas"), 0) for r in timeline_rows)
pods_requested_by_group = defaultdict(int)
for r in timeline_rows:
    pods_requested_by_group[(r.get("profile"), r.get("target"))] += _to_int(
        r.get("up_replicas"), 0
    )

scenario_durations = [
    _to_float(r.get("scenario_total_seconds"))
    for r in timeline_rows
    if r.get("scenario_total_seconds") is not None
]
scenario_durations = [v for v in scenario_durations if v is not None]

all_starts = [
    _parse_ts(r.get("scenario_start_utc"))
    for r in timeline_rows
    if r.get("scenario_start_utc")
]
all_starts = [v for v in all_starts if v]
all_ends = [
    _parse_ts(r.get("scenario_end_utc"))
    for r in timeline_rows
    if r.get("scenario_end_utc")
]
all_ends = [v for v in all_ends if v]
window = {
    "start_utc": _fmt_ts(min(all_starts)) if all_starts else None,
    "end_utc": _fmt_ts(max(all_ends)) if all_ends else None,
}

# Aggregates lookup for report tables
agg_map = {(g.get("target"), g.get("profile")): g for g in agg_by_target_profile}

# Deltas where both ca and kpo exist for same profile
profiles = sorted({r.get("profile") for r in timeline_rows if r.get("profile")})
primary_delta_by_profile = []
for profile in profiles:
    ca = agg_map.get(("ca", profile), {})
    kpo = agg_map.get(("kpo", profile), {})
    if not ca or not kpo:
        continue

    def _m(obj, key):
        return (obj.get(key) or {}).get("mean")

    primary_delta_by_profile.append(
        {
            "profile": profile,
            "scenario_type": _scenario_type(profile),
            "delta_scale_up_to_first_new_node_ready_seconds": _delta(
                _m(kpo, "scale_up_to_first_new_node_ready_seconds"),
                _m(ca, "scale_up_to_first_new_node_ready_seconds"),
            ),
            "delta_scale_up_to_first_pod_running_seconds": _delta(
                _m(kpo, "scale_up_to_first_pod_running_seconds"),
                _m(ca, "scale_up_to_first_pod_running_seconds"),
            ),
            "delta_scale_up_to_all_pods_running_seconds": _delta(
                _m(kpo, "scale_up_to_all_pods_running_seconds"),
                _m(ca, "scale_up_to_all_pods_running_seconds"),
            ),
            "delta_scale_down_to_workload_floor_seconds": _delta(
                _m(kpo, "scale_down_to_workload_floor_seconds"),
                _m(ca, "scale_down_to_workload_floor_seconds"),
            ),
            "delta_scale_down_to_capacity_floor_seconds": _delta(
                _m(kpo, "scale_down_to_capacity_floor_seconds"),
                _m(ca, "scale_down_to_capacity_floor_seconds"),
            ),
        }
    )

# Incomplete runs for explicit reporting
incomplete_runs = [
    {
        "profile": r.get("profile"),
        "scenario_type": r.get("scenario_type"),
        "target": r.get("target"),
        "repeat": r.get("repeat"),
        "status": r.get("status"),
        "missing_primary_events": (
            ((r.get("data_quality") or {}).get("missing_primary_events"))
            if isinstance(r.get("data_quality"), dict)
            else []
        ),
    }
    for r in per_run
    if not bool(r.get("primary_kpi_complete"))
]

attempts_without_events = (kpi_summary.get("data_quality") or {}).get(
    "attempts_without_scenario_events"
) or []

# Region + context
targets_obj = provisioning.get("targets", {}) or {}
region_context = {
    "region": _first_nonempty(
        provisioning.get("region"),
        telemetry_summary.get("region"),
        _env("OCI_REGION", ""),
    ),
    "benchmark_compartment_id": _first_nonempty(
        _env("BENCHMARK_COMPARTMENT_OCID", ""),
        (targets_obj.get("ca") or {}).get("compartment_id"),
        (targets_obj.get("kpo") or {}).get("compartment_id"),
    ),
    "target_cluster": _first_nonempty(
        provisioning.get("target_cluster"),
        _env("TARGET_CLUSTER", "both"),
    ),
    "selected_targets": selected_targets,
}

capacity_selection = provisioning.get("capacity_selection", {}) or {}
selected_capacity = {}
for t, tcfg in capacity_selection.items():
    selected_capacity[t] = {
        "cluster_name": tcfg.get("cluster_name"),
        "kubernetes_version": tcfg.get("kubernetes_version"),
        "worker_pools": tcfg.get("worker_pools", {}),
    }

versions = {
    "kubernetes": {
        "ca": (capacity_selection.get("ca") or {}).get("kubernetes_version"),
        "kpo": (capacity_selection.get("kpo") or {}).get("kubernetes_version"),
    },
    "controllers": {
        "ca_namespace": _env("CA_NAMESPACE", "kube-system"),
        "kpo_namespace": _env("KPO_NAMESPACE", "karpenter"),
    },
    "benchmark_mode": benchmark_mode,
}

cluster_identifiers = {}
for t in ["ca", "kpo"]:
    tp = targets_obj.get(t, {}) or {}
    cluster_identifiers[t] = {
        "cluster_id": tp.get("cluster_id"),
        "vcn_id": tp.get("vcn_id"),
        "apiserver_public_endpoint": tp.get("apiserver_public_endpoint"),
        "apiserver_private_endpoint": tp.get("apiserver_private_endpoint"),
        "kubeconfig_path": tp.get("kubeconfig_path"),
        "worker_pool_ids": tp.get("worker_pool_ids", {}),
    }

# Prepare output dirs
reporting_dir = artifact_dir / "reporting"
reporting_dir.mkdir(parents=True, exist_ok=True)

artifacts = {
    "run_manifest_json": str(artifact_dir / "run_manifest.json"),
    "comparison_report_md": str(artifact_dir / "comparison_report.md"),
    "reporting_summary_json": str(artifact_dir / "reporting_summary.json"),
    "execution_timeline_csv": str(reporting_dir / "scenario_timeline.csv"),
    "execution_matrix_csv": str(reporting_dir / "execution_matrix.csv"),
    "timeline_summary_json": str(reporting_dir / "timeline_summary.json"),
    "kpi_summary_json": str(artifact_dir / "kpi" / "kpi_summary.json"),
    "kpi_per_run_csv": str(artifact_dir / "kpi" / "kpi_per_run.csv"),
    "telemetry_summary_json": str(
        artifact_dir / "telemetry" / "telemetry_collection_summary.json"
    ),
    "telemetry_metrics_csv": str(artifact_dir / "telemetry" / "oci_metrics.csv"),
    "k8s_events_jsonl": str(artifact_dir / "telemetry" / "k8s_events.jsonl"),
    "controller_logs_dir": str(artifact_dir / "telemetry" / "controller_logs"),
    "custom_runs_index_json": str(artifact_dir / "scenario_custom_runs_index.json"),
}

# run_manifest
run_manifest = {
    "generated_at_utc": _now_utc(),
    "run_id": run_id,
    "benchmark_mode": benchmark_mode,
    "kpi_schema_version": kpi_schema_version,
    "primary_event_contract_version": kpi_summary.get("primary_event_contract_version"),
    "region_context": region_context,
    "selected_shapes_sizes_images_flex": selected_capacity,
    "resolved_images": {
        "image_mode": _first_nonempty(
            provisioning.get("image_mode"),
            generation_summary.get("image_mode"),
        ),
        "image_selection_artifact": provisioning.get("image_selection_artifact"),
    },
    "versions": versions,
    "scenario_window": window,
    "cluster_and_nodepool_identifiers": cluster_identifiers,
    "execution_overview": {
        "profiles_discovered": len(profiles),
        "total_runs": len(timeline_rows),
        "successful_runs": len(
            [r for r in timeline_rows if str(r.get("status")) == "success"]
        ),
        "primary_complete_runs": len(
            [r for r in timeline_rows if bool(r.get("primary_kpi_complete"))]
        ),
        "runs_with_missing_primary_events": len(incomplete_runs),
        "attempts_without_scenario_events": len(attempts_without_events),
        "pods_requested_total": pods_requested_total,
    },
    "artifacts": artifacts,
}

_write_json(artifact_dir / "run_manifest.json", run_manifest)

# Export CSV artifacts
if timeline_rows:
    with (reporting_dir / "scenario_timeline.csv").open(
        "w", encoding="utf-8", newline=""
    ) as f:
        w = csv.DictWriter(f, fieldnames=list(timeline_rows[0].keys()))
        w.writeheader()
        w.writerows(timeline_rows)

if execution_matrix:
    with (reporting_dir / "execution_matrix.csv").open(
        "w", encoding="utf-8", newline=""
    ) as f:
        w = csv.DictWriter(f, fieldnames=list(execution_matrix[0].keys()))
        w.writeheader()
        w.writerows(execution_matrix)

# timeline_summary.json
telemetry_counts = telemetry_summary.get("counts") or {}
summary_obj = {
    "generated_at_utc": _now_utc(),
    "run_id": run_id,
    "benchmark_mode": benchmark_mode,
    "kpi_schema_version": kpi_schema_version,
    "execution_matrix": execution_matrix,
    "timeline_rows": timeline_rows,
    "primary_deltas_by_profile": primary_delta_by_profile,
    "incomplete_runs": incomplete_runs,
    "attempts_without_scenario_events": attempts_without_events,
    "totals": {
        "runs": len(timeline_rows),
        "success_runs": len(
            [r for r in timeline_rows if str(r.get("status")) == "success"]
        ),
        "primary_complete_runs": len(
            [r for r in timeline_rows if bool(r.get("primary_kpi_complete"))]
        ),
        "pods_requested_total": pods_requested_total,
        "pods_requested_by_profile_target": [
            {"profile": k[0], "target": k[1], "pods_requested": v}
            for k, v in sorted(pods_requested_by_group.items())
        ],
        "scenario_total_seconds_mean": (
            (sum(scenario_durations) / len(scenario_durations))
            if scenario_durations
            else None
        ),
    },
    "telemetry": {
        "metrics_rows": telemetry_counts.get("metrics_rows"),
        "k8s_events": telemetry_counts.get("k8s_events"),
        "controller_log_files": telemetry_counts.get("controller_log_files"),
        "errors": telemetry_summary.get("errors", []),
    },
}

_write_json(reporting_dir / "timeline_summary.json", summary_obj)

# Markdown comparison report
lines = []
lines.append("# CA vs KPO Comparison Report")
lines.append("")
lines.append(f"Run ID: `{run_id}`")
lines.append(f"Generated: `{_now_utc()}`")
lines.append(f"Benchmark Mode: `{benchmark_mode}`")
lines.append(f"KPI Schema: `{kpi_schema_version}`")
lines.append("")

lines.append("## Scope")
lines.append(f"- Targets: `{', '.join(selected_targets)}`")
lines.append(f"- Region: `{region_context.get('region')}`")
lines.append(
    f"- Scenario window: `{window.get('start_utc')}` to `{window.get('end_utc')}`"
)
lines.append(f"- Profiles discovered: `{len(profiles)}`")
lines.append("")

if benchmark_mode == "native":
    lines.append("## Headline (Native Mode)")
    lines.append(
        "Primary headline metrics are request-based and symmetric. Native efficiency metrics are shown as secondary context."
    )
else:
    lines.append("## Headline (Parity Mode)")
    lines.append(
        "Primary headline metrics are request-based and symmetric for strict cross-target comparison."
    )
lines.append("")

lines.append("## Execution Matrix")
lines.append(
    "| Profile | Type | Target | Repeats | Success | Failure | Up Replicas | Down Replicas | Hold (s) | Poll Up (s) | Poll Down (s) |"
)
lines.append("|---|---|---|---:|---:|---:|---:|---:|---:|---:|---:|")
for m in execution_matrix:
    lines.append(
        f"| {m.get('profile')} | {m.get('scenario_type')} | {m.get('target')} | {m.get('repeats')} | {m.get('successes')} | {m.get('failures')} | {m.get('up_replicas')} | {m.get('down_replicas')} | {m.get('hold_seconds')} | {m.get('poll_up_seconds')} | {m.get('poll_down_seconds')} |"
    )
lines.append("")

lines.append("## Primary KPI Means by Profile/Target")
lines.append(
    "| Profile | Type | Target | Up->FirstNodeReady | Up->FirstPodRunning | Up->AllPodsRunning | Down->WorkloadFloor | Down->CapacityFloor |"
)
lines.append("|---|---|---|---:|---:|---:|---:|---:|")
for g in sorted(
    agg_by_target_profile,
    key=lambda x: (
        x.get("scenario_type") or "",
        x.get("profile") or "",
        x.get("target") or "",
    ),
):
    lines.append(
        f"| {g.get('profile')} | {g.get('scenario_type')} | {g.get('target')} | {_fmt_sec(((g.get('scale_up_to_first_new_node_ready_seconds') or {}).get('mean')))} | {_fmt_sec(((g.get('scale_up_to_first_pod_running_seconds') or {}).get('mean')))} | {_fmt_sec(((g.get('scale_up_to_all_pods_running_seconds') or {}).get('mean')))} | {_fmt_sec(((g.get('scale_down_to_workload_floor_seconds') or {}).get('mean')))} | {_fmt_sec(((g.get('scale_down_to_capacity_floor_seconds') or {}).get('mean')))} |"
    )
lines.append("")

if primary_delta_by_profile:
    lines.append("## Primary KPI Delta (KPO - CA)")
    lines.append(
        "| Profile | Type | FirstNodeReady Δ | FirstPodRunning Δ | AllPodsRunning Δ | WorkloadFloor Δ | CapacityFloor Δ |"
    )
    lines.append("|---|---|---:|---:|---:|---:|---:|")
    for d in sorted(
        primary_delta_by_profile,
        key=lambda x: (x.get("scenario_type") or "", x.get("profile") or ""),
    ):
        lines.append(
            f"| {d.get('profile')} | {d.get('scenario_type')} | {_fmt_sec(d.get('delta_scale_up_to_first_new_node_ready_seconds'))} | {_fmt_sec(d.get('delta_scale_up_to_first_pod_running_seconds'))} | {_fmt_sec(d.get('delta_scale_up_to_all_pods_running_seconds'))} | {_fmt_sec(d.get('delta_scale_down_to_workload_floor_seconds'))} | {_fmt_sec(d.get('delta_scale_down_to_capacity_floor_seconds'))} |"
        )
    lines.append("")

if agg_by_target_custom_replicas:
    lines.append("## Custom Run Scaling View")
    lines.append(
        "| Target | Up Replicas | Max Nodes | Runs | Success | Primary Complete | Up->AllPodsRunning Mean | Down->CapacityFloor Mean |"
    )
    lines.append("|---|---:|---:|---:|---:|---:|---:|---:|")
    for g in sorted(
        agg_by_target_custom_replicas,
        key=lambda x: (
            x.get("target") or "",
            _to_int(x.get("up_replicas"), 0),
            _to_int(x.get("custom_max_node_count"), 0),
        ),
    ):
        lines.append(
            f"| {g.get('target')} | {g.get('up_replicas')} | {g.get('custom_max_node_count')} | {g.get('run_count')} | {g.get('success_count')} | {g.get('primary_complete_count')} | {_fmt_sec(((g.get('scale_up_to_all_pods_running_seconds') or {}).get('mean')))} | {_fmt_sec(((g.get('scale_down_to_capacity_floor_seconds') or {}).get('mean')))} |"
        )
    lines.append("")

lines.append("## Per-Run Timeline")
lines.append(
    "| Profile | Type | Target | Repeat | Status | Primary Complete | ScaleUpRequested | FirstNodeReady | FirstPodRunning | AllPodsRunning | ScaleDownRequested | WorkloadFloor | CapacityFloor |"
)
lines.append("|---|---|---|---:|---|---|---|---|---|---|---|---|---|")
for r in timeline_rows:
    lines.append(
        f"| {r.get('profile')} | {r.get('scenario_type')} | {r.get('target')} | {r.get('repeat')} | {r.get('status')} | {r.get('primary_kpi_complete')} | {r.get('scale_up_requested_utc') or 'n/a'} | {r.get('first_new_node_ready_utc') or 'n/a'} | {r.get('first_pod_running_utc') or 'n/a'} | {r.get('all_pods_running_utc') or 'n/a'} | {r.get('scale_down_requested_utc') or 'n/a'} | {r.get('workload_floor_reached_utc') or 'n/a'} | {r.get('capacity_floor_reached_utc') or 'n/a'} |"
    )
lines.append("")

if incomplete_runs:
    lines.append("## Incomplete Primary Runs")
    lines.append(
        "| Profile | Type | Target | Repeat | Status | Missing Primary Events |"
    )
    lines.append("|---|---|---|---:|---|---|")
    for r in sorted(
        incomplete_runs,
        key=lambda x: (
            x.get("scenario_type") or "",
            x.get("profile") or "",
            x.get("target") or "",
            _to_int(x.get("repeat"), 0),
        ),
    ):
        miss = ", ".join(r.get("missing_primary_events") or [])
        lines.append(
            f"| {r.get('profile')} | {r.get('scenario_type')} | {r.get('target')} | {r.get('repeat')} | {r.get('status')} | {miss or 'n/a'} |"
        )
    lines.append("")

if attempts_without_events:
    lines.append("## Attempts Without Scenario Event Files")
    lines.append(
        "| Profile | Queue ID | Status | Replicas | Max Nodes | Started | Ended |"
    )
    lines.append("|---|---|---|---:|---:|---|---|")
    for a in attempts_without_events:
        lines.append(
            f"| {a.get('profile')} | {a.get('queue_id') or 'n/a'} | {a.get('status')} | {a.get('replicas') or 'n/a'} | {a.get('max_node_count') or 'n/a'} | {a.get('started_at_utc') or 'n/a'} | {a.get('ended_at_utc') or 'n/a'} |"
        )
    lines.append("")

lines.append("## Telemetry Diagnostics")
lines.append(
    f"- Metrics rows: `{(telemetry_summary.get('counts') or {}).get('metrics_rows')}`"
)
lines.append(
    f"- Kubernetes events rows: `{(telemetry_summary.get('counts') or {}).get('k8s_events')}`"
)
lines.append(
    f"- Controller log files: `{(telemetry_summary.get('counts') or {}).get('controller_log_files')}`"
)
lines.append(f"- Telemetry errors: `{len(telemetry_summary.get('errors') or [])}`")
lines.append("")

unsched = (kpi_summary.get("aggregates") or {}).get("unschedulable") or []
if unsched:
    lines.append("### UnschedulablePods")
    lines.append(
        "| Target | Datapoints | Unschedulable Datapoints | Duration (min) | Max Pods |"
    )
    lines.append("|---|---:|---:|---:|---:|")
    for u in sorted(unsched, key=lambda x: x.get("target") or ""):
        lines.append(
            f"| {u.get('target')} | {u.get('datapoints_total')} | {u.get('unschedulable_datapoints')} | {u.get('unschedulable_duration_minutes')} | {_fmt_num(u.get('unschedulable_max_pods'), 1)} |"
        )
    lines.append("")

controller = (kpi_summary.get("aggregates") or {}).get(
    "controller_log_severity_by_target"
) or {}
if controller:
    lines.append("### Controller Log Severity")
    lines.append("| Target | Info | Warning | Error | Files |")
    lines.append("|---|---:|---:|---:|---:|")
    for t in sorted(controller.keys()):
        c = controller.get(t) or {}
        lines.append(
            f"| {t} | {c.get('info', 0)} | {c.get('warning', 0)} | {c.get('error', 0)} | {c.get('files', 0)} |"
        )
    lines.append("")

k8s_counts = (kpi_summary.get("aggregates") or {}).get("k8s_event_counts") or {}
if k8s_counts:
    lines.append("### Kubernetes Events")
    lines.append("| Target | Total | Warning | Normal | Other |")
    lines.append("|---|---:|---:|---:|---:|")
    for t in sorted(k8s_counts.keys()):
        c = k8s_counts.get(t) or {}
        lines.append(
            f"| {t} | {c.get('total', 0)} | {c.get('warning', 0)} | {c.get('normal', 0)} | {c.get('other', 0)} |"
        )
    lines.append("")

if benchmark_mode == "native":
    lines.append("## Native Efficiency KPIs (Secondary)")
    lines.append(
        "| Profile | Type | Target | Peak Nodes (mean) | Node-Minutes (mean) | Packing Efficiency (mean) |"
    )
    lines.append("|---|---|---|---:|---:|---:|")
    for g in sorted(
        agg_by_target_profile,
        key=lambda x: (
            x.get("scenario_type") or "",
            x.get("profile") or "",
            x.get("target") or "",
        ),
    ):
        lines.append(
            f"| {g.get('profile')} | {g.get('scenario_type')} | {g.get('target')} | {_fmt_num(((g.get('native_peak_node_count') or {}).get('mean')),1)} | {_fmt_num(((g.get('native_node_minutes_estimate') or {}).get('mean')),1)} | {_fmt_num(((g.get('native_packing_efficiency_estimate') or {}).get('mean')),3)} |"
        )
    lines.append("")

lines.append("## Artifact Index")
for k, v in artifacts.items():
    lines.append(f"- `{k}`: `{v}`")

report_path = artifact_dir / "comparison_report.md"
report_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

# reporting_summary.json
report_summary = {
    "generated_at_utc": _now_utc(),
    "run_id": run_id,
    "benchmark_mode": benchmark_mode,
    "kpi_schema_version": kpi_schema_version,
    "run_manifest": str(artifact_dir / "run_manifest.json"),
    "comparison_report": str(report_path),
    "reporting_dir": str(reporting_dir),
    "counts": {
        "timeline_rows": len(timeline_rows),
        "execution_matrix_rows": len(execution_matrix),
        "primary_delta_profiles": len(primary_delta_by_profile),
        "incomplete_runs": len(incomplete_runs),
        "attempts_without_scenario_events": len(attempts_without_events),
    },
    "artifacts": artifacts,
}
_write_json(artifact_dir / "reporting_summary.json", report_summary)

os.environ["RUN_MANIFEST_ARTIFACT"] = str(artifact_dir / "run_manifest.json")
os.environ["COMPARISON_REPORT_ARTIFACT"] = str(report_path)
os.environ["REPORTING_SUMMARY_ARTIFACT"] = str(artifact_dir / "reporting_summary.json")

print("\nCell 16 summary:")
print(json.dumps(report_summary, indent=2))
print(f"\nCell 16 complete. Artifact: {report_path}")
print("Next: Run Cell 17 (cleanup) if needed.")

In [ ]:
# Cell 17 — Cleanup Resources

import os
import json
import re
import time
import subprocess
from pathlib import Path
from datetime import datetime, timezone


# -------------------------------
# Helpers
# -------------------------------
def _env(k, d=""):
    return os.environ.get(k, d)


def _now_utc():
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def _parse_bool(v, default=False):
    if v is None:
        return default
    s = str(v).strip().lower()
    if s in {"1", "true", "yes", "y", "on"}:
        return True
    if s in {"0", "false", "no", "n", "off"}:
        return False
    return default


def _first_nonempty(*vals):
    for v in vals:
        if v is None:
            continue
        s = str(v).strip()
        if s:
            return s
    return ""


def _json(path: Path, default=None):
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding="utf-8"))


def _candidate_output_roots():
    roots = []
    out_env = _env("OUTPUT_DIR", "").strip()
    if out_env:
        roots.append(Path(os.path.expanduser(out_env)).resolve())
    cwd = Path.cwd().resolve()
    roots.append((cwd / "OCIK8sClusterAutoscalerVsKarpenter" / "artifacts").resolve())
    roots.append((cwd / "artifacts").resolve())

    uniq = []
    seen = set()
    for r in roots:
        k = str(r)
        if k not in seen:
            seen.add(k)
            uniq.append(r)
    return uniq


def _is_valid_run_dir(run_dir: Path):
    if not run_dir.is_dir():
        return False
    # Keep cleanup available even for partially completed runs.
    return (run_dir / "provisioning.json").exists()


def _discover_run_context():
    run_id_env = _env("RUN_ID", "").strip()
    out_env = _env("OUTPUT_DIR", "").strip()
    roots = _candidate_output_roots()

    if run_id_env and out_env:
        out = Path(os.path.expanduser(out_env)).resolve()
        ad = out / run_id_env
        if _is_valid_run_dir(ad):
            return out, run_id_env, ad

    if run_id_env:
        for r in roots:
            ad = r / run_id_env
            if _is_valid_run_dir(ad):
                return r, run_id_env, ad

    if out_env:
        out = Path(os.path.expanduser(out_env)).resolve()
        if out.exists():
            runs = [p for p in out.glob("ca-vs-kpo-*") if _is_valid_run_dir(p)]
            if runs:
                runs.sort(key=lambda p: p.stat().st_mtime, reverse=True)
                ad = runs[0]
                return out, ad.name, ad

    candidates = []
    for r in roots:
        if not r.exists():
            continue
        for p in r.glob("ca-vs-kpo-*"):
            if _is_valid_run_dir(p):
                candidates.append((p.stat().st_mtime, r, p))

    if candidates:
        candidates.sort(key=lambda x: x[0], reverse=True)
        _, out, ad = candidates[0]
        return out, ad.name, ad

    raise RuntimeError("Could not resolve run context for Cell 17.")


def _run(cmd, timeout=None):
    p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    return {
        "cmd": cmd,
        "rc": p.returncode,
        "stdout": p.stdout,
        "stderr": p.stderr,
    }


def _run_json(cmd, timeout=None):
    r = _run(cmd, timeout=timeout)
    if r["rc"] != 0:
        return r, None
    txt = (r.get("stdout") or "").strip()
    if not txt:
        return r, None
    try:
        return r, json.loads(txt)
    except Exception:
        return r, None


def _extract_manifest_name(path: Path):
    if not path.exists():
        return ""
    txt = path.read_text(encoding="utf-8")
    m = re.search(r"(?m)^metadata:\s*$[\s\S]*?^\s*name:\s*([A-Za-z0-9._-]+)\s*$", txt)
    if m:
        return m.group(1)
    m2 = re.search(r"(?m)^\s*name:\s*([A-Za-z0-9._-]+)\s*$", txt)
    return m2.group(1) if m2 else ""


def _kubectl_json(kubeconfig, args):
    cmd = ["kubectl", "--kubeconfig", str(kubeconfig)] + args + ["-o", "json"]
    r, j = _run_json(cmd)
    return r, j


def _wait_until(fn, timeout_seconds, poll_seconds=10):
    started = time.time()
    history = []
    while True:
        ok, payload = fn()
        history.append({"time_utc": _now_utc(), "ok": bool(ok), "payload": payload})
        if ok:
            return True, history
        if time.time() - started >= timeout_seconds:
            return False, history
        time.sleep(poll_seconds)


def _is_not_found(stderr_or_stdout):
    s = (stderr_or_stdout or "").lower()
    needles = [
        "notauthorizedornotfound",
        "not found",
        "404",
        "entity does not exist",
        "no such",
    ]
    return any(n in s for n in needles)


# -------------------------------
# Context discovery
# -------------------------------
output_dir, run_id, artifact_dir = _discover_run_context()
os.environ["OUTPUT_DIR"] = str(output_dir)
os.environ["RUN_ID"] = run_id

provisioning = _json(artifact_dir / "provisioning.json", {})
post_prov = _json(artifact_dir / "post_provisioning.json", {})
workload_deployments = _json(artifact_dir / "workload_deployments.json", {})
generation_summary = _json(Path("tf") / "generation-summary.json", {})
reporting_summary = _json(artifact_dir / "reporting_summary.json", {})
run_manifest = _json(artifact_dir / "run_manifest.json", {})

# Target resolution priority: env -> run_manifest -> provisioning
target_cluster = _env("TARGET_CLUSTER", "").strip().lower()
if target_cluster in {"ca", "kpo"}:
    selected_targets = [target_cluster]
elif target_cluster == "both":
    selected_targets = ["ca", "kpo"]
else:
    selected_targets = (
        (run_manifest.get("region_context") or {}).get("selected_targets")
        or provisioning.get("selected_targets")
        or ["ca", "kpo"]
    )
selected_targets = [t for t in selected_targets if t in {"ca", "kpo"}]
if not selected_targets:
    selected_targets = ["ca", "kpo"]

region = _first_nonempty(
    _env("OCI_REGION", ""),
    provisioning.get("region"),
    generation_summary.get("region"),
    "us-ashburn-1",
)
oci_profile = _first_nonempty(_env("OCI_PROFILE", ""), "DEFAULT")
oci_config_file = str(
    Path(
        os.path.expanduser(
            _first_nonempty(_env("OCI_CONFIG_FILE", "~/.oci/config"), "~/.oci/config")
        )
    ).resolve()
)

cleanup_timeout_nodeclaims = int(
    _first_nonempty(_env("CLEANUP_NODECLAIMS_TIMEOUT_SECONDS", "1800"), "1800")
)
cleanup_timeout_nodes = int(
    _first_nonempty(_env("CLEANUP_NODES_TIMEOUT_SECONDS", "1200"), "1200")
)
terraform_destroy_timeout_seconds = int(
    _first_nonempty(_env("CLEANUP_TERRAFORM_DESTROY_TIMEOUT_SECONDS", "7200"), "7200")
)
poll_seconds = int(_first_nonempty(_env("CLEANUP_POLL_SECONDS", "10"), "10"))
dry_run = _parse_bool(_env("CLEANUP_DRY_RUN", "false"), default=False)
skip_terraform_destroy = _parse_bool(
    _env("CLEANUP_SKIP_TERRAFORM_DESTROY", "false"), default=False
)
cleanup_delete_iam_bootstrap = _parse_bool(
    _env("CLEANUP_DELETE_IAM_BOOTSTRAP", "true"), default=True
)
terraform_bin = _first_nonempty(_env("TERRAFORM_BIN", ""), "terraform")

print("=== Cell 17 Cleanup ===")
print(f"run_id: {run_id}")
print(f"artifact_dir: {artifact_dir}")
print(f"selected_targets: {selected_targets}")
print(f"region: {region}")
print(f"oci_profile: {oci_profile}")
print(f"oci_config_file: {oci_config_file}")
print(f"terraform_bin: {terraform_bin}")
print(f"dry_run: {dry_run}")
print(f"skip_terraform_destroy: {skip_terraform_destroy}")
print(f"cleanup_delete_iam_bootstrap: {cleanup_delete_iam_bootstrap}")

# -------------------------------
# Prepare per-target runtime context
# -------------------------------
kubeconfigs = {}
stack_dirs = {}
cluster_ids = {}
worker_pool_ids = {}

for t in ["ca", "kpo"]:
    tp = (provisioning.get("targets") or {}).get(t, {}) or {}
    wd = (workload_deployments.get("targets") or {}).get(t, {}) or {}

    kubeconfigs[t] = _first_nonempty(
        wd.get("kubeconfig"),
        tp.get("kubeconfig_path"),
    )
    stack_dirs[t] = _first_nonempty(
        tp.get("stack_dir"),
        generation_summary.get(f"{t}_stack_dir"),
        str((Path("tf") / t).resolve()),
    )
    cluster_ids[t] = _first_nonempty(tp.get("cluster_id"))
    worker_pool_ids[t] = list(((tp.get("worker_pool_ids") or {}).values()))

# -------------------------------
# Cleanup execution log
# -------------------------------
cleanup_actions = []
errors = []


def _record(action, data):
    cleanup_actions.append({"time_utc": _now_utc(), "action": action, "data": data})


# -------------------------------
# KPO pre-destroy cleanup: NodePool -> NodeClaims -> Nodes -> OCINodeClass
# -------------------------------
kpo_cleanup = {
    "executed": False,
    "kubeconfig": kubeconfigs.get("kpo"),
    "nodepool_candidates": [],
    "nodepools_deleted": [],
    "nodeclaims_wait": {},
    "nodes_wait": {},
    "ocinodeclass_candidates": [],
    "ocinodeclasses_deleted": [],
}

if "kpo" in selected_targets:
    kpo_cleanup["executed"] = True
    kpo_kubeconfig = kubeconfigs.get("kpo")

    if not kpo_kubeconfig:
        msg = "Missing KPO kubeconfig path in artifacts/provisioning outputs."
        errors.append(msg)
        _record("kpo_cleanup_skipped", {"reason": msg})
    else:
        np_candidates = set()
        nc_candidates = set()

        # Candidate from workload selector
        kpo_node_selector = (
            (workload_deployments.get("targets") or {}).get("kpo") or {}
        ).get("node_selector") or {}
        if kpo_node_selector.get("karpenter.sh/nodepool"):
            np_candidates.add(kpo_node_selector.get("karpenter.sh/nodepool"))

        # Candidate names from rendered manifests
        applied = (post_prov.get("applied_manifests") or {}).get("kpo") or {}
        nodepool_manifest = (
            Path(applied.get("nodepool")) if applied.get("nodepool") else None
        )
        ocinodeclass_manifest = (
            Path(applied.get("ocinodeclass")) if applied.get("ocinodeclass") else None
        )

        if nodepool_manifest:
            nm = _extract_manifest_name(nodepool_manifest)
            if nm:
                np_candidates.add(nm)
        if ocinodeclass_manifest:
            cm = _extract_manifest_name(ocinodeclass_manifest)
            if cm:
                nc_candidates.add(cm)

        # Discover live NodePools and OCINodeClass refs
        r_np, j_np = _kubectl_json(kpo_kubeconfig, ["get", "nodepool", "-A"])
        if r_np["rc"] == 0 and isinstance(j_np, dict):
            for item in j_np.get("items", []) or []:
                md = item.get("metadata", {}) or {}
                name = md.get("name")
                ns = md.get("namespace") or "default"
                ann = md.get("annotations", {}) or {}
                tmpl_labels = (
                    ((item.get("spec") or {}).get("template") or {}).get("metadata")
                    or {}
                ).get("labels", {}) or {}
                ncr = (
                    ((item.get("spec") or {}).get("template") or {}).get("spec") or {}
                ).get("nodeClassRef") or {}
                ncr_name = ncr.get("name")
                if ncr_name:
                    nc_candidates.add(ncr_name)

                is_benchmark = bool(
                    (name in np_candidates)
                    or (ann.get("benchmark.oracle.com/min-size") is not None)
                    or (tmpl_labels.get("benchmark.oracle.com/autoscaler") == "kpo")
                )
                if is_benchmark and name:
                    kpo_cleanup["nodepool_candidates"].append(
                        {"namespace": ns, "name": name}
                    )
        else:
            _record("kpo_get_nodepools_failed", r_np)

        # Fallback: if not discovered but we have name candidate, assume default ns
        if not kpo_cleanup["nodepool_candidates"]:
            for n in sorted(np_candidates):
                if n:
                    kpo_cleanup["nodepool_candidates"].append(
                        {"namespace": "default", "name": n}
                    )

        kpo_cleanup["ocinodeclass_candidates"] = sorted(x for x in nc_candidates if x)

        # Delete NodePools first
        for np in kpo_cleanup["nodepool_candidates"]:
            cmd = [
                "kubectl",
                "--kubeconfig",
                str(kpo_kubeconfig),
                "delete",
                "nodepool",
                np["name"],
                "-n",
                np["namespace"],
                "--ignore-not-found=true",
            ]
            if dry_run:
                _record("dry_run_kpo_delete_nodepool", {"cmd": cmd})
                kpo_cleanup["nodepools_deleted"].append(
                    {"namespace": np["namespace"], "name": np["name"], "dry_run": True}
                )
            else:
                r = _run(cmd)
                _record("kpo_delete_nodepool", {"target": np, "result": r})
                if r["rc"] != 0:
                    errors.append(
                        f"Failed deleting NodePool {np['namespace']}/{np['name']}: {r['stderr'].strip()}"
                    )
                else:
                    kpo_cleanup["nodepools_deleted"].append(
                        {"namespace": np["namespace"], "name": np["name"]}
                    )

        # Wait for NodeClaims to clear
        def _nodeclaims_empty():
            r, j = _kubectl_json(kpo_kubeconfig, ["get", "nodeclaims", "-A"])
            if r["rc"] != 0:
                return False, {"error": r["stderr"].strip(), "rc": r["rc"]}
            items = (j or {}).get("items", []) or []
            count = len(items)
            return count == 0, {"count": count}

        if dry_run:
            kpo_cleanup["nodeclaims_wait"] = {"skipped": True, "reason": "dry_run"}
        else:
            ok, history = _wait_until(
                _nodeclaims_empty, cleanup_timeout_nodeclaims, poll_seconds=poll_seconds
            )
            kpo_cleanup["nodeclaims_wait"] = {
                "timeout_seconds": cleanup_timeout_nodeclaims,
                "poll_seconds": poll_seconds,
                "ok": ok,
                "history_tail": history[-20:],
            }
            if not ok:
                errors.append("Timed out waiting for KPO NodeClaims to clear.")

        # Wait for KPO nodes to terminate (best effort)
        nodepool_names = [
            x["name"] for x in kpo_cleanup["nodepool_candidates"] if x.get("name")
        ]

        def _nodes_gone():
            counts = {}
            total = 0
            for name in nodepool_names:
                cmd = [
                    "kubectl",
                    "--kubeconfig",
                    str(kpo_kubeconfig),
                    "get",
                    "nodes",
                    "-l",
                    f"karpenter.sh/nodepool={name}",
                    "-o",
                    "json",
                ]
                r, j = _run_json(cmd)
                if r["rc"] != 0:
                    return False, {"error": r["stderr"].strip(), "rc": r["rc"]}
                c = len((j or {}).get("items", []) or [])
                counts[name] = c
                total += c
            return total == 0, {"total": total, "by_nodepool": counts}

        if dry_run:
            kpo_cleanup["nodes_wait"] = {"skipped": True, "reason": "dry_run"}
        elif nodepool_names:
            ok, history = _wait_until(
                _nodes_gone, cleanup_timeout_nodes, poll_seconds=poll_seconds
            )
            kpo_cleanup["nodes_wait"] = {
                "timeout_seconds": cleanup_timeout_nodes,
                "poll_seconds": poll_seconds,
                "ok": ok,
                "history_tail": history[-20:],
            }
            if not ok:
                errors.append("Timed out waiting for KPO nodes to terminate.")
        else:
            kpo_cleanup["nodes_wait"] = {
                "skipped": True,
                "reason": "no_nodepool_candidates",
            }

        # Delete OCINodeClass resources after NodePool cleanup
        for nc in kpo_cleanup["ocinodeclass_candidates"]:
            cmd = [
                "kubectl",
                "--kubeconfig",
                str(kpo_kubeconfig),
                "delete",
                "ocinodeclass",
                nc,
                "--ignore-not-found=true",
            ]
            if dry_run:
                _record("dry_run_kpo_delete_ocinodeclass", {"cmd": cmd})
                kpo_cleanup["ocinodeclasses_deleted"].append(
                    {"name": nc, "dry_run": True}
                )
            else:
                r = _run(cmd)
                _record("kpo_delete_ocinodeclass", {"target": nc, "result": r})
                if r["rc"] != 0:
                    errors.append(
                        f"Failed deleting OCINodeClass {nc}: {r['stderr'].strip()}"
                    )
                else:
                    kpo_cleanup["ocinodeclasses_deleted"].append({"name": nc})


# -------------------------------
# Terraform destroy selected targets
# -------------------------------
terraform_cleanup = {
    "executed": not skip_terraform_destroy,
    "targets": {},
}

for t in selected_targets:
    d = Path(stack_dirs.get(t, "")).resolve()
    entry = {
        "stack_dir": str(d),
        "exists": d.exists(),
        "init": None,
        "destroy": None,
        "state_after": None,
    }
    terraform_cleanup["targets"][t] = entry

    if skip_terraform_destroy:
        entry["skipped"] = True
        entry["reason"] = "CLEANUP_SKIP_TERRAFORM_DESTROY=true"
        continue

    if not d.exists():
        errors.append(f"Terraform stack dir not found for {t}: {d}")
        continue

    init_cmd = [
        terraform_bin,
        f"-chdir={d}",
        "init",
        "-input=false",
        "-no-color",
    ]
    destroy_cmd = [
        terraform_bin,
        f"-chdir={d}",
        "destroy",
        "-auto-approve",
        "-input=false",
        "-no-color",
        "-lock-timeout=20m",
    ]
    state_cmd = [terraform_bin, f"-chdir={d}", "state", "list"]

    if dry_run:
        entry["init"] = {"dry_run": True, "cmd": init_cmd}
        entry["destroy"] = {"dry_run": True, "cmd": destroy_cmd}
        entry["state_after"] = {"dry_run": True, "cmd": state_cmd}
        _record(
            "dry_run_terraform", {"target": t, "init": init_cmd, "destroy": destroy_cmd}
        )
        continue

    r_init = _run(init_cmd, timeout=1800)
    entry["init"] = {
        "rc": r_init["rc"],
        "stderr_tail": (r_init["stderr"] or "")[-4000:],
    }
    _record("terraform_init", {"target": t, "result": entry["init"]})
    if r_init["rc"] != 0:
        errors.append(
            f"terraform init failed for {t}: {(r_init['stderr'] or '').strip()}"
        )
        continue

    r_destroy = _run(destroy_cmd, timeout=terraform_destroy_timeout_seconds)
    entry["destroy"] = {
        "rc": r_destroy["rc"],
        "stderr_tail": (r_destroy["stderr"] or "")[-4000:],
    }
    _record("terraform_destroy", {"target": t, "result": entry["destroy"]})
    if r_destroy["rc"] != 0:
        errors.append(
            f"terraform destroy failed for {t}: {(r_destroy['stderr'] or '').strip()}"
        )

    r_state = _run(state_cmd, timeout=600)
    stdout_lines = [
        x.strip() for x in (r_state["stdout"] or "").splitlines() if x.strip()
    ]
    stderr_text = r_state["stderr"] or ""
    no_state_file = "No state file was found" in stderr_text
    empty = (r_state["rc"] == 0 and len(stdout_lines) == 0) or no_state_file
    entry["state_after"] = {
        "rc": r_state["rc"],
        "empty": empty,
        "resource_count": len(stdout_lines),
        "resources": stdout_lines[:200],
        "stderr_tail": stderr_text[-2000:],
    }


# -------------------------------
# Residual OCI resource checks
# -------------------------------
oci_checks = []


def _oci_cmd(args):
    base = [
        "oci",
        "--config-file",
        oci_config_file,
        "--profile",
        oci_profile,
        "--region",
        region,
    ]
    return base + args + ["--output", "json"]


def _check_oci_resource(resource_type, resource_id, args):
    cmd = _oci_cmd(args)
    if dry_run:
        return {
            "resource_type": resource_type,
            "id": resource_id,
            "exists": None,
            "dry_run": True,
            "cmd": cmd,
        }

    r = _run(cmd, timeout=300)
    if r["rc"] == 0:
        return {
            "resource_type": resource_type,
            "id": resource_id,
            "exists": True,
            "rc": r["rc"],
        }

    combined = f"{r.get('stderr', '')}\n{r.get('stdout', '')}"
    if _is_not_found(combined):
        return {
            "resource_type": resource_type,
            "id": resource_id,
            "exists": False,
            "rc": r["rc"],
        }

    return {
        "resource_type": resource_type,
        "id": resource_id,
        "exists": None,
        "rc": r["rc"],
        "error": combined[-2000:],
    }


# IAM bootstrap resources created post-provision are not in Terraform state.
# Delete them here for true clean-slate reruns.
iam_cleanup = {
    "executed": cleanup_delete_iam_bootstrap,
    "dry_run": dry_run,
    "policies_deleted": [],
    "dynamic_groups_deleted": [],
    "skipped": False,
    "reason": "",
}

if cleanup_delete_iam_bootstrap:
    iam_obj = post_prov.get("iam") or {}
    policy_ids = []
    dynamic_group_ids = []
    for _, val in iam_obj.items():
        rid = str((val or {}).get("id", "")).strip()
        if not rid:
            continue
        if "dynamicgroup" in rid:
            dynamic_group_ids.append(rid)
        elif "policy" in rid:
            policy_ids.append(rid)

    # Delete policies first, then dynamic groups
    for pid in sorted(set(policy_ids)):
        cmd = _oci_cmd(["iam", "policy", "delete", "--policy-id", pid, "--force"])
        if dry_run:
            _record("dry_run_iam_delete_policy", {"cmd": cmd, "policy_id": pid})
            iam_cleanup["policies_deleted"].append({"id": pid, "dry_run": True})
            continue

        r = _run(cmd, timeout=300)
        _record("iam_delete_policy", {"policy_id": pid, "result": r})
        combined = f"{r.get('stderr', '')}\n{r.get('stdout', '')}"
        if r["rc"] == 0 or _is_not_found(combined):
            iam_cleanup["policies_deleted"].append(
                {
                    "id": pid,
                    "deleted": r["rc"] == 0,
                    "already_absent": _is_not_found(combined),
                }
            )
        else:
            errors.append(f"Failed deleting IAM policy {pid}: {combined[-1000:]}")

    for dgid in sorted(set(dynamic_group_ids)):
        cmd = _oci_cmd(
            ["iam", "dynamic-group", "delete", "--dynamic-group-id", dgid, "--force"]
        )
        if dry_run:
            _record(
                "dry_run_iam_delete_dynamic_group",
                {"cmd": cmd, "dynamic_group_id": dgid},
            )
            iam_cleanup["dynamic_groups_deleted"].append({"id": dgid, "dry_run": True})
            continue

        r = _run(cmd, timeout=300)
        _record("iam_delete_dynamic_group", {"dynamic_group_id": dgid, "result": r})
        combined = f"{r.get('stderr', '')}\n{r.get('stdout', '')}"
        if r["rc"] == 0 or _is_not_found(combined):
            iam_cleanup["dynamic_groups_deleted"].append(
                {
                    "id": dgid,
                    "deleted": r["rc"] == 0,
                    "already_absent": _is_not_found(combined),
                }
            )
        else:
            errors.append(
                f"Failed deleting IAM dynamic group {dgid}: {combined[-1000:]}"
            )
else:
    iam_cleanup["skipped"] = True
    iam_cleanup["reason"] = "CLEANUP_DELETE_IAM_BOOTSTRAP=false"

# Cluster checks
for t in selected_targets:
    cid = cluster_ids.get(t)
    if cid:
        oci_checks.append(
            _check_oci_resource(
                "cluster", cid, ["ce", "cluster", "get", "--cluster-id", cid]
            )
        )

# Worker nodepool checks
for t in selected_targets:
    for npid in worker_pool_ids.get(t, []) or []:
        oci_checks.append(
            _check_oci_resource(
                "nodepool", npid, ["ce", "node-pool", "get", "--node-pool-id", npid]
            )
        )

# IAM resource checks from post-provisioning
iam_obj = post_prov.get("iam") or {}
if iam_obj:
    for key, val in iam_obj.items():
        rid = (val or {}).get("id")
        if not rid:
            continue
        if "dynamicgroup" in rid:
            oci_checks.append(
                _check_oci_resource(
                    key, rid, ["iam", "dynamic-group", "get", "--dynamic-group-id", rid]
                )
            )
        elif "policy" in rid:
            oci_checks.append(
                _check_oci_resource(
                    key, rid, ["iam", "policy", "get", "--policy-id", rid]
                )
            )


# -------------------------------
# Residual report
# -------------------------------
residuals = {
    "terraform_state_residuals": {},
    "oci_existing_resources": [],
    "oci_unknown_state_resources": [],
}

for t, info in (terraform_cleanup.get("targets") or {}).items():
    state_after = info.get("state_after") or {}
    if state_after.get("empty") is False:
        residuals["terraform_state_residuals"][t] = state_after

for chk in oci_checks:
    if chk.get("exists") is True:
        residuals["oci_existing_resources"].append(chk)
    elif chk.get("exists") is None and not chk.get("dry_run"):
        residuals["oci_unknown_state_resources"].append(chk)

summary = {
    "generated_at_utc": _now_utc(),
    "run_id": run_id,
    "artifact_dir": str(artifact_dir),
    "selected_targets": selected_targets,
    "region": region,
    "oci_profile": oci_profile,
    "oci_config_file": oci_config_file,
    "terraform_bin": terraform_bin,
    "dry_run": dry_run,
    "skip_terraform_destroy": skip_terraform_destroy,
    "cleanup_delete_iam_bootstrap": cleanup_delete_iam_bootstrap,
    "timeouts": {
        "nodeclaims_seconds": cleanup_timeout_nodeclaims,
        "nodes_seconds": cleanup_timeout_nodes,
        "terraform_destroy_seconds": terraform_destroy_timeout_seconds,
        "poll_seconds": poll_seconds,
    },
    "kpo_cleanup": kpo_cleanup,
    "terraform_cleanup": terraform_cleanup,
    "iam_cleanup": iam_cleanup,
    "oci_resource_checks": oci_checks,
    "residuals": residuals,
    "errors": errors,
    "status": "success" if not errors else "completed_with_errors",
}

cleanup_summary_path = artifact_dir / "cleanup_summary.json"
actions_path = artifact_dir / "cleanup_actions.jsonl"

cleanup_summary_path.write_text(
    json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
with actions_path.open("w", encoding="utf-8") as f:
    for a in cleanup_actions:
        f.write(json.dumps(a, sort_keys=True) + "\n")

os.environ["CLEANUP_SUMMARY_ARTIFACT"] = str(cleanup_summary_path)
os.environ["CLEANUP_ACTIONS_ARTIFACT"] = str(actions_path)

print("\nCell 17 summary:")
print(
    json.dumps(
        {
            "run_id": run_id,
            "status": summary["status"],
            "errors": len(errors),
            "residual_terraform_targets": sorted(
                list(residuals["terraform_state_residuals"].keys())
            ),
            "residual_oci_resources": len(residuals["oci_existing_resources"]),
            "unknown_oci_states": len(residuals["oci_unknown_state_resources"]),
            "iam_policies_deleted": len(iam_cleanup.get("policies_deleted", [])),
            "iam_dynamic_groups_deleted": len(
                iam_cleanup.get("dynamic_groups_deleted", [])
            ),
            "cleanup_summary": str(cleanup_summary_path),
            "cleanup_actions": str(actions_path),
        },
        indent=2,
    )
)
print(f"\nCell 17 complete. Artifact: {cleanup_summary_path}")